# Notebook 20 — Connections and Ownership Identity

## Bounded question

> What do the runner-level `jockey`, `trainer` and `owner` fields represent in the source, how stable and complete are their labels, and which source-internal identity relationships can be preserved safely without inventing equivalence between people, partnerships, syndicates or organisations?

## Initial governed scope

This notebook investigates three runner-level source-text fields:

- `jockey`
- `trainer`
- `owner`

The source-field governance register assigns all three to the `connections_and_ownership` family, requires their raw values to be preserved and leaves their semantics pending.

The investigation begins with source lineage and inherited governance only. At this stage, no assumption is made that:

- a raw label is a stable real-world entity identifier;
- identical strings always identify the same person, organisation or ownership account;
- different strings always identify different entities;
- initials uniquely identify a person;
- punctuation, spacing or title differences imply equivalence;
- trainer and jockey names follow the same identity rules;
- owner partnership or syndicate labels can be decomposed safely;
- a person, organisation, ownership account, partnership and syndicate can share one undifferentiated entity model;
- name similarity alone justifies merging.

Raw source labels, display-name parsing, exact-label identity, provisional source-internal identity, externally verified identity, role assertions and ownership-entity type will remain separate concepts.

## Stage 1 — Source lineage and governed population

This stage establishes the immutable source, read-only controls, complete governed runner population and provisional race key before interpreting any connection or ownership label.

The source is:

- database: `data/raw/form_2015-present/form_2015-present/raceform.db`
- table: `data`
- governed row predicate: `rowid <> 1`
- provisional race identity: `date + course + off`

The established source population is expected to contain:

- 1,851,285 governed runner rows;
- 189,043 provisional races;
- 37 source columns.

The first code cell opens SQLite in read-only mode, confirms the source schema, reconciles the governed runner and provisional-race counts, and confirms that `jockey`, `trainer` and `owner` are present.

It does not parse, normalise or interpret any name.

In [1]:
from pathlib import Path
import sqlite3

import pandas as pd


PROJECT_ROOT = Path.cwd().resolve().parent

SOURCE_DB_PATH = (
    PROJECT_ROOT
    / "data"
    / "raw"
    / "form_2015-present"
    / "form_2015-present"
    / "raceform.db"
)

SOURCE_TABLE = "data"
DATA_ROW_PREDICATE = "rowid <> 1"
RACE_KEY_COLUMNS = ["date", "course", "off"]
CONNECTION_IDENTITY_FIELDS = ["jockey", "trainer", "owner"]

EXPECTED_RUNNER_ROWS = 1_851_285
EXPECTED_PROVISIONAL_RACES = 189_043
EXPECTED_SOURCE_COLUMNS = 37

if not SOURCE_DB_PATH.exists():
    raise FileNotFoundError(f"Source database not found: {SOURCE_DB_PATH}")

connection = sqlite3.connect(
    f"file:{SOURCE_DB_PATH}?mode=ro",
    uri=True,
)

try:
    schema = pd.read_sql_query(
        f"PRAGMA table_info({SOURCE_TABLE})",
        connection,
    )

    source_columns = schema["name"].tolist()

    missing_fields = [
        field
        for field in CONNECTION_IDENTITY_FIELDS
        if field not in source_columns
    ]

    if missing_fields:
        raise AssertionError(
            f"Missing connection-identity fields: {missing_fields}"
        )

    runner_rows = connection.execute(
        f"""
        SELECT COUNT(*)
        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}
        """
    ).fetchone()[0]

    provisional_races = connection.execute(
        f"""
        SELECT COUNT(*)
        FROM (
            SELECT DISTINCT date, course, off
            FROM {SOURCE_TABLE}
            WHERE {DATA_ROW_PREDICATE}
        )
        """
    ).fetchone()[0]

finally:
    connection.close()

assert runner_rows == EXPECTED_RUNNER_ROWS
assert provisional_races == EXPECTED_PROVISIONAL_RACES
assert len(source_columns) == EXPECTED_SOURCE_COLUMNS

source_lineage_summary = pd.DataFrame(
    [
        (
            "source database",
            SOURCE_DB_PATH.relative_to(PROJECT_ROOT).as_posix(),
        ),
        ("source table", SOURCE_TABLE),
        ("data-row predicate", DATA_ROW_PREDICATE),
        ("runner rows", runner_rows),
        ("provisional races", provisional_races),
        ("source columns", len(source_columns)),
        (
            "provisional race key",
            " + ".join(RACE_KEY_COLUMNS),
        ),
        (
            "connection-identity fields present",
            ", ".join(CONNECTION_IDENTITY_FIELDS),
        ),
    ],
    columns=["measure", "value"],
)

print("Governed source population confirmed")
source_lineage_summary

Governed source population confirmed


,measure,value
0,source database,data/raw/form_2015-present/form_2015-present/r...
1,source table,data
2,data-row predicate,rowid <> 1
3,runner rows,1851285
4,provisional races,189043
5,source columns,37
6,provisional race key,date + course + off
7,connection-identity fields present,"jockey, trainer, owner"


## Stage 2 — Confirm inherited source-field governance

Before profiling the contents of the three fields, this stage reads their existing rows from `data/reference/source_field_governance.csv`.

The check is limited to confirming the inherited starting position:

- each field is recorded at runner grain;
- each belongs to `connections_and_ownership`;
- raw preservation is required;
- blank values retain the governed `field_not_supplied` policy;
- semantic status remains pending;
- the existing Notebook 10 attribution is preserved.

This stage does not revise the register or infer anything from the field names.

In [2]:
SOURCE_FIELD_GOVERNANCE_PATH = (
    PROJECT_ROOT
    / "data"
    / "reference"
    / "source_field_governance.csv"
)

if not SOURCE_FIELD_GOVERNANCE_PATH.exists():
    raise FileNotFoundError(
        "Source-field governance register not found: "
        f"{SOURCE_FIELD_GOVERNANCE_PATH}"
    )

source_field_governance = pd.read_csv(
    SOURCE_FIELD_GOVERNANCE_PATH
)

connection_identity_governance = (
    source_field_governance.loc[
        source_field_governance["source_field"].isin(
            CONNECTION_IDENTITY_FIELDS
        ),
        [
            "ordinal",
            "source_field",
            "declared_type",
            "grain",
            "field_family",
            "raw_preservation",
            "blank_policy",
            "dash_policy",
            "zero_policy",
            "governed_by",
            "status",
        ],
    ]
    .sort_values("ordinal")
    .reset_index(drop=True)
)

assert connection_identity_governance["source_field"].tolist() == [
    "jockey",
    "trainer",
    "owner",
]

assert connection_identity_governance["declared_type"].eq(
    "TEXT"
).all()

assert connection_identity_governance["grain"].eq(
    "runner"
).all()

assert connection_identity_governance["field_family"].eq(
    "connections_and_ownership"
).all()

assert connection_identity_governance["raw_preservation"].eq(
    "required"
).all()

assert connection_identity_governance["blank_policy"].eq(
    "field_not_supplied"
).all()

assert connection_identity_governance["dash_policy"].eq(
    "not_expected"
).all()

assert connection_identity_governance["zero_policy"].eq(
    "contextual_value"
).all()

assert connection_identity_governance["governed_by"].eq(
    "Notebook 10"
).all()

assert connection_identity_governance["status"].eq(
    "pending_semantics"
).all()

print("Inherited source-field governance confirmed")
connection_identity_governance

Inherited source-field governance confirmed


,ordinal,source_field,declared_type,grain,field_family,raw_preservation,blank_policy,dash_policy,zero_policy,governed_by,status
0,27,jockey,TEXT,runner,connections_and_ownership,required,field_not_supplied,not_expected,contextual_value,Notebook 10,pending_semantics
1,28,trainer,TEXT,runner,connections_and_ownership,required,field_not_supplied,not_expected,contextual_value,Notebook 10,pending_semantics
2,36,owner,TEXT,runner,connections_and_ownership,required,field_not_supplied,not_expected,contextual_value,Notebook 10,pending_semantics


## Stage 3 — Raw field population and physical text profile

This stage profiles the three raw source fields without cleaning, parsing or joining labels.

For each field, it measures:

- governed runner rows;
- populated rows;
- blank rows;
- distinct populated raw labels;
- first and last observed dates;
- maximum rows attached to one exact label;
- median rows per exact label;
- leading whitespace;
- trailing whitespace;
- repeated internal whitespace.

These measures establish physical storage behaviour and exact-label frequency only.

They do not establish whether:

- one exact label identifies one real-world entity;
- two different labels identify the same entity;
- a populated value is correct;
- a blank value has one universal cause;
- whitespace or formatting differences may safely be normalised.

In [3]:
def quote_identifier(identifier: str) -> str:
    """Safely quote a known SQLite identifier."""
    return '"' + identifier.replace('"', '""') + '"'


profile_rows = []

connection = sqlite3.connect(
    f"file:{SOURCE_DB_PATH}?mode=ro",
    uri=True,
)

try:
    for field in CONNECTION_IDENTITY_FIELDS:
        quoted_field = quote_identifier(field)

        summary = connection.execute(
            f"""
            SELECT
                COUNT(*) AS governed_runner_rows,
                SUM(
                    CASE
                        WHEN {quoted_field} IS NOT NULL
                         AND {quoted_field} <> ''
                        THEN 1
                        ELSE 0
                    END
                ) AS populated_rows,
                SUM(
                    CASE
                        WHEN {quoted_field} IS NULL
                          OR {quoted_field} = ''
                        THEN 1
                        ELSE 0
                    END
                ) AS blank_rows,
                COUNT(
                    DISTINCT CASE
                        WHEN {quoted_field} IS NOT NULL
                         AND {quoted_field} <> ''
                        THEN {quoted_field}
                    END
                ) AS distinct_populated_labels,
                MIN(
                    CASE
                        WHEN {quoted_field} IS NOT NULL
                         AND {quoted_field} <> ''
                        THEN date
                    END
                ) AS first_observed_date,
                MAX(
                    CASE
                        WHEN {quoted_field} IS NOT NULL
                         AND {quoted_field} <> ''
                        THEN date
                    END
                ) AS last_observed_date,
                SUM(
                    CASE
                        WHEN {quoted_field} IS NOT NULL
                         AND {quoted_field} <> ''
                         AND {quoted_field} <> LTRIM({quoted_field})
                        THEN 1
                        ELSE 0
                    END
                ) AS leading_whitespace_rows,
                SUM(
                    CASE
                        WHEN {quoted_field} IS NOT NULL
                         AND {quoted_field} <> ''
                         AND {quoted_field} <> RTRIM({quoted_field})
                        THEN 1
                        ELSE 0
                    END
                ) AS trailing_whitespace_rows,
                SUM(
                    CASE
                        WHEN {quoted_field} IS NOT NULL
                         AND INSTR({quoted_field}, '  ') > 0
                        THEN 1
                        ELSE 0
                    END
                ) AS repeated_internal_whitespace_rows
            FROM {SOURCE_TABLE}
            WHERE {DATA_ROW_PREDICATE}
            """
        ).fetchone()

        label_counts = pd.read_sql_query(
            f"""
            SELECT
                {quoted_field} AS raw_label,
                COUNT(*) AS runner_rows
            FROM {SOURCE_TABLE}
            WHERE {DATA_ROW_PREDICATE}
              AND {quoted_field} IS NOT NULL
              AND {quoted_field} <> ''
            GROUP BY {quoted_field}
            """,
            connection,
        )

        if label_counts.empty:
            maximum_rows_per_label = 0
            median_rows_per_label = 0.0
        else:
            maximum_rows_per_label = int(
                label_counts["runner_rows"].max()
            )
            median_rows_per_label = float(
                label_counts["runner_rows"].median()
            )

        profile_rows.append(
            {
                "source_field": field,
                "governed_runner_rows": int(summary[0]),
                "populated_rows": int(summary[1] or 0),
                "blank_rows": int(summary[2] or 0),
                "distinct_populated_labels": int(summary[3] or 0),
                "first_observed_date": summary[4],
                "last_observed_date": summary[5],
                "maximum_rows_per_label": maximum_rows_per_label,
                "median_rows_per_label": median_rows_per_label,
                "leading_whitespace_rows": int(summary[6] or 0),
                "trailing_whitespace_rows": int(summary[7] or 0),
                "repeated_internal_whitespace_rows": int(
                    summary[8] or 0
                ),
            }
        )

finally:
    connection.close()

raw_field_profile = pd.DataFrame(profile_rows)

assert raw_field_profile["source_field"].tolist() == (
    CONNECTION_IDENTITY_FIELDS
)

assert raw_field_profile["governed_runner_rows"].eq(
    EXPECTED_RUNNER_ROWS
).all()

assert (
    raw_field_profile["populated_rows"]
    + raw_field_profile["blank_rows"]
).eq(EXPECTED_RUNNER_ROWS).all()

print("Raw connection and ownership fields profiled")
raw_field_profile

Raw connection and ownership fields profiled


,source_field,governed_runner_rows,populated_rows,blank_rows,distinct_populated_labels,first_observed_date,last_observed_date,maximum_rows_per_label,median_rows_per_label,leading_whitespace_rows,trailing_whitespace_rows,repeated_internal_whitespace_rows
0,jockey,1851285,1851283,2,7917,2015-01-01,2026-05-27,14241,10.0,0,0,0
1,trainer,1851285,1851276,9,10708,2015-01-01,2026-05-27,15140,7.0,0,0,0
2,owner,1851285,1851250,35,98234,2015-01-01,2026-05-27,15384,5.0,1,0,6300


## Stage 4 — Inspect blanks and whitespace anomalies

The physical profile found:

- 2 blank `jockey` rows;
- 9 blank `trainer` rows;
- 35 blank `owner` rows;
- 1 `owner` row with leading whitespace;
- 6,300 `owner` rows containing repeated internal spaces.

This stage inspects those finite anomaly sets while preserving the raw source strings exactly.

The purpose is to determine whether the anomalies represent:

- genuine missing values;
- systematic source formatting;
- isolated extraction artefacts;
- meaningful spacing within registered owner labels;
- or cases requiring later review.

No trimming or whitespace collapsing is applied at this stage.

In [4]:
connection = sqlite3.connect(
    f"file:{SOURCE_DB_PATH}?mode=ro",
    uri=True,
)

try:
    blank_rows = pd.read_sql_query(
        f"""
        SELECT
            rowid AS source_rowid,
            date,
            course,
            off,
            horse,
            jockey,
            trainer,
            owner
        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}
          AND (
                jockey IS NULL
             OR jockey = ''
             OR trainer IS NULL
             OR trainer = ''
             OR owner IS NULL
             OR owner = ''
          )
        ORDER BY date, course, off, rowid
        """,
        connection,
    )

    owner_whitespace_summary = pd.read_sql_query(
        f"""
        SELECT
            owner AS raw_owner_label,
            COUNT(*) AS runner_rows,
            MIN(date) AS first_observed_date,
            MAX(date) AS last_observed_date,
            SUM(
                CASE
                    WHEN owner <> LTRIM(owner)
                    THEN 1
                    ELSE 0
                END
            ) AS leading_whitespace_rows,
            SUM(
                CASE
                    WHEN owner <> RTRIM(owner)
                    THEN 1
                    ELSE 0
                END
            ) AS trailing_whitespace_rows,
            SUM(
                CASE
                    WHEN INSTR(owner, '  ') > 0
                    THEN 1
                    ELSE 0
                END
            ) AS repeated_internal_whitespace_rows
        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}
          AND owner IS NOT NULL
          AND owner <> ''
          AND (
                owner <> LTRIM(owner)
             OR owner <> RTRIM(owner)
             OR INSTR(owner, '  ') > 0
          )
        GROUP BY owner
        ORDER BY runner_rows DESC, raw_owner_label
        """,
        connection,
    )

finally:
    connection.close()

assert len(blank_rows) <= 46

assert (
    owner_whitespace_summary["leading_whitespace_rows"].sum()
    == 1
)

assert (
    owner_whitespace_summary[
        "repeated_internal_whitespace_rows"
    ].sum()
    == 6_300
)

print("Blank rows")
display(blank_rows)

print("Distinct owner labels with whitespace anomalies")
display(owner_whitespace_summary)

Blank rows


,source_rowid,date,course,off,horse,jockey,trainer,owner
0,1443,2015-01-03,Santa Anita (USA),11:30,Rattataptap (USA),Edwin A Maldonado,Jeff Bonde,
1,33009,2015-04-04,Caulfield (AUS),6:10,Post DFrance (NZ),Patrick Moloney,Stephen Brown,
2,48891,2015-05-03,San Siro (ITY),12:07,Balami Fan (ITY),Luca Maniezzi,V Cangiano,
3,50160,2015-05-06,Sonoda (JPN),11:07,Maximum Kaiser (JPN),Shoichi Kawahara,,Tsuru Nishimori
4,71791,2015-06-18,Longchamp (FR),11:15,Rappeur Des Mottes (FR),Anthony Crastus,E Lellouche,
5,189632,2016-04-08,Auteuil (FR),2:15,Ahzana (FR),,L Viel,Laurent Viel
6,203870,2016-05-07,Maisons-Laffitte (FR),3:30,Star White (FR),Tony Piccone,,
7,203991,2016-05-07,Maisons-Laffitte (FR),3:30,Colombia DEmra (FR),Richard Juteau,,
8,599152,2018-09-10,Chantilly (FR),3:30,Numbers Talk (IRE),Ronan Thomas,,Ecurie Avant Garde
9,600778,2018-09-14,Saint-Cloud (FR),3:10,Valley Kid (FR),Michelle Swinnens,,Stalt Neerhof


Distinct owner labels with whitespace anomalies


,raw_owner_label,runner_rows,first_observed_date,last_observed_date,leading_whitespace_rows,trailing_whitespace_rows,repeated_internal_whitespace_rows
0,J L Wetherald M M Glover,70,2015-05-28,2023-10-19,0,0,70
1,Ursa Major England,65,2023-02-21,2026-05-25,0,0,65
2,Kingsley Park 1 Ready To Run,63,2015-02-02,2015-10-24,0,0,63
3,Pau Perth Partnership,60,2019-04-19,2025-04-12,0,0,60
4,Kildare Stud Frankie Oconnor,55,2016-12-15,2025-03-07,0,0,55
...,...,...,...,...,...,...,...
641,Value Racing Eternal Angel,1,2026-05-25,2026-05-25,0,0,1
642,Value Racing Mirror Of Illusion,1,2025-09-07,2025-09-07,0,0,1
643,Vl Racing Syndikat Kogut,1,2025-10-12,2025-10-12,0,0,1
644,Wil A Way Farm Gail C Jewsbury Pat Jarvis,1,2024-09-15,2024-09-15,0,0,1


## Stage 5 — Distinguish missing-value storage and whitespace forms

The anomaly inspection found 46 blank field occurrences across 44 runner rows and 646 distinct owner labels containing leading or repeated internal whitespace.

Before interpreting those cases, this stage establishes:

- whether missing values are stored as SQLite `NULL`, empty strings, or both;
- how many fields are missing on each affected runner row;
- the maximum consecutive-space run in each anomalous owner label;
- the distribution of two-space, three-space and longer runs;
- the exact label containing leading whitespace;
- whether a whitespace-collapsed form also exists independently as a raw source label.

The collapsed form is used only as a comparison key.

It is not adopted as a canonical label and does not establish that two labels identify the same ownership entity.

In [5]:
import re


connection = sqlite3.connect(
    f"file:{SOURCE_DB_PATH}?mode=ro",
    uri=True,
)

try:
    missing_storage_rows = []

    for field in CONNECTION_IDENTITY_FIELDS:
        quoted_field = quote_identifier(field)

        null_count, empty_string_count = connection.execute(
            f"""
            SELECT
                SUM(
                    CASE
                        WHEN {quoted_field} IS NULL
                        THEN 1
                        ELSE 0
                    END
                ),
                SUM(
                    CASE
                        WHEN {quoted_field} = ''
                        THEN 1
                        ELSE 0
                    END
                )
            FROM {SOURCE_TABLE}
            WHERE {DATA_ROW_PREDICATE}
            """
        ).fetchone()

        missing_storage_rows.append(
            {
                "source_field": field,
                "null_rows": int(null_count or 0),
                "empty_string_rows": int(empty_string_count or 0),
                "total_blank_rows": int(
                    (null_count or 0) + (empty_string_count or 0)
                ),
            }
        )

    affected_row_missingness = pd.read_sql_query(
        f"""
        SELECT
            (
                CASE WHEN jockey IS NULL OR jockey = '' THEN 1 ELSE 0 END
              + CASE WHEN trainer IS NULL OR trainer = '' THEN 1 ELSE 0 END
              + CASE WHEN owner IS NULL OR owner = '' THEN 1 ELSE 0 END
            ) AS missing_field_count,
            COUNT(*) AS runner_rows
        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}
          AND (
                jockey IS NULL
             OR jockey = ''
             OR trainer IS NULL
             OR trainer = ''
             OR owner IS NULL
             OR owner = ''
          )
        GROUP BY missing_field_count
        ORDER BY missing_field_count
        """,
        connection,
    )

    all_raw_owner_labels = set(
        pd.read_sql_query(
            f"""
            SELECT DISTINCT owner AS raw_owner_label
            FROM {SOURCE_TABLE}
            WHERE {DATA_ROW_PREDICATE}
              AND owner IS NOT NULL
              AND owner <> ''
            """,
            connection,
        )["raw_owner_label"]
    )

finally:
    connection.close()


missing_storage_profile = pd.DataFrame(missing_storage_rows)

owner_whitespace_detail = owner_whitespace_summary.copy()

owner_whitespace_detail["maximum_space_run"] = (
    owner_whitespace_detail["raw_owner_label"]
    .map(
        lambda value: max(
            (len(match.group(0)) for match in re.finditer(r" +", value)),
            default=0,
        )
    )
)

owner_whitespace_detail["collapsed_owner_label"] = (
    owner_whitespace_detail["raw_owner_label"]
    .str.strip()
    .str.replace(r" +", " ", regex=True)
)

owner_whitespace_detail["collapsed_form_exists_as_raw_label"] = (
    owner_whitespace_detail["collapsed_owner_label"]
    .isin(all_raw_owner_labels)
)

space_run_distribution = (
    owner_whitespace_detail.groupby(
        "maximum_space_run",
        as_index=False,
    )
    .agg(
        distinct_raw_labels=("raw_owner_label", "size"),
        runner_rows=("runner_rows", "sum"),
        collapsed_forms_also_present=(
            "collapsed_form_exists_as_raw_label",
            "sum",
        ),
    )
    .sort_values("maximum_space_run")
    .reset_index(drop=True)
)

leading_whitespace_labels = owner_whitespace_detail.loc[
    owner_whitespace_detail["leading_whitespace_rows"] > 0,
    [
        "raw_owner_label",
        "runner_rows",
        "first_observed_date",
        "last_observed_date",
        "collapsed_owner_label",
        "collapsed_form_exists_as_raw_label",
    ],
].reset_index(drop=True)

assert missing_storage_profile["total_blank_rows"].tolist() == [
    2,
    9,
    35,
]

assert affected_row_missingness["runner_rows"].sum() == 44

assert len(owner_whitespace_detail) == 646

assert len(leading_whitespace_labels) == 1

print("Missing-value storage")
display(missing_storage_profile)

print("Missing fields per affected runner row")
display(affected_row_missingness)

print("Owner consecutive-space distribution")
display(space_run_distribution)

print("Leading-whitespace owner label")
display(leading_whitespace_labels)

print("Whitespace labels whose collapsed form also exists")
display(
    owner_whitespace_detail.loc[
        owner_whitespace_detail[
            "collapsed_form_exists_as_raw_label"
        ],
        [
            "raw_owner_label",
            "runner_rows",
            "first_observed_date",
            "last_observed_date",
            "collapsed_owner_label",
        ],
    ]
    .sort_values(
        ["runner_rows", "raw_owner_label"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

Missing-value storage


,source_field,null_rows,empty_string_rows,total_blank_rows
0,jockey,0,2,2
1,trainer,0,9,9
2,owner,0,35,35


Missing fields per affected runner row


,missing_field_count,runner_rows
0,1,42
1,2,2


Owner consecutive-space distribution


,maximum_space_run,distinct_raw_labels,runner_rows,collapsed_forms_also_present
0,1,1,1,0
1,2,51,476,0
2,3,594,5824,0


Leading-whitespace owner label


,raw_owner_label,runner_rows,first_observed_date,last_observed_date,collapsed_owner_label,collapsed_form_exists_as_raw_label
0,Bradley Thoroughbreds Llc O Ghrghar C Y Lerner,1,2026-05-10,2026-05-10,Bradley Thoroughbreds Llc O Ghrghar C Y Lerner,False


Whitespace labels whose collapsed form also exists


,raw_owner_label,runner_rows,first_observed_date,last_observed_date,collapsed_owner_label


## Stage 6 — Raw punctuation and structural vocabulary

The physical-profile stages established that:

- missing values are stored as empty strings;
- whitespace anomalies are concentrated in owner labels;
- repeated owner spaces do not currently collide with independently observed collapsed labels;
- no whitespace normalisation has yet been adopted.

This stage measures the visible structural features of exact raw labels.

For each field, it counts distinct populated labels and runner rows containing:

- apostrophes;
- hyphens;
- periods;
- commas;
- ampersands;
- slashes;
- round brackets;
- square brackets;
- numerals;
- colons;
- semicolons;
- plus signs;
- repeated spaces;
- non-ASCII characters.

It also measures labels containing words or phrases that may indicate:

- partnerships;
- syndicates;
- clubs;
- companies;
- studs;
- farms;
- estates;
- trusts;
- racing groups;
- ownership constructions.

These are lexical indicators only.

They do not prove entity type, legal status, membership, identity equivalence or safe decomposition.

In [6]:
# Directly observable punctuation tokens.
#
# These are counted exactly as stored. No label is trimmed, case-folded,
# transliterated, split or otherwise normalised.
PUNCTUATION_TOKENS = {
    "apostrophe": "'",
    "hyphen": "-",
    "period": ".",
    "comma": ",",
    "ampersand": "&",
    "slash": "/",
    "round_bracket_open": "(",
    "round_bracket_close": ")",
    "square_bracket_open": "[",
    "square_bracket_close": "]",
    "repeated_space": "  ",
}


def contains_ascii_digit(value: str) -> bool:
    """Return True when the exact raw label contains an ASCII digit."""
    return any(character in "0123456789" for character in value)


def contains_non_ascii_character(value: str) -> bool:
    """Return True when the exact raw label contains a non-ASCII character."""
    return any(ord(character) > 127 for character in value)


# Build one exact-label frequency table per source field.
#
# The query preserves each raw label exactly and records its source frequency
# and observed date range. Empty strings are excluded because their storage
# behaviour was already established in Stage 5.
connection = sqlite3.connect(
    f"file:{SOURCE_DB_PATH}?mode=ro",
    uri=True,
)

try:
    exact_label_profiles = {}

    for field in CONNECTION_IDENTITY_FIELDS:
        quoted_field = quote_identifier(field)

        exact_label_profiles[field] = pd.read_sql_query(
            f"""
            SELECT
                {quoted_field} AS raw_label,
                COUNT(*) AS runner_rows,
                MIN(date) AS first_observed_date,
                MAX(date) AS last_observed_date
            FROM {SOURCE_TABLE}
            WHERE {DATA_ROW_PREDICATE}
              AND {quoted_field} IS NOT NULL
              AND {quoted_field} <> ''
            GROUP BY {quoted_field}
            """,
            connection,
        )

finally:
    connection.close()


# Reconcile the exact-label tables with the Stage 3 field profile.
#
# This prevents later feature counts from being based on an incomplete or
# accidentally altered source population.
for field in CONNECTION_IDENTITY_FIELDS:
    labels = exact_label_profiles[field]

    expected_distinct_labels = int(
        raw_field_profile.loc[
            raw_field_profile["source_field"] == field,
            "distinct_populated_labels",
        ].iloc[0]
    )

    expected_populated_rows = int(
        raw_field_profile.loc[
            raw_field_profile["source_field"] == field,
            "populated_rows",
        ].iloc[0]
    )

    assert len(labels) == expected_distinct_labels
    assert int(labels["runner_rows"].sum()) == expected_populated_rows


feature_rows = []

for field in CONNECTION_IDENTITY_FIELDS:
    labels = exact_label_profiles[field]

    # Count each literal punctuation token without regex interpretation.
    for feature, token in PUNCTUATION_TOKENS.items():
        feature_mask = labels["raw_label"].str.contains(
            token,
            regex=False,
            na=False,
        )

        feature_rows.append(
            {
                "source_field": field,
                "feature": feature,
                "distinct_raw_labels": int(feature_mask.sum()),
                "runner_rows": int(
                    labels.loc[feature_mask, "runner_rows"].sum()
                ),
            }
        )

    # Numerals and non-ASCII characters require character-level checks rather
    # than one fixed literal token.
    numeral_mask = labels["raw_label"].map(
        contains_ascii_digit
    )

    feature_rows.append(
        {
            "source_field": field,
            "feature": "ascii_numeral",
            "distinct_raw_labels": int(numeral_mask.sum()),
            "runner_rows": int(
                labels.loc[numeral_mask, "runner_rows"].sum()
            ),
        }
    )

    non_ascii_mask = labels["raw_label"].map(
        contains_non_ascii_character
    )

    feature_rows.append(
        {
            "source_field": field,
            "feature": "non_ascii_character",
            "distinct_raw_labels": int(non_ascii_mask.sum()),
            "runner_rows": int(
                labels.loc[non_ascii_mask, "runner_rows"].sum()
            ),
        }
    )


raw_character_profile = pd.DataFrame(feature_rows)

# Every field must contribute one row for each literal punctuation token plus
# the numeral and non-ASCII checks.
expected_features_per_field = len(PUNCTUATION_TOKENS) + 2

assert (
    raw_character_profile.groupby("source_field")
    .size()
    .eq(expected_features_per_field)
    .all()
)

distinct_label_character_profile = (
    raw_character_profile.pivot(
        index="feature",
        columns="source_field",
        values="distinct_raw_labels",
    )
    .fillna(0)
    .astype(int)
)

runner_row_character_profile = (
    raw_character_profile.pivot(
        index="feature",
        columns="source_field",
        values="runner_rows",
    )
    .fillna(0)
    .astype(int)
)

print("Raw character features by distinct exact label")
display(distinct_label_character_profile)

print("Raw character features by governed runner rows")
display(runner_row_character_profile)

Raw character features by distinct exact label


source_field,jockey,owner,trainer
feature,,,
ampersand,0,0,305
apostrophe,0,0,0
ascii_numeral,0,2500,0
comma,0,0,0
hyphen,186,0,260
non_ascii_character,0,0,0
period,0,0,0
repeated_space,0,645,0
round_bracket_close,0,0,1


Raw character features by governed runner rows


source_field,jockey,owner,trainer
feature,,,
ampersand,0,0,53656
apostrophe,0,0,0
ascii_numeral,0,36588,0
comma,0,0,0
hyphen,24259,0,46903
non_ascii_character,0,0,0
period,0,0,0
repeated_space,0,6300,0
round_bracket_close,0,0,1


## Stage 7 — Inspect observed structural label forms

The raw character profile showed that the three fields use markedly different visible conventions:

- jockey labels contain hyphens but no other measured punctuation;
- trainer labels contain hyphens, ampersands and one bracketed label;
- owner labels contain numerals and repeated spaces, but none of the measured punctuation;
- no field contains a non-ASCII character.

These observations may reflect source formatting or source transformation rather than real-world naming conventions.

This stage inspects the exact raw labels behind the observed features.

It asks:

- what forms the trainer ampersand labels take;
- whether trainer hyphens appear within personal names or other constructions;
- what the one bracketed trainer label contains;
- what forms the jockey hyphen labels take;
- how numerals appear in owner labels;
- whether repeated owner spaces appear to separate otherwise distinct components.

The examples remain exact source labels.

No punctuation is removed, no owner label is decomposed, and no entity equivalence is inferred.

In [7]:
# Helper used only to display exact-label examples for one observed feature.
#
# The function does not transform the raw label. It filters an already
# reconciled exact-label table and returns the most frequent examples.
def most_frequent_feature_examples(
    labels: pd.DataFrame,
    mask: pd.Series,
    limit: int = 30,
) -> pd.DataFrame:
    """Return the most frequent exact raw labels selected by a Boolean mask."""
    return (
        labels.loc[
            mask,
            [
                "raw_label",
                "runner_rows",
                "first_observed_date",
                "last_observed_date",
            ],
        ]
        .sort_values(
            ["runner_rows", "raw_label"],
            ascending=[False, True],
        )
        .head(limit)
        .reset_index(drop=True)
    )


# Retain direct references to the three exact-label tables built in Stage 6.
jockey_labels = exact_label_profiles["jockey"]
trainer_labels = exact_label_profiles["trainer"]
owner_labels = exact_label_profiles["owner"]


# Trainer ampersands may represent joint licences, partnerships or another
# source convention. Display them before assigning any interpretation.
trainer_ampersand_examples = most_frequent_feature_examples(
    trainer_labels,
    trainer_labels["raw_label"].str.contains(
        "&",
        regex=False,
        na=False,
    ),
)


# Trainer hyphens may occur inside personal names or in other label forms.
# They must not be treated as removable punctuation without inspection.
trainer_hyphen_examples = most_frequent_feature_examples(
    trainer_labels,
    trainer_labels["raw_label"].str.contains(
        "-",
        regex=False,
        na=False,
    ),
)


# Only one trainer exact label contains round brackets, so preserve and inspect
# the complete finite set rather than sampling it.
trainer_bracket_examples = most_frequent_feature_examples(
    trainer_labels,
    trainer_labels["raw_label"].str.contains(
        "(",
        regex=False,
        na=False,
    )
    | trainer_labels["raw_label"].str.contains(
        ")",
        regex=False,
        na=False,
    ),
    limit=len(trainer_labels),
)


# Jockey hyphens are the only measured punctuation found in jockey labels.
# Inspect the most frequent labels before considering whether the hyphen is
# part of a given name, family name or source presentation convention.
jockey_hyphen_examples = most_frequent_feature_examples(
    jockey_labels,
    jockey_labels["raw_label"].str.contains(
        "-",
        regex=False,
        na=False,
    ),
)


# Owner numerals may identify numbered syndicates, racing groups, companies,
# account labels, initials or another construction. This mask merely selects
# labels containing at least one ASCII digit.
owner_numeral_mask = owner_labels["raw_label"].map(
    contains_ascii_digit
)

owner_numeral_examples = most_frequent_feature_examples(
    owner_labels,
    owner_numeral_mask,
)


# Repeated owner spaces may encode a lost separator, but that remains only a
# hypothesis. Display the most frequent exact labels without collapsing them.
owner_repeated_space_examples = most_frequent_feature_examples(
    owner_labels,
    owner_labels["raw_label"].str.contains(
        "  ",
        regex=False,
        na=False,
    ),
)


# Reconcile each displayed population with the counts already reported in the
# raw character profile. These checks guard against inspecting a different or
# incomplete subset.
def expected_distinct_feature_count(
    source_field: str,
    feature: str,
) -> int:
    """Read one previously calculated distinct-label feature count."""
    return int(
        raw_character_profile.loc[
            (
                raw_character_profile["source_field"]
                == source_field
            )
            & (
                raw_character_profile["feature"]
                == feature
            ),
            "distinct_raw_labels",
        ].iloc[0]
    )


assert int(
    trainer_labels["raw_label"]
    .str.contains("&", regex=False, na=False)
    .sum()
) == expected_distinct_feature_count(
    "trainer",
    "ampersand",
)

assert int(
    trainer_labels["raw_label"]
    .str.contains("-", regex=False, na=False)
    .sum()
) == expected_distinct_feature_count(
    "trainer",
    "hyphen",
)

assert len(trainer_bracket_examples) == expected_distinct_feature_count(
    "trainer",
    "round_bracket_open",
)

assert int(
    jockey_labels["raw_label"]
    .str.contains("-", regex=False, na=False)
    .sum()
) == expected_distinct_feature_count(
    "jockey",
    "hyphen",
)

assert int(owner_numeral_mask.sum()) == expected_distinct_feature_count(
    "owner",
    "ascii_numeral",
)

assert int(
    owner_labels["raw_label"]
    .str.contains("  ", regex=False, na=False)
    .sum()
) == expected_distinct_feature_count(
    "owner",
    "repeated_space",
)


print("Most frequent trainer labels containing ampersands")
display(trainer_ampersand_examples)

print("Most frequent trainer labels containing hyphens")
display(trainer_hyphen_examples)

print("All trainer labels containing round brackets")
display(trainer_bracket_examples)

print("Most frequent jockey labels containing hyphens")
display(jockey_hyphen_examples)

print("Most frequent owner labels containing numerals")
display(owner_numeral_examples)

print("Most frequent owner labels containing repeated spaces")
display(owner_repeated_space_examples)

Most frequent trainer labels containing ampersands


,raw_label,runner_rows,first_observed_date,last_observed_date
0,John & Thady Gosden,3230,2021-03-26,2026-05-27
1,Michael & David Easterby,2589,2021-06-08,2026-05-27
2,Simon & Ed Crisford,2533,2020-06-01,2026-05-27
3,Oliver Greenall & Josh Guerriero,1768,2022-04-30,2026-05-26
4,A & G Botti,1615,2015-01-03,2026-01-16
5,Gary & Josh Moore,1603,2024-05-02,2026-05-27
6,Gai Waterhouse & Adrian Bott,1335,2016-08-20,2026-05-23
7,Charlie & Mark Johnston,1280,2022-01-02,2022-12-31
8,D & P ProdHomme,1233,2015-01-03,2026-04-26
9,Peter & Paul Snowden,1205,2015-01-09,2024-10-19


Most frequent trainer labels containing hyphens


,raw_label,runner_rows,first_observed_date,last_observed_date
0,Nigel Twiston-Davies,5220,2015-01-01,2025-04-26
1,H-A Pantall,4167,2015-01-03,2026-05-24
2,F-H Graffard,2812,2015-01-10,2026-05-25
3,J-C Rouget,2482,2015-01-14,2026-05-23
4,Jane Chapple-Hyam,2195,2015-01-11,2026-05-27
5,F-M Cottin,1813,2015-02-24,2021-07-05
6,C Laffon-Parias,1627,2015-02-26,2024-12-18
7,A Chaille-Chaille,1519,2015-02-19,2025-09-23
8,H-F Devin,1347,2015-03-09,2026-05-26
9,Mme P Butel & J-L Beaunez,1078,2017-09-18,2026-05-26


All trainer labels containing round brackets


...

,raw_label,runner_rows,first_observed_date,last_observed_date
0,Sam Twiston-Davies,7420,2015-01-01,2026-05-27
1,Pierre-Charles Boudot,4544,2015-01-10,2026-02-14
2,Pierre-Louis Jamin,1503,2015-09-21,2026-05-27
3,Conor Stone-Walsh,1032,2022-05-24,2026-05-27
4,William Twiston-Davies,913,2015-01-02,2018-09-12
5,Christophe-Patrice Lemaire,832,2015-04-05,2026-05-24
6,Emma Smith-Chaston,599,2017-05-31,2025-04-22
7,Tom Kiely-Marshall,486,2023-05-03,2026-05-27
8,Elle-May Croot,439,2021-01-02,2026-04-05
9,Leo-Paul Brechet,402,2023-09-03,2026-05-26


Most frequent owner labels containing numerals


,raw_label,runner_rows,first_observed_date,last_observed_date
0,Melbourne 10 Racing,515,2016-09-13,2023-06-05
1,The 119 Partnership,354,2020-06-14,2026-05-27
2,Ontoawinner 10 Partner,308,2015-04-11,2026-05-23
3,G1 Racing Co Ltd,291,2015-01-18,2026-05-02
4,Bronte Collection 1,261,2022-04-09,2024-10-31
5,Nick Bradley Racing 3 And Partner,217,2021-03-26,2025-10-24
6,Rebel Racing 2,199,2015-06-25,2024-05-18
7,Nick Bradley Racing 5 E Burke,189,2016-07-07,2026-05-27
8,Ontoawinner 14 Mrs E Burke,183,2016-02-01,2023-09-22
9,Nickbradleyracing45 Alfa Site Services,175,2023-04-30,2026-04-07


Most frequent owner labels containing repeated spaces


,raw_label,runner_rows,first_observed_date,last_observed_date
0,J L Wetherald M M Glover,70,2015-05-28,2023-10-19
1,Ursa Major England,65,2023-02-21,2026-05-25
2,Kingsley Park 1 Ready To Run,63,2015-02-02,2015-10-24
3,Pau Perth Partnership,60,2019-04-19,2025-04-12
4,Kildare Stud Frankie Oconnor,55,2016-12-15,2025-03-07
5,Kingsley Park 7 Ready To Run,48,2016-11-28,2017-10-27
6,Muir Racing Partnership Saint Cloud,48,2018-07-16,2024-08-16
7,Muir Racing Partnership Santa Anita,45,2017-05-02,2021-11-11
8,A Graham Bankruptcy Trustee M Stanley,44,2019-04-07,2020-08-12
9,Newsells Park Stud Ossie Ardiles Synd,44,2022-04-22,2023-12-12


## Stage 8 — Profile explicit role and ownership vocabulary

The structural examples show that visible punctuation alone is insufficient for interpreting these fields.

In particular:

- trainer ampersands commonly occur in joint-training labels;
- trainer hyphens have several unrelated naming functions;
- owner numerals occur in many different label constructions;
- repeated owner spaces appear to divide meaningful components.

This stage searches the exact raw labels for explicit words already present in the source.

For trainer labels, it measures titles and role-related terms such as:

- `Mr`;
- `Mrs`;
- `Mme`;
- `Mlle`;
- `Dr`;
- `Jnr`.

For owner labels, it measures explicit terms such as:

- `Partnership`;
- `Partner` or `Partners`;
- `Syndicate`;
- `Club`;
- `Racing`;
- `Stud`;
- `Farm`;
- `Stable` or `Stables`;
- `Ltd`;
- `Limited`;
- `LLC`;
- `Inc`;
- `Company` or `Co`;
- `Trust`;
- `Estate`.

The matching is descriptive and source-internal.

A lexical match does not prove:

- legal entity type;
- registered ownership-account type;
- partnership membership;
- syndicate membership;
- equivalence between similarly named labels;
- that labels without a keyword are individuals;
- that labels with a keyword may be decomposed safely.

The results will be used only to describe visible source vocabulary and to identify later investigation groups.

In [8]:
# Match complete source words rather than arbitrary substrings.
#
# For example, the token "club" should match "Racing Club" but should not
# match a longer unrelated word containing those letters.
def contains_source_word(
    value: str,
    source_word: str,
) -> bool:
    """Return True when a case-insensitive source word occurs as a token."""
    pattern = rf"(?<![A-Za-z0-9]){re.escape(source_word)}(?![A-Za-z0-9])"
    return bool(
        re.search(
            pattern,
            value,
            flags=re.IGNORECASE,
        )
    )


# Each indicator retains its component source words.
#
# Indicators that combine several visible variants are descriptive groups
# only. The source labels themselves remain unchanged.
TRAINER_LEXICAL_INDICATORS = {
    "mr": ["Mr"],
    "mrs": ["Mrs"],
    "mme": ["Mme"],
    "mlle": ["Mlle"],
    "dr": ["Dr"],
    "jnr": ["Jnr"],
}

OWNER_LEXICAL_INDICATORS = {
    "partnership": ["Partnership"],
    "partner_or_partners": ["Partner", "Partners"],
    "syndicate": ["Syndicate", "Syndikat", "Syndicat"],
    "club": ["Club"],
    "racing": ["Racing"],
    "stud": ["Stud"],
    "farm": ["Farm"],
    "stable_or_stables": ["Stable", "Stables", "Stalt"],
    "limited_company_marker": ["Ltd", "Limited", "LLC", "Inc"],
    "company_or_co": ["Company", "Co"],
    "trust": ["Trust"],
    "estate": ["Estate"],
}


def indicator_mask(
    labels: pd.Series,
    source_words: list[str],
) -> pd.Series:
    """Return a mask for labels containing any governed source word."""
    return labels.map(
        lambda value: any(
            contains_source_word(value, source_word)
            for source_word in source_words
        )
    )


lexical_profile_rows = []

# Trainer vocabulary is calculated only from the exact trainer-label table.
for indicator, source_words in TRAINER_LEXICAL_INDICATORS.items():
    mask = indicator_mask(
        trainer_labels["raw_label"],
        source_words,
    )

    lexical_profile_rows.append(
        {
            "source_field": "trainer",
            "indicator": indicator,
            "distinct_raw_labels": int(mask.sum()),
            "runner_rows": int(
                trainer_labels.loc[mask, "runner_rows"].sum()
            ),
        }
    )


# Owner vocabulary is calculated independently because owner labels may
# represent people, groups, accounts or organisations rather than licensed
# racing professionals.
for indicator, source_words in OWNER_LEXICAL_INDICATORS.items():
    mask = indicator_mask(
        owner_labels["raw_label"],
        source_words,
    )

    lexical_profile_rows.append(
        {
            "source_field": "owner",
            "indicator": indicator,
            "distinct_raw_labels": int(mask.sum()),
            "runner_rows": int(
                owner_labels.loc[mask, "runner_rows"].sum()
            ),
        }
    )


lexical_vocabulary_profile = pd.DataFrame(
    lexical_profile_rows
)

# Ensure that each configured indicator produced exactly one summary row.
assert (
    lexical_vocabulary_profile.loc[
        lexical_vocabulary_profile["source_field"] == "trainer"
    ].shape[0]
    == len(TRAINER_LEXICAL_INDICATORS)
)

assert (
    lexical_vocabulary_profile.loc[
        lexical_vocabulary_profile["source_field"] == "owner"
    ].shape[0]
    == len(OWNER_LEXICAL_INDICATORS)
)


trainer_lexical_profile = (
    lexical_vocabulary_profile.loc[
        lexical_vocabulary_profile["source_field"] == "trainer",
        [
            "indicator",
            "distinct_raw_labels",
            "runner_rows",
        ],
    ]
    .sort_values(
        ["distinct_raw_labels", "indicator"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

owner_lexical_profile = (
    lexical_vocabulary_profile.loc[
        lexical_vocabulary_profile["source_field"] == "owner",
        [
            "indicator",
            "distinct_raw_labels",
            "runner_rows",
        ],
    ]
    .sort_values(
        ["distinct_raw_labels", "indicator"],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)

print("Trainer lexical vocabulary")
display(trainer_lexical_profile)

print("Owner lexical vocabulary")
display(owner_lexical_profile)

Trainer lexical vocabulary


,indicator,distinct_raw_labels,runner_rows
0,mrs,229,16432
1,mme,192,13473
2,mlle,100,6238
3,dr,8,3927
4,jnr,4,878
5,mr,0,0


Owner lexical vocabulary


,indicator,distinct_raw_labels,runner_rows
0,racing,12107,215549
1,limited_company_marker,6835,126611
2,syndicate,5720,95515
3,partnership,3947,72106
4,partner_or_partners,3247,46113
5,stable_or_stables,2924,22435
6,stud,2188,41161
7,club,1331,43108
8,farm,1025,14544
9,company_or_co,351,16035


## Stage 9 — Measure overlap in owner vocabulary

The owner vocabulary profile confirms that the field contains labels associated with several visibly different constructions, including racing groups, companies, syndicates, partnerships, clubs, studs, farms, stables, trusts and estates.

The indicator counts overlap.

For example, one raw label may contain several terms such as:

- `Racing`;
- `Club`;
- `Ltd`.

This stage measures:

- how many configured owner indicators each exact raw label matches;
- the number of labels and runner rows matching no indicator;
- the number matching one indicator;
- the number matching several indicators;
- the most frequent combinations of indicators;
- examples from the residual group with no measured indicator.

The indicator count is not an entity classification.

A label matching no configured word is not assumed to be an individual person. It may still represent:

- an individual;
- a family;
- a partnership without explicit partnership vocabulary;
- a syndicate or organisation using a proper name;
- an ownership account;
- a translated or abbreviated organisational label;
- another unresolved construction.

Similarly, a label matching several indicators is not decomposed into several entities.

In [9]:
# Build one Boolean indicator column per owner vocabulary group.
#
# These columns record only whether the exact source label contains one of the
# configured words. The raw owner label itself remains unchanged.
owner_indicator_detail = owner_labels.copy()

for indicator, source_words in OWNER_LEXICAL_INDICATORS.items():
    owner_indicator_detail[indicator] = indicator_mask(
        owner_indicator_detail["raw_label"],
        source_words,
    )


# Count how many configured vocabulary groups each exact owner label matches.
#
# This is a description of visible source wording, not a count of real-world
# entities or members.
owner_indicator_columns = list(
    OWNER_LEXICAL_INDICATORS
)

owner_indicator_detail["indicator_count"] = (
    owner_indicator_detail[owner_indicator_columns]
    .sum(axis=1)
    .astype(int)
)


# Create a stable text description of each matched indicator combination.
#
# Labels with no configured indicator are retained explicitly as
# "no_measured_indicator" rather than being interpreted as individuals.
def describe_indicator_combination(row: pd.Series) -> str:
    """Describe the visible owner-vocabulary indicators matched by one label."""
    matched_indicators = [
        indicator
        for indicator in owner_indicator_columns
        if bool(row[indicator])
    ]

    if not matched_indicators:
        return "no_measured_indicator"

    return " | ".join(matched_indicators)


owner_indicator_detail["indicator_combination"] = (
    owner_indicator_detail.apply(
        describe_indicator_combination,
        axis=1,
    )
)


# Summarise the distribution by number of matched vocabulary groups.
owner_indicator_count_distribution = (
    owner_indicator_detail.groupby(
        "indicator_count",
        as_index=False,
    )
    .agg(
        distinct_raw_labels=("raw_label", "size"),
        runner_rows=("runner_rows", "sum"),
    )
    .sort_values("indicator_count")
    .reset_index(drop=True)
)


# Summarise exact combinations so that common overlaps such as
# "racing | limited_company_marker" remain visible.
owner_indicator_combination_profile = (
    owner_indicator_detail.groupby(
        "indicator_combination",
        as_index=False,
    )
    .agg(
        distinct_raw_labels=("raw_label", "size"),
        runner_rows=("runner_rows", "sum"),
    )
    .sort_values(
        ["distinct_raw_labels", "runner_rows", "indicator_combination"],
        ascending=[False, False, True],
    )
    .reset_index(drop=True)
)


# Retain examples from labels with no measured organisational vocabulary.
#
# These are a residual investigation group only. They are not classified as
# people merely because none of the configured words appears.
owner_no_indicator_examples = (
    owner_indicator_detail.loc[
        owner_indicator_detail["indicator_count"] == 0,
        [
            "raw_label",
            "runner_rows",
            "first_observed_date",
            "last_observed_date",
        ],
    ]
    .sort_values(
        ["runner_rows", "raw_label"],
        ascending=[False, True],
    )
    .head(40)
    .reset_index(drop=True)
)


# Also retain examples of labels matching several indicators because these
# show why the vocabulary groups cannot be treated as mutually exclusive
# entity classes.
owner_multiple_indicator_examples = (
    owner_indicator_detail.loc[
        owner_indicator_detail["indicator_count"] >= 3,
        [
            "raw_label",
            "runner_rows",
            "first_observed_date",
            "last_observed_date",
            "indicator_count",
            "indicator_combination",
        ],
    ]
    .sort_values(
        ["runner_rows", "raw_label"],
        ascending=[False, True],
    )
    .head(40)
    .reset_index(drop=True)
)


# Reconcile the detailed table to the complete populated owner population.
assert len(owner_indicator_detail) == int(
    raw_field_profile.loc[
        raw_field_profile["source_field"] == "owner",
        "distinct_populated_labels",
    ].iloc[0]
)

assert int(
    owner_indicator_detail["runner_rows"].sum()
) == int(
    raw_field_profile.loc[
        raw_field_profile["source_field"] == "owner",
        "populated_rows",
    ].iloc[0]
)

assert (
    owner_indicator_count_distribution[
        "distinct_raw_labels"
    ].sum()
    == len(owner_indicator_detail)
)

assert (
    owner_indicator_count_distribution[
        "runner_rows"
    ].sum()
    == owner_indicator_detail["runner_rows"].sum()
)


print("Owner labels by number of matched vocabulary indicators")
display(owner_indicator_count_distribution)

print("Most frequent owner vocabulary combinations")
display(owner_indicator_combination_profile.head(30))

print("Most frequent owner labels with no measured indicator")
display(owner_no_indicator_examples)

print("Most frequent owner labels matching at least three indicators")
display(owner_multiple_indicator_examples)

Owner labels by number of matched vocabulary indicators


,indicator_count,distinct_raw_labels,runner_rows
0,0,67874,1323718
1,1,22076,380861
2,2,7058,125099
3,3,1174,21356
4,4,52,216


Most frequent owner vocabulary combinations


,indicator_combination,distinct_raw_labels,runner_rows
0,no_measured_indicator,67874,1323718
1,racing,6122,105813
2,syndicate,4312,75305
3,partnership,3305,60082
4,limited_company_marker,2667,58111
5,partner_or_partners,2237,31473
6,stud,1592,29318
7,racing | limited_company_marker,1510,25781
8,stable_or_stables,955,8349
9,syndicate | racing,908,15441


Most frequent owner labels with no measured indicator


,raw_label,runner_rows,first_observed_date,last_observed_date
0,John P Mcmanus,15384,2015-01-01,2026-05-27
1,Godolphin,13416,2015-01-01,2026-05-27
2,Hamdan Al Maktoum,6867,2015-01-04,2021-03-23
3,Sheikh Hamdan Bin Mohammed Al Maktoum,4539,2015-01-04,2026-05-27
4,Mrs J S Bolger,3779,2015-01-02,2026-05-26
5,Simon Munir Isaac Souede,3410,2015-01-01,2026-05-26
6,H H Aga Khan,2970,2015-01-09,2025-03-04
7,Wertheimer Frere,2797,2015-01-12,2026-05-24
8,Sheikh Ahmed Al Maktoum,2769,2015-01-04,2026-05-26
9,K Abdullah,2486,2015-01-03,2021-01-09


Most frequent owner labels matching at least three indicators


,raw_label,runner_rows,first_observed_date,last_observed_date,indicator_count,indicator_combination
0,King Power Racing Co Ltd,2563,2017-07-07,2026-05-27,3,racing | limited_company_marker | company_or_co
1,Shadwell Estate Company Ltd,1604,2021-03-26,2026-05-27,3,limited_company_marker | company_or_co | estate
2,Newtown Anner Stud Farm Ltd,1594,2015-01-10,2026-05-27,3,stud | farm | limited_company_marker
3,Sunday Racing Co Ltd,1183,2015-01-04,2026-05-24,3,racing | limited_company_marker | company_or_co
4,Carrot Farm Co Ltd,1025,2015-01-18,2026-05-24,3,farm | limited_company_marker | company_or_co
5,Richard Fahey Ebor Racing Club Ltd,961,2015-01-07,2026-05-27,3,club | racing | limited_company_marker
6,Silk Racing Co Ltd,702,2015-01-18,2026-05-17,3,racing | limited_company_marker | company_or_co
7,Thoroughbred Club Ruffian Co Ltd,432,2015-01-04,2026-05-24,3,club | limited_company_marker | company_or_co
8,Ballantines Racing Stud Ltd,424,2015-02-20,2024-02-02,3,racing | stud | limited_company_marker
9,Hkjc Racing Club Limited,365,2016-11-16,2026-05-24,3,club | racing | limited_company_marker


## Stage 10 — Cross-field exact-label overlap

The source stores `jockey`, `trainer` and `owner` as separate runner-level role assertions.

The same exact raw string may nevertheless occur in more than one field.

This may happen because:

- one person performs more than one racing role;
- an individual trains and owns horses;
- a jockey later becomes a trainer;
- an owner label happens to resemble a professional’s name;
- initials or abbreviated names collide;
- the same source string represents different real-world entities;
- the source has copied or misassigned a connection.

Exact string equality across fields therefore does not prove shared real-world identity.

This stage measures:

- exact labels appearing as both jockey and trainer;
- exact labels appearing as both jockey and owner;
- exact labels appearing as both trainer and owner;
- exact labels appearing in all three fields;
- the runner-row frequency and date range of each label within each role.

The results remain role-specific source assertions.

Any cross-role entity link is deferred unless later evidence supports it.

In [10]:
# Prepare one role-specific summary per exact raw label.
#
# Each table preserves the raw source label and records only its frequency and
# observed date range within that role. No cross-role identity is assumed.
role_label_summaries = {}

for role in CONNECTION_IDENTITY_FIELDS:
    role_labels = exact_label_profiles[role].rename(
        columns={
            "runner_rows": f"{role}_runner_rows",
            "first_observed_date": f"{role}_first_observed_date",
            "last_observed_date": f"{role}_last_observed_date",
        }
    )

    role_label_summaries[role] = role_labels


def exact_role_overlap(
    first_role: str,
    second_role: str,
) -> pd.DataFrame:
    """
    Return exact raw labels appearing in two source roles.

    The merge uses literal source-string equality only. It does not claim that
    the two role assertions identify the same real-world entity.
    """
    return (
        role_label_summaries[first_role]
        .merge(
            role_label_summaries[second_role],
            on="raw_label",
            how="inner",
            validate="one_to_one",
        )
        .sort_values(
            [
                f"{first_role}_runner_rows",
                f"{second_role}_runner_rows",
                "raw_label",
            ],
            ascending=[False, False, True],
        )
        .reset_index(drop=True)
    )


# Build the three pairwise exact-label overlap sets.
jockey_trainer_overlap = exact_role_overlap(
    "jockey",
    "trainer",
)

jockey_owner_overlap = exact_role_overlap(
    "jockey",
    "owner",
)

trainer_owner_overlap = exact_role_overlap(
    "trainer",
    "owner",
)


# Build the all-three overlap independently.
#
# This is not inferred from pairwise counts because a label may appear in one
# pair without appearing in the third role.
all_three_role_overlap = (
    role_label_summaries["jockey"]
    .merge(
        role_label_summaries["trainer"],
        on="raw_label",
        how="inner",
        validate="one_to_one",
    )
    .merge(
        role_label_summaries["owner"],
        on="raw_label",
        how="inner",
        validate="one_to_one",
    )
    .sort_values(
        [
            "jockey_runner_rows",
            "trainer_runner_rows",
            "owner_runner_rows",
            "raw_label",
        ],
        ascending=[False, False, False, True],
    )
    .reset_index(drop=True)
)


# Summarise the scale of exact-string overlap.
cross_role_overlap_summary = pd.DataFrame(
    [
        {
            "role_overlap": "jockey_and_trainer",
            "distinct_exact_labels": len(jockey_trainer_overlap),
        },
        {
            "role_overlap": "jockey_and_owner",
            "distinct_exact_labels": len(jockey_owner_overlap),
        },
        {
            "role_overlap": "trainer_and_owner",
            "distinct_exact_labels": len(trainer_owner_overlap),
        },
        {
            "role_overlap": "jockey_trainer_and_owner",
            "distinct_exact_labels": len(all_three_role_overlap),
        },
    ]
)


# Confirm that every all-three label appears in each relevant pairwise set.
all_three_labels = set(all_three_role_overlap["raw_label"])

assert all_three_labels.issubset(
    set(jockey_trainer_overlap["raw_label"])
)

assert all_three_labels.issubset(
    set(jockey_owner_overlap["raw_label"])
)

assert all_three_labels.issubset(
    set(trainer_owner_overlap["raw_label"])
)


print("Exact raw-label overlap across roles")
display(cross_role_overlap_summary)

print("Most frequent labels appearing as both jockey and trainer")
display(jockey_trainer_overlap.head(40))

print("Most frequent labels appearing as both jockey and owner")
display(jockey_owner_overlap.head(40))

print("Most frequent labels appearing as both trainer and owner")
display(trainer_owner_overlap.head(40))

print("All labels appearing in all three roles")
display(all_three_role_overlap)

Exact raw-label overlap across roles


,role_overlap,distinct_exact_labels
0,jockey_and_trainer,226
1,jockey_and_owner,198
2,trainer_and_owner,1857
3,jockey_trainer_and_owner,65


Most frequent labels appearing as both jockey and trainer


,raw_label,jockey_runner_rows,jockey_first_observed_date,jockey_last_observed_date,trainer_runner_rows,trainer_first_observed_date,trainer_last_observed_date
0,Adam Kirby,5962,2015-01-02,2025-09-14,55,2026-03-11,2026-05-26
1,Tom Scudamore,4530,2015-01-01,2023-09-17,1,2026-05-15,2026-05-15
2,Brendan Powell,4096,2015-01-01,2026-05-27,896,2015-01-03,2019-01-26
3,David Noonan,3421,2015-01-01,2026-05-27,2,2024-09-28,2024-10-12
4,Nick Scholfield,3324,2015-01-01,2025-04-05,139,2025-11-06,2026-05-26
5,Andrew Slattery,2944,2017-10-07,2026-05-26,3573,2015-01-02,2026-05-26
6,Gerald Mosse,2387,2015-01-25,2024-09-15,262,2024-10-15,2026-05-24
7,Sean Davis,2002,2015-10-10,2026-05-04,166,2023-04-14,2026-05-04
8,Phillip Makin,1967,2015-01-15,2018-08-25,1071,2019-03-27,2026-05-26
9,Richard Patrick,1703,2015-03-26,2026-04-16,1,2015-03-26,2015-03-26


Most frequent labels appearing as both jockey and owner


,raw_label,jockey_runner_rows,jockey_first_observed_date,jockey_last_observed_date,owner_runner_rows,owner_first_observed_date,owner_last_observed_date
0,W J Lee,6317,2015-01-02,2026-05-26,3,2015-12-10,2016-02-21
1,Adam Kirby,5962,2015-01-02,2025-09-14,15,2026-03-11,2026-05-26
2,Richard Johnson,5361,2015-01-03,2024-09-15,1,2026-05-13,2026-05-13
3,Shane Gray,3944,2015-01-01,2026-05-27,4,2025-08-29,2026-03-14
4,Paddy Brennan,3672,2015-01-01,2024-04-17,2,2025-01-17,2025-01-31
5,Nick Scholfield,3324,2015-01-01,2025-04-05,5,2026-01-15,2026-04-06
6,Mark Enright,2295,2015-01-01,2024-09-15,5,2025-01-28,2025-05-16
7,Noel Fehily,2281,2015-01-01,2021-09-08,16,2022-02-27,2024-04-30
8,William Kennedy,2134,2015-01-01,2023-09-17,6,2025-04-11,2026-05-13
9,James Ryan,1777,2021-05-30,2026-05-26,1,2016-03-18,2016-03-18


Most frequent labels appearing as both trainer and owner


,raw_label,trainer_runner_rows,trainer_first_observed_date,trainer_last_observed_date,owner_runner_rows,owner_first_observed_date,owner_last_observed_date
0,Gordon Elliott,15140,2015-01-01,2026-05-27,181,2015-07-17,2026-05-15
1,Richard Hannon,13165,2015-01-03,2026-05-27,323,2015-01-03,2026-05-18
2,Michael Appleby,9584,2015-01-01,2026-05-27,384,2015-01-11,2025-10-29
3,Dan Skelton,8939,2015-01-01,2026-05-27,221,2015-03-29,2026-05-16
4,Mrs John Harrington,8320,2015-01-02,2026-05-26,215,2015-04-09,2026-05-03
5,Ian Williams,7333,2015-01-01,2026-05-27,394,2015-04-18,2026-05-22
6,Alan King,6770,2015-01-01,2026-05-24,97,2015-06-18,2026-05-09
7,Paul Nicholls,6712,2015-01-01,2026-05-27,11,2016-04-07,2024-10-23
8,Keith Dalgleish,5986,2015-01-01,2023-07-08,52,2015-12-17,2022-01-31
9,Brian Ellison,5855,2015-01-01,2026-05-27,62,2015-01-05,2018-04-23


All labels appearing in all three roles


,raw_label,jockey_runner_rows,jockey_first_observed_date,jockey_last_observed_date,trainer_runner_rows,trainer_first_observed_date,trainer_last_observed_date,owner_runner_rows,owner_first_observed_date,owner_last_observed_date
0,Adam Kirby,5962,2015-01-02,2025-09-14,55,2026-03-11,2026-05-26,15,2026-03-11,2026-05-26
1,Nick Scholfield,3324,2015-01-01,2025-04-05,139,2025-11-06,2026-05-26,5,2026-01-15,2026-04-06
2,Richard Patrick,1703,2015-03-26,2026-04-16,1,2015-03-26,2015-03-26,1,2015-03-26,2015-03-26
3,George Baker,1520,2015-01-02,2017-02-26,2625,2015-01-03,2026-05-27,46,2015-05-19,2026-05-23
4,Connor King,1052,2015-01-02,2023-04-21,26,2024-05-13,2026-04-29,2,2025-01-05,2025-01-29
...,...,...,...,...,...,...,...,...,...,...
60,Mme Laura Salton,1,2025-05-15,2025-05-15,3,2025-05-15,2025-12-29,3,2025-05-15,2025-12-29
61,T Doumen,1,2015-08-22,2015-08-22,3,2021-11-07,2022-05-16,3,2021-11-07,2022-05-16
62,Chris Williams,1,2018-01-26,2018-01-26,1,2016-01-28,2016-01-28,20,2016-01-28,2025-04-17
63,Frau Valentina Stefutti,1,2017-03-11,2017-03-11,1,2017-03-11,2017-03-11,3,2015-12-28,2017-03-11


## Stage 11 — Temporal relationships between exact cross-role labels

Exact raw-label overlap does not establish shared real-world identity.

However, chronology can help divide the overlap sets into more useful investigation groups.

For each pair of roles, this stage classifies the observed date ranges as:

- `first_role_before_second_role` — the first role ends before the second begins;
- `second_role_before_first_role` — the second role ends before the first begins;
- `date_ranges_overlap` — the two role date ranges overlap;
- `same_single_date` — both roles occur only on the same single date.

These categories describe source chronology only.

They do not prove:

- a genuine career transition;
- that overlapping labels identify different people;
- that non-overlapping labels identify the same person;
- that one role ended permanently;
- that a one-day overlap is a correct source assignment.

The stage also checks whether an exact label appears in two roles within the same provisional race.

A same-race occurrence may be informative, but it still does not by itself establish identity or error.

In [11]:
def classify_role_date_relationship(
    row: pd.Series,
    first_role: str,
    second_role: str,
) -> str:
    """
    Classify the observed date-range relationship for one exact raw label.

    The result describes source chronology only. It is not an entity decision.
    """
    first_start = pd.Timestamp(
        row[f"{first_role}_first_observed_date"]
    )
    first_end = pd.Timestamp(
        row[f"{first_role}_last_observed_date"]
    )
    second_start = pd.Timestamp(
        row[f"{second_role}_first_observed_date"]
    )
    second_end = pd.Timestamp(
        row[f"{second_role}_last_observed_date"]
    )

    # Preserve the narrow case where both roles are observed only once and on
    # exactly the same date. This may indicate a source anomaly, a genuine
    # dual-role case, or two different people sharing one exact label.
    if (
        first_start == first_end
        and second_start == second_end
        and first_start == second_start
    ):
        return "same_single_date"

    if first_end < second_start:
        return f"{first_role}_before_{second_role}"

    if second_end < first_start:
        return f"{second_role}_before_{first_role}"

    return "date_ranges_overlap"


def add_temporal_relationship(
    overlap: pd.DataFrame,
    first_role: str,
    second_role: str,
) -> pd.DataFrame:
    """Add a source-chronology classification to one pairwise overlap table."""
    result = overlap.copy()

    result["temporal_relationship"] = result.apply(
        classify_role_date_relationship,
        axis=1,
        first_role=first_role,
        second_role=second_role,
    )

    return result


# Add temporal classifications independently to each pairwise overlap set.
jockey_trainer_temporal = add_temporal_relationship(
    jockey_trainer_overlap,
    "jockey",
    "trainer",
)

jockey_owner_temporal = add_temporal_relationship(
    jockey_owner_overlap,
    "jockey",
    "owner",
)

trainer_owner_temporal = add_temporal_relationship(
    trainer_owner_overlap,
    "trainer",
    "owner",
)


# Summarise date-range relationships by role pair.
temporal_relationship_summary = pd.concat(
    [
        (
            jockey_trainer_temporal.groupby(
                "temporal_relationship",
                as_index=False,
            )
            .size()
            .rename(columns={"size": "distinct_exact_labels"})
            .assign(role_pair="jockey_and_trainer")
        ),
        (
            jockey_owner_temporal.groupby(
                "temporal_relationship",
                as_index=False,
            )
            .size()
            .rename(columns={"size": "distinct_exact_labels"})
            .assign(role_pair="jockey_and_owner")
        ),
        (
            trainer_owner_temporal.groupby(
                "temporal_relationship",
                as_index=False,
            )
            .size()
            .rename(columns={"size": "distinct_exact_labels"})
            .assign(role_pair="trainer_and_owner")
        ),
    ],
    ignore_index=True,
)[
    [
        "role_pair",
        "temporal_relationship",
        "distinct_exact_labels",
    ]
].sort_values(
    [
        "role_pair",
        "temporal_relationship",
    ]
).reset_index(drop=True)


# Query exact labels used in two roles within the same provisional race.
#
# The provisional race key remains date + course + off. Each role assertion is
# preserved separately, and no shared entity is inferred from the match.
connection = sqlite3.connect(
    f"file:{SOURCE_DB_PATH}?mode=ro",
    uri=True,
)

try:
    same_race_jockey_trainer = pd.read_sql_query(
        f"""
        SELECT
            jockey AS raw_label,
            date,
            course,
            off,
            COUNT(*) AS matching_runner_rows
        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}
          AND jockey IS NOT NULL
          AND jockey <> ''
          AND trainer IS NOT NULL
          AND trainer <> ''
          AND jockey = trainer
        GROUP BY jockey, date, course, off
        ORDER BY date, course, off, raw_label
        """,
        connection,
    )

    same_race_jockey_owner = pd.read_sql_query(
        f"""
        SELECT
            jockey AS raw_label,
            date,
            course,
            off,
            COUNT(*) AS matching_runner_rows
        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}
          AND jockey IS NOT NULL
          AND jockey <> ''
          AND owner IS NOT NULL
          AND owner <> ''
          AND jockey = owner
        GROUP BY jockey, date, course, off
        ORDER BY date, course, off, raw_label
        """,
        connection,
    )

    same_race_trainer_owner = pd.read_sql_query(
        f"""
        SELECT
            trainer AS raw_label,
            date,
            course,
            off,
            COUNT(*) AS matching_runner_rows
        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}
          AND trainer IS NOT NULL
          AND trainer <> ''
          AND owner IS NOT NULL
          AND owner <> ''
          AND trainer = owner
        GROUP BY trainer, date, course, off
        ORDER BY date, course, off, raw_label
        """,
        connection,
    )

finally:
    connection.close()


same_race_overlap_summary = pd.DataFrame(
    [
        {
            "role_pair": "jockey_and_trainer",
            "provisional_races": len(same_race_jockey_trainer),
            "distinct_exact_labels": (
                same_race_jockey_trainer["raw_label"].nunique()
            ),
            "matching_runner_rows": int(
                same_race_jockey_trainer[
                    "matching_runner_rows"
                ].sum()
            ),
        },
        {
            "role_pair": "jockey_and_owner",
            "provisional_races": len(same_race_jockey_owner),
            "distinct_exact_labels": (
                same_race_jockey_owner["raw_label"].nunique()
            ),
            "matching_runner_rows": int(
                same_race_jockey_owner[
                    "matching_runner_rows"
                ].sum()
            ),
        },
        {
            "role_pair": "trainer_and_owner",
            "provisional_races": len(same_race_trainer_owner),
            "distinct_exact_labels": (
                same_race_trainer_owner["raw_label"].nunique()
            ),
            "matching_runner_rows": int(
                same_race_trainer_owner[
                    "matching_runner_rows"
                ].sum()
            ),
        },
    ]
)


# Reconcile the temporal classifications to the complete overlap sets.
assert (
    temporal_relationship_summary.loc[
        temporal_relationship_summary["role_pair"]
        == "jockey_and_trainer",
        "distinct_exact_labels",
    ].sum()
    == len(jockey_trainer_overlap)
)

assert (
    temporal_relationship_summary.loc[
        temporal_relationship_summary["role_pair"]
        == "jockey_and_owner",
        "distinct_exact_labels",
    ].sum()
    == len(jockey_owner_overlap)
)

assert (
    temporal_relationship_summary.loc[
        temporal_relationship_summary["role_pair"]
        == "trainer_and_owner",
        "distinct_exact_labels",
    ].sum()
    == len(trainer_owner_overlap)
)


print("Temporal relationships between exact cross-role labels")
display(temporal_relationship_summary)

print("Same-race exact-label overlap")
display(same_race_overlap_summary)

print("Jockey–trainer labels with non-overlapping date ranges")
display(
    jockey_trainer_temporal.loc[
        jockey_trainer_temporal["temporal_relationship"]
        != "date_ranges_overlap"
    ]
    .sort_values(
        ["jockey_runner_rows", "trainer_runner_rows"],
        ascending=[False, False],
    )
    .head(40)
    .reset_index(drop=True)
)

print("Jockey–trainer exact matches within one provisional race")
display(same_race_jockey_trainer.head(40))

print("Trainer–owner exact matches within one provisional race")
display(same_race_trainer_owner.head(40))

Temporal relationships between exact cross-role labels


,role_pair,temporal_relationship,distinct_exact_labels
0,jockey_and_owner,date_ranges_overlap,99
1,jockey_and_owner,jockey_before_owner,81
2,jockey_and_owner,owner_before_jockey,14
3,jockey_and_owner,same_single_date,4
4,jockey_and_trainer,date_ranges_overlap,94
5,jockey_and_trainer,jockey_before_trainer,113
6,jockey_and_trainer,same_single_date,3
7,jockey_and_trainer,trainer_before_jockey,16
8,trainer_and_owner,date_ranges_overlap,1573
9,trainer_and_owner,owner_before_trainer,38


Same-race exact-label overlap


,role_pair,provisional_races,distinct_exact_labels,matching_runner_rows
0,jockey_and_trainer,2227,70,2227
1,jockey_and_owner,193,54,193
2,trainer_and_owner,45861,1690,46810


Jockey–trainer labels with non-overlapping date ranges


,raw_label,jockey_runner_rows,jockey_first_observed_date,jockey_last_observed_date,trainer_runner_rows,trainer_first_observed_date,trainer_last_observed_date,temporal_relationship
0,Adam Kirby,5962,2015-01-02,2025-09-14,55,2026-03-11,2026-05-26,jockey_before_trainer
1,Tom Scudamore,4530,2015-01-01,2023-09-17,1,2026-05-15,2026-05-15,jockey_before_trainer
2,Nick Scholfield,3324,2015-01-01,2025-04-05,139,2025-11-06,2026-05-26,jockey_before_trainer
3,Gerald Mosse,2387,2015-01-25,2024-09-15,262,2024-10-15,2026-05-24,jockey_before_trainer
4,Phillip Makin,1967,2015-01-15,2018-08-25,1071,2019-03-27,2026-05-26,jockey_before_trainer
5,Douglas Whyte,1058,2015-01-25,2019-02-10,3042,2019-09-01,2026-05-27,jockey_before_trainer
6,Connor King,1052,2015-01-02,2023-04-21,26,2024-05-13,2026-04-29,jockey_before_trainer
7,Adam Nicol,832,2015-01-01,2020-03-15,448,2020-12-09,2026-05-24,jockey_before_trainer
8,Regis Schmidlin,792,2015-02-24,2021-12-21,157,2022-05-05,2026-05-17,jockey_before_trainer
9,Jack Foley,755,2015-09-06,2024-11-30,45,2025-08-13,2026-05-26,jockey_before_trainer


Jockey–trainer exact matches within one provisional race


,raw_label,date,course,off,matching_runner_rows
0,Ann Stokell,2015-01-02,Southwell (AW),12:20,1
1,Anthony Clement,2015-01-03,Deauville (FR),3:25,1
2,Brendan Powell,2015-01-03,Sandown,1:15,1
3,Ann Stokell,2015-01-04,Southwell (AW),1:40,1
4,Ann Stokell,2015-01-05,Wolverhampton (AW),3:20,1
5,Ann Stokell,2015-01-06,Southwell (AW),1:20,1
6,Ann Stokell,2015-01-08,Southwell (AW),1:20,1
7,Ann Stokell,2015-01-08,Southwell (AW),1:50,1
8,Brendan Powell,2015-01-09,Huntingdon,1:55,1
9,Ann Stokell,2015-01-12,Wolverhampton (AW),3:25,1


Trainer–owner exact matches within one provisional race


,raw_label,date,course,off,matching_runner_rows
0,Andrew J Martin,2015-01-01,Exeter,12:35,1
1,Andrew J Martin,2015-01-01,Exeter,1:45,1
2,Mrs Denise Foster,2015-01-01,Fairyhouse (IRE),12:25,2
3,Denis Hickey,2015-01-01,Fairyhouse (IRE),12:55,1
4,Edward Cawley,2015-01-01,Fairyhouse (IRE),12:55,1
5,J P Dempsey,2015-01-01,Fairyhouse (IRE),12:55,1
6,Edward Cawley,2015-01-01,Fairyhouse (IRE),2:00,1
7,John F Robinson,2015-01-01,Fairyhouse (IRE),2:00,1
8,Leonard Whitmore,2015-01-01,Fairyhouse (IRE),2:35,1
9,Edward Cawley,2015-01-01,Fairyhouse (IRE),3:10,1


## Stage 12 — Test exact jockey labels within one provisional race

The source records `jockey` at runner grain.

Within one correctly reconstructed race, one real jockey cannot ride more than one runner.

This creates a stronger source-internal constraint than ordinary name similarity or chronology:

> An exact jockey label should normally occur no more than once within one provisional race.

This stage identifies provisional races where the same exact populated jockey label appears on multiple runner rows.

A violation does not automatically prove that the jockey label identifies multiple people. It may instead arise from:

- duplicate source rows;
- an incorrectly reconstructed provisional race;
- source extraction errors;
- two distinct races sharing `date + course + off`;
- a wrongly assigned jockey;
- a same-name collision;
- another unresolved source anomaly.

Each candidate retains:

- provisional race identity;
- exact jockey label;
- number of matching runner rows;
- distinct horses;
- source row IDs;
- supplied race IDs.

No row is removed or corrected in this stage.

In [12]:
# Query exact jockey labels attached to more than one runner row within the
# same provisional race.
#
# The provisional race key remains date + course + off. This query does not
# assume that the reconstructed race is always correct; it identifies cases
# where the expected jockey cardinality requires inspection.
connection = sqlite3.connect(
    f"file:{SOURCE_DB_PATH}?mode=ro",
    uri=True,
)

try:
    duplicate_jockey_race_candidates = pd.read_sql_query(
        f"""
        SELECT
            date,
            course,
            off,
            jockey AS raw_jockey_label,
            COUNT(*) AS runner_rows,
            COUNT(DISTINCT horse) AS distinct_horse_labels,
            COUNT(DISTINCT race_id) AS distinct_supplied_race_ids,
            MIN(rowid) AS first_source_rowid,
            MAX(rowid) AS last_source_rowid
        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}
          AND jockey IS NOT NULL
          AND jockey <> ''
        GROUP BY
            date,
            course,
            off,
            jockey
        HAVING COUNT(*) > 1
        ORDER BY
            runner_rows DESC,
            date,
            course,
            off,
            raw_jockey_label
        """,
        connection,
    )

    # Retrieve every underlying source row for the finite candidate set.
    #
    # A temporary table is avoided so that the source connection remains
    # strictly read-only. The join repeats the candidate grouping inside a
    # common table expression.
    duplicate_jockey_candidate_rows = pd.read_sql_query(
        f"""
        WITH duplicate_jockey_groups AS (
            SELECT
                date,
                course,
                off,
                jockey
            FROM {SOURCE_TABLE}
            WHERE {DATA_ROW_PREDICATE}
              AND jockey IS NOT NULL
              AND jockey <> ''
            GROUP BY
                date,
                course,
                off,
                jockey
            HAVING COUNT(*) > 1
        )
        SELECT
            source.rowid AS source_rowid,
            source.race_id AS supplied_race_id,
            source.date,
            source.course,
            source.off,
            source.race_name,
            source.horse,
            source.jockey,
            source.trainer,
            source.owner,
            source.num,
            source.pos
        FROM {SOURCE_TABLE} AS source
        INNER JOIN duplicate_jockey_groups AS candidate
            ON source.date = candidate.date
           AND source.course = candidate.course
           AND source.off = candidate.off
           AND source.jockey = candidate.jockey
        WHERE source.{DATA_ROW_PREDICATE}
        ORDER BY
            source.date,
            source.course,
            source.off,
            source.jockey,
            source.rowid
        """,
        connection,
    )

finally:
    connection.close()


# Separate exact duplicated rows from groups containing genuinely different
# runner labels. Repeated source rows and multiple horses have different
# implications and must not be conflated.
duplicate_jockey_race_candidates[
    "candidate_structure"
] = duplicate_jockey_race_candidates.apply(
    lambda row: (
        "repeated_same_horse_label"
        if row["distinct_horse_labels"] == 1
        else "multiple_horse_labels"
    ),
    axis=1,
)


duplicate_jockey_candidate_summary = (
    duplicate_jockey_race_candidates.groupby(
        "candidate_structure",
        as_index=False,
    )
    .agg(
        provisional_race_jockey_groups=(
            "raw_jockey_label",
            "size",
        ),
        runner_rows=("runner_rows", "sum"),
        distinct_jockey_labels=(
            "raw_jockey_label",
            "nunique",
        ),
    )
    .sort_values("candidate_structure")
    .reset_index(drop=True)
)


# Reconcile the candidate-level and row-level outputs.
assert (
    duplicate_jockey_race_candidates["runner_rows"].sum()
    == len(duplicate_jockey_candidate_rows)
)

assert (
    duplicate_jockey_candidate_summary[
        "provisional_race_jockey_groups"
    ].sum()
    == len(duplicate_jockey_race_candidates)
)

print("Duplicate exact jockey labels within provisional races")
display(duplicate_jockey_candidate_summary)

print("Candidate groups")
display(duplicate_jockey_race_candidates)

print("Underlying source rows")
display(duplicate_jockey_candidate_rows)

Duplicate exact jockey labels within provisional races


,candidate_structure,provisional_race_jockey_groups,runner_rows,distinct_jockey_labels


Candidate groups


,date,course,off,raw_jockey_label,runner_rows,distinct_horse_labels,distinct_supplied_race_ids,first_source_rowid,last_source_rowid,candidate_structure


Underlying source rows


,source_rowid,supplied_race_id,date,course,off,race_name,horse,jockey,trainer,owner,num,pos


## Stage 13 — Test case-only label variants

The exact-jockey cardinality test found no provisional race where one populated exact jockey label was attached to more than one runner.

The next investigation concerns a narrower form of label variation:

> Do two or more distinct raw labels become identical when compared without letter case?

Examples of possible case-only variation might include:

- different capitalisation of initials;
- inconsistent capitalisation of surnames;
- company suffixes such as `Ltd` and `LTD`;
- source labels entered in title case and uppercase;
- jurisdiction-specific presentation differences.

This stage uses Unicode case-folding only as a comparison key.

It does not:

- replace the raw label;
- establish that case-only variants identify the same entity;
- remove titles, punctuation, spaces or numerals;
- transliterate characters;
- create permanent entity identifiers.

Every collision retains all exact raw labels, role-specific frequencies and observed date ranges.

A case-folded collision is only a candidate formatting relationship until its context has been inspected.

In [13]:
# Build a case-folded comparison key without altering the exact raw labels.
#
# Python's casefold method is used rather than lower because it is the more
# explicit Unicode-aware operation for caseless comparison. The source labels
# themselves remain unchanged in every output.
casefold_collision_frames = {}
casefold_collision_summary_rows = []

for field in CONNECTION_IDENTITY_FIELDS:
    labels = exact_label_profiles[field].copy()

    labels["casefold_key"] = labels["raw_label"].str.casefold()

    # Count how many distinct exact source labels share each comparison key.
    casefold_key_counts = (
        labels.groupby(
            "casefold_key",
            as_index=False,
        )
        .agg(
            distinct_raw_labels=("raw_label", "nunique"),
            runner_rows=("runner_rows", "sum"),
        )
    )

    collision_keys = set(
        casefold_key_counts.loc[
            casefold_key_counts["distinct_raw_labels"] > 1,
            "casefold_key",
        ]
    )

    # Preserve every exact raw label participating in a collision.
    collisions = (
        labels.loc[
            labels["casefold_key"].isin(collision_keys),
            [
                "casefold_key",
                "raw_label",
                "runner_rows",
                "first_observed_date",
                "last_observed_date",
            ],
        ]
        .sort_values(
            [
                "casefold_key",
                "runner_rows",
                "raw_label",
            ],
            ascending=[True, False, True],
        )
        .reset_index(drop=True)
    )

    casefold_collision_frames[field] = collisions

    casefold_collision_summary_rows.append(
        {
            "source_field": field,
            "collision_keys": len(collision_keys),
            "distinct_raw_labels_in_collisions": (
                collisions["raw_label"].nunique()
            ),
            "runner_rows_in_collisions": int(
                collisions["runner_rows"].sum()
            ),
        }
    )


casefold_collision_summary = pd.DataFrame(
    casefold_collision_summary_rows
)


# Reconcile each collision table to its summary.
for field in CONNECTION_IDENTITY_FIELDS:
    collisions = casefold_collision_frames[field]

    summary_row = casefold_collision_summary.loc[
        casefold_collision_summary["source_field"] == field
    ].iloc[0]

    assert (
        collisions["casefold_key"].nunique()
        == summary_row["collision_keys"]
    )

    assert (
        collisions["raw_label"].nunique()
        == summary_row["distinct_raw_labels_in_collisions"]
    )

    assert (
        int(collisions["runner_rows"].sum())
        == summary_row["runner_rows_in_collisions"]
    )


print("Case-folded exact-label collisions")
display(casefold_collision_summary)

for field in CONNECTION_IDENTITY_FIELDS:
    print(f"{field.title()} case-only collision labels")
    display(casefold_collision_frames[field])

Case-folded exact-label collisions


,source_field,collision_keys,distinct_raw_labels_in_collisions,runner_rows_in_collisions
0,jockey,1,2,14
1,trainer,2,4,1248
2,owner,0,0,0


Jockey case-only collision labels


,casefold_key,raw_label,runner_rows,first_observed_date,last_observed_date
0,esentur turganaaly uulu,Esentur Turganaaly Uulu,10,2025-10-26,2026-05-25
1,esentur turganaaly uulu,Esentur Turganaaly UUlu,4,2025-07-06,2025-08-03


Trainer case-only collision labels


,casefold_key,raw_label,runner_rows,first_observed_date,last_observed_date
0,d & p prodhomme,D & P ProdHomme,1233,2015-01-03,2026-04-26
1,d & p prodhomme,D & P Prodhomme,8,2026-05-05,2026-05-15
2,marcelo a degregorio,Marcelo A Degregorio,5,2025-04-05,2025-11-19
3,marcelo a degregorio,Marcelo A DeGregorio,2,2015-10-22,2024-11-19


Owner case-only collision labels


,casefold_key,raw_label,runner_rows,first_observed_date,last_observed_date


## Stage 14 — Inspect case-only collision context

The case-fold comparison found only three collision groups:

- one jockey group;
- two trainer groups;
- no owner groups.

The collision labels are:

- `Esentur Turganaaly Uulu`
- `Esentur Turganaaly UUlu`
- `D & P ProdHomme`
- `D & P Prodhomme`
- `Marcelo A Degregorio`
- `Marcelo A DeGregorio`

These differences may be formatting variants, but case-fold equality does not prove entity equivalence.

This stage inspects each exact label in source context.

For every collision label, it measures:

- runner rows;
- provisional races;
- distinct courses;
- distinct horses;
- first and last observed dates;
- most frequent courses;
- most frequent horses;
- whether two variants occur in the same provisional race;
- whether their observed date ranges overlap;
- whether their course and horse populations overlap.

The purpose is to determine whether the labels are consistent with a formatting variant or whether they require separate unresolved treatment.

No labels are merged or replaced in this stage.

In [14]:
# Collect the complete finite case-collision residue identified in Stage 13.
#
# The exact raw labels are retained. The case-fold key is used only to group
# labels that differ in letter case.
case_collision_labels = pd.concat(
    [
        collisions.assign(source_field=field)
        for field, collisions in casefold_collision_frames.items()
        if not collisions.empty
    ],
    ignore_index=True,
)

assert len(case_collision_labels) == 6
assert case_collision_labels["casefold_key"].nunique() == 3


# Build a SQL-safe list of the six exact labels.
#
# Query parameters are used rather than interpolating label values into SQL.
collision_label_values = case_collision_labels[
    "raw_label"
].tolist()

label_placeholders = ", ".join(
    ["?"] * len(collision_label_values)
)


connection = sqlite3.connect(
    f"file:{SOURCE_DB_PATH}?mode=ro",
    uri=True,
)

try:
    # Retrieve every source row in which one of the collision labels appears
    # in its relevant role.
    collision_source_rows = []

    for field in CONNECTION_IDENTITY_FIELDS:
        field_labels = (
            case_collision_labels.loc[
                case_collision_labels["source_field"] == field,
                "raw_label",
            ]
            .tolist()
        )

        if not field_labels:
            continue

        quoted_field = quote_identifier(field)
        field_placeholders = ", ".join(
            ["?"] * len(field_labels)
        )

        field_rows = pd.read_sql_query(
            f"""
            SELECT
                rowid AS source_rowid,
                race_id AS supplied_race_id,
                date,
                course,
                off,
                race_name,
                horse,
                jockey,
                trainer,
                owner,
                {quoted_field} AS raw_label
            FROM {SOURCE_TABLE}
            WHERE {DATA_ROW_PREDICATE}
              AND {quoted_field} IN ({field_placeholders})
            ORDER BY
                date,
                course,
                off,
                rowid
            """,
            connection,
            params=field_labels,
        )

        field_rows["source_field"] = field

        collision_source_rows.append(field_rows)

finally:
    connection.close()


collision_source_rows = pd.concat(
    collision_source_rows,
    ignore_index=True,
)


# Summarise each exact label independently.
#
# Distinct provisional races use the established date + course + off key.
collision_context_summary = (
    collision_source_rows.assign(
        provisional_race_key=lambda frame: (
            frame["date"].astype(str)
            + " | "
            + frame["course"].astype(str)
            + " | "
            + frame["off"].astype(str)
        )
    )
    .groupby(
        [
            "source_field",
            "raw_label",
        ],
        as_index=False,
    )
    .agg(
        runner_rows=("source_rowid", "size"),
        provisional_races=("provisional_race_key", "nunique"),
        distinct_courses=("course", "nunique"),
        distinct_horses=("horse", "nunique"),
        first_observed_date=("date", "min"),
        last_observed_date=("date", "max"),
    )
    .sort_values(
        [
            "source_field",
            "runner_rows",
            "raw_label",
        ],
        ascending=[True, False, True],
    )
    .reset_index(drop=True)
)


# Build compact course and horse frequency tables for each exact label.
collision_course_profile = (
    collision_source_rows.groupby(
        [
            "source_field",
            "raw_label",
            "course",
        ],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "runner_rows"})
    .sort_values(
        [
            "source_field",
            "raw_label",
            "runner_rows",
            "course",
        ],
        ascending=[True, True, False, True],
    )
    .groupby(
        [
            "source_field",
            "raw_label",
        ],
        as_index=False,
        group_keys=False,
    )
    .head(10)
    .reset_index(drop=True)
)

collision_horse_profile = (
    collision_source_rows.groupby(
        [
            "source_field",
            "raw_label",
            "horse",
        ],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "runner_rows"})
    .sort_values(
        [
            "source_field",
            "raw_label",
            "runner_rows",
            "horse",
        ],
        ascending=[True, True, False, True],
    )
    .groupby(
        [
            "source_field",
            "raw_label",
        ],
        as_index=False,
        group_keys=False,
    )
    .head(10)
    .reset_index(drop=True)
)


# Compare the two exact variants within each case-fold group.
#
# The comparison records overlap in dates, courses, horses and provisional
# races. These are source-internal consistency signals only.
comparison_rows = []

for (
    source_field,
    casefold_key,
), group in case_collision_labels.groupby(
    [
        "source_field",
        "casefold_key",
    ]
):
    raw_labels = group["raw_label"].tolist()

    assert len(raw_labels) == 2

    first_label, second_label = raw_labels

    first_rows = collision_source_rows.loc[
        (
            collision_source_rows["source_field"]
            == source_field
        )
        & (
            collision_source_rows["raw_label"]
            == first_label
        )
    ].copy()

    second_rows = collision_source_rows.loc[
        (
            collision_source_rows["source_field"]
            == source_field
        )
        & (
            collision_source_rows["raw_label"]
            == second_label
        )
    ].copy()

    first_dates = set(first_rows["date"])
    second_dates = set(second_rows["date"])

    first_courses = set(first_rows["course"])
    second_courses = set(second_rows["course"])

    first_horses = set(first_rows["horse"])
    second_horses = set(second_rows["horse"])

    first_races = set(
        zip(
            first_rows["date"],
            first_rows["course"],
            first_rows["off"],
        )
    )

    second_races = set(
        zip(
            second_rows["date"],
            second_rows["course"],
            second_rows["off"],
        )
    )

    comparison_rows.append(
        {
            "source_field": source_field,
            "casefold_key": casefold_key,
            "first_raw_label": first_label,
            "second_raw_label": second_label,
            "shared_dates": len(
                first_dates & second_dates
            ),
            "shared_courses": len(
                first_courses & second_courses
            ),
            "shared_horses": len(
                first_horses & second_horses
            ),
            "shared_provisional_races": len(
                first_races & second_races
            ),
        }
    )


case_collision_comparison = pd.DataFrame(
    comparison_rows
)


# Reconcile the source-row population to the Stage 13 collision summary.
assert len(collision_source_rows) == int(
    casefold_collision_summary[
        "runner_rows_in_collisions"
    ].sum()
)

assert collision_context_summary["runner_rows"].sum() == len(
    collision_source_rows
)

assert len(case_collision_comparison) == 3


print("Case-only collision context summary")
display(collision_context_summary)

print("Case-only variant overlap comparison")
display(case_collision_comparison)

print("Most frequent courses by exact collision label")
display(collision_course_profile)

print("Most frequent horses by exact collision label")
display(collision_horse_profile)

print("Complete source rows for the six collision labels")
display(collision_source_rows)

Case-only collision context summary


,source_field,raw_label,runner_rows,provisional_races,distinct_courses,distinct_horses,first_observed_date,last_observed_date
0,jockey,Esentur Turganaaly Uulu,10,10,6,9,2025-10-26,2026-05-25
1,jockey,Esentur Turganaaly UUlu,4,4,2,4,2025-07-06,2025-08-03
2,trainer,D & P ProdHomme,1233,1128,21,134,2015-01-03,2026-04-26
3,trainer,D & P Prodhomme,8,8,2,8,2026-05-05,2026-05-15
4,trainer,Marcelo A Degregorio,5,5,3,1,2025-04-05,2025-11-19
5,trainer,Marcelo A DeGregorio,2,2,1,2,2015-10-22,2024-11-19


Case-only variant overlap comparison


,source_field,casefold_key,first_raw_label,second_raw_label,shared_dates,shared_courses,shared_horses,shared_provisional_races
0,jockey,esentur turganaaly uulu,Esentur Turganaaly Uulu,Esentur Turganaaly UUlu,0,0,0,0
1,trainer,d & p prodhomme,D & P ProdHomme,D & P Prodhomme,0,2,7,0
2,trainer,marcelo a degregorio,Marcelo A Degregorio,Marcelo A DeGregorio,0,0,1,0


Most frequent courses by exact collision label


,source_field,raw_label,course,runner_rows
0,jockey,Esentur Turganaaly UUlu,Hamburg (GER),3
1,jockey,Esentur Turganaaly UUlu,Dusseldorf (GER),1
2,jockey,Esentur Turganaaly Uulu,Munich,4
3,jockey,Esentur Turganaaly Uulu,St Moritz,2
4,jockey,Esentur Turganaaly Uulu,Chantilly,1
5,jockey,Esentur Turganaaly Uulu,Cologne,1
6,jockey,Esentur Turganaaly Uulu,Hanover,1
7,jockey,Esentur Turganaaly Uulu,San Siro,1
8,trainer,D & P ProdHomme,Saint-Cloud (FR),322
9,trainer,D & P ProdHomme,Deauville (FR),301


Most frequent horses by exact collision label


,source_field,raw_label,horse,runner_rows
0,jockey,Esentur Turganaaly UUlu,Mabel (GER),1
1,jockey,Esentur Turganaaly UUlu,No Stopping Her (IRE),1
2,jockey,Esentur Turganaaly UUlu,Tamino (GER),1
3,jockey,Esentur Turganaaly UUlu,Zakaria (GER),1
4,jockey,Esentur Turganaaly Uulu,Principe (GER),2
5,jockey,Esentur Turganaaly Uulu,Abachi (GER),1
6,jockey,Esentur Turganaaly Uulu,Avola (GER),1
7,jockey,Esentur Turganaaly Uulu,Beau Gars (FR),1
8,jockey,Esentur Turganaaly Uulu,Ferrari Twister (CRO),1
9,jockey,Esentur Turganaaly Uulu,Love Me Tender (GER),1


Complete source rows for the six collision labels


,source_rowid,supplied_race_id,date,course,off,race_name,horse,jockey,trainer,owner,raw_label,source_field
0,1702855,899291,2025-07-06,Hamburg (GER),11:50,Sparkasse Holstein-Rennen - HKJC World Pool Ha...,No Stopping Her (IRE),Esentur Turganaaly UUlu,Frank Fuhrmann,Stall Royal Blue,Esentur Turganaaly UUlu,jockey
1,1702916,899296,2025-07-06,Hamburg (GER),1:03,Rudolf August Oetker-Gedachtnisrennen-BBAG Auk...,Mabel (GER),Esentur Turganaaly UUlu,R Dzubasz,Yvonne Militzer,Esentur Turganaaly UUlu,jockey
2,1702872,899292,2025-07-06,Hamburg (GER),1:54,HKJC World Pool Handicap (Handicap) (3yo+) (Turf),Tamino (GER),Esentur Turganaaly UUlu,R Dzubasz,Rennstall Labinsky,Esentur Turganaaly UUlu,jockey
3,1714773,900948,2025-08-03,Dusseldorf (GER),1:53,Schauma-Rennen - HKJC World Pool Handicap (3yo...,Zakaria (GER),Esentur Turganaaly UUlu,A Kleinkorres,Stefan Schumacher,Esentur Turganaaly UUlu,jockey
4,1756136,906918,2025-10-26,Hanover,13:05,Shadwell Stallions - Herbst-Stutenpreis (3yo+ ...,Avola (GER),Esentur Turganaaly Uulu,P Schiergen,Stall Nizza,Esentur Turganaaly Uulu,jockey
...,...,...,...,...,...,...,...,...,...,...,...,...
1257,1842391,920452,2026-05-10,Longchamp,14:15,Super Handicap du Printemps (Turf),Big Log (FR),Antoine Hamelin,D & P Prodhomme,Bryan Lynam,D & P Prodhomme,trainer
1258,1842344,920453,2026-05-10,Longchamp,16:00,Prix du Petit Chatelet (Handicap) (Turf),Paolino (FR),Rosario Mangione,D & P Prodhomme,Anton Krauliger,D & P Prodhomme,trainer
1259,1844190,920667,2026-05-14,Longchamp,19:15,Prix du Palais des Glaces (Handicap) (Fillies ...,Romance Marine (FR),Leonard Plommee,D & P Prodhomme,Bernard Giraudon,D & P Prodhomme,trainer
1260,1844229,920668,2026-05-14,Longchamp,19:50,Prix du Passage des Princes (Handicap) (Turf),Pergolor (FR),Eddy Hardouin,D & P Prodhomme,Bernard Giraudon,D & P Prodhomme,trainer


## Stage 15 — Test whitespace-only comparison variants

The case-only investigation found three small collision groups and no owner case-fold collisions.

The earlier physical profile also found:

- one owner label with leading whitespace;
- 645 owner labels containing repeated internal spaces;
- no jockey or trainer whitespace anomalies;
- no whitespace-collapsed owner form already present as an exact raw label.

The next bounded question is:

> Do distinct raw labels become identical when only outer whitespace is trimmed and repeated internal spaces are collapsed to one space?

This stage creates a whitespace comparison key by:

1. removing leading and trailing ASCII spaces;
2. replacing each run of internal ASCII spaces with one space;
3. preserving case, punctuation, numerals and word order.

The comparison key is not a canonical label.

A collision would identify a possible formatting relationship only. It would not prove:

- shared ownership identity;
- shared ownership-account identity;
- partnership membership;
- safe decomposition;
- that repeated spaces are meaningless;
- that a source separator may be discarded permanently.

All exact raw labels and source frequencies remain preserved.

In [15]:
def build_whitespace_comparison_key(value: str) -> str:
    """
    Build a reversible comparison key from ASCII-space formatting only.

    The exact raw label remains unchanged. This function trims outer spaces
    and collapses each internal run of ASCII spaces to one space. It does not
    alter case, punctuation, numerals or word order.
    """
    return re.sub(r" +", " ", value.strip())


whitespace_collision_frames = {}
whitespace_collision_summary_rows = []

for field in CONNECTION_IDENTITY_FIELDS:
    labels = exact_label_profiles[field].copy()

    labels["whitespace_key"] = labels["raw_label"].map(
        build_whitespace_comparison_key
    )

    # Count how many distinct exact source labels share each whitespace-only
    # comparison key.
    whitespace_key_counts = (
        labels.groupby(
            "whitespace_key",
            as_index=False,
        )
        .agg(
            distinct_raw_labels=("raw_label", "nunique"),
            runner_rows=("runner_rows", "sum"),
        )
    )

    collision_keys = set(
        whitespace_key_counts.loc[
            whitespace_key_counts["distinct_raw_labels"] > 1,
            "whitespace_key",
        ]
    )

    # Preserve every exact raw label participating in a collision.
    collisions = (
        labels.loc[
            labels["whitespace_key"].isin(collision_keys),
            [
                "whitespace_key",
                "raw_label",
                "runner_rows",
                "first_observed_date",
                "last_observed_date",
            ],
        ]
        .sort_values(
            [
                "whitespace_key",
                "runner_rows",
                "raw_label",
            ],
            ascending=[True, False, True],
        )
        .reset_index(drop=True)
    )

    whitespace_collision_frames[field] = collisions

    whitespace_collision_summary_rows.append(
        {
            "source_field": field,
            "collision_keys": len(collision_keys),
            "distinct_raw_labels_in_collisions": (
                collisions["raw_label"].nunique()
            ),
            "runner_rows_in_collisions": int(
                collisions["runner_rows"].sum()
            ),
        }
    )


whitespace_collision_summary = pd.DataFrame(
    whitespace_collision_summary_rows
)


# Reconcile every collision frame to its summary row.
for field in CONNECTION_IDENTITY_FIELDS:
    collisions = whitespace_collision_frames[field]

    summary_row = whitespace_collision_summary.loc[
        whitespace_collision_summary["source_field"] == field
    ].iloc[0]

    assert (
        collisions["whitespace_key"].nunique()
        == summary_row["collision_keys"]
    )

    assert (
        collisions["raw_label"].nunique()
        == summary_row["distinct_raw_labels_in_collisions"]
    )

    assert (
        int(collisions["runner_rows"].sum())
        == summary_row["runner_rows_in_collisions"]
    )


# Confirm that the one known leading-space label changes under the comparison
# key even if it does not collide with another exact source label.
leading_space_raw_label = leading_whitespace_labels[
    "raw_owner_label"
].iloc[0]

leading_space_comparison = pd.DataFrame(
    [
        {
            "raw_owner_label": leading_space_raw_label,
            "whitespace_key": build_whitespace_comparison_key(
                leading_space_raw_label
            ),
            "label_changed": (
                leading_space_raw_label
                != build_whitespace_comparison_key(
                    leading_space_raw_label
                )
            ),
        }
    ]
)

assert leading_space_comparison["label_changed"].all()


print("Whitespace-only exact-label collisions")
display(whitespace_collision_summary)

for field in CONNECTION_IDENTITY_FIELDS:
    print(f"{field.title()} whitespace collision labels")
    display(whitespace_collision_frames[field])

print("Known leading-space owner comparison")
display(leading_space_comparison)

Whitespace-only exact-label collisions


,source_field,collision_keys,distinct_raw_labels_in_collisions,runner_rows_in_collisions
0,jockey,0,0,0
1,trainer,0,0,0
2,owner,0,0,0


Jockey whitespace collision labels


,whitespace_key,raw_label,runner_rows,first_observed_date,last_observed_date


Trainer whitespace collision labels


,whitespace_key,raw_label,runner_rows,first_observed_date,last_observed_date


Owner whitespace collision labels


,whitespace_key,raw_label,runner_rows,first_observed_date,last_observed_date


Known leading-space owner comparison


,raw_owner_label,whitespace_key,label_changed
0,Bradley Thoroughbreds Llc O Ghrghar C Y Lerner,Bradley Thoroughbreds Llc O Ghrghar C Y Lerner,True


## Stage 16 — Exact-label recurrence and observed lifespan

The whitespace-only comparison produced no collisions.

This supports retaining a reversible whitespace comparison value if needed later, while preserving every raw source label unchanged.

The next bounded question is:

> How frequently and over what observed periods do exact `jockey`, `trainer` and `owner` labels recur?

This stage measures, for each field:

- labels observed once;
- labels observed on multiple runner rows;
- labels observed on one date only;
- labels observed across multiple dates;
- labels observed within one calendar year;
- labels observed across multiple calendar years;
- observed lifespan in days;
- maximum gap between consecutive observed dates.

These measures describe source recurrence only.

They do not prove:

- that a frequently repeated label identifies one entity;
- that a long observed span belongs to one person or organisation;
- that a long gap indicates retirement, inactivity or succession;
- that a short-lived label is erroneous;
- that two separated periods belong to different entities;
- that chronology alone justifies merging or splitting labels.

The maximum-gap calculation uses distinct observed dates, so multiple runner rows on one date do not create artificial zero-day intervals.

In [16]:
# Build one temporal recurrence table per source field.
#
# The SQL works from exact raw labels and distinct observed dates. It does not
# trim, case-fold, split or otherwise alter any source label.
temporal_label_profiles = {}

connection = sqlite3.connect(
    f"file:{SOURCE_DB_PATH}?mode=ro",
    uri=True,
)

try:
    for field in CONNECTION_IDENTITY_FIELDS:
        quoted_field = quote_identifier(field)

        temporal_label_profiles[field] = pd.read_sql_query(
            f"""
            WITH populated_rows AS (
                SELECT
                    {quoted_field} AS raw_label,
                    date
                FROM {SOURCE_TABLE}
                WHERE {DATA_ROW_PREDICATE}
                  AND {quoted_field} IS NOT NULL
                  AND {quoted_field} <> ''
            ),
            label_summary AS (
                SELECT
                    raw_label,
                    COUNT(*) AS runner_rows,
                    COUNT(DISTINCT date) AS observed_dates,
                    MIN(date) AS first_observed_date,
                    MAX(date) AS last_observed_date,
                    COUNT(
                        DISTINCT SUBSTR(CAST(date AS TEXT), 1, 4)
                    ) AS observed_calendar_years
                FROM populated_rows
                GROUP BY raw_label
            ),
            distinct_label_dates AS (
                SELECT DISTINCT
                    raw_label,
                    date
                FROM populated_rows
            ),
            dated_with_previous AS (
                SELECT
                    raw_label,
                    date,
                    LAG(date) OVER (
                        PARTITION BY raw_label
                        ORDER BY date
                    ) AS previous_observed_date
                FROM distinct_label_dates
            ),
            maximum_gaps AS (
                SELECT
                    raw_label,
                    MAX(
                        CAST(
                            JULIANDAY(date)
                            - JULIANDAY(previous_observed_date)
                            AS INTEGER
                        )
                    ) AS maximum_gap_days
                FROM dated_with_previous
                WHERE previous_observed_date IS NOT NULL
                GROUP BY raw_label
            )
            SELECT
                summary.raw_label,
                summary.runner_rows,
                summary.observed_dates,
                summary.observed_calendar_years,
                summary.first_observed_date,
                summary.last_observed_date,
                CAST(
                    JULIANDAY(summary.last_observed_date)
                    - JULIANDAY(summary.first_observed_date)
                    AS INTEGER
                ) AS observed_lifespan_days,
                COALESCE(gaps.maximum_gap_days, 0)
                    AS maximum_gap_days
            FROM label_summary AS summary
            LEFT JOIN maximum_gaps AS gaps
                ON summary.raw_label = gaps.raw_label
            ORDER BY
                summary.runner_rows DESC,
                summary.raw_label
            """,
            connection,
        )

finally:
    connection.close()


# Reconcile each temporal table to the exact-label and populated-row counts
# already established in Stage 3.
for field in CONNECTION_IDENTITY_FIELDS:
    temporal_profile = temporal_label_profiles[field]

    expected_distinct_labels = int(
        raw_field_profile.loc[
            raw_field_profile["source_field"] == field,
            "distinct_populated_labels",
        ].iloc[0]
    )

    expected_populated_rows = int(
        raw_field_profile.loc[
            raw_field_profile["source_field"] == field,
            "populated_rows",
        ].iloc[0]
    )

    assert len(temporal_profile) == expected_distinct_labels

    assert int(
        temporal_profile["runner_rows"].sum()
    ) == expected_populated_rows

    assert (
        temporal_profile["observed_dates"] >= 1
    ).all()

    assert (
        temporal_profile["observed_calendar_years"] >= 1
    ).all()

    assert (
        temporal_profile["observed_lifespan_days"] >= 0
    ).all()

    assert (
        temporal_profile["maximum_gap_days"] >= 0
    ).all()


# Convert each exact-label table into mutually interpretable recurrence
# measures. These measures overlap by design; for example, a label can have
# multiple runner rows but still be observed on only one date.
recurrence_summary_rows = []

for field in CONNECTION_IDENTITY_FIELDS:
    temporal_profile = temporal_label_profiles[field]

    recurrence_summary_rows.append(
        {
            "source_field": field,
            "distinct_exact_labels": len(temporal_profile),
            "one_runner_row_labels": int(
                temporal_profile["runner_rows"].eq(1).sum()
            ),
            "multiple_runner_row_labels": int(
                temporal_profile["runner_rows"].gt(1).sum()
            ),
            "one_observed_date_labels": int(
                temporal_profile["observed_dates"].eq(1).sum()
            ),
            "multiple_observed_date_labels": int(
                temporal_profile["observed_dates"].gt(1).sum()
            ),
            "one_calendar_year_labels": int(
                temporal_profile[
                    "observed_calendar_years"
                ].eq(1).sum()
            ),
            "multiple_calendar_year_labels": int(
                temporal_profile[
                    "observed_calendar_years"
                ].gt(1).sum()
            ),
            "median_observed_lifespan_days": float(
                temporal_profile[
                    "observed_lifespan_days"
                ].median()
            ),
            "maximum_observed_lifespan_days": int(
                temporal_profile[
                    "observed_lifespan_days"
                ].max()
            ),
            "median_maximum_gap_days": float(
                temporal_profile[
                    "maximum_gap_days"
                ].median()
            ),
            "maximum_gap_days": int(
                temporal_profile[
                    "maximum_gap_days"
                ].max()
            ),
        }
    )


exact_label_recurrence_summary = pd.DataFrame(
    recurrence_summary_rows
)


# Partition checks for the non-overlapping recurrence pairs.
assert (
    exact_label_recurrence_summary[
        "one_runner_row_labels"
    ]
    + exact_label_recurrence_summary[
        "multiple_runner_row_labels"
    ]
).eq(
    exact_label_recurrence_summary[
        "distinct_exact_labels"
    ]
).all()

assert (
    exact_label_recurrence_summary[
        "one_observed_date_labels"
    ]
    + exact_label_recurrence_summary[
        "multiple_observed_date_labels"
    ]
).eq(
    exact_label_recurrence_summary[
        "distinct_exact_labels"
    ]
).all()

assert (
    exact_label_recurrence_summary[
        "one_calendar_year_labels"
    ]
    + exact_label_recurrence_summary[
        "multiple_calendar_year_labels"
    ]
).eq(
    exact_label_recurrence_summary[
        "distinct_exact_labels"
    ]
).all()


print("Exact-label recurrence and observed lifespan")
display(exact_label_recurrence_summary)

for field in CONNECTION_IDENTITY_FIELDS:
    print(
        f"{field.title()} labels with the longest gaps "
        "between observed dates"
    )

    display(
        temporal_label_profiles[field]
        .sort_values(
            [
                "maximum_gap_days",
                "runner_rows",
                "raw_label",
            ],
            ascending=[False, False, True],
        )
        .head(30)
        .reset_index(drop=True)
    )

Exact-label recurrence and observed lifespan


,source_field,distinct_exact_labels,one_runner_row_labels,multiple_runner_row_labels,one_observed_date_labels,multiple_observed_date_labels,one_calendar_year_labels,multiple_calendar_year_labels,median_observed_lifespan_days,maximum_observed_lifespan_days,median_maximum_gap_days,maximum_gap_days
0,jockey,7917,1331,6586,1385,6532,2256,5661,890.0,4164,211.0,3867
1,trainer,10708,2273,8435,2390,8318,3509,7199,897.0,4164,190.0,4073
2,owner,98234,20814,77420,20928,77306,40628,57606,312.0,4164,140.0,4046


Jockey labels with the longest gaps between observed dates


,raw_label,runner_rows,observed_dates,observed_calendar_years,first_observed_date,last_observed_date,observed_lifespan_days,maximum_gap_days
0,Israel Hernandez,2,2,2,2015-02-18,2025-09-20,3867,3867
1,Mark Lawson,37,16,3,2015-04-06,2026-02-19,3972,3771
2,Chris R Rosier,5,2,2,2015-02-08,2025-01-11,3625,3625
3,Xavier Bergeron,54,36,3,2015-02-22,2026-05-24,4109,3623
4,E Alvares,3,3,3,2015-10-10,2026-05-03,3858,3619
5,Gary Bartley,90,53,2,2015-01-08,2025-09-14,3902,3599
6,Pierre Boudvillain,7,7,2,2016-01-04,2026-03-21,3729,3530
7,Beany Panya,2,2,2,2015-04-16,2024-11-23,3509,3509
8,Ben Kennedy,5,5,3,2015-05-16,2026-05-04,4006,3439
9,Taichi Nishimura,5,5,3,2016-01-11,2026-02-22,3695,3394


Trainer labels with the longest gaps between observed dates


,raw_label,runner_rows,observed_dates,observed_calendar_years,first_observed_date,last_observed_date,observed_lifespan_days,maximum_gap_days
0,Paul Patrick Moloney,3,3,2,2015-02-10,2026-04-20,4087,4073
1,Kym Davison,2,2,2,2015-03-27,2026-04-04,4026,4026
2,Luis C Lopez,2,2,2,2015-05-01,2026-05-01,4018,4018
3,Diane Poidevin Laine,3,2,2,2015-03-21,2026-03-13,4010,4010
4,Lucy Longmire,3,3,2,2015-05-08,2026-04-18,3998,3977
5,Wade Rarick,2,2,2,2015-02-18,2025-09-29,3876,3876
6,Lindsay Gough,2,2,2,2015-05-16,2025-07-05,3703,3703
7,John Dann,2,2,2,2015-05-23,2025-05-24,3654,3654
8,Gabriel H Degregorio,2,2,2,2016-05-01,2026-05-01,3652,3652
9,Alberto M Lopez,4,4,2,2015-10-12,2025-12-13,3715,3617


Owner labels with the longest gaps between observed dates


,raw_label,runner_rows,observed_dates,observed_calendar_years,first_observed_date,last_observed_date,observed_lifespan_days,maximum_gap_days
0,D W Stutt,2,2,2,2015-03-07,2026-04-04,4046,4046
1,Foxtrot Racing Management Ltd,2,2,2,2015-04-13,2026-04-19,4024,4024
2,A C Cook Partner,3,3,2,2015-06-03,2026-05-22,4006,3993
3,Mrs N Hodge,6,6,2,2015-02-16,2026-04-17,4078,3985
4,Fateel Brothers Syndicate,6,5,3,2015-02-27,2026-03-27,4046,3955
5,Pump Technology Limited,3,3,2,2015-06-25,2026-05-23,3985,3949
6,Dermot Kelly,4,4,2,2015-01-12,2026-05-13,4139,3929
7,Let It Ride Equine Holdings Iii Llc,3,3,2,2015-07-19,2026-05-02,3940,3891
8,Yuka Mitsuoka,2,2,2,2015-07-08,2026-03-01,3889,3889
9,Scea Haras Dorfausse,2,2,2,2015-09-23,2026-05-05,3877,3877


## Stage 16 — Exact-label recurrence and observed lifespan

The whitespace-only comparison produced no collisions.

This supports retaining a reversible whitespace comparison value if needed later, while preserving every raw source label unchanged.

The next bounded question is:

> How frequently and over what observed periods do exact `jockey`, `trainer` and `owner` labels recur?

This stage measures, for each field:

- labels observed once;
- labels observed on multiple runner rows;
- labels observed on one date only;
- labels observed across multiple dates;
- labels observed within one calendar year;
- labels observed across multiple calendar years;
- observed lifespan in days;
- maximum gap between consecutive observed dates.

These measures describe source recurrence only.

They do not prove:

- that a frequently repeated label identifies one entity;
- that a long observed span belongs to one person or organisation;
- that a long gap indicates retirement, inactivity or succession;
- that a short-lived label is erroneous;
- that two separated periods belong to different entities;
- that chronology alone justifies merging or splitting labels.

The maximum-gap calculation uses distinct observed dates, so multiple runner rows on one date do not create artificial zero-day intervals.

In [17]:
# Build one temporal recurrence table per source field.
#
# The SQL works from exact raw labels and distinct observed dates. It does not
# trim, case-fold, split or otherwise alter any source label.
temporal_label_profiles = {}

connection = sqlite3.connect(
    f"file:{SOURCE_DB_PATH}?mode=ro",
    uri=True,
)

try:
    for field in CONNECTION_IDENTITY_FIELDS:
        quoted_field = quote_identifier(field)

        temporal_label_profiles[field] = pd.read_sql_query(
            f"""
            WITH populated_rows AS (
                SELECT
                    {quoted_field} AS raw_label,
                    date
                FROM {SOURCE_TABLE}
                WHERE {DATA_ROW_PREDICATE}
                  AND {quoted_field} IS NOT NULL
                  AND {quoted_field} <> ''
            ),
            label_summary AS (
                SELECT
                    raw_label,
                    COUNT(*) AS runner_rows,
                    COUNT(DISTINCT date) AS observed_dates,
                    MIN(date) AS first_observed_date,
                    MAX(date) AS last_observed_date,
                    COUNT(
                        DISTINCT SUBSTR(CAST(date AS TEXT), 1, 4)
                    ) AS observed_calendar_years
                FROM populated_rows
                GROUP BY raw_label
            ),
            distinct_label_dates AS (
                SELECT DISTINCT
                    raw_label,
                    date
                FROM populated_rows
            ),
            dated_with_previous AS (
                SELECT
                    raw_label,
                    date,
                    LAG(date) OVER (
                        PARTITION BY raw_label
                        ORDER BY date
                    ) AS previous_observed_date
                FROM distinct_label_dates
            ),
            maximum_gaps AS (
                SELECT
                    raw_label,
                    MAX(
                        CAST(
                            JULIANDAY(date)
                            - JULIANDAY(previous_observed_date)
                            AS INTEGER
                        )
                    ) AS maximum_gap_days
                FROM dated_with_previous
                WHERE previous_observed_date IS NOT NULL
                GROUP BY raw_label
            )
            SELECT
                summary.raw_label,
                summary.runner_rows,
                summary.observed_dates,
                summary.observed_calendar_years,
                summary.first_observed_date,
                summary.last_observed_date,
                CAST(
                    JULIANDAY(summary.last_observed_date)
                    - JULIANDAY(summary.first_observed_date)
                    AS INTEGER
                ) AS observed_lifespan_days,
                COALESCE(gaps.maximum_gap_days, 0)
                    AS maximum_gap_days
            FROM label_summary AS summary
            LEFT JOIN maximum_gaps AS gaps
                ON summary.raw_label = gaps.raw_label
            ORDER BY
                summary.runner_rows DESC,
                summary.raw_label
            """,
            connection,
        )

finally:
    connection.close()


# Reconcile each temporal table to the exact-label and populated-row counts
# already established in Stage 3.
for field in CONNECTION_IDENTITY_FIELDS:
    temporal_profile = temporal_label_profiles[field]

    expected_distinct_labels = int(
        raw_field_profile.loc[
            raw_field_profile["source_field"] == field,
            "distinct_populated_labels",
        ].iloc[0]
    )

    expected_populated_rows = int(
        raw_field_profile.loc[
            raw_field_profile["source_field"] == field,
            "populated_rows",
        ].iloc[0]
    )

    assert len(temporal_profile) == expected_distinct_labels

    assert int(
        temporal_profile["runner_rows"].sum()
    ) == expected_populated_rows

    assert (
        temporal_profile["observed_dates"] >= 1
    ).all()

    assert (
        temporal_profile["observed_calendar_years"] >= 1
    ).all()

    assert (
        temporal_profile["observed_lifespan_days"] >= 0
    ).all()

    assert (
        temporal_profile["maximum_gap_days"] >= 0
    ).all()


# Convert each exact-label table into mutually interpretable recurrence
# measures. These measures overlap by design; for example, a label can have
# multiple runner rows but still be observed on only one date.
recurrence_summary_rows = []

for field in CONNECTION_IDENTITY_FIELDS:
    temporal_profile = temporal_label_profiles[field]

    recurrence_summary_rows.append(
        {
            "source_field": field,
            "distinct_exact_labels": len(temporal_profile),
            "one_runner_row_labels": int(
                temporal_profile["runner_rows"].eq(1).sum()
            ),
            "multiple_runner_row_labels": int(
                temporal_profile["runner_rows"].gt(1).sum()
            ),
            "one_observed_date_labels": int(
                temporal_profile["observed_dates"].eq(1).sum()
            ),
            "multiple_observed_date_labels": int(
                temporal_profile["observed_dates"].gt(1).sum()
            ),
            "one_calendar_year_labels": int(
                temporal_profile[
                    "observed_calendar_years"
                ].eq(1).sum()
            ),
            "multiple_calendar_year_labels": int(
                temporal_profile[
                    "observed_calendar_years"
                ].gt(1).sum()
            ),
            "median_observed_lifespan_days": float(
                temporal_profile[
                    "observed_lifespan_days"
                ].median()
            ),
            "maximum_observed_lifespan_days": int(
                temporal_profile[
                    "observed_lifespan_days"
                ].max()
            ),
            "median_maximum_gap_days": float(
                temporal_profile[
                    "maximum_gap_days"
                ].median()
            ),
            "maximum_gap_days": int(
                temporal_profile[
                    "maximum_gap_days"
                ].max()
            ),
        }
    )


exact_label_recurrence_summary = pd.DataFrame(
    recurrence_summary_rows
)


# Partition checks for the non-overlapping recurrence pairs.
assert (
    exact_label_recurrence_summary[
        "one_runner_row_labels"
    ]
    + exact_label_recurrence_summary[
        "multiple_runner_row_labels"
    ]
).eq(
    exact_label_recurrence_summary[
        "distinct_exact_labels"
    ]
).all()

assert (
    exact_label_recurrence_summary[
        "one_observed_date_labels"
    ]
    + exact_label_recurrence_summary[
        "multiple_observed_date_labels"
    ]
).eq(
    exact_label_recurrence_summary[
        "distinct_exact_labels"
    ]
).all()

assert (
    exact_label_recurrence_summary[
        "one_calendar_year_labels"
    ]
    + exact_label_recurrence_summary[
        "multiple_calendar_year_labels"
    ]
).eq(
    exact_label_recurrence_summary[
        "distinct_exact_labels"
    ]
).all()


print("Exact-label recurrence and observed lifespan")
display(exact_label_recurrence_summary)

for field in CONNECTION_IDENTITY_FIELDS:
    print(
        f"{field.title()} labels with the longest gaps "
        "between observed dates"
    )

    display(
        temporal_label_profiles[field]
        .sort_values(
            [
                "maximum_gap_days",
                "runner_rows",
                "raw_label",
            ],
            ascending=[False, False, True],
        )
        .head(30)
        .reset_index(drop=True)
    )

Exact-label recurrence and observed lifespan


,source_field,distinct_exact_labels,one_runner_row_labels,multiple_runner_row_labels,one_observed_date_labels,multiple_observed_date_labels,one_calendar_year_labels,multiple_calendar_year_labels,median_observed_lifespan_days,maximum_observed_lifespan_days,median_maximum_gap_days,maximum_gap_days
0,jockey,7917,1331,6586,1385,6532,2256,5661,890.0,4164,211.0,3867
1,trainer,10708,2273,8435,2390,8318,3509,7199,897.0,4164,190.0,4073
2,owner,98234,20814,77420,20928,77306,40628,57606,312.0,4164,140.0,4046


Jockey labels with the longest gaps between observed dates


,raw_label,runner_rows,observed_dates,observed_calendar_years,first_observed_date,last_observed_date,observed_lifespan_days,maximum_gap_days
0,Israel Hernandez,2,2,2,2015-02-18,2025-09-20,3867,3867
1,Mark Lawson,37,16,3,2015-04-06,2026-02-19,3972,3771
2,Chris R Rosier,5,2,2,2015-02-08,2025-01-11,3625,3625
3,Xavier Bergeron,54,36,3,2015-02-22,2026-05-24,4109,3623
4,E Alvares,3,3,3,2015-10-10,2026-05-03,3858,3619
5,Gary Bartley,90,53,2,2015-01-08,2025-09-14,3902,3599
6,Pierre Boudvillain,7,7,2,2016-01-04,2026-03-21,3729,3530
7,Beany Panya,2,2,2,2015-04-16,2024-11-23,3509,3509
8,Ben Kennedy,5,5,3,2015-05-16,2026-05-04,4006,3439
9,Taichi Nishimura,5,5,3,2016-01-11,2026-02-22,3695,3394


Trainer labels with the longest gaps between observed dates


,raw_label,runner_rows,observed_dates,observed_calendar_years,first_observed_date,last_observed_date,observed_lifespan_days,maximum_gap_days
0,Paul Patrick Moloney,3,3,2,2015-02-10,2026-04-20,4087,4073
1,Kym Davison,2,2,2,2015-03-27,2026-04-04,4026,4026
2,Luis C Lopez,2,2,2,2015-05-01,2026-05-01,4018,4018
3,Diane Poidevin Laine,3,2,2,2015-03-21,2026-03-13,4010,4010
4,Lucy Longmire,3,3,2,2015-05-08,2026-04-18,3998,3977
5,Wade Rarick,2,2,2,2015-02-18,2025-09-29,3876,3876
6,Lindsay Gough,2,2,2,2015-05-16,2025-07-05,3703,3703
7,John Dann,2,2,2,2015-05-23,2025-05-24,3654,3654
8,Gabriel H Degregorio,2,2,2,2016-05-01,2026-05-01,3652,3652
9,Alberto M Lopez,4,4,2,2015-10-12,2025-12-13,3715,3617


Owner labels with the longest gaps between observed dates


,raw_label,runner_rows,observed_dates,observed_calendar_years,first_observed_date,last_observed_date,observed_lifespan_days,maximum_gap_days
0,D W Stutt,2,2,2,2015-03-07,2026-04-04,4046,4046
1,Foxtrot Racing Management Ltd,2,2,2,2015-04-13,2026-04-19,4024,4024
2,A C Cook Partner,3,3,2,2015-06-03,2026-05-22,4006,3993
3,Mrs N Hodge,6,6,2,2015-02-16,2026-04-17,4078,3985
4,Fateel Brothers Syndicate,6,5,3,2015-02-27,2026-03-27,4046,3955
5,Pump Technology Limited,3,3,2,2015-06-25,2026-05-23,3985,3949
6,Dermot Kelly,4,4,2,2015-01-12,2026-05-13,4139,3929
7,Let It Ride Equine Holdings Iii Llc,3,3,2,2015-07-19,2026-05-02,3940,3891
8,Yuka Mitsuoka,2,2,2,2015-07-08,2026-03-01,3889,3889
9,Scea Haras Dorfausse,2,2,2,2015-09-23,2026-05-05,3877,3877


## Stage 17 — Exact-label jurisdiction breadth

The recurrence profile found long observed spans and long gaps in all three fields.

Chronology alone cannot distinguish:

- one entity returning after an absence;
- incomplete source coverage;
- movement between jurisdictions;
- two entities sharing one exact label;
- a reused organisation or ownership-account name;
- a source error.

The next bounded question is:

> Across how many governed course jurisdictions does each exact label appear?

This stage uses the project’s existing governed course and jurisdiction reference rather than parsing jurisdiction directly from `course` text.

For each exact `jockey`, `trainer` and `owner` label, it measures:

- distinct governed jurisdictions;
- distinct governed courses;
- runner rows;
- first and last observed dates;
- labels appearing in one jurisdiction only;
- labels appearing in several jurisdictions;
- the most jurisdictionally widespread exact labels.

Jurisdiction breadth remains descriptive.

It does not prove:

- that one person or organisation operated in every observed jurisdiction;
- that a cross-jurisdiction label identifies one global entity;
- that a single-jurisdiction label is unique;
- that movement between jurisdictions occurred;
- that identical labels in different jurisdictions should be merged or split.

The raw role assertions and exact source labels remain unchanged.


In [18]:
# Inspect the committed reference directory before choosing a governed course
# or jurisdiction source.
#
# This cell deliberately does not guess a filename or column name. It lists
# the available reference files and shows the schema of CSVs whose names or
# columns mention course, jurisdiction, country, authority, location or
# timezone.
REFERENCE_DIRECTORY = (
    PROJECT_ROOT
    / "data"
    / "reference"
)

if not REFERENCE_DIRECTORY.exists():
    raise FileNotFoundError(
        f"Reference directory not found: {REFERENCE_DIRECTORY}"
    )


reference_files = sorted(
    path
    for path in REFERENCE_DIRECTORY.iterdir()
    if path.is_file()
)


reference_inventory_rows = []

for path in reference_files:
    inventory_row = {
        "reference_file": path.name,
        "suffix": path.suffix.lower(),
        "size_bytes": path.stat().st_size,
    }

    # Read only CSV headers at this stage. We are inspecting repository
    # structure, not loading or interpreting reference contents yet.
    if path.suffix.lower() == ".csv":
        try:
            columns = pd.read_csv(
                path,
                nrows=0,
            ).columns.tolist()

            inventory_row["columns"] = ", ".join(columns)

        except Exception as error:
            inventory_row["columns"] = (
                f"<header read failed: {type(error).__name__}: {error}>"
            )

    else:
        inventory_row["columns"] = ""

    reference_inventory_rows.append(inventory_row)


reference_inventory = pd.DataFrame(
    reference_inventory_rows
)


# Retain files whose filename or CSV schema contains terminology relevant to
# governed course, jurisdiction, location or timezone context.
relevant_terms = [
    "course",
    "jurisdiction",
    "country",
    "authority",
    "location",
    "timezone",
    "surface",
]


def contains_relevant_reference_term(row: pd.Series) -> bool:
    """Return True when a reference filename or schema contains a target term."""
    searchable_text = (
        f"{row['reference_file']} {row['columns']}"
    ).casefold()

    return any(
        term in searchable_text
        for term in relevant_terms
    )


relevant_reference_inventory = (
    reference_inventory.loc[
        reference_inventory.apply(
            contains_relevant_reference_term,
            axis=1,
        )
    ]
    .reset_index(drop=True)
)


print("Relevant governed reference files and schemas")
display(relevant_reference_inventory)

Relevant governed reference files and schemas


,reference_file,suffix,size_bytes,columns
0,course_location_geocoding_run_summary.csv,.csv,34502,"candidate_course_label, candidate_jurisdiction..."
1,course_location_manual_review.csv,.csv,72316,"candidate_course_label, candidate_jurisdiction..."
2,course_location_manual_timezone_resolution.csv,.csv,15449,"candidate_course_label, candidate_jurisdiction..."
3,course_locations.csv,.csv,64353,"candidate_course_label, candidate_jurisdiction..."
4,manual_verifications.csv,.csv,25519,"verification_id, subject_type, source_date, so..."


In [19]:
# Load the existing governed course-location reference using its actual
# repository filename and actual committed schema.
#
# This cell still does not join the source data. It first establishes whether
# `raw_course_labels` can safely act as the source-course lookup column or
# whether it contains composite or non-unique values requiring a separate
# mapping step.
COURSE_LOCATION_REFERENCE_PATH = (
    PROJECT_ROOT
    / "data"
    / "reference"
    / "course_locations.csv"
)

if not COURSE_LOCATION_REFERENCE_PATH.exists():
    raise FileNotFoundError(
        "Governed course-location reference not found: "
        f"{COURSE_LOCATION_REFERENCE_PATH}"
    )


course_location_reference = pd.read_csv(
    COURSE_LOCATION_REFERENCE_PATH
)


# These names come from the committed file schema rather than an inferred or
# newly invented course model.
REQUIRED_COURSE_LOCATION_COLUMNS = [
    "candidate_course_label",
    "candidate_jurisdiction",
    "country",
    "iana_timezone",
    "raw_course_labels",
    "provisional_races",
    "meeting_dates",
    "earliest_date",
    "latest_date",
]

missing_course_location_columns = [
    column
    for column in REQUIRED_COURSE_LOCATION_COLUMNS
    if column not in course_location_reference.columns
]

if missing_course_location_columns:
    raise AssertionError(
        "Governed course-location reference is missing required columns: "
        f"{missing_course_location_columns}"
    )


# Profile the possible source-course lookup column before using it.
#
# We must establish whether each row contains exactly one raw source label and
# whether those labels are unique. A plural column name alone is not enough
# evidence to assume its storage convention.
raw_course_label_profile = pd.DataFrame(
    [
        {
            "reference_rows": len(course_location_reference),
            "blank_raw_course_label_rows": int(
                course_location_reference[
                    "raw_course_labels"
                ].isna().sum()
                + course_location_reference[
                    "raw_course_labels"
                ].fillna("").eq("").sum()
            ),
            "distinct_raw_course_label_values": int(
                course_location_reference[
                    "raw_course_labels"
                ].nunique(dropna=True)
            ),
            "duplicate_raw_course_label_rows": int(
                course_location_reference[
                    "raw_course_labels"
                ].duplicated(keep=False).sum()
            ),
            "distinct_candidate_jurisdictions": int(
                course_location_reference[
                    "candidate_jurisdiction"
                ].nunique(dropna=True)
            ),
            "distinct_countries": int(
                course_location_reference[
                    "country"
                ].nunique(dropna=True)
            ),
        }
    ]
)


# Display potentially composite raw-label values without assuming a delimiter.
#
# Values containing commas, semicolons, pipes or repeated spaces are shown
# because any of those could indicate multiple stored labels or merely be part
# of one legitimate source label.
possible_composite_raw_course_labels = (
    course_location_reference.loc[
        course_location_reference[
            "raw_course_labels"
        ]
        .fillna("")
        .str.contains(
            r"[,;|]|\s{2,}",
            regex=True,
        ),
        [
            "candidate_course_label",
            "candidate_jurisdiction",
            "raw_course_labels",
        ],
    ]
    .reset_index(drop=True)
)


print("Governed course-location reference")
print(
    COURSE_LOCATION_REFERENCE_PATH
    .relative_to(PROJECT_ROOT)
    .as_posix()
)

print("Course-location reference columns")
display(
    pd.DataFrame(
        {
            "column": course_location_reference.columns
        }
    )
)

print("Raw course-label lookup profile")
display(raw_course_label_profile)

print("Possible composite raw course-label values")
display(possible_composite_raw_course_labels)

Governed course-location reference
data/reference/course_locations.csv
Course-location reference columns


,column
0,candidate_course_label
1,candidate_jurisdiction
2,physical_venue_name
3,locality
4,region
5,country
6,latitude
7,longitude
8,iana_timezone
9,location_evidence


Raw course-label lookup profile


,reference_rows,blank_raw_course_label_rows,distinct_raw_course_label_values,duplicate_raw_course_label_rows,distinct_candidate_jurisdictions,distinct_countries
0,395,0,395,0,36,14


Possible composite raw course-label values


,candidate_course_label,candidate_jurisdiction,raw_course_labels


## Stage 17 — Exact-label jurisdiction breadth

The governed `course_locations.csv` reference provides a one-to-one mapping from each stored `raw_course_labels` value to:

- a governed course identity;
- a governed jurisdiction;
- a country;
- a timezone.

The lookup column is complete and unique within the reference:

- 395 reference rows;
- 395 distinct raw course-label values;
- no blank values;
- no duplicate lookup values;
- no composite raw-label values detected.

This stage therefore joins source runner rows to the governed course reference using:

`source.course = course_locations.raw_course_labels`

For each exact `jockey`, `trainer` and `owner` label, it measures:

- runner rows;
- distinct governed courses;
- distinct governed jurisdictions;
- distinct countries;
- first and last observed dates;
- whether all source rows received governed jurisdiction context.

Jurisdiction breadth is descriptive only.

It does not establish that:

- an exact label identifies one global entity;
- two labels in the same jurisdiction identify different entities;
- one label appearing in several jurisdictions represents physical movement;
- identical labels across jurisdictions should be merged;
- identical labels within one jurisdiction should be treated as unique.

Raw source labels and role assertions remain unchanged.

In [20]:
# Prepare the governed course lookup using only the columns required for this
# stage.
course_jurisdiction_lookup = (
    course_location_reference[
        [
            "raw_course_labels",
            "candidate_course_label",
            "candidate_jurisdiction",
            "country",
        ]
    ]
    .rename(
        columns={
            "raw_course_labels": "source_course",
            "candidate_course_label": "governed_course",
            "candidate_jurisdiction": "governed_jurisdiction",
            "country": "governed_country",
        }
    )
    .copy()
)


# Confirm the governed lookup assumptions.
assert course_jurisdiction_lookup["source_course"].notna().all()
assert course_jurisdiction_lookup["source_course"].ne("").all()
assert course_jurisdiction_lookup["source_course"].is_unique


# Create a Python lookup for the small governed reference.
#
# This is used only after SQLite has reduced the source data to compact
# label-by-course aggregates.
governed_course_set = set(
    course_jurisdiction_lookup["source_course"]
)


connection = sqlite3.connect(
    f"file:{SOURCE_DB_PATH}?mode=ro",
    uri=True,
)

try:
    # Aggregate source course coverage in SQLite rather than loading every
    # runner row into pandas.
    source_course_profile = pd.read_sql_query(
        f"""
        SELECT
            course,
            COUNT(*) AS runner_rows,
            MIN(date) AS first_observed_date,
            MAX(date) AS last_observed_date
        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}
        GROUP BY course
        ORDER BY
            runner_rows DESC,
            course
        """,
        connection,
    )

    # Build one compact exact-label-by-course table for each connection field.
    #
    # The largest grouping now happens inside SQLite. Pandas receives only one
    # row per exact label and source course, not one row per runner.
    connection_label_course_profiles = {}

    for field in CONNECTION_IDENTITY_FIELDS:
        quoted_field = quote_identifier(field)

        connection_label_course_profiles[field] = (
            pd.read_sql_query(
                f"""
                SELECT
                    {quoted_field} AS raw_label,
                    course,
                    COUNT(*) AS runner_rows,
                    MIN(date) AS first_observed_date,
                    MAX(date) AS last_observed_date
                FROM {SOURCE_TABLE}
                WHERE {DATA_ROW_PREDICATE}
                  AND {quoted_field} IS NOT NULL
                  AND {quoted_field} <> ''
                GROUP BY
                    {quoted_field},
                    course
                ORDER BY
                    {quoted_field},
                    course
                """,
                connection,
            )
        )

finally:
    connection.close()


# Join the compact source-course profile to the governed course reference.
source_course_with_jurisdiction = (
    source_course_profile.merge(
        course_jurisdiction_lookup,
        left_on="course",
        right_on="source_course",
        how="left",
        validate="one_to_one",
        indicator=True,
    )
)


# Reconcile source course coverage without constructing a runner-level merged
# DataFrame.
source_runner_rows = int(
    source_course_profile["runner_rows"].sum()
)

assert source_runner_rows == EXPECTED_RUNNER_ROWS


unmapped_course_rows = (
    source_course_with_jurisdiction.loc[
        source_course_with_jurisdiction[
            "_merge"
        ].ne("both"),
        [
            "course",
            "runner_rows",
            "first_observed_date",
            "last_observed_date",
        ],
    ]
    .sort_values(
        [
            "runner_rows",
            "course",
        ],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)


mapped_runner_rows = int(
    source_course_with_jurisdiction.loc[
        source_course_with_jurisdiction[
            "_merge"
        ].eq("both"),
        "runner_rows",
    ].sum()
)

unmapped_runner_rows = int(
    source_course_with_jurisdiction.loc[
        source_course_with_jurisdiction[
            "_merge"
        ].ne("both"),
        "runner_rows",
    ].sum()
)


course_join_summary = pd.DataFrame(
    [
        {
            "source_runner_rows": source_runner_rows,
            "mapped_runner_rows": mapped_runner_rows,
            "unmapped_runner_rows": unmapped_runner_rows,
            "distinct_source_courses": int(
                source_course_profile[
                    "course"
                ].nunique(dropna=True)
            ),
            "mapped_distinct_source_courses": int(
                source_course_with_jurisdiction.loc[
                    source_course_with_jurisdiction[
                        "_merge"
                    ].eq("both"),
                    "course",
                ].nunique()
            ),
            "unmapped_distinct_source_courses": int(
                unmapped_course_rows[
                    "course"
                ].nunique(dropna=True)
            ),
        }
    ]
)


assert (
    mapped_runner_rows
    + unmapped_runner_rows
    == EXPECTED_RUNNER_ROWS
)


# Build one exact-label jurisdiction profile for each role from the compact
# label-by-course aggregates.
jurisdiction_label_profiles = {}

for field in CONNECTION_IDENTITY_FIELDS:
    label_course_profile = (
        connection_label_course_profiles[
            field
        ].merge(
            course_jurisdiction_lookup,
            left_on="course",
            right_on="source_course",
            how="left",
            validate="many_to_one",
            indicator=True,
        )
    )

    # Retain the compact mapped table for later bounded checks. This replaces
    # the former 1.85-million-row connection_source_with_jurisdiction frame.
    connection_label_course_profiles[
        field
    ] = label_course_profile

    jurisdiction_profile = (
        label_course_profile.groupby(
            "raw_label",
            as_index=False,
        )
        .agg(
            runner_rows=("runner_rows", "sum"),
            distinct_governed_courses=(
                "governed_course",
                "nunique",
            ),
            distinct_governed_jurisdictions=(
                "governed_jurisdiction",
                "nunique",
            ),
            distinct_governed_countries=(
                "governed_country",
                "nunique",
            ),
            first_observed_date=(
                "first_observed_date",
                "min",
            ),
            last_observed_date=(
                "last_observed_date",
                "max",
            ),
            unmapped_runner_rows=(
                "runner_rows",
                lambda values: int(
                    values.loc[
                        label_course_profile.loc[
                            values.index,
                            "_merge",
                        ].ne("both")
                    ].sum()
                ),
            ),
        )
        .sort_values(
            [
                "distinct_governed_jurisdictions",
                "runner_rows",
                "raw_label",
            ],
            ascending=[False, False, True],
        )
        .reset_index(drop=True)
    )

    jurisdiction_label_profiles[
        field
    ] = jurisdiction_profile


# Summarise jurisdiction breadth across the three fields.
jurisdiction_breadth_summary_rows = []

for field in CONNECTION_IDENTITY_FIELDS:
    profile = jurisdiction_label_profiles[field]

    jurisdiction_breadth_summary_rows.append(
        {
            "source_field": field,
            "distinct_exact_labels": len(profile),
            "one_jurisdiction_labels": int(
                profile[
                    "distinct_governed_jurisdictions"
                ].eq(1).sum()
            ),
            "multiple_jurisdiction_labels": int(
                profile[
                    "distinct_governed_jurisdictions"
                ].gt(1).sum()
            ),
            "one_country_labels": int(
                profile[
                    "distinct_governed_countries"
                ].eq(1).sum()
            ),
            "multiple_country_labels": int(
                profile[
                    "distinct_governed_countries"
                ].gt(1).sum()
            ),
            "maximum_governed_jurisdictions": int(
                profile[
                    "distinct_governed_jurisdictions"
                ].max()
            ),
            "maximum_governed_countries": int(
                profile[
                    "distinct_governed_countries"
                ].max()
            ),
            "labels_with_unmapped_rows": int(
                profile[
                    "unmapped_runner_rows"
                ].gt(0).sum()
            ),
        }
    )


jurisdiction_breadth_summary = pd.DataFrame(
    jurisdiction_breadth_summary_rows
)


# Reconcile each compact profile to the exact-label and populated-row counts
# established earlier.
for field in CONNECTION_IDENTITY_FIELDS:
    profile = jurisdiction_label_profiles[field]

    expected_distinct_labels = int(
        raw_field_profile.loc[
            raw_field_profile[
                "source_field"
            ].eq(field),
            "distinct_populated_labels",
        ].iloc[0]
    )

    expected_populated_rows = int(
        raw_field_profile.loc[
            raw_field_profile[
                "source_field"
            ].eq(field),
            "populated_rows",
        ].iloc[0]
    )

    assert len(profile) == expected_distinct_labels

    assert int(
        profile["runner_rows"].sum()
    ) == expected_populated_rows


print("Governed course-reference join")
display(course_join_summary)

print("Unmapped source course labels")
display(unmapped_course_rows)

print("Exact-label jurisdiction breadth")
display(jurisdiction_breadth_summary)

for field in CONNECTION_IDENTITY_FIELDS:
    print(
        f"{field.title()} labels spanning the most "
        "governed jurisdictions"
    )

    display(
        jurisdiction_label_profiles[
            field
        ]
        .head(30)
        .reset_index(drop=True)
    )

Governed course-reference join


,source_runner_rows,mapped_runner_rows,unmapped_runner_rows,distinct_source_courses,mapped_distinct_source_courses,unmapped_distinct_source_courses
0,1851285,1800607,50678,528,395,133


Unmapped source course labels


,course,runner_rows,first_observed_date,last_observed_date
0,Sha Tin,4953,2025-10-19,2026-05-24
1,Chantilly,3659,2025-10-27,2026-05-19
2,Deauville,3396,2025-10-20,2026-04-08
3,Dundalk (AW),3166,2025-10-17,2026-05-26
4,Happy Valley,2954,2025-10-15,2026-05-27
...,...,...,...,...
128,Lyon Parilly,6,2025-11-07,2025-11-07
129,Pakenham,6,2026-01-23,2026-01-23
130,Sunland Park,6,2026-02-15,2026-02-15
131,Krefeld,5,2025-11-15,2025-11-15


Exact-label jurisdiction breadth


...

,raw_label,runner_rows,distinct_governed_courses,distinct_governed_jurisdictions,distinct_governed_countries,first_observed_date,last_observed_date,unmapped_runner_rows
0,Stephane Pasquier,6186,50,17,7,2015-01-10,2026-05-24,260
1,Frankie Dettori,2785,79,17,7,2015-01-24,2026-02-01,13
2,Oisin Murphy,10017,87,16,8,2015-01-04,2026-05-27,53
3,Tom Marquand,9918,75,16,7,2015-01-03,2026-05-27,35
4,Hollie Doyle,8594,82,16,6,2015-01-02,2026-05-22,112
5,Gerald Mosse,2387,76,16,4,2015-01-25,2024-09-15,0
6,Luke Morris,14241,75,15,6,2015-01-03,2026-05-27,6
7,Silvestre De Sousa,8382,68,15,4,2015-01-04,2026-05-27,187
8,William Buick,6799,80,15,7,2015-01-15,2026-05-24,86
9,Andrea Atzeni,6332,79,15,7,2015-02-13,2026-05-27,450


Trainer labels spanning the most governed jurisdictions


,raw_label,runner_rows,distinct_governed_courses,distinct_governed_jurisdictions,distinct_governed_countries,first_observed_date,last_observed_date,unmapped_runner_rows
0,Andrew Balding,9804,81,17,4,2015-01-01,2026-05-27,13
1,Marco Botti,4461,62,15,4,2015-01-03,2026-05-27,7
2,A Wohler,803,58,15,3,2015-01-10,2026-05-25,17
3,Joseph Patrick OBrien,11068,98,14,5,2016-06-06,2026-05-26,504
4,A P OBrien,8311,77,14,6,2015-01-05,2026-05-24,216
5,Archie Watson,5570,116,14,4,2016-08-29,2026-05-27,6
6,George Baker,2625,89,14,3,2015-01-03,2026-05-27,17
7,Richard Hannon,13165,73,13,5,2015-01-03,2026-05-27,6
8,Michael Appleby,9584,71,13,4,2015-01-01,2026-05-27,2
9,William Haggas,7421,73,13,4,2015-01-06,2026-05-27,15


Owner labels spanning the most governed jurisdictions


,raw_label,runner_rows,distinct_governed_courses,distinct_governed_jurisdictions,distinct_governed_countries,first_observed_date,last_observed_date,unmapped_runner_rows
0,Godolphin,13416,146,14,7,2015-01-01,2026-05-27,261
1,Al Shaqab Racing,2604,94,13,4,2015-01-10,2026-05-23,35
2,Hamdan Al Maktoum,6867,102,12,6,2015-01-04,2021-03-23,0
3,King Power Racing Co Ltd,2563,68,12,4,2017-07-07,2026-05-27,0
4,Qatar Racing Limited,2265,110,12,5,2015-01-09,2026-05-26,12
5,Sheikh Hamdan Bin Mohammed Al Maktoum,4539,64,11,5,2015-01-04,2026-05-27,211
6,Sheikh Ahmed Al Maktoum,2769,63,11,4,2015-01-04,2026-05-26,163
7,Mrs John Magnier Michael Tabor Derrick Smith,1673,77,11,5,2015-03-29,2026-05-19,28
8,Eckhard Sauren,456,39,11,2,2015-01-03,2026-05-25,27
9,Teruya Yoshida,324,45,11,4,2015-01-04,2026-05-16,13


## Stage 18 — Measure jurisdiction-reference coverage by exact role label

The governed course-location reference does not cover the complete current source population.

The source contains:

- 528 distinct course labels;
- 395 mapped course labels;
- 133 unmapped course labels;
- 50,678 runner rows without governed course context.

The missing labels are concentrated in the later source extension, including major courses such as:

- `Sha Tin`;
- `Happy Valley`;
- `Chantilly`;
- `Deauville`;
- `Dundalk (AW)`.

Consequently, the jurisdiction breadth calculated in Stage 17 is incomplete for many exact connection labels.

This stage separates labels into three coverage classes:

- `fully_mapped` — every populated role assertion has governed course context;
- `partially_mapped` — some, but not all, role assertions have governed course context;
- `entirely_unmapped` — none of the role assertions has governed course context.

It also measures:

- mapped and unmapped runner rows;
- mapped-row percentage;
- the provisional jurisdiction breadth among mapped rows only;
- examples with the largest unmapped populations.

No jurisdiction-based identity conclusion is drawn until the course reference is extended or the incomplete coverage is explicitly accepted as a limitation.

In [21]:
# Build mapped-source coverage profiles from the compact label-by-course
# aggregates created in the replacement jurisdiction cell.
#
# These profiles describe governed-reference coverage, not complete global
# jurisdiction breadth, because some source courses remain unmapped.
jurisdiction_coverage_profiles = {}


for field in CONNECTION_IDENTITY_FIELDS:
    label_course_profile = (
        connection_label_course_profiles[field]
        .copy()
    )

    # Build one coverage profile per exact populated label.
    coverage_profile = (
        label_course_profile.groupby(
            "raw_label",
            as_index=False,
        )
        .agg(
            runner_rows=("runner_rows", "sum"),
            mapped_runner_rows=(
                "runner_rows",
                lambda values: int(
                    values.loc[
                        label_course_profile.loc[
                            values.index,
                            "_merge",
                        ].eq("both")
                    ].sum()
                ),
            ),
            unmapped_runner_rows=(
                "runner_rows",
                lambda values: int(
                    values.loc[
                        label_course_profile.loc[
                            values.index,
                            "_merge",
                        ].ne("both")
                    ].sum()
                ),
            ),
            mapped_source_courses=(
                "governed_course",
                "nunique",
            ),
            mapped_jurisdictions=(
                "governed_jurisdiction",
                "nunique",
            ),
            mapped_countries=(
                "governed_country",
                "nunique",
            ),
            first_observed_date=(
                "first_observed_date",
                "min",
            ),
            last_observed_date=(
                "last_observed_date",
                "max",
            ),
        )
    )


    # Classify coverage using runner-row evidence.
    def classify_jurisdiction_coverage(
        row: pd.Series,
    ) -> str:
        """
        Describe whether an exact label's source rows are covered by the
        governed course reference.
        """
        mapped_rows = row["mapped_runner_rows"]
        unmapped_rows = row["unmapped_runner_rows"]

        if mapped_rows == 0:
            return "entirely_unmapped"

        if unmapped_rows == 0:
            return "fully_mapped"

        return "partially_mapped"


    coverage_profile[
        "coverage_class"
    ] = coverage_profile.apply(
        classify_jurisdiction_coverage,
        axis=1,
    )


    # Confirm mapped and unmapped rows reconcile to the full exact-label
    # population.
    assert (
        coverage_profile[
            "mapped_runner_rows"
        ]
        + coverage_profile[
            "unmapped_runner_rows"
        ]
        == coverage_profile[
            "runner_rows"
        ]
    ).all()


    jurisdiction_coverage_profiles[
        field
    ] = (
        coverage_profile.sort_values(
            [
                "coverage_class",
                "runner_rows",
                "raw_label",
            ],
            ascending=[True, False, True],
        )
        .reset_index(drop=True)
    )


# Summarise the coverage classes for each connection field.
jurisdiction_coverage_summary_rows = []


for field in CONNECTION_IDENTITY_FIELDS:
    profile = jurisdiction_coverage_profiles[field]

    for coverage_class in [
        "entirely_unmapped",
        "fully_mapped",
        "partially_mapped",
    ]:
        selected = profile.loc[
            profile[
                "coverage_class"
            ].eq(coverage_class)
        ]

        jurisdiction_coverage_summary_rows.append(
            {
                "source_field": field,
                "coverage_class": coverage_class,
                "distinct_exact_labels": len(selected),
                "runner_rows": int(
                    selected[
                        "runner_rows"
                    ].sum()
                ),
                "mapped_runner_rows": int(
                    selected[
                        "mapped_runner_rows"
                    ].sum()
                ),
                "unmapped_runner_rows": int(
                    selected[
                        "unmapped_runner_rows"
                    ].sum()
                ),
            }
        )


jurisdiction_coverage_summary = pd.DataFrame(
    jurisdiction_coverage_summary_rows
)


# Reconcile every field to the populated exact-label population established
# earlier in the notebook.
for field in CONNECTION_IDENTITY_FIELDS:
    profile = jurisdiction_coverage_profiles[field]

    expected_distinct_labels = int(
        raw_field_profile.loc[
            raw_field_profile[
                "source_field"
            ].eq(field),
            "distinct_populated_labels",
        ].iloc[0]
    )

    expected_populated_rows = int(
        raw_field_profile.loc[
            raw_field_profile[
                "source_field"
            ].eq(field),
            "populated_rows",
        ].iloc[0]
    )

    assert len(profile) == expected_distinct_labels

    assert int(
        profile[
            "runner_rows"
        ].sum()
    ) == expected_populated_rows

    assert int(
        profile[
            "mapped_runner_rows"
        ].sum()
        + profile[
            "unmapped_runner_rows"
        ].sum()
    ) == expected_populated_rows


print("Exact-label governed-reference coverage")
display(jurisdiction_coverage_summary)


for field in CONNECTION_IDENTITY_FIELDS:
    print(
        f"{field.title()} labels with entirely unmapped source rows"
    )

    display(
        jurisdiction_coverage_profiles[
            field
        ]
        .loc[
            jurisdiction_coverage_profiles[
                field
            ][
                "coverage_class"
            ].eq("entirely_unmapped")
        ]
        .sort_values(
            [
                "runner_rows",
                "raw_label",
            ],
            ascending=[False, True],
        )
        .head(30)
        .reset_index(drop=True)
    )


    print(
        f"{field.title()} labels with partially mapped source rows"
    )

    display(
        jurisdiction_coverage_profiles[
            field
        ]
        .loc[
            jurisdiction_coverage_profiles[
                field
            ][
                "coverage_class"
            ].eq("partially_mapped")
        ]
        .sort_values(
            [
                "unmapped_runner_rows",
                "runner_rows",
                "raw_label",
            ],
            ascending=[False, False, True],
        )
        .head(30)
        .reset_index(drop=True)
    )

Exact-label governed-reference coverage


,source_field,coverage_class,distinct_exact_labels,runner_rows,mapped_runner_rows,unmapped_runner_rows
0,jockey,entirely_unmapped,163,574,0,574
1,jockey,fully_mapped,6196,805188,805188,0
2,jockey,partially_mapped,1558,1045521,995417,50104
3,trainer,entirely_unmapped,218,941,0,941
4,trainer,fully_mapped,8276,862634,862634,0
5,trainer,partially_mapped,2214,987701,937964,49737
6,owner,entirely_unmapped,3202,6863,0,6863
7,owner,fully_mapped,86369,1332197,1332197,0
8,owner,partially_mapped,8663,512190,468375,43815


Jockey labels with entirely unmapped source rows


,raw_label,runner_rows,mapped_runner_rows,unmapped_runner_rows,mapped_source_courses,mapped_jurisdictions,mapped_countries,first_observed_date,last_observed_date,coverage_class
0,Kevin Healy,70,0,70,0,0,0,2025-11-04,2026-05-27,entirely_unmapped
1,Nichola Yuen,66,0,66,0,0,0,2026-04-01,2026-05-24,entirely_unmapped
2,Brian Barry,41,0,41,0,0,0,2025-10-16,2026-05-21,entirely_unmapped
3,Alan OSullivan,37,0,37,0,0,0,2026-05-01,2026-05-27,entirely_unmapped
4,Victor Salgado,33,0,33,0,0,0,2025-10-25,2026-03-15,entirely_unmapped
5,Gustavo Emiliano Calvente,11,0,11,0,0,0,2025-10-18,2026-05-25,entirely_unmapped
6,Mme Maelie Pien,11,0,11,0,0,0,2026-03-05,2026-05-22,entirely_unmapped
7,Esentur Turganaaly Uulu,10,0,10,0,0,0,2025-10-26,2026-05-25,entirely_unmapped
8,E E Ryan,8,0,8,0,0,0,2026-03-30,2026-05-20,entirely_unmapped
9,Lancelot Houssin,8,0,8,0,0,0,2026-03-14,2026-05-06,entirely_unmapped


Jockey labels with partially mapped source rows


...

,raw_label,runner_rows,mapped_runner_rows,unmapped_runner_rows,mapped_source_courses,mapped_jurisdictions,mapped_countries,first_observed_date,last_observed_date,coverage_class
0,Gordon Elliott,15140,14240,900,87,5,4,2015-01-01,2026-05-27,partially_mapped
1,W P Mullins,10807,10088,719,86,10,4,2015-01-01,2026-05-27,partially_mapped
2,Gavin Cromwell,7490,6803,687,84,6,4,2015-01-21,2026-05-27,partially_mapped
3,Joseph Patrick OBrien,11068,10564,504,98,14,5,2016-06-06,2026-05-26,partially_mapped
4,Henry De Bromhead,7479,6975,504,59,3,3,2015-01-01,2026-05-27,partially_mapped
5,David A Hayes,3027,2536,491,7,5,3,2015-02-21,2026-05-27,partially_mapped
6,A S Cruz,5536,5086,450,4,3,1,2015-01-25,2026-05-27,partially_mapped
7,J Size,5232,4789,443,4,3,1,2015-01-25,2026-05-27,partially_mapped
8,C Fownes,5059,4640,419,5,4,1,2015-03-01,2026-05-27,partially_mapped
9,Bhupat Seemar,754,335,419,5,2,1,2024-10-26,2026-05-02,partially_mapped


Owner labels with entirely unmapped source rows


,raw_label,runner_rows,mapped_runner_rows,unmapped_runner_rows,mapped_source_courses,mapped_jurisdictions,mapped_countries,first_observed_date,last_observed_date,coverage_class
0,Jacques Detre Detre Racing Ecurie Team Spirit,29,0,29,0,0,0,2026-03-01,2026-05-16,entirely_unmapped
1,Westminster Race Horses Sp Zoo,21,0,21,0,0,0,2025-12-13,2026-04-26,entirely_unmapped
2,Detre Racing Jacques Detre Ecurie Team Spirit,19,0,19,0,0,0,2026-02-14,2026-05-26,entirely_unmapped
3,Coolmara Stables Limited,18,0,18,0,0,0,2025-10-27,2026-05-17,entirely_unmapped
4,Jose Maria Maldonado,17,0,17,0,0,0,2026-03-12,2026-05-19,entirely_unmapped
5,Hermitage,15,0,15,0,0,0,2025-10-18,2026-05-13,entirely_unmapped
6,Mystery Downs Mrs C M Cook Et Al,15,0,15,0,0,0,2025-10-18,2026-04-11,entirely_unmapped
7,Haras Legacy,14,0,14,0,0,0,2025-10-18,2026-05-03,entirely_unmapped
8,Akhmat Racing,12,0,12,0,0,0,2025-11-15,2026-04-11,entirely_unmapped
9,Ardak Amirkulov Shamil Kurbanov Et Al,12,0,12,0,0,0,2025-11-15,2026-03-15,entirely_unmapped


Owner labels with partially mapped source rows


,raw_label,runner_rows,mapped_runner_rows,unmapped_runner_rows,mapped_source_courses,mapped_jurisdictions,mapped_countries,first_observed_date,last_observed_date,coverage_class
0,John P Mcmanus,15384,14855,529,93,5,3,2015-01-01,2026-05-27,partially_mapped
1,Gigginstown House Stud,6998,6686,312,48,4,3,2015-01-01,2026-05-27,partially_mapped
2,Godolphin,13416,13155,261,146,14,7,2015-01-01,2026-05-27,partially_mapped
3,Sheikh Hamdan Bin Mohammed Al Maktoum,4539,4328,211,64,11,5,2015-01-04,2026-05-27,partially_mapped
4,Sheikh Ahmed Al Maktoum,2769,2606,163,63,11,4,2015-01-04,2026-05-26,partially_mapped
5,Wertheimer Frere,2797,2638,159,58,9,3,2015-01-12,2026-05-24,partially_mapped
6,Simon Munir Isaac Souede,3410,3269,141,108,3,3,2015-01-01,2026-05-26,partially_mapped
7,Al Mohamediya Racing,821,686,135,44,6,2,2017-03-14,2026-05-18,partially_mapped
8,James Mcauley,1640,1525,115,28,2,2,2017-09-22,2026-05-26,partially_mapped
9,Robcour,1225,1111,114,41,4,3,2015-09-19,2026-05-26,partially_mapped


## Stage 19 — Owner token-order collision candidates

The jurisdiction-reference analysis is incomplete because 133 current source course labels are not yet present in the governed course-location reference.

Jurisdiction breadth will therefore remain provisional until that reference is extended.

The next source-internal identity question does not depend on course mapping:

> Do distinct exact owner labels contain the same words but present them in a different order?

Earlier examples included:

- `Mrs John Magnier Michael Tabor Derrick Smith`;
- `Michael Tabor Derrick Smith Mrs John Magnier`;
- `Derrick Smith Mrs John Magnier Michael Tabor`.

These labels may represent the same recorded ownership combination, but word-order equality does not itself prove that conclusion.

This stage creates an owner-only token-multiset comparison key by:

1. trimming outer ASCII spaces;
2. collapsing internal ASCII-space runs;
3. applying case-folding;
4. splitting on ASCII spaces;
5. sorting the complete token sequence;
6. retaining repeated tokens.

The comparison deliberately preserves:

- every word;
- repeated words;
- punctuation attached to words;
- numerals;
- company suffixes;
- titles.

It changes word order only for comparison.

A collision identifies a possible reordered-label relationship. It does not prove:

- shared ownership identity;
- equivalent partnership membership;
- equivalent legal ownership;
- safe canonicalisation;
- that one label is preferred;
- that reordered words always describe the same people or organisation.

All raw labels remain unchanged.

In [22]:
def build_token_multiset_key(value: str) -> str:
    """
    Build a comparison key that ignores word order only.

    The raw label is first compared using the already investigated reversible
    case and ASCII-whitespace treatments. Tokens are then sorted while
    retaining duplicates. Punctuation attached to a token remains attached.
    """
    whitespace_normalised = build_whitespace_comparison_key(value)

    tokens = whitespace_normalised.casefold().split(" ")

    return " | ".join(sorted(tokens))


# Work only from exact populated owner labels established earlier.
owner_token_order_profile = owner_labels.copy()

owner_token_order_profile["token_multiset_key"] = (
    owner_token_order_profile["raw_label"].map(
        build_token_multiset_key
    )
)

owner_token_order_profile["token_count"] = (
    owner_token_order_profile["raw_label"]
    .map(build_whitespace_comparison_key)
    .str.split(" ")
    .str.len()
)


# Count distinct exact labels sharing each order-insensitive token key.
owner_token_key_summary = (
    owner_token_order_profile.groupby(
        "token_multiset_key",
        as_index=False,
    )
    .agg(
        distinct_raw_labels=("raw_label", "nunique"),
        runner_rows=("runner_rows", "sum"),
        minimum_token_count=("token_count", "min"),
        maximum_token_count=("token_count", "max"),
    )
)


# Retain only keys shared by at least two exact raw labels.
owner_token_collision_keys = set(
    owner_token_key_summary.loc[
        owner_token_key_summary["distinct_raw_labels"] > 1,
        "token_multiset_key",
    ]
)


owner_token_order_collisions = (
    owner_token_order_profile.loc[
        owner_token_order_profile[
            "token_multiset_key"
        ].isin(owner_token_collision_keys),
        [
            "token_multiset_key",
            "raw_label",
            "token_count",
            "runner_rows",
            "first_observed_date",
            "last_observed_date",
        ],
    ]
    .sort_values(
        [
            "token_multiset_key",
            "runner_rows",
            "raw_label",
        ],
        ascending=[True, False, True],
    )
    .reset_index(drop=True)
)


# Summarise collision groups by the number of participating raw labels.
owner_token_collision_distribution = (
    owner_token_key_summary.loc[
        owner_token_key_summary["distinct_raw_labels"] > 1
    ]
    .groupby(
        "distinct_raw_labels",
        as_index=False,
    )
    .agg(
        collision_keys=("token_multiset_key", "size"),
        runner_rows=("runner_rows", "sum"),
    )
    .sort_values("distinct_raw_labels")
    .reset_index(drop=True)
)


# Produce a compact group-level display with all exact labels preserved.
def combine_collision_labels(group: pd.DataFrame) -> str:
    """Join exact labels for display without choosing a canonical form."""
    ordered = group.sort_values(
        [
            "runner_rows",
            "raw_label",
        ],
        ascending=[False, True],
    )

    return " || ".join(
        ordered["raw_label"].tolist()
    )


owner_token_collision_groups = (
    owner_token_order_collisions.groupby(
        "token_multiset_key",
        as_index=False,
    )
    .apply(
        lambda group: pd.Series(
            {
                "distinct_raw_labels": group[
                    "raw_label"
                ].nunique(),
                "token_count": int(
                    group["token_count"].iloc[0]
                ),
                "runner_rows": int(
                    group["runner_rows"].sum()
                ),
                "first_observed_date": group[
                    "first_observed_date"
                ].min(),
                "last_observed_date": group[
                    "last_observed_date"
                ].max(),
                "raw_labels": combine_collision_labels(group),
            }
        ),
        include_groups=False,
    )
    .reset_index(drop=True)
    .sort_values(
        [
            "runner_rows",
            "distinct_raw_labels",
            "raw_labels",
        ],
        ascending=[False, False, True],
    )
    .reset_index(drop=True)
)


# Reconcile the collision outputs.
assert (
    owner_token_order_collisions[
        "token_multiset_key"
    ].nunique()
    == len(owner_token_collision_keys)
)

assert (
    owner_token_order_collisions[
        "raw_label"
    ].nunique()
    == owner_token_collision_groups[
        "distinct_raw_labels"
    ].sum()
)

assert (
    owner_token_collision_groups[
        "runner_rows"
    ].sum()
    == owner_token_order_collisions[
        "runner_rows"
    ].sum()
)


print("Owner token-order collision distribution")
display(owner_token_collision_distribution)

print("Most frequent owner token-order collision groups")
display(owner_token_collision_groups.head(50))

print("Exact raw labels in owner token-order collision groups")
display(owner_token_order_collisions.head(100))

Owner token-order collision distribution


,distinct_raw_labels,collision_keys,runner_rows
0,2,906,26908
1,3,19,811
2,4,8,390
3,5,2,1033
4,6,1,5052


Most frequent owner token-order collision groups


,token_multiset_key,distinct_raw_labels,token_count,runner_rows,first_observed_date,last_observed_date,raw_labels
0,derrick | john | magnier | michael | mrs | smi...,6,7,5052,2015-01-23,2026-05-25,Mrs John Magnier Michael Tabor Derrick Smith |...
1,d | j | m | magnier | mrs | smith | tabor | we...,5,8,851,2020-08-08,2026-05-24,M Tabor D Smith Mrs J Magnier Westerberg || We...
2,anoj | daniel | don | macauliffe,2,4,696,2015-06-12,2026-05-21,Daniel Macauliffe Anoj Don || Anoj Don Daniel ...
3,heather | michael | yarrow,2,3,538,2015-01-09,2026-05-15,Michael Heather Yarrow || Heather Michael Yarrow
4,dineen | hughes | kerr | martin | michael,2,5,418,2015-04-13,2026-05-25,Martin Hughes Michael Kerr Dineen || Michael K...
5,julie | martin | phil,2,3,325,2018-02-03,2023-04-22,Phil Julie Martin || Julie Phil Martin
6,katsumi | yoshida,2,2,324,2015-01-04,2026-05-24,Katsumi Yoshida || Yoshida Katsumi
7,aguiar | amo | de | giselle | limited | racing,2,6,310,2022-03-27,2026-05-26,Amo Racing Limited Giselle De Aguiar || Gisell...
8,david | jones | lynne | lyons | mrs | sean | s...,3,7,306,2019-03-15,2026-05-16,David Spratt Sean Jones Mrs Lynne Lyons || Mrs...
9,andy | bell | fergus | lyons,2,4,276,2016-11-19,2026-05-12,Andy Bell Fergus Lyons || Fergus Lyons Andy Bell


Exact raw labels in owner token-order collision groups


,token_multiset_key,raw_label,token_count,runner_rows,first_observed_date,last_observed_date
0,1 | 2 | lees | newcastle | newcastle | racecou...,Newcastle Racecourse 1 Newcastle Racecourse 2 ...,8,11,2021-02-13,2022-10-22
1,1 | 2 | lees | newcastle | newcastle | racecou...,Lees Racing Newcastle Racecourse 1 Newcastle R...,8,1,2023-05-13,2023-05-13
2,100 | amy | biancone | club | diamond | dunne ...,Diamond 100 Racing Club Amy Dunne Patrick L Bi...,10,3,2019-06-29,2022-02-05
3,100 | amy | biancone | club | diamond | dunne ...,Diamond 100 Racing Club Llc Amy Dunne Patrick ...,10,1,2022-12-31,2022-12-31
4,3 | ailes | capital | caullery | compagnie | g...,3 Ailes Compagnie Gest Invest Capital Nicolas ...,8,23,2024-08-20,2026-01-09
...,...,...,...,...,...,...
95,a | bernsen | david | llc | ranch | rockingham,Rockingham Ranch David A Bernsen Llc,6,55,2017-10-07,2025-03-22
96,a | bernsen | david | llc | ranch | rockingham,David A Bernsen Llc Rockingham Ranch,6,5,2025-01-25,2026-01-17
97,a | bernsen | david | magdalena | moulton | ra...,David A Bernsen Susan Moulton Magdalena Racing,7,1,2019-07-28,2019-07-28
98,a | bernsen | david | magdalena | moulton | ra...,Susan Moulton David A Bernsen Magdalena Racing,7,1,2019-03-23,2019-03-23


## Stage 19 — Owner token-order collision candidates

The jurisdiction-reference analysis is incomplete because 133 current source course labels are not yet present in the governed course-location reference.

Jurisdiction breadth will therefore remain provisional until that reference is extended.

The next source-internal identity question does not depend on course mapping:

> Do distinct exact owner labels contain the same words but present them in a different order?

Earlier examples included:

- `Mrs John Magnier Michael Tabor Derrick Smith`;
- `Michael Tabor Derrick Smith Mrs John Magnier`;
- `Derrick Smith Mrs John Magnier Michael Tabor`.

These labels may represent the same recorded ownership combination, but word-order equality does not itself prove that conclusion.

This stage creates an owner-only token-multiset comparison key by:

1. trimming outer ASCII spaces;
2. collapsing internal ASCII-space runs;
3. applying case-folding;
4. splitting on ASCII spaces;
5. sorting the complete token sequence;
6. retaining repeated tokens.

The comparison deliberately preserves:

- every word;
- repeated words;
- punctuation attached to words;
- numerals;
- company suffixes;
- titles.

It changes word order only for comparison.

A collision identifies a possible reordered-label relationship. It does not prove:

- shared ownership identity;
- equivalent partnership membership;
- equivalent legal ownership;
- safe canonicalisation;
- that one label is preferred;
- that reordered words always describe the same people or organisation.

All raw labels remain unchanged.

In [23]:
def build_token_multiset_key(value: str) -> str:
    """
    Build a comparison key that ignores word order only.

    The raw label is first compared using the already investigated reversible
    case and ASCII-whitespace treatments. Tokens are then sorted while
    retaining duplicates. Punctuation attached to a token remains attached.
    """
    whitespace_normalised = build_whitespace_comparison_key(value)

    tokens = whitespace_normalised.casefold().split(" ")

    return " | ".join(sorted(tokens))


# Work only from exact populated owner labels established earlier.
owner_token_order_profile = owner_labels.copy()

owner_token_order_profile["token_multiset_key"] = (
    owner_token_order_profile["raw_label"].map(
        build_token_multiset_key
    )
)

owner_token_order_profile["token_count"] = (
    owner_token_order_profile["raw_label"]
    .map(build_whitespace_comparison_key)
    .str.split(" ")
    .str.len()
)


# Count distinct exact labels sharing each order-insensitive token key.
owner_token_key_summary = (
    owner_token_order_profile.groupby(
        "token_multiset_key",
        as_index=False,
    )
    .agg(
        distinct_raw_labels=("raw_label", "nunique"),
        runner_rows=("runner_rows", "sum"),
        minimum_token_count=("token_count", "min"),
        maximum_token_count=("token_count", "max"),
    )
)


# Retain only keys shared by at least two exact raw labels.
owner_token_collision_keys = set(
    owner_token_key_summary.loc[
        owner_token_key_summary["distinct_raw_labels"] > 1,
        "token_multiset_key",
    ]
)


owner_token_order_collisions = (
    owner_token_order_profile.loc[
        owner_token_order_profile[
            "token_multiset_key"
        ].isin(owner_token_collision_keys),
        [
            "token_multiset_key",
            "raw_label",
            "token_count",
            "runner_rows",
            "first_observed_date",
            "last_observed_date",
        ],
    ]
    .sort_values(
        [
            "token_multiset_key",
            "runner_rows",
            "raw_label",
        ],
        ascending=[True, False, True],
    )
    .reset_index(drop=True)
)


# Summarise collision groups by the number of participating raw labels.
owner_token_collision_distribution = (
    owner_token_key_summary.loc[
        owner_token_key_summary["distinct_raw_labels"] > 1
    ]
    .groupby(
        "distinct_raw_labels",
        as_index=False,
    )
    .agg(
        collision_keys=("token_multiset_key", "size"),
        runner_rows=("runner_rows", "sum"),
    )
    .sort_values("distinct_raw_labels")
    .reset_index(drop=True)
)


# Produce a compact group-level display with all exact labels preserved.
def combine_collision_labels(group: pd.DataFrame) -> str:
    """Join exact labels for display without choosing a canonical form."""
    ordered = group.sort_values(
        [
            "runner_rows",
            "raw_label",
        ],
        ascending=[False, True],
    )

    return " || ".join(
        ordered["raw_label"].tolist()
    )


owner_token_collision_groups = (
    owner_token_order_collisions.groupby(
        "token_multiset_key",
        as_index=False,
    )
    .apply(
        lambda group: pd.Series(
            {
                "distinct_raw_labels": group[
                    "raw_label"
                ].nunique(),
                "token_count": int(
                    group["token_count"].iloc[0]
                ),
                "runner_rows": int(
                    group["runner_rows"].sum()
                ),
                "first_observed_date": group[
                    "first_observed_date"
                ].min(),
                "last_observed_date": group[
                    "last_observed_date"
                ].max(),
                "raw_labels": combine_collision_labels(group),
            }
        ),
        include_groups=False,
    )
    .reset_index(drop=True)
    .sort_values(
        [
            "runner_rows",
            "distinct_raw_labels",
            "raw_labels",
        ],
        ascending=[False, False, True],
    )
    .reset_index(drop=True)
)


# Reconcile the collision outputs.
assert (
    owner_token_order_collisions[
        "token_multiset_key"
    ].nunique()
    == len(owner_token_collision_keys)
)

assert (
    owner_token_order_collisions[
        "raw_label"
    ].nunique()
    == owner_token_collision_groups[
        "distinct_raw_labels"
    ].sum()
)

assert (
    owner_token_collision_groups[
        "runner_rows"
    ].sum()
    == owner_token_order_collisions[
        "runner_rows"
    ].sum()
)


print("Owner token-order collision distribution")
display(owner_token_collision_distribution)

print("Most frequent owner token-order collision groups")
display(owner_token_collision_groups.head(50))

print("Exact raw labels in owner token-order collision groups")
display(owner_token_order_collisions.head(100))

Owner token-order collision distribution


,distinct_raw_labels,collision_keys,runner_rows
0,2,906,26908
1,3,19,811
2,4,8,390
3,5,2,1033
4,6,1,5052


Most frequent owner token-order collision groups


,token_multiset_key,distinct_raw_labels,token_count,runner_rows,first_observed_date,last_observed_date,raw_labels
0,derrick | john | magnier | michael | mrs | smi...,6,7,5052,2015-01-23,2026-05-25,Mrs John Magnier Michael Tabor Derrick Smith |...
1,d | j | m | magnier | mrs | smith | tabor | we...,5,8,851,2020-08-08,2026-05-24,M Tabor D Smith Mrs J Magnier Westerberg || We...
2,anoj | daniel | don | macauliffe,2,4,696,2015-06-12,2026-05-21,Daniel Macauliffe Anoj Don || Anoj Don Daniel ...
3,heather | michael | yarrow,2,3,538,2015-01-09,2026-05-15,Michael Heather Yarrow || Heather Michael Yarrow
4,dineen | hughes | kerr | martin | michael,2,5,418,2015-04-13,2026-05-25,Martin Hughes Michael Kerr Dineen || Michael K...
5,julie | martin | phil,2,3,325,2018-02-03,2023-04-22,Phil Julie Martin || Julie Phil Martin
6,katsumi | yoshida,2,2,324,2015-01-04,2026-05-24,Katsumi Yoshida || Yoshida Katsumi
7,aguiar | amo | de | giselle | limited | racing,2,6,310,2022-03-27,2026-05-26,Amo Racing Limited Giselle De Aguiar || Gisell...
8,david | jones | lynne | lyons | mrs | sean | s...,3,7,306,2019-03-15,2026-05-16,David Spratt Sean Jones Mrs Lynne Lyons || Mrs...
9,andy | bell | fergus | lyons,2,4,276,2016-11-19,2026-05-12,Andy Bell Fergus Lyons || Fergus Lyons Andy Bell


Exact raw labels in owner token-order collision groups


,token_multiset_key,raw_label,token_count,runner_rows,first_observed_date,last_observed_date
0,1 | 2 | lees | newcastle | newcastle | racecou...,Newcastle Racecourse 1 Newcastle Racecourse 2 ...,8,11,2021-02-13,2022-10-22
1,1 | 2 | lees | newcastle | newcastle | racecou...,Lees Racing Newcastle Racecourse 1 Newcastle R...,8,1,2023-05-13,2023-05-13
2,100 | amy | biancone | club | diamond | dunne ...,Diamond 100 Racing Club Amy Dunne Patrick L Bi...,10,3,2019-06-29,2022-02-05
3,100 | amy | biancone | club | diamond | dunne ...,Diamond 100 Racing Club Llc Amy Dunne Patrick ...,10,1,2022-12-31,2022-12-31
4,3 | ailes | capital | caullery | compagnie | g...,3 Ailes Compagnie Gest Invest Capital Nicolas ...,8,23,2024-08-20,2026-01-09
...,...,...,...,...,...,...
95,a | bernsen | david | llc | ranch | rockingham,Rockingham Ranch David A Bernsen Llc,6,55,2017-10-07,2025-03-22
96,a | bernsen | david | llc | ranch | rockingham,David A Bernsen Llc Rockingham Ranch,6,5,2025-01-25,2026-01-17
97,a | bernsen | david | magdalena | moulton | ra...,David A Bernsen Susan Moulton Magdalena Racing,7,1,2019-07-28,2019-07-28
98,a | bernsen | david | magdalena | moulton | ra...,Susan Moulton David A Bernsen Magdalena Racing,7,1,2019-03-23,2019-03-23


## Stage 20 — Horse continuity within owner token-order collisions

The token-order comparison found 936 groups of distinct owner labels containing the same case-folded token multiset.

The result is materially larger than a formatting-edge-case population.

Token-order equality alone is therefore not a safe owner-identity rule.

The next bounded question is:

> Within each token-order collision group, do the participating owner labels appear against the same exact horse labels?

Shared horse history may strengthen the interpretation that two reordered labels are source-presentation variants.

However, it still does not prove ownership identity because:

- ownership can change during a horse’s career;
- an exact horse label may represent different real horses;
- the source may contain duplicated or contaminated records;
- two ownership groups can contain the same words in different structural meanings;
- reordered personal names may represent one person or different people;
- legal ownership registrations may differ despite apparently identical members.

This stage measures for each token-order collision group:

- number of participating owner labels;
- runner rows;
- distinct horse labels;
- horse labels observed under more than one owner variant;
- runner rows attached to shared horse labels;
- whether every owner variant participates in at least one shared-horse relationship;
- the most frequent shared horse examples.

The output remains candidate evidence only.

No owner labels are merged, replaced or assigned a canonical form.

In [24]:
# Retrieve source owner and horse assertions only for the finite token-order
# collision population.
#
# Exact raw owner and horse labels remain unchanged. The comparison key is
# attached afterwards from the already constructed owner collision table.
token_collision_owner_labels = (
    owner_token_order_collisions["raw_label"]
    .drop_duplicates()
    .tolist()
)

owner_label_placeholders = ", ".join(
    ["?"] * len(token_collision_owner_labels)
)


connection = sqlite3.connect(
    f"file:{SOURCE_DB_PATH}?mode=ro",
    uri=True,
)

try:
    owner_token_collision_source_rows = pd.read_sql_query(
        f"""
        SELECT
            rowid AS source_rowid,
            race_id AS supplied_race_id,
            date,
            course,
            off,
            horse,
            owner AS raw_owner_label
        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}
          AND owner IN ({owner_label_placeholders})
        ORDER BY
            owner,
            date,
            course,
            off,
            rowid
        """,
        connection,
        params=token_collision_owner_labels,
    )

finally:
    connection.close()


# Attach each exact owner label to its token-order collision key.
owner_collision_key_lookup = (
    owner_token_order_collisions[
        [
            "raw_label",
            "token_multiset_key",
        ]
    ]
    .drop_duplicates()
    .rename(columns={"raw_label": "raw_owner_label"})
)

owner_token_collision_source_rows = (
    owner_token_collision_source_rows.merge(
        owner_collision_key_lookup,
        on="raw_owner_label",
        how="left",
        validate="many_to_one",
    )
)

assert (
    owner_token_collision_source_rows[
        "token_multiset_key"
    ].notna().all()
)


# Count how many distinct owner variants are attached to each exact horse label
# within each token-order collision group.
group_horse_usage = (
    owner_token_collision_source_rows.groupby(
        [
            "token_multiset_key",
            "horse",
        ],
        as_index=False,
    )
    .agg(
        distinct_owner_variants=(
            "raw_owner_label",
            "nunique",
        ),
        runner_rows=("source_rowid", "size"),
        first_observed_date=("date", "min"),
        last_observed_date=("date", "max"),
    )
)


# Shared horses are exact horse labels appearing under at least two raw owner
# variants within the same token-order collision group.
shared_group_horses = (
    group_horse_usage.loc[
        group_horse_usage[
            "distinct_owner_variants"
        ].gt(1)
    ]
    .sort_values(
        [
            "runner_rows",
            "distinct_owner_variants",
            "horse",
        ],
        ascending=[False, False, True],
    )
    .reset_index(drop=True)
)


# Identify which raw owner variants participate in at least one shared-horse
# relationship.
shared_horse_keys = shared_group_horses[
    [
        "token_multiset_key",
        "horse",
    ]
].drop_duplicates()

owner_variants_on_shared_horses = (
    owner_token_collision_source_rows.merge(
        shared_horse_keys.assign(
            is_shared_horse=True
        ),
        on=[
            "token_multiset_key",
            "horse",
        ],
        how="left",
        validate="many_to_one",
    )
)

owner_variants_on_shared_horses[
    "is_shared_horse"
] = owner_variants_on_shared_horses[
    "is_shared_horse"
].fillna(False)


# Build one evidence summary per token-order collision group.
owner_token_horse_continuity = (
    owner_variants_on_shared_horses.groupby(
        "token_multiset_key",
        as_index=False,
    )
    .agg(
        distinct_owner_variants=(
            "raw_owner_label",
            "nunique",
        ),
        runner_rows=("source_rowid", "size"),
        distinct_horse_labels=("horse", "nunique"),
        shared_horse_labels=(
            "horse",
            lambda values: values[
                owner_variants_on_shared_horses.loc[
                    values.index,
                    "is_shared_horse",
                ]
            ].nunique(),
        ),
        runner_rows_on_shared_horses=(
            "is_shared_horse",
            "sum",
        ),
        owner_variants_on_shared_horses=(
            "raw_owner_label",
            lambda values: values[
                owner_variants_on_shared_horses.loc[
                    values.index,
                    "is_shared_horse",
                ]
            ].nunique(),
        ),
        first_observed_date=("date", "min"),
        last_observed_date=("date", "max"),
    )
)


owner_token_horse_continuity[
    "all_owner_variants_share_horse_evidence"
] = (
    owner_token_horse_continuity[
        "owner_variants_on_shared_horses"
    ]
    == owner_token_horse_continuity[
        "distinct_owner_variants"
    ]
)

owner_token_horse_continuity[
    "shared_horse_runner_percentage"
] = (
    100
    * owner_token_horse_continuity[
        "runner_rows_on_shared_horses"
    ]
    / owner_token_horse_continuity[
        "runner_rows"
    ]
)


# Classify the strength of source-internal horse continuity.
#
# These are descriptive evidence classes, not identity decisions.
owner_token_horse_continuity[
    "horse_continuity_class"
] = owner_token_horse_continuity.apply(
    lambda row: (
        "no_shared_horse_label"
        if row["shared_horse_labels"] == 0
        else (
            "all_variants_share_horse_evidence"
            if row[
                "all_owner_variants_share_horse_evidence"
            ]
            else "some_variants_share_horse_evidence"
        )
    ),
    axis=1,
)


owner_token_horse_continuity_summary = (
    owner_token_horse_continuity.groupby(
        "horse_continuity_class",
        as_index=False,
    )
    .agg(
        collision_groups=("token_multiset_key", "size"),
        distinct_owner_variants=(
            "distinct_owner_variants",
            "sum",
        ),
        runner_rows=("runner_rows", "sum"),
        shared_horse_labels=("shared_horse_labels", "sum"),
        runner_rows_on_shared_horses=(
            "runner_rows_on_shared_horses",
            "sum",
        ),
    )
    .sort_values("horse_continuity_class")
    .reset_index(drop=True)
)


# Attach the readable raw-label combinations created in Stage 19.
owner_token_horse_continuity_display = (
    owner_token_horse_continuity.merge(
        owner_token_collision_groups[
            [
                "token_multiset_key",
                "raw_labels",
            ]
        ],
        on="token_multiset_key",
        how="left",
        validate="one_to_one",
    )
    .sort_values(
        [
            "shared_horse_labels",
            "runner_rows_on_shared_horses",
            "runner_rows",
            "raw_labels",
        ],
        ascending=[False, False, False, True],
    )
    .reset_index(drop=True)
)


# Show the exact owner variants attached to the most frequent shared horses.
shared_horse_owner_examples = (
    owner_token_collision_source_rows.merge(
        shared_horse_keys,
        on=[
            "token_multiset_key",
            "horse",
        ],
        how="inner",
        validate="many_to_many",
    )
    .groupby(
        [
            "token_multiset_key",
            "horse",
            "raw_owner_label",
        ],
        as_index=False,
    )
    .agg(
        runner_rows=("source_rowid", "size"),
        first_observed_date=("date", "min"),
        last_observed_date=("date", "max"),
    )
    .sort_values(
        [
            "token_multiset_key",
            "horse",
            "runner_rows",
            "raw_owner_label",
        ],
        ascending=[True, True, False, True],
    )
    .reset_index(drop=True)
)


# Reconcile the finite source population to the Stage 19 collision totals.
assert len(owner_token_collision_source_rows) == int(
    owner_token_order_collisions["runner_rows"].sum()
)

assert (
    owner_token_horse_continuity[
        "runner_rows"
    ].sum()
    == len(owner_token_collision_source_rows)
)

assert len(owner_token_horse_continuity) == len(
    owner_token_collision_groups
)

assert (
    owner_token_horse_continuity_summary[
        "collision_groups"
    ].sum()
    == len(owner_token_collision_groups)
)


print("Owner token-order groups by horse-continuity evidence")
display(owner_token_horse_continuity_summary)

print("Token-order groups with the strongest shared-horse evidence")
display(owner_token_horse_continuity_display.head(50))

print("Most frequent exact horse labels shared across owner variants")
display(shared_group_horses.head(50))

print("Owner variants attached to shared horse labels")
display(shared_horse_owner_examples.head(100))

Owner token-order groups by horse-continuity evidence


,horse_continuity_class,collision_groups,distinct_owner_variants,runner_rows,shared_horse_labels,runner_rows_on_shared_horses
0,all_variants_share_horse_evidence,166,334,6505,202,2011
1,no_shared_horse_label,766,1566,21696,0,0
2,some_variants_share_horse_evidence,4,17,5993,25,181


Token-order groups with the strongest shared-horse evidence


,token_multiset_key,distinct_owner_variants,runner_rows,distinct_horse_labels,shared_horse_labels,runner_rows_on_shared_horses,owner_variants_on_shared_horses,first_observed_date,last_observed_date,all_owner_variants_share_horse_evidence,shared_horse_runner_percentage,horse_continuity_class,raw_labels
0,derrick | john | magnier | michael | mrs | smi...,6,5052,786,21,137,5,2015-01-23,2026-05-25,False,2.711797,some_variants_share_horse_evidence,Mrs John Magnier Michael Tabor Derrick Smith |...
1,dineen | hughes | kerr | martin | michael,2,418,56,8,57,2,2015-04-13,2026-05-25,True,13.636364,all_variants_share_horse_evidence,Martin Hughes Michael Kerr Dineen || Michael K...
2,bloodstock | chris | giles | ltd | potensis,2,75,16,5,40,2,2015-01-03,2017-01-28,True,53.333333,all_variants_share_horse_evidence,Chris Giles Potensis Bloodstock Ltd || Potensi...
3,al | bin | khalifa | kuwari | sheail,2,146,78,5,16,2,2015-05-14,2025-12-20,True,10.958904,all_variants_share_horse_evidence,Khalifa Bin Sheail Al Kuwari || Sheail Bin Kha...
4,bryceland | family | mcneill | patrick | scott,2,58,5,3,39,2,2023-01-09,2026-05-11,True,67.241379,all_variants_share_horse_evidence,Mcneill Family Patrick Scott Bryceland || Patr...
5,finch | james | samuel | sutton,2,41,9,3,12,2,2022-09-24,2025-11-15,True,29.268293,all_variants_share_horse_evidence,Samuel Sutton James Finch || James Finch Samue...
6,bun | lee | marces | tze,2,187,12,2,73,2,2016-01-06,2026-05-24,True,39.037433,all_variants_share_horse_evidence,Lee Tze Bun Marces || Marces Lee Tze Bun
7,fyffe | fyffe | james | scott,2,121,9,2,50,2,2018-08-11,2026-03-28,True,41.322314,all_variants_share_horse_evidence,James Fyffe Scott Fyffe || Scott Fyffe James F...
8,chris | dan | giles | macdonald,2,47,2,2,47,2,2015-10-25,2020-02-22,True,100.0,all_variants_share_horse_evidence,Chris Giles Dan Macdonald || Dan Macdonald Chr...
9,john | lynch | patrick | sheridan,2,57,4,2,46,2,2023-03-26,2025-11-02,True,80.701754,all_variants_share_horse_evidence,Patrick Sheridan John Lynch || John Lynch Patr...


Most frequent exact horse labels shared across owner variants


,token_multiset_key,horse,distinct_owner_variants,runner_rows,first_observed_date,last_observed_date
0,finegan | hugh | morgan | paul | ryan,Futurum Regem (IRE),2,63,2019-10-26,2025-12-29
1,anthony | dawson | dr | f | j | mccoubrey | pa...,Magic Sea (IRE),2,45,2017-10-14,2020-11-09
2,bun | lee | marces | tze,Private Rocket (IRE),2,40,2018-06-11,2022-11-06
3,delany | diane | flanagan | lisa | mrs | mrs,Serpolette (IRE),2,38,2019-09-28,2023-12-02
4,ann | h | kennedy | michael | ms | nolan | odo...,Rebel Gold (IRE),3,36,2020-11-14,2026-03-07
5,lam | suet | wan,Turin Warrior (AUS),2,34,2022-12-21,2026-05-20
6,bun | lee | marces | tze,Captain Win (AUS),2,33,2021-10-24,2024-11-09
7,anne | browne | coffey | k | mrs | mrs,Crazyheart (IRE),2,33,2015-10-26,2021-04-18
8,al | christophe | et | herve | kambrun | louis...,Twin Boy (FR),2,32,2021-05-01,2026-01-23
9,fyffe | fyffe | james | scott,Geremia (IRE),2,31,2022-05-08,2024-08-23


Owner variants attached to shared horse labels


,token_multiset_key,horse,raw_owner_label,runner_rows,first_observed_date,last_observed_date
0,1 | 2 | lees | newcastle | newcastle | racecou...,Never Talk (AUS),Newcastle Racecourse 1 Newcastle Racecourse 2 ...,11,2021-02-13,2022-10-22
1,1 | 2 | lees | newcastle | newcastle | racecou...,Never Talk (AUS),Lees Racing Newcastle Racecourse 1 Newcastle R...,1,2023-05-13,2023-05-13
2,100 | amy | biancone | club | diamond | dunne ...,Diamond Wow (USA),Diamond 100 Racing Club Amy Dunne Patrick L Bi...,2,2021-10-13,2022-02-05
3,100 | amy | biancone | club | diamond | dunne ...,Diamond Wow (USA),Diamond 100 Racing Club Llc Amy Dunne Patrick ...,1,2022-12-31,2022-12-31
4,a | a | b | m | n | newsom | p | wales | white...,The Frontman (NZ),P B Newsom A N Wales A M Whitehouse,6,2021-03-07,2022-09-16
...,...,...,...,...,...,...
95,andrew | friel | gemmell | thomas,Discorama (FR),Thomas Friel Andrew Gemmell,10,2017-03-25,2021-10-23
96,ann | h | kennedy | michael | ms | nolan | odo...,Rebel Gold (IRE),T H Stanley Michael Odowd Ms Ann P Nolan R Ken...,16,2020-11-14,2026-03-07
97,ann | h | kennedy | michael | ms | nolan | odo...,Rebel Gold (IRE),Michael Odowd T H Stanley Ms Ann P Nolan R Ken...,13,2022-02-06,2024-12-21
98,ann | h | kennedy | michael | ms | nolan | odo...,Rebel Gold (IRE),R Kennedy Michael Odowd T H Stanley Ms Ann P N...,7,2025-01-11,2025-11-08


## Stage 21 — Temporal sequencing of reordered owner labels on shared horses

Stage 20 found 170 token-order collision groups with at least one exact horse label appearing under multiple owner variants.

Shared-horse evidence strengthens the possibility that reordered labels are source-presentation variants.

However, the temporal relationship still matters.

For each shared exact horse label, this stage examines whether the participating owner variants are:

- `non_overlapping_sequence` — one variant ends before the next begins;
- `same_date_handoff_or_overlap` — different variants occur on at least one common date;
- `interleaved_periods` — variant date ranges overlap without sharing an exact date;
- `complex_multi_variant_sequence` — three or more variants require a broader sequence description.

These categories describe source timing only.

They do not prove:

- that ownership legally remained unchanged;
- that a later label corrected an earlier label;
- that same-date variants are duplicates;
- that non-overlapping variants are equivalent;
- that exact horse labels identify one real horse globally;
- that one variant should replace another.

The purpose is to distinguish strong formatting-transition candidates from ambiguous or potentially substantive ownership changes.

In [25]:
# Build one exact owner-variant period for each shared horse.
#
# Multiple runner rows under the same owner label are condensed to their
# observed date range for that exact horse label.
shared_horse_variant_periods = (
    owner_token_collision_source_rows.merge(
        shared_horse_keys,
        on=[
            "token_multiset_key",
            "horse",
        ],
        how="inner",
        validate="many_to_many",
    )
    .groupby(
        [
            "token_multiset_key",
            "horse",
            "raw_owner_label",
        ],
        as_index=False,
    )
    .agg(
        runner_rows=("source_rowid", "size"),
        first_observed_date=("date", "min"),
        last_observed_date=("date", "max"),
        observed_dates=("date", "nunique"),
    )
)


# Convert date strings to pandas timestamps for interval comparison.
shared_horse_variant_periods[
    "first_observed_date"
] = pd.to_datetime(
    shared_horse_variant_periods[
        "first_observed_date"
    ]
)

shared_horse_variant_periods[
    "last_observed_date"
] = pd.to_datetime(
    shared_horse_variant_periods[
        "last_observed_date"
    ]
)


def classify_shared_horse_variant_timing(
    group: pd.DataFrame,
) -> pd.Series:
    """
    Classify temporal relationships among owner variants for one exact horse.

    Grouping keys are read from group.name because pandas removes them from the
    group DataFrame when include_groups=False is used.
    """
    token_multiset_key, horse = group.name

    ordered = group.sort_values(
        [
            "first_observed_date",
            "last_observed_date",
            "raw_owner_label",
        ]
    ).reset_index(drop=True)

    variant_count = len(ordered)

    # Retain groups with three or more owner variants as complex cases rather
    # than forcing several pairwise relationships into one simplified label.
    if variant_count > 2:
        has_any_range_overlap = False

        for first_index in range(variant_count):
            for second_index in range(
                first_index + 1,
                variant_count,
            ):
                first_row = ordered.iloc[first_index]
                second_row = ordered.iloc[second_index]

                if (
                    first_row["first_observed_date"]
                    <= second_row["last_observed_date"]
                    and second_row["first_observed_date"]
                    <= first_row["last_observed_date"]
                ):
                    has_any_range_overlap = True

        return pd.Series(
            {
                "variant_count": variant_count,
                "timing_class": (
                    "complex_multi_variant_overlap"
                    if has_any_range_overlap
                    else "complex_multi_variant_sequence"
                ),
                "first_variant": ordered.iloc[0][
                    "raw_owner_label"
                ],
                "last_variant": ordered.iloc[-1][
                    "raw_owner_label"
                ],
                "earliest_observed_date": ordered[
                    "first_observed_date"
                ].min(),
                "latest_observed_date": ordered[
                    "last_observed_date"
                ].max(),
            }
        )

    if variant_count != 2:
        raise AssertionError(
            "Expected at least two owner variants for shared horse "
            f"{horse!r}; found {variant_count}."
        )

    first = ordered.iloc[0]
    second = ordered.iloc[1]

    # Retrieve exact observed dates for both variants from the uncondensed
    # source rows. This distinguishes shared dates from date ranges that merely
    # overlap.
    first_dates = set(
        owner_token_collision_source_rows.loc[
            (
                owner_token_collision_source_rows[
                    "token_multiset_key"
                ]
                == token_multiset_key
            )
            & (
                owner_token_collision_source_rows["horse"]
                == horse
            )
            & (
                owner_token_collision_source_rows[
                    "raw_owner_label"
                ]
                == first["raw_owner_label"]
            ),
            "date",
        ]
    )

    second_dates = set(
        owner_token_collision_source_rows.loc[
            (
                owner_token_collision_source_rows[
                    "token_multiset_key"
                ]
                == token_multiset_key
            )
            & (
                owner_token_collision_source_rows["horse"]
                == horse
            )
            & (
                owner_token_collision_source_rows[
                    "raw_owner_label"
                ]
                == second["raw_owner_label"]
            ),
            "date",
        ]
    )

    shared_dates = first_dates & second_dates

    if shared_dates:
        timing_class = "same_date_handoff_or_overlap"

    elif (
        first["last_observed_date"]
        < second["first_observed_date"]
    ):
        timing_class = "non_overlapping_sequence"

    else:
        timing_class = "interleaved_periods"

    return pd.Series(
        {
            "variant_count": variant_count,
            "timing_class": timing_class,
            "first_variant": first["raw_owner_label"],
            "last_variant": second["raw_owner_label"],
            "earliest_observed_date": min(
                first["first_observed_date"],
                second["first_observed_date"],
            ),
            "latest_observed_date": max(
                first["last_observed_date"],
                second["last_observed_date"],
            ),
        }
    )


# Apply the timing classification independently to each shared exact horse
# within each token-order collision group.
shared_horse_timing_profile = (
    shared_horse_variant_periods.groupby(
        [
            "token_multiset_key",
            "horse",
        ],
        as_index=False,
    )
    .apply(
        classify_shared_horse_variant_timing,
        include_groups=False,
    )
    .reset_index(drop=True)
)


# Attach source-volume measures for interpretation.
shared_horse_timing_profile = (
    shared_horse_timing_profile.merge(
        shared_group_horses[
            [
                "token_multiset_key",
                "horse",
                "distinct_owner_variants",
                "runner_rows",
                "first_observed_date",
                "last_observed_date",
            ]
        ].rename(
            columns={
                "first_observed_date":
                    "shared_horse_first_observed_date",
                "last_observed_date":
                    "shared_horse_last_observed_date",
            }
        ),
        on=[
            "token_multiset_key",
            "horse",
        ],
        how="left",
        validate="one_to_one",
    )
)


# Summarise the timing classes.
shared_horse_timing_summary = (
    shared_horse_timing_profile.groupby(
        "timing_class",
        as_index=False,
    )
    .agg(
        shared_horse_cases=("horse", "size"),
        token_order_groups=(
            "token_multiset_key",
            "nunique",
        ),
        runner_rows=("runner_rows", "sum"),
    )
    .sort_values("timing_class")
    .reset_index(drop=True)
)


# Build a readable owner-variant sequence for each shared horse.
shared_horse_variant_sequence = (
    shared_horse_variant_periods.sort_values(
        [
            "token_multiset_key",
            "horse",
            "first_observed_date",
            "last_observed_date",
            "raw_owner_label",
        ]
    )
    .assign(
        variant_period=lambda frame: (
            frame["raw_owner_label"]
            + " ["
            + frame["first_observed_date"].dt.strftime(
                "%Y-%m-%d"
            )
            + " to "
            + frame["last_observed_date"].dt.strftime(
                "%Y-%m-%d"
            )
            + "; "
            + frame["runner_rows"].astype(str)
            + " rows]"
        )
    )
    .groupby(
        [
            "token_multiset_key",
            "horse",
        ],
        as_index=False,
    )
    .agg(
        variant_sequence=(
            "variant_period",
            " || ".join,
        )
    )
)


shared_horse_timing_display = (
    shared_horse_timing_profile.merge(
        shared_horse_variant_sequence,
        on=[
            "token_multiset_key",
            "horse",
        ],
        how="left",
        validate="one_to_one",
    )
    .sort_values(
        [
            "runner_rows",
            "variant_count",
            "horse",
        ],
        ascending=[False, False, True],
    )
    .reset_index(drop=True)
)


# Reconcile the timing population to every exact horse label already identified
# as shared across owner variants.
assert len(shared_horse_timing_profile) == len(
    shared_group_horses
)

assert (
    shared_horse_timing_summary[
        "shared_horse_cases"
    ].sum()
    == len(shared_group_horses)
)


print("Shared-horse owner-variant timing classes")
display(shared_horse_timing_summary)

print("Most frequent shared horses and owner-variant sequences")
display(shared_horse_timing_display.head(75))

for timing_class in sorted(
    shared_horse_timing_profile[
        "timing_class"
    ].unique()
):
    print(f"Examples: {timing_class}")

    display(
        shared_horse_timing_display.loc[
            shared_horse_timing_display[
                "timing_class"
            ].eq(timing_class)
        ]
        .head(30)
        .reset_index(drop=True)
    )

Shared-horse owner-variant timing classes


,timing_class,shared_horse_cases,token_order_groups,runner_rows
0,complex_multi_variant_overlap,1,1,36
1,complex_multi_variant_sequence,1,1,12
2,interleaved_periods,28,20,319
3,non_overlapping_sequence,197,154,1825


Most frequent shared horses and owner-variant sequences


,token_multiset_key,horse,variant_count,timing_class,first_variant,last_variant,earliest_observed_date,latest_observed_date,distinct_owner_variants,runner_rows,shared_horse_first_observed_date,shared_horse_last_observed_date,variant_sequence
0,finegan | hugh | morgan | paul | ryan,Futurum Regem (IRE),2,non_overlapping_sequence,Hugh Paul Finegan Morgan Ryan,Morgan Ryan Hugh Paul Finegan,2019-10-26,2025-12-29,2,63,2019-10-26,2025-12-29,Hugh Paul Finegan Morgan Ryan [2019-10-26 to 2...
1,anthony | dawson | dr | f | j | mccoubrey | pa...,Magic Sea (IRE),2,non_overlapping_sequence,Dr J F Dawson Anthony Peter Passmore R Mccoubrey,Anthony Peter Passmore Dr J F Dawson R Mccoubrey,2017-10-14,2020-11-09,2,45,2017-10-14,2020-11-09,Dr J F Dawson Anthony Peter Passmore R Mccoubr...
2,bun | lee | marces | tze,Private Rocket (IRE),2,non_overlapping_sequence,Lee Tze Bun Marces,Marces Lee Tze Bun,2018-06-11,2022-11-06,2,40,2018-06-11,2022-11-06,Lee Tze Bun Marces [2018-06-11 to 2022-07-10; ...
3,delany | diane | flanagan | lisa | mrs | mrs,Serpolette (IRE),2,non_overlapping_sequence,Mrs Lisa Delany Mrs Diane Flanagan,Mrs Diane Flanagan Mrs Lisa Delany,2019-09-28,2023-12-02,2,38,2019-09-28,2023-12-02,Mrs Lisa Delany Mrs Diane Flanagan [2019-09-28...
4,ann | h | kennedy | michael | ms | nolan | odo...,Rebel Gold (IRE),3,complex_multi_variant_overlap,T H Stanley Michael Odowd Ms Ann P Nolan R Ken...,R Kennedy Michael Odowd T H Stanley Ms Ann P N...,2020-11-14,2026-03-07,3,36,2020-11-14,2026-03-07,T H Stanley Michael Odowd Ms Ann P Nolan R Ken...
...,...,...,...,...,...,...,...,...,...,...,...,...,...
70,d | j | m | mchale | munroe | walshe,Elizabeths Legacy (GB),2,non_overlapping_sequence,M Mchale J Munroe D Walshe,J Munroe D Walshe M Mchale,2024-08-25,2025-10-27,2,10,2024-08-25,2025-10-27,M Mchale J Munroe D Walshe [2024-08-25 to 2024...
71,derrick | john | magnier | michael | mrs | smi...,First Approach (IRE),2,non_overlapping_sequence,Michael Tabor Derrick Smith Mrs John Magnier,Mrs John Magnier Michael Tabor Derrick Smith,2025-04-11,2025-10-11,2,10,2025-04-11,2025-10-11,Michael Tabor Derrick Smith Mrs John Magnier [...
72,hung | kei | kin | ltd | qatar | racing,Furious (GB),2,non_overlapping_sequence,Kin Hung Kei Qatar Racing Ltd,Qatar Racing Ltd Kin Hung Kei,2018-08-02,2020-02-22,2,10,2018-08-02,2020-02-22,Kin Hung Kei Qatar Racing Ltd [2018-08-02 to 2...
73,dineen | hughes | kerr | martin | michael,Great Bedwyn (GB),2,non_overlapping_sequence,Martin Hughes Michael Kerr Dineen,Michael Kerr Dineen Martin Hughes,2024-11-12,2025-09-21,2,10,2024-11-12,2025-09-21,Martin Hughes Michael Kerr Dineen [2024-11-12 ...


Examples: complex_multi_variant_overlap


...

,token_multiset_key,horse,variant_count,timing_class,first_variant,last_variant,earliest_observed_date,latest_observed_date,distinct_owner_variants,runner_rows,shared_horse_first_observed_date,shared_horse_last_observed_date,variant_sequence
0,barber | c | d | fogg | i | macdonald | r | webb,Antartica De Thaix (FR),3,complex_multi_variant_sequence,I Fogg R Webb D Macdonald C Barber,I Fogg C Barber D Macdonald R Webb,2015-02-06,2017-03-05,3,12,2015-02-06,2017-03-05,I Fogg R Webb D Macdonald C Barber [2015-02-06...


Examples: interleaved_periods


,token_multiset_key,horse,variant_count,timing_class,first_variant,last_variant,earliest_observed_date,latest_observed_date,distinct_owner_variants,runner_rows,shared_horse_first_observed_date,shared_horse_last_observed_date,variant_sequence
0,chris | dan | giles | macdonald,Romain De Senam (FR),2,interleaved_periods,Chris Giles Dan Macdonald,Dan Macdonald Chris Giles,2015-10-25,2020-02-22,2,30,2015-10-25,2020-02-22,Chris Giles Dan Macdonald [2015-10-25 to 2020-...
1,andrew | friel | gemmell | thomas,Discorama (FR),2,interleaved_periods,Thomas Friel Andrew Gemmell,Andrew Gemmell Thomas Friel,2017-03-25,2022-04-09,2,22,2017-03-25,2022-04-09,Thomas Friel Andrew Gemmell [2017-03-25 to 202...
2,dineen | hughes | kerr | martin | michael,Ouzo (GB),2,interleaved_periods,Michael Kerr Dineen Martin Hughes,Martin Hughes Michael Kerr Dineen,2018-09-21,2021-10-08,2,19,2018-09-21,2021-10-08,Michael Kerr Dineen Martin Hughes [2018-09-21 ...
3,d | j | m | magnier | mrs | smith | tabor | we...,Point Lonsdale (IRE),2,interleaved_periods,M Tabor D Smith Mrs J Magnier Westerberg,D Smith Mrs J Magnier M Tabor Westerberg,2021-06-02,2024-11-15,2,19,2021-06-02,2024-11-15,M Tabor D Smith Mrs J Magnier Westerberg [2021...
4,bryceland | family | mcneill | patrick | scott,Three Card Brag (IRE),2,interleaved_periods,Mcneill Family Patrick Scott Bryceland,Patrick Scott Bryceland Mcneill Family,2023-01-25,2026-05-11,2,18,2023-01-25,2026-05-11,Mcneill Family Patrick Scott Bryceland [2023-0...
5,chris | dan | giles | macdonald,Connetable (FR),2,interleaved_periods,Chris Giles Dan Macdonald,Dan Macdonald Chris Giles,2016-03-18,2018-04-14,2,17,2016-03-18,2018-04-14,Chris Giles Dan Macdonald [2016-03-18 to 2018-...
6,bryceland | family | mcneill | patrick | scott,Where It All Began (IRE),2,interleaved_periods,Mcneill Family Patrick Scott Bryceland,Patrick Scott Bryceland Mcneill Family,2023-02-18,2026-01-24,2,17,2023-02-18,2026-01-24,Mcneill Family Patrick Scott Bryceland [2023-0...
7,chris | colm | donlon | giles,Le Mercurey (FR),2,interleaved_periods,Chris Giles Colm Donlon,Colm Donlon Chris Giles,2015-01-17,2017-03-04,2,14,2015-01-17,2017-03-04,Chris Giles Colm Donlon [2015-01-17 to 2017-03...
8,charlie | fry | phil | walker,Oneupmanship (IRE),2,interleaved_periods,Charlie Walker Phil Fry,Phil Fry Charlie Walker,2019-11-16,2023-05-17,2,14,2019-11-16,2023-05-17,Charlie Walker Phil Fry [2019-11-16 to 2023-05...
9,head | lapenta | of | plains | robert | southe...,Whitmore (USA),2,interleaved_periods,Southern Springs Stables Robert V Lapenta Head...,Robert V Lapenta Southern Springs Stables Head...,2017-05-20,2021-04-11,2,14,2017-05-20,2021-04-11,Southern Springs Stables Robert V Lapenta Head...


Examples: non_overlapping_sequence


,token_multiset_key,horse,variant_count,timing_class,first_variant,last_variant,earliest_observed_date,latest_observed_date,distinct_owner_variants,runner_rows,shared_horse_first_observed_date,shared_horse_last_observed_date,variant_sequence
0,finegan | hugh | morgan | paul | ryan,Futurum Regem (IRE),2,non_overlapping_sequence,Hugh Paul Finegan Morgan Ryan,Morgan Ryan Hugh Paul Finegan,2019-10-26,2025-12-29,2,63,2019-10-26,2025-12-29,Hugh Paul Finegan Morgan Ryan [2019-10-26 to 2...
1,anthony | dawson | dr | f | j | mccoubrey | pa...,Magic Sea (IRE),2,non_overlapping_sequence,Dr J F Dawson Anthony Peter Passmore R Mccoubrey,Anthony Peter Passmore Dr J F Dawson R Mccoubrey,2017-10-14,2020-11-09,2,45,2017-10-14,2020-11-09,Dr J F Dawson Anthony Peter Passmore R Mccoubr...
2,bun | lee | marces | tze,Private Rocket (IRE),2,non_overlapping_sequence,Lee Tze Bun Marces,Marces Lee Tze Bun,2018-06-11,2022-11-06,2,40,2018-06-11,2022-11-06,Lee Tze Bun Marces [2018-06-11 to 2022-07-10; ...
3,delany | diane | flanagan | lisa | mrs | mrs,Serpolette (IRE),2,non_overlapping_sequence,Mrs Lisa Delany Mrs Diane Flanagan,Mrs Diane Flanagan Mrs Lisa Delany,2019-09-28,2023-12-02,2,38,2019-09-28,2023-12-02,Mrs Lisa Delany Mrs Diane Flanagan [2019-09-28...
4,lam | suet | wan,Turin Warrior (AUS),2,non_overlapping_sequence,Suet Wan Lam,Lam Suet Wan,2022-12-21,2026-05-20,2,34,2022-12-21,2026-05-20,Suet Wan Lam [2022-12-21 to 2025-04-30; 23 row...
5,bun | lee | marces | tze,Captain Win (AUS),2,non_overlapping_sequence,Lee Tze Bun Marces,Marces Lee Tze Bun,2021-10-24,2024-11-09,2,33,2021-10-24,2024-11-09,Lee Tze Bun Marces [2021-10-24 to 2022-07-10; ...
6,anne | browne | coffey | k | mrs | mrs,Crazyheart (IRE),2,non_overlapping_sequence,Mrs Anne Coffey Mrs K Browne,Mrs K Browne Mrs Anne Coffey,2015-10-26,2021-04-18,2,33,2015-10-26,2021-04-18,Mrs Anne Coffey Mrs K Browne [2015-10-26 to 20...
7,al | christophe | et | herve | kambrun | louis...,Twin Boy (FR),2,non_overlapping_sequence,Louis Rene Kambrun Herve Christophe Plisson Et Al,Christophe Plisson Louis Rene Kambrun Herve Et Al,2021-05-01,2026-01-23,2,32,2021-05-01,2026-01-23,Louis Rene Kambrun Herve Christophe Plisson Et...
8,fyffe | fyffe | james | scott,Geremia (IRE),2,non_overlapping_sequence,James Fyffe Scott Fyffe,Scott Fyffe James Fyffe,2022-05-08,2024-08-23,2,31,2022-05-08,2024-08-23,James Fyffe Scott Fyffe [2022-05-08 to 2022-11...
9,john | lynch | patrick | sheridan,Mickey The Steel (GB),2,non_overlapping_sequence,John Lynch Patrick Sheridan,Patrick Sheridan John Lynch,2023-03-26,2025-11-02,2,28,2023-03-26,2025-11-02,John Lynch Patrick Sheridan [2023-03-26 to 202...


## Stage 22 — Switching patterns between reordered owner labels

Shared-horse timing found:

- 197 non-overlapping owner-variant sequences;
- 28 interleaved periods;
- two cases involving three variants;
- no exact same-date use of different reordered labels.

A non-overlapping date range does not necessarily mean a single permanent transition.

For example, a horse’s source history might use:

- variant A;
- then variant B;
- then variant A again.

That pattern would show that the labels are alternative source presentations rather than a straightforward one-way correction.

This stage orders every runner appearance for each shared horse and measures:

- the number of owner-label changes;
- the number of distinct owner variants;
- whether an earlier variant returns after another variant appears;
- the first and last observed owner label;
- the complete sequence of consecutive label runs.

The resulting classes are:

- `single_transition` — exactly one change between two variants;
- `repeated_switching` — two or more changes between variants;
- `multi_variant_sequence` — three or more owner variants;
- `unexpected_no_change` — retained only as a validation failure category.

These patterns remain source-presentation evidence.

They do not establish legal ownership continuity or justify replacing raw owner labels.

In [26]:
# Retain one owner assertion per exact horse, date and raw owner label.
#
# Multiple source rows on the same date with the same label do not represent
# additional label transitions and are therefore condensed before sequencing.
shared_horse_owner_dates = (
    owner_token_collision_source_rows.merge(
        shared_horse_keys,
        on=[
            "token_multiset_key",
            "horse",
        ],
        how="inner",
        validate="many_to_many",
    )
    [
        [
            "token_multiset_key",
            "horse",
            "date",
            "raw_owner_label",
        ]
    ]
    .drop_duplicates()
    .copy()
)


shared_horse_owner_dates["date"] = pd.to_datetime(
    shared_horse_owner_dates["date"]
)


# Confirm the Stage 21 result that no shared horse has different reordered
# owner variants recorded on the same source date.
same_date_variant_counts = (
    shared_horse_owner_dates.groupby(
        [
            "token_multiset_key",
            "horse",
            "date",
        ],
        as_index=False,
    )
    .agg(
        distinct_owner_variants=(
            "raw_owner_label",
            "nunique",
        )
    )
)

assert (
    same_date_variant_counts[
        "distinct_owner_variants"
    ].le(1).all()
)


def build_owner_switching_profile(
    group: pd.DataFrame,
) -> pd.Series:
    """
    Describe consecutive owner-label runs for one exact shared horse.

    The result records source presentation changes only. It does not infer a
    change in legal or beneficial ownership.
    """
    ordered = group.sort_values(
        [
            "date",
            "raw_owner_label",
        ]
    ).reset_index(drop=True)

    labels = ordered["raw_owner_label"].tolist()

    # Start the first consecutive run.
    runs = [
        {
            "raw_owner_label": labels[0],
            "first_date": ordered.iloc[0]["date"],
            "last_date": ordered.iloc[0]["date"],
            "observed_dates": 1,
        }
    ]

    # Extend the current run when the label is unchanged. Otherwise start a
    # new run and therefore record one source-label switch.
    for row in ordered.iloc[1:].itertuples(index=False):
        current_run = runs[-1]

        if row.raw_owner_label == current_run["raw_owner_label"]:
            current_run["last_date"] = row.date
            current_run["observed_dates"] += 1

        else:
            runs.append(
                {
                    "raw_owner_label": row.raw_owner_label,
                    "first_date": row.date,
                    "last_date": row.date,
                    "observed_dates": 1,
                }
            )

    distinct_variants = ordered[
        "raw_owner_label"
    ].nunique()

    switch_count = len(runs) - 1

    # A variant returns when the sequence of runs contains a label already
    # observed in an earlier, non-adjacent run.
    run_labels = [
        run["raw_owner_label"]
        for run in runs
    ]

    variant_returned = (
        len(run_labels)
        > len(set(run_labels))
    )

    if distinct_variants >= 3:
        switching_class = "multi_variant_sequence"

    elif switch_count == 1:
        switching_class = "single_transition"

    elif switch_count >= 2:
        switching_class = "repeated_switching"

    else:
        switching_class = "unexpected_no_change"

    run_sequence = " || ".join(
        (
            f"{run['raw_owner_label']} "
            f"[{run['first_date'].strftime('%Y-%m-%d')} to "
            f"{run['last_date'].strftime('%Y-%m-%d')}; "
            f"{run['observed_dates']} dates]"
        )
        for run in runs
    )

    return pd.Series(
        {
            "distinct_owner_variants": distinct_variants,
            "observed_dates": len(ordered),
            "consecutive_label_runs": len(runs),
            "owner_label_switches": switch_count,
            "variant_returned": variant_returned,
            "first_owner_variant": run_labels[0],
            "last_owner_variant": run_labels[-1],
            "switching_class": switching_class,
            "run_sequence": run_sequence,
        }
    )


# Apply the switching analysis independently to every exact horse label shared
# by reordered owner variants.
shared_horse_switching_profile = (
    shared_horse_owner_dates.groupby(
        [
            "token_multiset_key",
            "horse",
        ],
        as_index=False,
    )
    .apply(
        build_owner_switching_profile,
        include_groups=False,
    )
    .reset_index(drop=True)
)


# Attach the source-volume and timing evidence already established.
shared_horse_switching_profile = (
    shared_horse_switching_profile.merge(
        shared_horse_timing_profile[
            [
                "token_multiset_key",
                "horse",
                "timing_class",
                "runner_rows",
            ]
        ],
        on=[
            "token_multiset_key",
            "horse",
        ],
        how="left",
        validate="one_to_one",
    )
)


# Summarise the switching classes.
shared_horse_switching_summary = (
    shared_horse_switching_profile.groupby(
        "switching_class",
        as_index=False,
    )
    .agg(
        shared_horse_cases=("horse", "size"),
        token_order_groups=(
            "token_multiset_key",
            "nunique",
        ),
        runner_rows=("runner_rows", "sum"),
        cases_with_returning_variant=(
            "variant_returned",
            "sum",
        ),
        maximum_owner_label_switches=(
            "owner_label_switches",
            "max",
        ),
    )
    .sort_values("switching_class")
    .reset_index(drop=True)
)


# Reconcile to all 227 shared-horse cases from Stage 21.
assert len(shared_horse_switching_profile) == len(
    shared_horse_timing_profile
)

assert (
    shared_horse_switching_summary[
        "shared_horse_cases"
    ].sum()
    == len(shared_horse_timing_profile)
)

assert (
    shared_horse_switching_profile[
        "owner_label_switches"
    ].ge(1).all()
)


print("Reordered owner-label switching patterns")
display(shared_horse_switching_summary)

print("Shared horses with the most owner-label switches")
display(
    shared_horse_switching_profile.sort_values(
        [
            "owner_label_switches",
            "observed_dates",
            "runner_rows",
            "horse",
        ],
        ascending=[False, False, False, True],
    )
    .head(75)
    .reset_index(drop=True)
)

print("Cases where an earlier owner variant later returns")
display(
    shared_horse_switching_profile.loc[
        shared_horse_switching_profile[
            "variant_returned"
        ]
    ]
    .sort_values(
        [
            "owner_label_switches",
            "observed_dates",
            "runner_rows",
        ],
        ascending=[False, False, False],
    )
    .reset_index(drop=True)
)

Reordered owner-label switching patterns


,switching_class,shared_horse_cases,token_order_groups,runner_rows,cases_with_returning_variant,maximum_owner_label_switches
0,multi_variant_sequence,2,2,48,1,5
1,repeated_switching,28,20,319,28,5
2,single_transition,197,154,1825,0,1


Shared horses with the most owner-label switches


,token_multiset_key,horse,distinct_owner_variants,observed_dates,consecutive_label_runs,owner_label_switches,variant_returned,first_owner_variant,last_owner_variant,switching_class,run_sequence,timing_class,runner_rows
0,ann | h | kennedy | michael | ms | nolan | odo...,Rebel Gold (IRE),3,36,6,5,True,T H Stanley Michael Odowd Ms Ann P Nolan R Ken...,T H Stanley Michael Odowd Ms Ann P Nolan R Ken...,multi_variant_sequence,T H Stanley Michael Odowd Ms Ann P Nolan R Ken...,complex_multi_variant_overlap,36
1,andrew | friel | gemmell | thomas,Discorama (FR),2,22,6,5,True,Thomas Friel Andrew Gemmell,Andrew Gemmell Thomas Friel,repeated_switching,Thomas Friel Andrew Gemmell [2017-03-25 to 201...,interleaved_periods,22
2,derrick | john | magnier | michael | mrs | smi...,Legatissimo (IRE),2,8,5,4,True,Michael Tabor Mrs John Magnier Derrick Smith,Michael Tabor Mrs John Magnier Derrick Smith,repeated_switching,Michael Tabor Mrs John Magnier Derrick Smith [...,interleaved_periods,8
3,alan | ann | partnership | potts,Sizing Coal (IRE),2,8,5,4,True,Ann Alan Potts Partnership,Ann Alan Potts Partnership,repeated_switching,Ann Alan Potts Partnership [2015-05-02 to 2016...,interleaved_periods,8
4,d | j | m | magnier | mrs | smith | tabor | we...,Point Lonsdale (IRE),2,19,4,3,True,M Tabor D Smith Mrs J Magnier Westerberg,D Smith Mrs J Magnier M Tabor Westerberg,repeated_switching,M Tabor D Smith Mrs J Magnier Westerberg [2021...,interleaved_periods,19
...,...,...,...,...,...,...,...,...,...,...,...,...,...
70,b | davis | hoggarth | hull | s | s,La Rav (IRE),2,12,2,1,False,B Hoggarth S Davis S Hull,B Hoggarth S Hull S Davis,single_transition,B Hoggarth S Davis S Hull [2019-08-30 to 2019-...,non_overlapping_sequence,12
71,1 | 2 | lees | newcastle | newcastle | racecou...,Never Talk (AUS),2,12,2,1,False,Newcastle Racecourse 1 Newcastle Racecourse 2 ...,Lees Racing Newcastle Racecourse 1 Newcastle R...,single_transition,Newcastle Racecourse 1 Newcastle Racecourse 2 ...,non_overlapping_sequence,12
72,bloodstock | farms | j | lincoln | ltd | mcali...,Platinum Invador (NZ),2,12,2,1,False,N J Mcalister Lincoln Farms Bloodstock Ltd,Lincoln Farms Bloodstock Ltd N J Mcalister,single_transition,N J Mcalister Lincoln Farms Bloodstock Ltd [20...,non_overlapping_sequence,12
73,d | j | m | magnier | mrs | smith | tabor | we...,Democracy (IRE),2,11,2,1,False,D Smith Mrs J Magnier M Tabor Westerberg,M Tabor D Smith Mrs J Magnier Westerberg,single_transition,D Smith Mrs J Magnier M Tabor Westerberg [2023...,non_overlapping_sequence,11


Cases where an earlier owner variant later returns


,token_multiset_key,horse,distinct_owner_variants,observed_dates,consecutive_label_runs,owner_label_switches,variant_returned,first_owner_variant,last_owner_variant,switching_class,run_sequence,timing_class,runner_rows
0,ann | h | kennedy | michael | ms | nolan | odo...,Rebel Gold (IRE),3,36,6,5,True,T H Stanley Michael Odowd Ms Ann P Nolan R Ken...,T H Stanley Michael Odowd Ms Ann P Nolan R Ken...,multi_variant_sequence,T H Stanley Michael Odowd Ms Ann P Nolan R Ken...,complex_multi_variant_overlap,36
1,andrew | friel | gemmell | thomas,Discorama (FR),2,22,6,5,True,Thomas Friel Andrew Gemmell,Andrew Gemmell Thomas Friel,repeated_switching,Thomas Friel Andrew Gemmell [2017-03-25 to 201...,interleaved_periods,22
2,alan | ann | partnership | potts,Sizing Coal (IRE),2,8,5,4,True,Ann Alan Potts Partnership,Ann Alan Potts Partnership,repeated_switching,Ann Alan Potts Partnership [2015-05-02 to 2016...,interleaved_periods,8
3,derrick | john | magnier | michael | mrs | smi...,Legatissimo (IRE),2,8,5,4,True,Michael Tabor Mrs John Magnier Derrick Smith,Michael Tabor Mrs John Magnier Derrick Smith,repeated_switching,Michael Tabor Mrs John Magnier Derrick Smith [...,interleaved_periods,8
4,d | j | m | magnier | mrs | smith | tabor | we...,Point Lonsdale (IRE),2,19,4,3,True,M Tabor D Smith Mrs J Magnier Westerberg,D Smith Mrs J Magnier M Tabor Westerberg,repeated_switching,M Tabor D Smith Mrs J Magnier Westerberg [2021...,interleaved_periods,19
5,bryceland | family | mcneill | patrick | scott,Where It All Began (IRE),2,17,4,3,True,Mcneill Family Patrick Scott Bryceland,Patrick Scott Bryceland Mcneill Family,repeated_switching,Mcneill Family Patrick Scott Bryceland [2023-0...,interleaved_periods,17
6,chris | dan | giles | macdonald,Romain De Senam (FR),2,30,3,2,True,Chris Giles Dan Macdonald,Chris Giles Dan Macdonald,repeated_switching,Chris Giles Dan Macdonald [2015-10-25 to 2016-...,interleaved_periods,30
7,dineen | hughes | kerr | martin | michael,Ouzo (GB),2,19,3,2,True,Michael Kerr Dineen Martin Hughes,Michael Kerr Dineen Martin Hughes,repeated_switching,Michael Kerr Dineen Martin Hughes [2018-09-21 ...,interleaved_periods,19
8,bryceland | family | mcneill | patrick | scott,Three Card Brag (IRE),2,18,3,2,True,Mcneill Family Patrick Scott Bryceland,Mcneill Family Patrick Scott Bryceland,repeated_switching,Mcneill Family Patrick Scott Bryceland [2023-0...,interleaved_periods,18
9,chris | dan | giles | macdonald,Connetable (FR),2,17,3,2,True,Chris Giles Dan Macdonald,Chris Giles Dan Macdonald,repeated_switching,Chris Giles Dan Macdonald [2016-03-18 to 2016-...,interleaved_periods,17


## Stage 23 — Ampersand structure in trainer labels

Trainer punctuation profiling found 305 exact labels containing an ampersand, covering 53,656 runner rows.

Many appear to represent joint trainers, but punctuation alone cannot establish that interpretation.

This stage asks:

> When an ampersand-bearing trainer label is divided around `&`, do its component strings also occur as standalone exact trainer labels?

For each populated trainer label containing `&`, the analysis records:

- the complete raw trainer label;
- the number of ampersands;
- trimmed left and right component text for labels containing exactly one ampersand;
- whether each component also occurs as an exact standalone trainer label;
- the runner-row history of any standalone matches;
- whether both, one or neither component is independently observed.

Component overlap is descriptive evidence only.

It does not establish that:

- the ampersand always denotes joint trainers;
- standalone and joint-role strings identify the same people;
- the components are correctly segmented personal names;
- a joint trainer label should be decomposed in the database;
- independently observed components justify entity creation;
- labels containing multiple ampersands have a stable structure.

The complete raw trainer assertion remains the governed source value.

In [27]:
# Select exact populated trainer labels containing at least one ampersand.
trainer_ampersand_labels = (
    trainer_labels.loc[
        trainer_labels["raw_label"].str.contains(
            "&",
            regex=False,
        )
    ]
    .copy()
    .reset_index(drop=True)
)


# Count ampersands before attempting any bounded component comparison.
trainer_ampersand_labels["ampersand_count"] = (
    trainer_ampersand_labels["raw_label"]
    .str.count("&")
)


trainer_ampersand_count_summary = (
    trainer_ampersand_labels.groupby(
        "ampersand_count",
        as_index=False,
    )
    .agg(
        distinct_raw_labels=("raw_label", "size"),
        runner_rows=("runner_rows", "sum"),
    )
    .sort_values("ampersand_count")
    .reset_index(drop=True)
)


# Only labels with exactly one ampersand are eligible for a simple left/right
# structural comparison.
single_ampersand_trainers = (
    trainer_ampersand_labels.loc[
        trainer_ampersand_labels[
            "ampersand_count"
        ].eq(1)
    ]
    .copy()
)


single_ampersand_trainers[
    [
        "left_component",
        "right_component",
    ]
] = (
    single_ampersand_trainers["raw_label"]
    .str.split(
        "&",
        n=1,
        expand=True,
    )
)


# Apply outer trimming only to the comparison components.
#
# The complete raw trainer label is preserved unchanged.
single_ampersand_trainers["left_component"] = (
    single_ampersand_trainers[
        "left_component"
    ].str.strip()
)

single_ampersand_trainers["right_component"] = (
    single_ampersand_trainers[
        "right_component"
    ].str.strip()
)


# Check for malformed or empty sides before comparing against standalone
# trainer labels.
single_ampersand_trainers[
    "left_component_blank"
] = single_ampersand_trainers[
    "left_component"
].eq("")

single_ampersand_trainers[
    "right_component_blank"
] = single_ampersand_trainers[
    "right_component"
].eq("")


# Build an exact-label lookup for independently observed trainer strings.
standalone_trainer_lookup = (
    trainer_labels[
        [
            "raw_label",
            "runner_rows",
            "first_observed_date",
            "last_observed_date",
        ]
    ]
    .rename(
        columns={
            "raw_label": "standalone_component",
            "runner_rows": "standalone_runner_rows",
            "first_observed_date":
                "standalone_first_observed_date",
            "last_observed_date":
                "standalone_last_observed_date",
        }
    )
)


# Attach any exact standalone match for the left component.
single_ampersand_trainers = (
    single_ampersand_trainers.merge(
        standalone_trainer_lookup.add_prefix(
            "left_"
        ),
        left_on="left_component",
        right_on="left_standalone_component",
        how="left",
        validate="many_to_one",
    )
)


# Attach any exact standalone match for the right component.
single_ampersand_trainers = (
    single_ampersand_trainers.merge(
        standalone_trainer_lookup.add_prefix(
            "right_"
        ),
        left_on="right_component",
        right_on="right_standalone_component",
        how="left",
        validate="many_to_one",
    )
)


single_ampersand_trainers[
    "left_observed_standalone"
] = single_ampersand_trainers[
    "left_standalone_component"
].notna()

single_ampersand_trainers[
    "right_observed_standalone"
] = single_ampersand_trainers[
    "right_standalone_component"
].notna()


def classify_ampersand_component_overlap(
    row: pd.Series,
) -> str:
    """Classify exact standalone overlap for a simple ampersand label."""
    if (
        row["left_component_blank"]
        or row["right_component_blank"]
    ):
        return "blank_component"

    if (
        row["left_observed_standalone"]
        and row["right_observed_standalone"]
    ):
        return "both_components_observed_standalone"

    if (
        row["left_observed_standalone"]
        or row["right_observed_standalone"]
    ):
        return "one_component_observed_standalone"

    return "neither_component_observed_standalone"


single_ampersand_trainers[
    "component_overlap_class"
] = (
    single_ampersand_trainers.apply(
        classify_ampersand_component_overlap,
        axis=1,
    )
)


trainer_ampersand_overlap_summary = (
    single_ampersand_trainers.groupby(
        "component_overlap_class",
        as_index=False,
    )
    .agg(
        distinct_joint_labels=("raw_label", "size"),
        joint_runner_rows=("runner_rows", "sum"),
    )
    .sort_values("component_overlap_class")
    .reset_index(drop=True)
)


# Create a readable inspection table without treating either component as a
# governed entity.
trainer_ampersand_component_examples = (
    single_ampersand_trainers[
        [
            "raw_label",
            "runner_rows",
            "first_observed_date",
            "last_observed_date",
            "left_component",
            "left_observed_standalone",
            "left_standalone_runner_rows",
            "left_standalone_first_observed_date",
            "left_standalone_last_observed_date",
            "right_component",
            "right_observed_standalone",
            "right_standalone_runner_rows",
            "right_standalone_first_observed_date",
            "right_standalone_last_observed_date",
            "component_overlap_class",
        ]
    ]
    .sort_values(
        [
            "component_overlap_class",
            "runner_rows",
            "raw_label",
        ],
        ascending=[True, False, True],
    )
    .reset_index(drop=True)
)


# Reconcile to the previously measured ampersand population.
assert len(trainer_ampersand_labels) == 305
assert int(
    trainer_ampersand_labels[
        "runner_rows"
    ].sum()
) == 53656

assert (
    trainer_ampersand_count_summary[
        "distinct_raw_labels"
    ].sum()
    == len(trainer_ampersand_labels)
)

assert (
    trainer_ampersand_overlap_summary[
        "distinct_joint_labels"
    ].sum()
    == len(single_ampersand_trainers)
)


print("Trainer ampersand-count distribution")
display(trainer_ampersand_count_summary)

print("Standalone component overlap for single-ampersand trainer labels")
display(trainer_ampersand_overlap_summary)

for overlap_class in trainer_ampersand_overlap_summary[
    "component_overlap_class"
]:
    print(f"Examples: {overlap_class}")

    display(
        trainer_ampersand_component_examples.loc[
            trainer_ampersand_component_examples[
                "component_overlap_class"
            ].eq(overlap_class)
        ]
        .head(40)
        .reset_index(drop=True)
    )

Trainer ampersand-count distribution


,ampersand_count,distinct_raw_labels,runner_rows
0,1,297,52507
1,2,8,1149


Standalone component overlap for single-ampersand trainer labels


,component_overlap_class,distinct_joint_labels,joint_runner_rows
0,both_components_observed_standalone,29,5323
1,neither_component_observed_standalone,157,26721
2,one_component_observed_standalone,111,20463


Examples: both_components_observed_standalone


,raw_label,runner_rows,first_observed_date,last_observed_date,left_component,left_observed_standalone,left_standalone_runner_rows,left_standalone_first_observed_date,left_standalone_last_observed_date,right_component,right_observed_standalone,right_standalone_runner_rows,right_standalone_first_observed_date,right_standalone_last_observed_date,component_overlap_class
0,H De Lageneste & G Macaire,835,2021-01-21,2026-05-26,H De Lageneste,True,74.0,2015-04-24,2020-12-24,G Macaire,True,1562.0,2015-02-24,2020-12-24,both_components_observed_standalone
1,Ciaron Maher & David Eustace,814,2018-08-27,2024-08-31,Ciaron Maher,True,922.0,2015-02-07,2026-05-23,David Eustace,True,767.0,2024-10-01,2026-05-27,both_components_observed_standalone
2,Roger Fell & Sean Murray,651,2023-06-10,2024-10-04,Roger Fell,True,3390.0,2016-09-28,2026-05-27,Sean Murray,True,2.0,2015-05-28,2016-05-04,both_components_observed_standalone
3,Lucinda Russell & Michael Scudamore,603,2025-08-09,2026-05-25,Lucinda Russell,True,4854.0,2015-01-01,2025-08-02,Michael Scudamore,True,1400.0,2015-01-02,2024-05-30,both_components_observed_standalone
4,Mike Murphy & Michael Keady,522,2022-06-08,2024-11-02,Mike Murphy,True,1185.0,2015-01-15,2026-05-25,Michael Keady,True,372.0,2024-12-19,2026-05-27,both_components_observed_standalone
5,David A Hayes & Tom Dabernig,329,2015-01-26,2016-07-30,David A Hayes,True,3027.0,2015-02-21,2026-05-27,Tom Dabernig,True,37.0,2021-08-28,2025-11-29,both_components_observed_standalone
6,Murray Baker & Andrew Forsman,305,2015-01-01,2022-04-16,Murray Baker,True,31.0,2015-03-07,2018-03-10,Andrew Forsman,True,89.0,2022-05-07,2026-05-23,both_components_observed_standalone
7,Mathew Ellerton & Simon Zahra,275,2015-01-24,2022-01-26,Mathew Ellerton,True,38.0,2022-01-26,2024-11-16,Simon Zahra,True,28.0,2022-03-05,2026-04-25,both_components_observed_standalone
8,John Best & Karen Jewell,188,2021-10-19,2023-03-10,John Best,True,1115.0,2015-01-03,2021-10-12,Karen Jewell,True,354.0,2023-03-15,2026-05-26,both_components_observed_standalone
9,Harriet Graham & Gary Rutherford,175,2022-02-01,2024-08-02,Harriet Graham,True,232.0,2015-01-08,2022-01-29,Gary Rutherford,True,202.0,2016-04-30,2026-05-25,both_components_observed_standalone


Examples: neither_component_observed_standalone


,raw_label,runner_rows,first_observed_date,last_observed_date,left_component,left_observed_standalone,left_standalone_runner_rows,left_standalone_first_observed_date,left_standalone_last_observed_date,right_component,right_observed_standalone,right_standalone_runner_rows,right_standalone_first_observed_date,right_standalone_last_observed_date,component_overlap_class
0,John & Thady Gosden,3230,2021-03-26,2026-05-27,John,False,NaN,NaN,NaN,Thady Gosden,False,NaN,NaN,NaN,neither_component_observed_standalone
1,Michael & David Easterby,2589,2021-06-08,2026-05-27,Michael,False,NaN,NaN,NaN,David Easterby,False,NaN,NaN,NaN,neither_component_observed_standalone
2,Simon & Ed Crisford,2533,2020-06-01,2026-05-27,Simon,False,NaN,NaN,NaN,Ed Crisford,False,NaN,NaN,NaN,neither_component_observed_standalone
3,Gary & Josh Moore,1603,2024-05-02,2026-05-27,Gary,False,NaN,NaN,NaN,Josh Moore,False,NaN,NaN,NaN,neither_component_observed_standalone
4,D & P ProdHomme,1233,2015-01-03,2026-04-26,D,False,NaN,NaN,NaN,P ProdHomme,False,NaN,NaN,NaN,neither_component_observed_standalone
5,Daniel & Claire Kubler,1189,2020-07-25,2026-05-26,Daniel,False,NaN,NaN,NaN,Claire Kubler,False,NaN,NaN,NaN,neither_component_observed_standalone
6,Jonjo & A J ONeill,1093,2024-05-02,2026-05-24,Jonjo,False,NaN,NaN,NaN,A J ONeill,False,NaN,NaN,NaN,neither_component_observed_standalone
7,Michael Wayne & John Hawkes,1058,2015-01-31,2026-05-23,Michael Wayne,False,NaN,NaN,NaN,John Hawkes,False,NaN,NaN,NaN,neither_component_observed_standalone
8,David & Nicola Barron,988,2022-03-26,2026-05-26,David,False,NaN,NaN,NaN,Nicola Barron,False,NaN,NaN,NaN,neither_component_observed_standalone
9,P & J Brandt,957,2020-08-01,2026-05-24,P,False,NaN,NaN,NaN,J Brandt,False,NaN,NaN,NaN,neither_component_observed_standalone


Examples: one_component_observed_standalone


,raw_label,runner_rows,first_observed_date,last_observed_date,left_component,left_observed_standalone,left_standalone_runner_rows,left_standalone_first_observed_date,left_standalone_last_observed_date,right_component,right_observed_standalone,right_standalone_runner_rows,right_standalone_first_observed_date,right_standalone_last_observed_date,component_overlap_class
0,Oliver Greenall & Josh Guerriero,1768,2022-04-30,2026-05-26,Oliver Greenall,True,1447.0,2015-05-01,2022-04-21,Josh Guerriero,False,NaN,NaN,NaN,one_component_observed_standalone
1,A & G Botti,1615,2015-01-03,2026-01-16,A,False,NaN,NaN,NaN,G Botti,True,444.0,2015-05-31,2022-04-21,one_component_observed_standalone
2,Gai Waterhouse & Adrian Bott,1335,2016-08-20,2026-05-23,Gai Waterhouse,True,273.0,2015-01-31,2016-06-25,Adrian Bott,False,NaN,NaN,NaN,one_component_observed_standalone
3,Charlie & Mark Johnston,1280,2022-01-02,2022-12-31,Charlie,False,NaN,NaN,NaN,Mark Johnston,True,9501.0,2015-01-07,2021-12-31,one_component_observed_standalone
4,Peter & Paul Snowden,1205,2015-01-09,2024-10-19,Peter,False,NaN,NaN,NaN,Paul Snowden,True,1.0,2026-03-07,2026-03-07,one_component_observed_standalone
5,William Muir & Chris Grassick,1173,2021-04-12,2026-05-25,William Muir,True,1384.0,2015-01-01,2021-04-10,Chris Grassick,False,NaN,NaN,NaN,one_component_observed_standalone
6,Dr Richard Newland & Jamie Insole,1149,2023-12-13,2026-05-27,Dr Richard Newland,True,2231.0,2015-01-01,2023-12-12,Jamie Insole,False,NaN,NaN,NaN,one_component_observed_standalone
7,Mme P Butel & J-L Beaunez,1078,2017-09-18,2026-05-26,Mme P Butel,True,383.0,2015-01-31,2017-10-27,J-L Beaunez,False,NaN,NaN,NaN,one_component_observed_standalone
8,Philip Hobbs & Johnson White,878,2023-03-15,2026-05-23,Philip Hobbs,True,4024.0,2015-01-01,2023-03-13,Johnson White,False,NaN,NaN,NaN,one_component_observed_standalone
9,Paul & Oliver Cole,824,2020-06-02,2025-08-04,Paul,False,NaN,NaN,NaN,Oliver Cole,True,111.0,2025-07-19,2026-05-23,one_component_observed_standalone


## Stage 24 — Horse continuity between joint trainer labels and standalone components

Literal ampersand splitting is not a general trainer-identity rule.

Of 297 single-ampersand labels:

- 29 have both literal components observed as standalone trainer labels;
- 111 have only one component observed;
- 157 have neither component observed.

Many labels use compressed shared-surname structures, so their literal left side is not a complete trainer label.

This stage is therefore restricted to the 29 labels where both complete literal components are independently observed.

For each joint label and each component, it measures:

- joint-label runner rows;
- standalone component runner rows;
- exact horse labels under the joint label;
- exact horse labels under the standalone component;
- shared exact horse labels;
- runner rows associated with shared horses;
- whether the standalone component appears before, after or across the joint-label period.

Shared horse continuity may support a source-internal relationship between a joint label and a standalone component.

It does not establish:

- legal or regulatory trainer identity;
- that the joint label can be decomposed safely;
- that each component refers to one permanent person;
- that a horse moving between labels retained the same responsible trainer;
- that one label should replace another;
- that compressed-surname ampersand labels can be parsed using the same method.

The complete raw trainer label remains the governed source assertion.

In [28]:
# Restrict the analysis to the finite set of single-ampersand trainer labels
# where both literal components are independently observed as exact trainer
# labels.
both_component_joint_trainers = (
    single_ampersand_trainers.loc[
        single_ampersand_trainers[
            "component_overlap_class"
        ].eq("both_components_observed_standalone"),
        [
            "raw_label",
            "runner_rows",
            "first_observed_date",
            "last_observed_date",
            "left_component",
            "right_component",
        ],
    ]
    .copy()
    .reset_index(drop=True)
)


# Convert each joint label into two candidate component relationships without
# treating either component as a governed entity.
joint_component_relationships = pd.concat(
    [
        both_component_joint_trainers[
            [
                "raw_label",
                "runner_rows",
                "first_observed_date",
                "last_observed_date",
                "left_component",
            ]
        ]
        .rename(
            columns={
                "raw_label": "joint_trainer_label",
                "runner_rows": "joint_runner_rows",
                "first_observed_date":
                    "joint_first_observed_date",
                "last_observed_date":
                    "joint_last_observed_date",
                "left_component": "component_trainer_label",
            }
        )
        .assign(component_side="left"),

        both_component_joint_trainers[
            [
                "raw_label",
                "runner_rows",
                "first_observed_date",
                "last_observed_date",
                "right_component",
            ]
        ]
        .rename(
            columns={
                "raw_label": "joint_trainer_label",
                "runner_rows": "joint_runner_rows",
                "first_observed_date":
                    "joint_first_observed_date",
                "last_observed_date":
                    "joint_last_observed_date",
                "right_component": "component_trainer_label",
            }
        )
        .assign(component_side="right"),
    ],
    ignore_index=True,
)


assert len(joint_component_relationships) == (
    2 * len(both_component_joint_trainers)
)


# Retrieve source rows for every relevant joint and standalone trainer label.
relevant_trainer_labels = sorted(
    set(
        joint_component_relationships[
            "joint_trainer_label"
        ]
    )
    | set(
        joint_component_relationships[
            "component_trainer_label"
        ]
    )
)

trainer_label_placeholders = ", ".join(
    ["?"] * len(relevant_trainer_labels)
)


connection = sqlite3.connect(
    f"file:{SOURCE_DB_PATH}?mode=ro",
    uri=True,
)

try:
    joint_component_source_rows = pd.read_sql_query(
        f"""
        SELECT
            rowid AS source_rowid,
            date,
            course,
            off,
            horse,
            trainer AS raw_trainer_label
        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}
          AND trainer IN ({trainer_label_placeholders})
        ORDER BY
            trainer,
            horse,
            date,
            course,
            off,
            rowid
        """,
        connection,
        params=relevant_trainer_labels,
    )

finally:
    connection.close()


# Summarise exact horse use under each relevant trainer label.
trainer_label_horse_usage = (
    joint_component_source_rows.groupby(
        [
            "raw_trainer_label",
            "horse",
        ],
        as_index=False,
    )
    .agg(
        runner_rows=("source_rowid", "size"),
        first_observed_date=("date", "min"),
        last_observed_date=("date", "max"),
    )
)


# Attach horse histories for the joint labels.
joint_horse_usage = (
    joint_component_relationships.merge(
        trainer_label_horse_usage,
        left_on="joint_trainer_label",
        right_on="raw_trainer_label",
        how="left",
        validate="many_to_many",
    )
    .drop(columns="raw_trainer_label")
    .rename(
        columns={
            "runner_rows": "joint_horse_runner_rows",
            "first_observed_date":
                "joint_horse_first_observed_date",
            "last_observed_date":
                "joint_horse_last_observed_date",
        }
    )
)


# Attach matching exact horse histories for each standalone component.
joint_component_horse_comparison = (
    joint_horse_usage.merge(
        trainer_label_horse_usage.rename(
            columns={
                "raw_trainer_label":
                    "component_trainer_label",
                "runner_rows":
                    "component_horse_runner_rows",
                "first_observed_date":
                    "component_horse_first_observed_date",
                "last_observed_date":
                    "component_horse_last_observed_date",
            }
        ),
        on=[
            "component_trainer_label",
            "horse",
        ],
        how="left",
        validate="many_to_one",
    )
)


joint_component_horse_comparison[
    "horse_observed_under_component"
] = joint_component_horse_comparison[
    "component_horse_runner_rows"
].notna()


# Build one continuity profile for each joint-label/component relationship.
joint_component_horse_continuity = (
    joint_component_horse_comparison.groupby(
        [
            "joint_trainer_label",
            "component_side",
            "component_trainer_label",
            "joint_runner_rows",
            "joint_first_observed_date",
            "joint_last_observed_date",
        ],
        as_index=False,
        dropna=False,
    )
    .agg(
        joint_distinct_horse_labels=("horse", "nunique"),
        shared_exact_horse_labels=(
            "horse",
            lambda values: values[
                joint_component_horse_comparison.loc[
                    values.index,
                    "horse_observed_under_component",
                ]
            ].nunique(),
        ),
        joint_runner_rows_on_shared_horses=(
            "joint_horse_runner_rows",
            lambda values: int(
                values[
                    joint_component_horse_comparison.loc[
                        values.index,
                        "horse_observed_under_component",
                    ]
                ].sum()
            ),
        ),
    )
)


# Attach the complete standalone component profile already established.
component_profile_lookup = (
    trainer_labels[
        [
            "raw_label",
            "runner_rows",
            "first_observed_date",
            "last_observed_date",
        ]
    ]
    .rename(
        columns={
            "raw_label": "component_trainer_label",
            "runner_rows": "component_runner_rows",
            "first_observed_date":
                "component_first_observed_date",
            "last_observed_date":
                "component_last_observed_date",
        }
    )
)


joint_component_horse_continuity = (
    joint_component_horse_continuity.merge(
        component_profile_lookup,
        on="component_trainer_label",
        how="left",
        validate="many_to_one",
    )
)


joint_component_horse_continuity[
    "shared_horse_percentage"
] = (
    100
    * joint_component_horse_continuity[
        "shared_exact_horse_labels"
    ]
    / joint_component_horse_continuity[
        "joint_distinct_horse_labels"
    ]
)


def classify_component_period_relationship(
    row: pd.Series,
) -> str:
    """Describe component chronology relative to the joint-label period."""
    component_first = row[
        "component_first_observed_date"
    ]
    component_last = row[
        "component_last_observed_date"
    ]
    joint_first = row[
        "joint_first_observed_date"
    ]
    joint_last = row[
        "joint_last_observed_date"
    ]

    if component_last < joint_first:
        return "component_before_joint_only"

    if component_first > joint_last:
        return "component_after_joint_only"

    if (
        component_first < joint_first
        and component_last > joint_last
    ):
        return "component_spans_joint_period"

    return "component_period_overlaps_joint"


joint_component_horse_continuity[
    "period_relationship"
] = (
    joint_component_horse_continuity.apply(
        classify_component_period_relationship,
        axis=1,
    )
)


joint_component_horse_continuity[
    "horse_continuity_class"
] = (
    joint_component_horse_continuity[
        "shared_exact_horse_labels"
    ].map(
        lambda count: (
            "shared_exact_horse_evidence"
            if count > 0
            else "no_shared_exact_horse_evidence"
        )
    )
)


# Summarise the 58 bounded component relationships.
joint_component_horse_summary = (
    joint_component_horse_continuity.groupby(
        [
            "horse_continuity_class",
            "period_relationship",
        ],
        as_index=False,
    )
    .agg(
        joint_component_relationships=(
            "component_trainer_label",
            "size",
        ),
        joint_labels=("joint_trainer_label", "nunique"),
        shared_exact_horse_labels=(
            "shared_exact_horse_labels",
            "sum",
        ),
        joint_runner_rows_on_shared_horses=(
            "joint_runner_rows_on_shared_horses",
            "sum",
        ),
    )
    .sort_values(
        [
            "horse_continuity_class",
            "period_relationship",
        ]
    )
    .reset_index(drop=True)
)


# Build detailed shared-horse examples.
joint_component_shared_horse_examples = (
    joint_component_horse_comparison.loc[
        joint_component_horse_comparison[
            "horse_observed_under_component"
        ],
        [
            "joint_trainer_label",
            "component_side",
            "component_trainer_label",
            "horse",
            "joint_horse_runner_rows",
            "joint_horse_first_observed_date",
            "joint_horse_last_observed_date",
            "component_horse_runner_rows",
            "component_horse_first_observed_date",
            "component_horse_last_observed_date",
        ],
    ]
    .sort_values(
        [
            "joint_horse_runner_rows",
            "component_horse_runner_rows",
            "joint_trainer_label",
            "component_trainer_label",
            "horse",
        ],
        ascending=[False, False, True, True, True],
    )
    .reset_index(drop=True)
)


# Reconcile the bounded population.
assert len(both_component_joint_trainers) == 29
assert len(joint_component_relationships) == 58
assert len(joint_component_horse_continuity) == 58

assert (
    joint_component_horse_summary[
        "joint_component_relationships"
    ].sum()
    == 58
)


print("Joint-trainer/component horse-continuity summary")
display(joint_component_horse_summary)

print("Joint-trainer/component continuity profiles")
display(
    joint_component_horse_continuity.sort_values(
        [
            "shared_exact_horse_labels",
            "joint_runner_rows_on_shared_horses",
            "joint_runner_rows",
            "joint_trainer_label",
            "component_side",
        ],
        ascending=[False, False, False, True, True],
    )
    .reset_index(drop=True)
)

print("Exact horses shared by joint labels and standalone components")
display(joint_component_shared_horse_examples.head(100))

Joint-trainer/component horse-continuity summary


,horse_continuity_class,period_relationship,joint_component_relationships,joint_labels,shared_exact_horse_labels,joint_runner_rows_on_shared_horses
0,no_shared_exact_horse_evidence,component_after_joint_only,9,9,0,0
1,no_shared_exact_horse_evidence,component_before_joint_only,6,6,0,0
2,no_shared_exact_horse_evidence,component_spans_joint_period,3,3,0,0
3,shared_exact_horse_evidence,component_after_joint_only,9,9,49,274
4,shared_exact_horse_evidence,component_before_joint_only,8,6,182,892
5,shared_exact_horse_evidence,component_period_overlaps_joint,8,8,44,192
6,shared_exact_horse_evidence,component_spans_joint_period,15,14,176,1136


Joint-trainer/component continuity profiles


,joint_trainer_label,component_side,component_trainer_label,joint_runner_rows,joint_first_observed_date,joint_last_observed_date,joint_distinct_horse_labels,shared_exact_horse_labels,joint_runner_rows_on_shared_horses,component_runner_rows,component_first_observed_date,component_last_observed_date,shared_horse_percentage,period_relationship,horse_continuity_class
0,Lucinda Russell & Michael Scudamore,left,Lucinda Russell,603,2025-08-09,2026-05-25,146,105,483,4854,2015-01-01,2025-08-02,71.917808,component_before_joint_only,shared_exact_horse_evidence
1,Roger Fell & Sean Murray,left,Roger Fell,651,2023-06-10,2024-10-04,83,58,509,3390,2016-09-28,2026-05-27,69.879518,component_spans_joint_period,shared_exact_horse_evidence
2,Ciaron Maher & David Eustace,left,Ciaron Maher,814,2018-08-27,2024-08-31,241,52,207,922,2015-02-07,2026-05-23,21.576763,component_spans_joint_period,shared_exact_horse_evidence
3,H De Lageneste & G Macaire,right,G Macaire,835,2021-01-21,2026-05-26,253,45,171,1562,2015-02-24,2020-12-24,17.786561,component_before_joint_only,shared_exact_horse_evidence
4,Mike Murphy & Michael Keady,left,Mike Murphy,522,2022-06-08,2024-11-02,68,23,224,1185,2015-01-15,2026-05-25,33.823529,component_spans_joint_period,shared_exact_horse_evidence
5,John Best & Karen Jewell,right,Karen Jewell,188,2021-10-19,2023-03-10,36,17,104,354,2023-03-15,2026-05-26,47.222222,component_after_joint_only,shared_exact_horse_evidence
6,John OShea & Tom Charlton,right,Tom Charlton,143,2024-08-10,2026-03-28,44,15,80,27,2026-03-21,2026-05-16,34.090909,component_period_overlaps_joint,shared_exact_horse_evidence
7,Michael Moroney & Pam Gerard,left,Michael Moroney,62,2016-11-05,2024-04-06,25,14,49,428,2015-02-09,2024-11-16,56.000000,component_spans_joint_period,shared_exact_horse_evidence
8,Harriet Graham & Gary Rutherford,left,Harriet Graham,175,2022-02-01,2024-08-02,27,12,123,232,2015-01-08,2022-01-29,44.444444,component_before_joint_only,shared_exact_horse_evidence
9,Mike Murphy & Michael Keady,right,Michael Keady,522,2022-06-08,2024-11-02,68,10,89,372,2024-12-19,2026-05-27,14.705882,component_after_joint_only,shared_exact_horse_evidence


Exact horses shared by joint labels and standalone components


,joint_trainer_label,component_side,component_trainer_label,horse,joint_horse_runner_rows,joint_horse_first_observed_date,joint_horse_last_observed_date,component_horse_runner_rows,component_horse_first_observed_date,component_horse_last_observed_date
0,Mike Murphy & Michael Keady,left,Mike Murphy,Antiphon (IRE),34,2022-06-18,2024-08-09,8.0,2022-04-16,2025-10-16
1,Mike Murphy & Michael Keady,right,Michael Keady,Antiphon (IRE),34,2022-06-18,2024-08-09,7.0,2025-01-04,2025-05-20
2,Murray Baker & Andrew Forsman,right,Andrew Forsman,The Chosen One (NZ),29,2018-11-10,2022-04-09,2.0,2022-05-28,2022-06-11
3,Roger Fell & Sean Murray,left,Roger Fell,Oso Rapido (IRE),25,2023-06-12,2024-09-28,29.0,2021-08-04,2024-11-04
4,Mike Murphy & Michael Keady,left,Mike Murphy,Temple Bruer (GB),24,2022-08-21,2024-10-24,11.0,2024-12-03,2025-08-15
...,...,...,...,...,...,...,...,...,...,...
95,Lucinda Russell & Michael Scudamore,left,Lucinda Russell,If Not For Dylan (IRE),8,2025-10-26,2026-05-25,3.0,2024-12-30,2025-04-25
96,Lucinda Russell & Michael Scudamore,left,Lucinda Russell,Walk On Son (IRE),8,2025-09-24,2026-05-24,3.0,2024-10-12,2025-04-05
97,Roger Fell & Sean Murray,left,Roger Fell,Yeulan (IRE),8,2023-06-20,2024-05-02,2.0,2023-05-06,2023-05-30
98,Ciaron Maher & David Eustace,left,Ciaron Maher,Explosive Jack (NZ),8,2021-03-13,2021-11-02,1.0,2024-03-16,2024-03-16


## Stage 25 — Conclusions from trainer ampersand structure

The trainer field contains 305 exact labels with at least one ampersand, covering 53,656 runner rows.

The source uses ampersands across several different text structures:

- full-name joint labels, such as `Ciaron Maher & David Eustace`;
- shared-surname compression, such as `John & Thady Gosden`;
- initial-based compression, such as `D & P ProdHomme`;
- labels containing more than one ampersand.

Literal splitting is not a safe general rule.

Of the 297 labels containing exactly one ampersand:

- 29 had both literal sides independently observed as exact trainer labels;
- 111 had only one side independently observed;
- 157 had neither side independently observed.

Many unmatched components are incomplete strings created by shared-surname compression rather than meaningful standalone trainer labels.

The 29 strongest literal full-name cases produced 58 joint-label/component relationships:

- 40 had at least one shared exact horse label;
- 18 had no shared exact horse evidence.

Continuity was often asymmetric.

A joint label could share many horses with one component but few or none with the other. Component chronology also varied between:

- standalone activity before the joint label;
- standalone activity after the joint label;
- overlapping periods;
- activity spanning the whole joint-label period.

These results support the following governance position:

1. The complete raw trainer label is the source assertion and must be preserved atomically.
2. An ampersand must not trigger automatic trainer decomposition.
3. Literal component strings may be retained only as analytical relationship candidates.
4. Shared exact horses and chronology can strengthen a candidate relationship but do not establish entity equivalence.
5. Shared-surname and initial-compressed labels require external or separately governed identity resolution.
6. Standalone exact-label matches may be namesakes and are not proof of component identity.
7. No canonical trainer entity or partnership identity should be created from these labels in this notebook.

In [29]:
# Persist a compact evidence table for every single-ampersand trainer label.
#
# This table preserves the complete raw trainer label and records only the
# bounded source-internal evidence established in Stages 23 and 24.
trainer_ampersand_evidence = (
    single_ampersand_trainers[
        [
            "raw_label",
            "runner_rows",
            "first_observed_date",
            "last_observed_date",
            "ampersand_count",
            "left_component",
            "right_component",
            "left_observed_standalone",
            "right_observed_standalone",
            "component_overlap_class",
        ]
    ]
    .copy()
)


# Attach aggregate shared-horse evidence where both literal components were
# independently observed.
component_continuity_lookup = (
    joint_component_horse_continuity[
        [
            "joint_trainer_label",
            "component_side",
            "component_trainer_label",
            "shared_exact_horse_labels",
            "joint_runner_rows_on_shared_horses",
            "period_relationship",
            "horse_continuity_class",
        ]
    ]
    .copy()
)


left_component_continuity = (
    component_continuity_lookup.loc[
        component_continuity_lookup[
            "component_side"
        ].eq("left")
    ]
    .drop(columns="component_side")
    .rename(
        columns={
            "joint_trainer_label": "raw_label",
            "component_trainer_label":
                "left_evaluated_component",
            "shared_exact_horse_labels":
                "left_shared_exact_horse_labels",
            "joint_runner_rows_on_shared_horses":
                "left_joint_rows_on_shared_horses",
            "period_relationship":
                "left_period_relationship",
            "horse_continuity_class":
                "left_horse_continuity_class",
        }
    )
)


right_component_continuity = (
    component_continuity_lookup.loc[
        component_continuity_lookup[
            "component_side"
        ].eq("right")
    ]
    .drop(columns="component_side")
    .rename(
        columns={
            "joint_trainer_label": "raw_label",
            "component_trainer_label":
                "right_evaluated_component",
            "shared_exact_horse_labels":
                "right_shared_exact_horse_labels",
            "joint_runner_rows_on_shared_horses":
                "right_joint_rows_on_shared_horses",
            "period_relationship":
                "right_period_relationship",
            "horse_continuity_class":
                "right_horse_continuity_class",
        }
    )
)


trainer_ampersand_evidence = (
    trainer_ampersand_evidence.merge(
        left_component_continuity,
        on="raw_label",
        how="left",
        validate="one_to_one",
    )
    .merge(
        right_component_continuity,
        on="raw_label",
        how="left",
        validate="one_to_one",
    )
)


# Assign a governance outcome without inferring people or partnerships.
def classify_trainer_ampersand_governance(
    row: pd.Series,
) -> str:
    """
    Assign the safest source-level treatment for one ampersand trainer label.
    """
    if row["component_overlap_class"] != (
        "both_components_observed_standalone"
    ):
        return "preserve_atomic_no_safe_literal_decomposition"

    left_shared = (
        row.get("left_shared_exact_horse_labels", 0)
        if pd.notna(
            row.get("left_shared_exact_horse_labels")
        )
        else 0
    )

    right_shared = (
        row.get("right_shared_exact_horse_labels", 0)
        if pd.notna(
            row.get("right_shared_exact_horse_labels")
        )
        else 0
    )

    if left_shared > 0 and right_shared > 0:
        return (
            "preserve_atomic_relationship_candidates_both_sides"
        )

    if left_shared > 0 or right_shared > 0:
        return (
            "preserve_atomic_relationship_candidate_one_side"
        )

    return (
        "preserve_atomic_literal_matches_without_horse_continuity"
    )


trainer_ampersand_evidence[
    "governance_outcome"
] = trainer_ampersand_evidence.apply(
    classify_trainer_ampersand_governance,
    axis=1,
)


trainer_ampersand_governance_summary = (
    trainer_ampersand_evidence.groupby(
        "governance_outcome",
        as_index=False,
    )
    .agg(
        distinct_trainer_labels=("raw_label", "size"),
        runner_rows=("runner_rows", "sum"),
    )
    .sort_values(
        [
            "runner_rows",
            "governance_outcome",
        ],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)


# Reconcile to all single-ampersand labels.
assert len(trainer_ampersand_evidence) == 297

assert int(
    trainer_ampersand_evidence[
        "runner_rows"
    ].sum()
) == 52507

assert (
    trainer_ampersand_governance_summary[
        "distinct_trainer_labels"
    ].sum()
    == 297
)


print("Trainer ampersand governance outcomes")
display(trainer_ampersand_governance_summary)

print("Trainer ampersand evidence table")
display(
    trainer_ampersand_evidence.sort_values(
        [
            "governance_outcome",
            "runner_rows",
            "raw_label",
        ],
        ascending=[True, False, True],
    )
    .reset_index(drop=True)
)

Trainer ampersand governance outcomes


,governance_outcome,distinct_trainer_labels,runner_rows
0,preserve_atomic_no_safe_literal_decomposition,268,47184
1,preserve_atomic_relationship_candidates_both_s...,13,3190
2,preserve_atomic_relationship_candidate_one_side,14,2122
3,preserve_atomic_literal_matches_without_horse_...,2,11


Trainer ampersand evidence table


,raw_label,runner_rows,first_observed_date,last_observed_date,ampersand_count,left_component,right_component,left_observed_standalone,right_observed_standalone,component_overlap_class,...,left_shared_exact_horse_labels,left_joint_rows_on_shared_horses,left_period_relationship,left_horse_continuity_class,right_evaluated_component,right_shared_exact_horse_labels,right_joint_rows_on_shared_horses,right_period_relationship,right_horse_continuity_class,governance_outcome
0,Mitchell Beer & Max Hinton,9,2016-02-26,2017-02-24,1,Mitchell Beer,Max Hinton,True,True,both_components_observed_standalone,...,0.0,0.0,component_after_joint_only,no_shared_exact_horse_evidence,Max Hinton,0.0,0.0,component_before_joint_only,no_shared_exact_horse_evidence,preserve_atomic_literal_matches_without_horse_...
1,Ryan Tyrell & Tom Button,2,2022-12-17,2023-07-01,1,Ryan Tyrell,Tom Button,True,True,both_components_observed_standalone,...,0.0,0.0,component_spans_joint_period,no_shared_exact_horse_evidence,Tom Button,0.0,0.0,component_before_joint_only,no_shared_exact_horse_evidence,preserve_atomic_literal_matches_without_horse_...
2,John & Thady Gosden,3230,2021-03-26,2026-05-27,1,John,Thady Gosden,False,False,neither_component_observed_standalone,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,preserve_atomic_no_safe_literal_decomposition
3,Michael & David Easterby,2589,2021-06-08,2026-05-27,1,Michael,David Easterby,False,False,neither_component_observed_standalone,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,preserve_atomic_no_safe_literal_decomposition
4,Simon & Ed Crisford,2533,2020-06-01,2026-05-27,1,Simon,Ed Crisford,False,False,neither_component_observed_standalone,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,preserve_atomic_no_safe_literal_decomposition
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
292,P Chemin & C Herpin,64,2017-05-07,2022-06-22,1,P Chemin,C Herpin,True,True,both_components_observed_standalone,...,7.0,29.0,component_before_joint_only,shared_exact_horse_evidence,C Herpin,4.0,7.0,component_period_overlaps_joint,shared_exact_horse_evidence,preserve_atomic_relationship_candidates_both_s...
293,Michael Moroney & Pam Gerard,62,2016-11-05,2024-04-06,1,Michael Moroney,Pam Gerard,True,True,both_components_observed_standalone,...,14.0,49.0,component_spans_joint_period,shared_exact_horse_evidence,Pam Gerard,1.0,2.0,component_after_joint_only,shared_exact_horse_evidence,preserve_atomic_relationship_candidates_both_s...
294,Donna Logan & Chris Gibbs,43,2016-01-01,2018-03-10,1,Donna Logan,Chris Gibbs,True,True,both_components_observed_standalone,...,2.0,15.0,component_spans_joint_period,shared_exact_horse_evidence,Chris Gibbs,2.0,2.0,component_after_joint_only,shared_exact_horse_evidence,preserve_atomic_relationship_candidates_both_s...
295,Michael Moroney & Glen Thompson,34,2024-08-17,2025-03-01,1,Michael Moroney,Glen Thompson,True,True,both_components_observed_standalone,...,7.0,17.0,component_period_overlaps_joint,shared_exact_horse_evidence,Glen Thompson,4.0,11.0,component_spans_joint_period,shared_exact_horse_evidence,preserve_atomic_relationship_candidates_both_s...


## Stage 26 — Trainer labels containing multiple ampersands

Eight exact trainer labels contain two ampersands, covering 1,149 runner rows.

These labels were excluded from the earlier left/right component analysis because splitting once around `&` would not represent their complete structure.

This stage inspects the finite population directly.

For each label, it records:

- the complete raw trainer label;
- runner-row volume;
- first and last observed dates;
- the number of ampersands;
- the literal trimmed segments produced by splitting at every ampersand;
- the number of resulting segments;
- whether each literal segment occurs as a standalone exact trainer label.

The resulting segments are diagnostic only.

They must not be treated as trainer identities because multiple-ampersand labels may contain:

- three named trainers;
- compressed shared surnames;
- initials that depend on a later surname;
- stable partnership or organisational wording;
- punctuation whose exact structure cannot be inferred internally.

The complete raw trainer label remains the governed source assertion.

Any component relationship must be established separately through manual or external verification.

In [30]:
# Restrict inspection to the complete finite population of trainer labels
# containing more than one ampersand.
multiple_ampersand_trainers = (
    trainer_ampersand_labels.loc[
        trainer_ampersand_labels[
            "ampersand_count"
        ].gt(1)
    ]
    .copy()
    .reset_index(drop=True)
)


# Split every label at each ampersand and trim only the resulting diagnostic
# segments. The complete raw label is preserved unchanged.
multiple_ampersand_trainers[
    "literal_segments"
] = (
    multiple_ampersand_trainers[
        "raw_label"
    ].map(
        lambda label: [
            segment.strip()
            for segment in label.split("&")
        ]
    )
)


multiple_ampersand_trainers[
    "literal_segment_count"
] = (
    multiple_ampersand_trainers[
        "literal_segments"
    ].map(len)
)


# Build an exact set of all independently observed trainer labels.
exact_trainer_label_set = set(
    trainer_labels["raw_label"]
)


# Record whether each literal segment is itself observed as an exact standalone
# trainer label. These matches remain text evidence only.
multiple_ampersand_trainers[
    "segment_standalone_matches"
] = (
    multiple_ampersand_trainers[
        "literal_segments"
    ].map(
        lambda segments: [
            segment in exact_trainer_label_set
            for segment in segments
        ]
    )
)


multiple_ampersand_trainers[
    "standalone_segment_count"
] = (
    multiple_ampersand_trainers[
        "segment_standalone_matches"
    ].map(sum)
)


def classify_multiple_ampersand_structure(
    row: pd.Series,
) -> str:
    """
    Classify only the literal standalone overlap of a multi-ampersand label.

    This does not classify people, partnerships or legal training structures.
    """
    segment_count = row[
        "literal_segment_count"
    ]

    standalone_count = row[
        "standalone_segment_count"
    ]

    if standalone_count == segment_count:
        return "all_literal_segments_observed_standalone"

    if standalone_count == 0:
        return "no_literal_segments_observed_standalone"

    return "some_literal_segments_observed_standalone"


multiple_ampersand_trainers[
    "literal_overlap_class"
] = (
    multiple_ampersand_trainers.apply(
        classify_multiple_ampersand_structure,
        axis=1,
    )
)


multiple_ampersand_display = (
    multiple_ampersand_trainers[
        [
            "raw_label",
            "runner_rows",
            "first_observed_date",
            "last_observed_date",
            "ampersand_count",
            "literal_segment_count",
            "literal_segments",
            "segment_standalone_matches",
            "standalone_segment_count",
            "literal_overlap_class",
        ]
    ]
    .sort_values(
        [
            "runner_rows",
            "raw_label",
        ],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)


multiple_ampersand_summary = (
    multiple_ampersand_display.groupby(
        "literal_overlap_class",
        as_index=False,
    )
    .agg(
        distinct_trainer_labels=("raw_label", "size"),
        runner_rows=("runner_rows", "sum"),
    )
    .sort_values(
        "literal_overlap_class"
    )
    .reset_index(drop=True)
)


# Reconcile to the Stage 23 population.
assert len(multiple_ampersand_trainers) == 8

assert int(
    multiple_ampersand_trainers[
        "runner_rows"
    ].sum()
) == 1149

assert (
    multiple_ampersand_summary[
        "distinct_trainer_labels"
    ].sum()
    == 8
)


print("Multiple-ampersand trainer-label summary")
display(multiple_ampersand_summary)

print("Complete multiple-ampersand trainer-label inspection")
display(multiple_ampersand_display)

Multiple-ampersand trainer-label summary


,literal_overlap_class,distinct_trainer_labels,runner_rows
0,no_literal_segments_observed_standalone,6,126
1,some_literal_segments_observed_standalone,2,1023


Complete multiple-ampersand trainer-label inspection


,raw_label,runner_rows,first_observed_date,last_observed_date,ampersand_count,literal_segment_count,literal_segments,segment_standalone_matches,standalone_segment_count,literal_overlap_class
0,David A & B Hayes & Tom Dabernig,1015,2016-08-06,2020-04-25,2,3,"[David A, B Hayes, Tom Dabernig]","[False, False, True]",1,some_literal_segments_observed_standalone
1,C & G Hue & Taupin,50,2023-01-20,2026-04-24,2,3,"[C, G Hue, Taupin]","[False, False, False]",0,no_literal_segments_observed_standalone
2,Leon & Troy Corstens & Will Larkin,43,2024-09-14,2026-05-23,2,3,"[Leon, Troy Corstens, Will Larkin]","[False, False, False]",0,no_literal_segments_observed_standalone
3,H & T Hue & Taupin,28,2022-02-28,2022-12-26,2,3,"[H, T Hue, Taupin]","[False, False, False]",0,no_literal_segments_observed_standalone
4,Peter & Dawn Williams & Paul Richards,8,2015-10-24,2017-01-01,2,3,"[Peter, Dawn Williams, Paul Richards]","[False, False, True]",1,some_literal_segments_observed_standalone
5,Ken & Bev Kelso & Mark Donoghue,3,2019-01-01,2019-04-06,2,3,"[Ken, Bev Kelso, Mark Donoghue]","[False, False, False]",0,no_literal_segments_observed_standalone
6,G & N Searle & B Callanan,1,2025-10-25,2025-10-25,2,3,"[G, N Searle, B Callanan]","[False, False, False]",0,no_literal_segments_observed_standalone
7,Kenny & Lisa Rae & Krystal Williams,1,2018-03-10,2018-03-10,2,3,"[Kenny, Lisa Rae, Krystal Williams]","[False, False, False]",0,no_literal_segments_observed_standalone


## Stage 27 — Boundary between source semantics and connection identity resolution

The trainer ampersand investigation demonstrates that connection-label semantics and entity resolution are separate problems.

The source provides complete runner-level trainer assertions such as:

- `John & Thady Gosden`;
- `Michael & David Easterby`;
- `Ciaron Maher & David Eustace`;
- `David A & B Hayes & Tom Dabernig`.

These labels contain several structures:

- complete names joined by ampersands;
- compressed shared surnames;
- initial-based compressed names;
- labels involving three apparent participants;
- standalone components with strong horse continuity;
- standalone components with weak or no continuity;
- independently observed strings that may be namesakes.

The source does not provide enough information to reconstruct every participant safely.

Therefore this notebook establishes the following boundary:

### In scope here

- preserve the complete raw `trainer` value;
- describe punctuation and structural vocabularies;
- identify candidate relationships supported by exact-label and horse-history evidence;
- document why automatic splitting or canonicalisation is unsafe;
- define the raw-field database treatment.

### Deferred to later identity-resolution work

- reconstructing omitted shared surnames;
- verifying formal joint-training arrangements;
- assigning canonical person identifiers;
- distinguishing namesakes;
- establishing partnership start and end dates;
- resolving trainer organisations or licences externally;
- linking joint labels to verified individual trainer entities.

The same boundary will apply to complex owner labels, where partnerships, syndicates, companies and named co-owners may be represented within one source string.

The current notebook should not become a complete connections entity-resolution study.

Its responsibility is to define what the source safely says and what must remain unresolved.

## Stage 28 — Within-race cardinality of connection labels

The three connection fields do not necessarily have the same relationship to a race.

A jockey normally rides no more than one runner in a race, while a trainer or owner may be attached to several runners.

This stage measures, for each exact populated label:

- how many provisional races contain the label;
- the maximum number of runner rows carrying that label in one provisional race;
- how often the label appears more than once in the same race;
- the largest observed same-race groups.

The provisional race key remains:

- `date`;
- `course`;
- `off`.

The results describe source-row behaviour only.

They do not establish:

- legal ownership;
- beneficial ownership;
- trainer licensing structures;
- whether repeated owner labels represent one person or a collective;
- whether provisional race keys are globally authoritative race identifiers.

The purpose is to establish the safe source-level cardinality of each field.

In [31]:
# Measure within-race exact-label cardinality using SQLite aggregation.
#
# This avoids loading the full runner-level source table into pandas.
# The provisional race key remains:
# - date
# - course
# - off


def build_within_race_cardinality_sql(
    connection: sqlite3.Connection,
    field: str,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Measure exact-label repetition within provisional races for one field.

    Empty strings and NULL values are excluded because they represent
    field-not-supplied values rather than connection labels.
    """
    quoted_field = quote_identifier(field)
    quoted_table = quote_identifier(SOURCE_TABLE)

    # Produce one compact row per provisional race and exact label.
    label_race_counts_sql = f"""
        WITH label_race_counts AS (
            SELECT
                date,
                course,
                off,
                {quoted_field} AS raw_label,
                COUNT(*) AS runner_rows,
                COUNT(DISTINCT horse) AS distinct_horse_labels
            FROM {quoted_table}
            WHERE {DATA_ROW_PREDICATE}
              AND {quoted_field} IS NOT NULL
              AND {quoted_field} <> ''
            GROUP BY
                date,
                course,
                off,
                {quoted_field}
        )
    """

    # Build one profile for every exact populated label.
    label_profile = pd.read_sql_query(
        label_race_counts_sql
        + """
        SELECT
            raw_label,
            COUNT(*) AS provisional_races,
            SUM(runner_rows) AS runner_rows,
            MAX(runner_rows) AS maximum_rows_in_one_race,
            SUM(
                CASE
                    WHEN runner_rows > 1 THEN 1
                    ELSE 0
                END
            ) AS races_with_multiple_rows,
            SUM(
                CASE
                    WHEN runner_rows > 1 THEN runner_rows
                    ELSE 0
                END
            ) AS rows_in_multi_runner_races
        FROM label_race_counts
        GROUP BY raw_label
        ORDER BY raw_label
        """,
        connection,
    )

    label_profile["has_same_race_repetition"] = (
        label_profile[
            "maximum_rows_in_one_race"
        ].gt(1)
    )

    # Summarise labels by their maximum observed within-race multiplicity.
    multiplicity_summary = (
        label_profile.groupby(
            "maximum_rows_in_one_race",
            as_index=False,
        )
        .agg(
            distinct_labels=("raw_label", "size"),
            runner_rows=("runner_rows", "sum"),
            provisional_races=("provisional_races", "sum"),
            races_with_multiple_rows=(
                "races_with_multiple_rows",
                "sum",
            ),
        )
        .sort_values("maximum_rows_in_one_race")
        .reset_index(drop=True)
    )

    # Return only the largest repeated groups for inspection.
    largest_same_race_groups = pd.read_sql_query(
        label_race_counts_sql
        + """
        SELECT
            date,
            course,
            off,
            raw_label,
            runner_rows,
            distinct_horse_labels
        FROM label_race_counts
        WHERE runner_rows > 1
        ORDER BY
            runner_rows DESC,
            date,
            course,
            off,
            raw_label
        LIMIT 50
        """,
        connection,
    )

    return (
        label_profile,
        multiplicity_summary,
        largest_same_race_groups,
    )


connection_race_cardinality_profiles = {}
connection_race_multiplicity_summaries = {}
connection_largest_same_race_groups = {}


connection = sqlite3.connect(
    f"file:{SOURCE_DB_PATH}?mode=ro",
    uri=True,
)

try:
    for field in CONNECTION_IDENTITY_FIELDS:
        (
            connection_race_cardinality_profiles[field],
            connection_race_multiplicity_summaries[field],
            connection_largest_same_race_groups[field],
        ) = build_within_race_cardinality_sql(
            connection,
            field,
        )

finally:
    connection.close()


# Reconfirm the previously established jockey rule.
assert (
    connection_race_cardinality_profiles[
        "jockey"
    ]["maximum_rows_in_one_race"]
    .le(1)
    .all()
)


# Reconcile exact-label populations.
assert len(
    connection_race_cardinality_profiles["jockey"]
) == len(jockey_labels)

assert len(
    connection_race_cardinality_profiles["trainer"]
) == len(trainer_labels)

assert len(
    connection_race_cardinality_profiles["owner"]
) == len(owner_labels)


# Build one compact cross-field summary.
connection_within_race_summary = pd.DataFrame(
    [
        {
            "field": field,
            "distinct_labels": len(
                connection_race_cardinality_profiles[field]
            ),
            "labels_repeated_within_race": int(
                connection_race_cardinality_profiles[
                    field
                ]["has_same_race_repetition"].sum()
            ),
            "maximum_rows_for_one_label_in_one_race": int(
                connection_race_cardinality_profiles[
                    field
                ]["maximum_rows_in_one_race"].max()
            ),
            "races_with_repeated_exact_label": int(
                connection_race_cardinality_profiles[
                    field
                ]["races_with_multiple_rows"].sum()
            ),
            "runner_rows_in_repeated_groups": int(
                connection_race_cardinality_profiles[
                    field
                ]["rows_in_multi_runner_races"].sum()
            ),
        }
        for field in CONNECTION_IDENTITY_FIELDS
    ]
)


print("Cross-field within-race cardinality summary")
display(connection_within_race_summary)


for field in CONNECTION_IDENTITY_FIELDS:
    print(
        f"Maximum same-race multiplicity distribution: {field}"
    )

    display(
        connection_race_multiplicity_summaries[field]
    )

    print(
        f"Largest repeated exact-label race groups: {field}"
    )

    display(
        connection_largest_same_race_groups[field]
    )

Cross-field within-race cardinality summary


,field,distinct_labels,labels_repeated_within_race,maximum_rows_for_one_label_in_one_race,races_with_repeated_exact_label,runner_rows_in_repeated_groups
0,jockey,7917,0,1,0,0
1,trainer,10708,3600,14,120906,263062
2,owner,98234,3895,13,28080,61377


Maximum same-race multiplicity distribution: jockey


,maximum_rows_in_one_race,distinct_labels,runner_rows,provisional_races,races_with_multiple_rows
0,1,7917,1851283,1851283,0


Largest repeated exact-label race groups: jockey


...

,date,course,off,raw_label,runner_rows,distinct_horse_labels
0,2023-11-19,Navan (IRE),2:30,Gordon Elliott,14,14
1,2018-04-02,Fairyhouse (IRE),5:00,Gordon Elliott,13,13
2,2018-04-28,Punchestown (IRE),5:35,W P Mullins,13,13
3,2019-04-22,Fairyhouse (IRE),5:00,Gordon Elliott,12,12
4,2015-09-05,Randwick (AUS),6:45,Chris Waller,11,11
5,2016-11-27,Navan (IRE),2:20,Gordon Elliott,11,11
6,2018-11-25,Navan (IRE),2:30,Gordon Elliott,11,11
7,2019-04-06,Aintree,5:15,Gordon Elliott,11,11
8,2025-03-14,Cheltenham,1:20,W P Mullins,11,11
9,2018-04-17,Fairyhouse (IRE),5:50,W P Mullins,10,10


Maximum same-race multiplicity distribution: owner


,maximum_rows_in_one_race,distinct_labels,runner_rows,provisional_races,races_with_multiple_rows
0,1,94339,1218315,1218315,0
1,2,3557,439080,429829,9251
2,3,271,107467,100552,6324
3,4,39,24005,21302,2289
4,5,17,13333,10768,1946
5,6,5,5517,4075,1043
6,7,3,7735,5941,1199
7,8,1,13416,10644,2097
8,10,1,15384,11473,2567
9,13,1,6998,5054,1364


Largest repeated exact-label race groups: owner


,date,course,off,raw_label,runner_rows,distinct_horse_labels
0,2017-04-17,Fairyhouse (IRE),5:00,Gigginstown House Stud,13,13
1,2019-04-22,Fairyhouse (IRE),5:00,Gigginstown House Stud,12,12
2,2018-04-02,Fairyhouse (IRE),5:00,Gigginstown House Stud,10,10
3,2020-02-02,Leopardstown (IRE),4:00,John P Mcmanus,10,10
4,2016-04-30,Punchestown (IRE),5:35,John P Mcmanus,9,9
5,2018-08-02,Galway (IRE),4:35,John P Mcmanus,9,9
6,2019-11-24,Navan (IRE),2:30,John P Mcmanus,9,9
7,2020-11-08,Navan (IRE),1:35,Gigginstown House Stud,9,9
8,2015-05-02,Punchestown (IRE),5:35,John P Mcmanus,8,8
9,2015-12-27,Leopardstown (IRE),2:55,John P Mcmanus,8,8


## Stage 29 — Distinct-horse consistency within repeated connection groups

Stage 28 established different within-race cardinalities:

- an exact jockey label appears on no more than one runner in a provisional race;
- an exact trainer label may appear on many runners;
- an exact owner label may appear on many runners.

The largest repeated trainer and owner groups contained one distinct horse label for every runner row.

This stage checks that relationship source-wide.

For every provisional race and exact populated connection label, it compares:

- runner-row count;
- distinct exact horse-label count.

A repeated trainer or owner group is `distinct_horse_consistent` when every row has a different exact horse label.

A group is `horse_label_repetition_present` when runner rows exceed distinct horse labels.

Such a mismatch could reflect:

- duplicate source rows;
- repeated exact horse labels;
- provisional race-key collisions;
- unresolved horse-label ambiguity;
- another source anomaly.

This remains a source-consistency check.

Exact horse labels are not treated as globally resolved horse entities.

In [32]:
# Check source-wide whether repeated connection labels within provisional races
# are attached to distinct exact horse labels.
#
# All large grouping remains inside SQLite.

def build_repeated_group_horse_consistency_sql(
    connection: sqlite3.Connection,
    field: str,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Compare runner rows and distinct exact horse labels within every repeated
    provisional-race / connection-label group.
    """
    quoted_field = quote_identifier(field)
    quoted_table = quote_identifier(SOURCE_TABLE)

    repeated_group_cte = f"""
        WITH repeated_label_groups AS (
            SELECT
                date,
                course,
                off,
                {quoted_field} AS raw_label,
                COUNT(*) AS runner_rows,
                COUNT(DISTINCT horse) AS distinct_horse_labels
            FROM {quoted_table}
            WHERE {DATA_ROW_PREDICATE}
              AND {quoted_field} IS NOT NULL
              AND {quoted_field} <> ''
            GROUP BY
                date,
                course,
                off,
                {quoted_field}
            HAVING COUNT(*) > 1
        )
    """

    # Return one compact summary row for the field.
    summary = pd.read_sql_query(
        repeated_group_cte
        + """
        SELECT
            COUNT(*) AS repeated_exact_label_groups,
            SUM(runner_rows) AS runner_rows,
            SUM(
                CASE
                    WHEN runner_rows = distinct_horse_labels
                    THEN 1
                    ELSE 0
                END
            ) AS distinct_horse_consistent_groups,
            SUM(
                CASE
                    WHEN runner_rows > distinct_horse_labels
                    THEN 1
                    ELSE 0
                END
            ) AS groups_with_repeated_horse_labels,
            SUM(
                runner_rows - distinct_horse_labels
            ) AS excess_rows_over_distinct_horses,
            MAX(
                runner_rows - distinct_horse_labels
            ) AS maximum_excess_rows_in_one_group
        FROM repeated_label_groups
        """,
        connection,
    )

    # Retrieve only anomalous groups for inspection.
    anomalies = pd.read_sql_query(
        repeated_group_cte
        + """
        SELECT
            date,
            course,
            off,
            raw_label,
            runner_rows,
            distinct_horse_labels,
            runner_rows - distinct_horse_labels
                AS excess_rows_over_distinct_horses
        FROM repeated_label_groups
        WHERE runner_rows > distinct_horse_labels
        ORDER BY
            excess_rows_over_distinct_horses DESC,
            runner_rows DESC,
            date,
            course,
            off,
            raw_label
        LIMIT 100
        """,
        connection,
    )

    return summary, anomalies


connection_repeated_group_horse_summaries = {}
connection_repeated_group_horse_anomalies = {}


connection = sqlite3.connect(
    f"file:{SOURCE_DB_PATH}?mode=ro",
    uri=True,
)

try:
    for field in CONNECTION_IDENTITY_FIELDS:
        (
            connection_repeated_group_horse_summaries[field],
            connection_repeated_group_horse_anomalies[field],
        ) = build_repeated_group_horse_consistency_sql(
            connection,
            field,
        )

finally:
    connection.close()


# Combine the three field summaries.
connection_repeated_group_horse_summary = pd.concat(
    [
        connection_repeated_group_horse_summaries[
            field
        ].assign(field=field)
        for field in CONNECTION_IDENTITY_FIELDS
    ],
    ignore_index=True,
)


connection_repeated_group_horse_summary = (
    connection_repeated_group_horse_summary[
        [
            "field",
            "repeated_exact_label_groups",
            "runner_rows",
            "distinct_horse_consistent_groups",
            "groups_with_repeated_horse_labels",
            "excess_rows_over_distinct_horses",
            "maximum_excess_rows_in_one_group",
        ]
    ]
)


# Reconcile the repeated-group counts to Stage 28.
for field in CONNECTION_IDENTITY_FIELDS:
    expected_repeated_groups = int(
        connection_within_race_summary.loc[
            connection_within_race_summary[
                "field"
            ].eq(field),
            "races_with_repeated_exact_label",
        ].iloc[0]
    )

    observed_repeated_groups = int(
        connection_repeated_group_horse_summaries[
            field
        ][
            "repeated_exact_label_groups"
        ].iloc[0]
        or 0
    )

    assert (
        observed_repeated_groups
        == expected_repeated_groups
    )


print("Distinct-horse consistency in repeated connection groups")
display(connection_repeated_group_horse_summary)


for field in CONNECTION_IDENTITY_FIELDS:
    print(
        f"Repeated {field} groups with repeated exact horse labels"
    )

    display(
        connection_repeated_group_horse_anomalies[
            field
        ]
    )

Distinct-horse consistency in repeated connection groups


,field,repeated_exact_label_groups,runner_rows,distinct_horse_consistent_groups,groups_with_repeated_horse_labels,excess_rows_over_distinct_horses,maximum_excess_rows_in_one_group
0,jockey,0,None,None,None,None,None
1,trainer,120906,263062,120906,0,0,0
2,owner,28080,61377,28080,0,0,0


Repeated jockey groups with repeated exact horse labels


,date,course,off,raw_label,runner_rows,distinct_horse_labels,excess_rows_over_distinct_horses


Repeated trainer groups with repeated exact horse labels


,date,course,off,raw_label,runner_rows,distinct_horse_labels,excess_rows_over_distinct_horses


Repeated owner groups with repeated exact horse labels


,date,course,off,raw_label,runner_rows,distinct_horse_labels,excess_rows_over_distinct_horses


## Stage 30 — Conclusion from within-race connection cardinality

Within provisional races defined by `date`, `course` and `off`:

- no exact populated jockey label appears on more than one runner;
- 3,600 exact trainer labels repeat within at least one race;
- 3,895 exact owner labels repeat within at least one race;
- the maximum observed trainer multiplicity is 14 runners;
- the maximum observed owner multiplicity is 13 runners.

All repeated trainer and owner groups are exact-horse consistent:

- 120,906 repeated trainer groups contain 263,062 runner rows, with one distinct exact horse label per row;
- 28,080 repeated owner groups contain 61,377 runner rows, with one distinct exact horse label per row;
- no repeated group contains fewer exact horse labels than runner rows.

This supports different source-level cardinalities:

- `jockey` — maximum one populated exact label occurrence per provisional race;
- `trainer` — one exact label may relate to multiple runners in one race;
- `owner` — one exact label may relate to multiple runners in one race.

These results do not establish canonical person, partnership or organisation identities.

They establish only the runner-level behaviour of the exact source labels.

## Stage 31 — Context of blank connection fields

The raw-field profile found very few blank connection values:

- 2 blank jockey values;
- 9 blank trainer values;
- 35 blank owner values.

All are empty strings rather than SQL `NULL`.

Because the populations are so small, the correct next step is direct inspection rather than broad statistical modelling.

This stage retrieves every row with at least one blank connection field and records:

- race context;
- horse label;
- which connection fields are blank;
- the populated companion connection labels;
- result and race-description fields useful for understanding the row.

The aim is to determine whether blanks appear to represent:

- field not supplied;
- jurisdiction-specific source incompleteness;
- abandoned or non-standard records;
- malformed rows;
- another bounded source condition.

A blank value will not be replaced with an inferred person, trainer or owner.

In [33]:
# Retrieve the complete finite population of rows with at least one blank
# connection field.
#
# The source database remains read-only.
connection = sqlite3.connect(
    f"file:{SOURCE_DB_PATH}?mode=ro",
    uri=True,
)

try:
    blank_connection_rows = pd.read_sql_query(
        f"""
        SELECT
            rowid AS source_rowid,
            race_id,
            date,
            course,
            off,
            horse,
            pos,
            ran,
            race_name,
            type,
            jockey,
            trainer,
            owner
        FROM {SOURCE_TABLE}
        WHERE {DATA_ROW_PREDICATE}
          AND (
                jockey = ''
             OR trainer = ''
             OR owner = ''
          )
        ORDER BY
            date,
            course,
            off,
            rowid
        """,
        connection,
    )

finally:
    connection.close()


# Record exactly which connection fields are blank on each source row.
for field in CONNECTION_IDENTITY_FIELDS:
    blank_connection_rows[
        f"{field}_is_blank"
    ] = blank_connection_rows[field].eq("")


blank_connection_rows[
    "blank_connection_field_count"
] = (
    blank_connection_rows[
        [
            "jockey_is_blank",
            "trainer_is_blank",
            "owner_is_blank",
        ]
    ]
    .sum(axis=1)
)


blank_connection_rows[
    "blank_connection_fields"
] = (
    blank_connection_rows.apply(
        lambda row: " | ".join(
            field
            for field in CONNECTION_IDENTITY_FIELDS
            if row[f"{field}_is_blank"]
        ),
        axis=1,
    )
)


# Summarise blank combinations.
blank_connection_combination_summary = (
    blank_connection_rows.groupby(
        [
            "blank_connection_fields",
            "blank_connection_field_count",
        ],
        as_index=False,
    )
    .agg(
        runner_rows=("source_rowid", "size"),
        distinct_race_ids=("race_id", "nunique"),
        distinct_courses=("course", "nunique"),
        first_observed_date=("date", "min"),
        last_observed_date=("date", "max"),
    )
    .sort_values(
        [
            "runner_rows",
            "blank_connection_fields",
        ],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)


# Summarise the finite population by course.
blank_connection_course_summary = (
    blank_connection_rows.groupby(
        "course",
        as_index=False,
        dropna=False,
    )
    .agg(
        runner_rows=("source_rowid", "size"),
        distinct_race_ids=("race_id", "nunique"),
        blank_jockey_rows=("jockey_is_blank", "sum"),
        blank_trainer_rows=("trainer_is_blank", "sum"),
        blank_owner_rows=("owner_is_blank", "sum"),
        first_observed_date=("date", "min"),
        last_observed_date=("date", "max"),
    )
    .sort_values(
        [
            "runner_rows",
            "course",
        ],
        ascending=[False, True],
    )
    .reset_index(drop=True)
)


# Reconcile to the raw-field profile established earlier.
expected_blank_counts = {
    field: int(
        raw_field_profile.loc[
            raw_field_profile[
                "source_field"
            ].eq(field),
            "blank_rows",
        ].iloc[0]
    )
    for field in CONNECTION_IDENTITY_FIELDS
}


for field in CONNECTION_IDENTITY_FIELDS:
    assert int(
        blank_connection_rows[
            f"{field}_is_blank"
        ].sum()
    ) == expected_blank_counts[field]


print("Blank connection-field combinations")
display(blank_connection_combination_summary)

print("Blank connection rows by course")
display(blank_connection_course_summary)

print("Complete blank connection-row inspection")
display(
    blank_connection_rows[
        [
            "source_rowid",
            "race_id",
            "date",
            "course",
            "off",
            "horse",
            "pos",
            "ran",
            "race_name",
            "type",
            "jockey",
            "trainer",
            "owner",
            "blank_connection_fields",
        ]
    ]
)

Blank connection-field combinations


,blank_connection_fields,blank_connection_field_count,runner_rows,distinct_race_ids,distinct_courses,first_observed_date,last_observed_date
0,owner,1,33,20,14,2015-01-03,2022-11-13
1,trainer,1,7,7,4,2015-05-06,2018-09-28
2,jockey,1,2,2,2,2016-04-08,2021-09-05
3,trainer | owner,2,2,1,1,2016-05-07,2016-05-07


Blank connection rows by course


,course,runner_rows,distinct_race_ids,blank_jockey_rows,blank_trainer_rows,blank_owner_rows,first_observed_date,last_observed_date
0,Newcastle (AUS),6,1,0,0,6,2020-10-03,2020-10-03
1,Mombetsu (JPN),5,1,0,0,5,2019-10-31,2019-10-31
2,Ohi (JPN),5,4,0,0,5,2020-12-29,2022-07-13
3,Maisons-Laffitte (FR),4,3,0,4,2,2016-05-07,2018-09-17
4,Saint-Cloud (FR),4,4,0,3,1,2018-09-14,2019-05-13
5,Funabashi (JPN),3,3,0,0,3,2021-04-07,2022-09-28
6,Flemington (AUS),2,1,0,0,2,2020-08-08,2020-08-08
7,Kawasaki (JPN),2,1,0,0,2,2021-12-15,2021-12-15
8,Longchamp (FR),2,2,0,0,2,2015-06-18,2019-06-03
9,Morioka (JPN),2,1,0,0,2,2022-07-18,2022-07-18


Complete blank connection-row inspection


,source_rowid,race_id,date,course,off,horse,pos,ran,race_name,type,jockey,trainer,owner,blank_connection_fields
0,1443,617158,2015-01-03,Santa Anita (USA),11:30,Rattataptap (USA),8,9,Santa Ynez Stakes (Fillies) (Dirt),Flat,Edwin A Maldonado,Jeff Bonde,,owner
1,33009,623280,2015-04-04,Caulfield (AUS),6:10,Post DFrance (NZ),6,11,Le Pine Funerals Easter Cup Handicap) (2yo+) (...,Flat,Patrick Moloney,Stephen Brown,,owner
2,48891,631223,2015-05-03,San Siro (ITY),12:07,Balami Fan (ITY),3,3,Premio Razza Ticino (Turf),Flat,Luca Maniezzi,V Cangiano,,owner
3,50160,627438,2015-05-06,Sonoda (JPN),11:07,Maximum Kaiser (JPN),10,12,Hyogo Championship (Local (Dirt),Flat,Shoichi Kawahara,,Tsuru Nishimori,trainer
4,71791,630083,2015-06-18,Longchamp (FR),11:15,Rappeur Des Mottes (FR),5,6,Prix de Bougival (Maiden) (Unraced 3yo Colts &...,Flat,Anthony Crastus,E Lellouche,,owner
5,189632,648524,2016-04-08,Auteuil (FR),2:15,Ahzana (FR),PU,12,Prix Guy Hunault (Hurdle) (Conditions) (5yo) (...,Hurdle,,L Viel,Laurent Viel,jockey
6,203870,651027,2016-05-07,Maisons-Laffitte (FR),3:30,Star White (FR),7,16,Prix de lEtoile dArtois (Handicap) (4yo) (Roun...,Flat,Tony Piccone,,,trainer | owner
7,203991,651027,2016-05-07,Maisons-Laffitte (FR),3:30,Colombia DEmra (FR),12,16,Prix de lEtoile dArtois (Handicap) (4yo) (Roun...,Flat,Richard Juteau,,,trainer | owner
8,599152,711601,2018-09-10,Chantilly (FR),3:30,Numbers Talk (IRE),15,18,Prix de Foulangues (Handicap) (4yo+) (Turf),Flat,Ronan Thomas,,Ecurie Avant Garde,trainer
9,600778,712047,2018-09-14,Saint-Cloud (FR),3:10,Valley Kid (FR),7,18,Prix de Lamarque (Handicap) (5yo+) (Turf),Flat,Michelle Swinnens,,Stalt Neerhof,trainer


## Stage 32 — Race-level pattern of blank connection fields

Blank connection values may represent different source conditions.

A race where every runner is blank for a field suggests race-level source omission.

A race where only some runners are blank suggests partial runner-level incompleteness.

This stage classifies every affected provisional race and connection field as:

- `whole_race_blank` — every runner row in the provisional race is blank;
- `partial_race_blank` — at least one runner is blank and at least one is populated.

The provisional race key remains:

- `date`;
- `course`;
- `off`.

The classification describes source coverage only.

It does not imply that the connection did not exist.

In [34]:
# Classify blank connection fields at provisional-race level.
#
# SQLite performs the runner-level aggregation so that only the small affected
# race population is returned to pandas.

def build_blank_race_coverage_sql(
    connection: sqlite3.Connection,
    field: str,
) -> pd.DataFrame:
    """
    Return every provisional race containing at least one blank value for the
    selected connection field.
    """
    quoted_field = quote_identifier(field)
    quoted_table = quote_identifier(SOURCE_TABLE)

    return pd.read_sql_query(
        f"""
        SELECT
            date,
            course,
            off,
            race_id,
            COUNT(*) AS race_runner_rows,
            SUM(
                CASE
                    WHEN {quoted_field} = ''
                    THEN 1
                    ELSE 0
                END
            ) AS blank_runner_rows,
            SUM(
                CASE
                    WHEN {quoted_field} IS NOT NULL
                     AND {quoted_field} <> ''
                    THEN 1
                    ELSE 0
                END
            ) AS populated_runner_rows
        FROM {quoted_table}
        WHERE {DATA_ROW_PREDICATE}
        GROUP BY
            date,
            course,
            off,
            race_id
        HAVING SUM(
            CASE
                WHEN {quoted_field} = ''
                THEN 1
                ELSE 0
            END
        ) > 0
        ORDER BY
            blank_runner_rows DESC,
            date,
            course,
            off
        """,
        connection,
    )


blank_race_coverage_profiles = {}


connection = sqlite3.connect(
    f"file:{SOURCE_DB_PATH}?mode=ro",
    uri=True,
)

try:
    for field in CONNECTION_IDENTITY_FIELDS:
        profile = build_blank_race_coverage_sql(
            connection,
            field,
        )

        # Classify the affected race without requiring numpy.
        profile[
            "blank_race_class"
        ] = profile.apply(
            lambda row: (
                "whole_race_blank"
                if row["blank_runner_rows"]
                == row["race_runner_rows"]
                else "partial_race_blank"
            ),
            axis=1,
        )

        blank_race_coverage_profiles[field] = profile

finally:
    connection.close()


# Build a compact cross-field summary.
blank_race_coverage_summary_rows = []


for field in CONNECTION_IDENTITY_FIELDS:
    profile = blank_race_coverage_profiles[field]

    for blank_race_class in [
        "whole_race_blank",
        "partial_race_blank",
    ]:
        selected = profile.loc[
            profile[
                "blank_race_class"
            ].eq(blank_race_class)
        ]

        blank_race_coverage_summary_rows.append(
            {
                "source_field": field,
                "blank_race_class": blank_race_class,
                "affected_races": len(selected),
                "blank_runner_rows": int(
                    selected[
                        "blank_runner_rows"
                    ].sum()
                ),
                "race_runner_rows": int(
                    selected[
                        "race_runner_rows"
                    ].sum()
                ),
            }
        )


blank_race_coverage_summary = pd.DataFrame(
    blank_race_coverage_summary_rows
)


# Reconcile blank-row totals to the earlier finite inspection.
for field in CONNECTION_IDENTITY_FIELDS:
    expected_blank_rows = int(
        blank_connection_rows[
            f"{field}_is_blank"
        ].sum()
    )

    observed_blank_rows = int(
        blank_race_coverage_profiles[
            field
        ][
            "blank_runner_rows"
        ].sum()
    )

    assert observed_blank_rows == expected_blank_rows


print("Race-level blank connection coverage")
display(blank_race_coverage_summary)


for field in CONNECTION_IDENTITY_FIELDS:
    print(
        f"Affected races for blank {field}"
    )

    display(
        blank_race_coverage_profiles[
            field
        ]
        .sort_values(
            [
                "blank_race_class",
                "blank_runner_rows",
                "date",
                "course",
                "off",
            ],
            ascending=[
                True,
                False,
                True,
                True,
                True,
            ],
        )
        .reset_index(drop=True)
    )

Race-level blank connection coverage


,source_field,blank_race_class,affected_races,blank_runner_rows,race_runner_rows
0,jockey,whole_race_blank,0,0,0
1,jockey,partial_race_blank,2,2,19
2,trainer,whole_race_blank,0,0,0
3,trainer,partial_race_blank,8,9,126
4,owner,whole_race_blank,1,5,5
5,owner,partial_race_blank,20,30,162


Affected races for blank jockey


,date,course,off,race_id,race_runner_rows,blank_runner_rows,populated_runner_rows,blank_race_class
0,2016-04-08,Auteuil (FR),2:15,648524,12,1,11,partial_race_blank
1,2021-09-05,Baden-Baden (GER),2:15,792961,7,1,6,partial_race_blank


Affected races for blank trainer


,date,course,off,race_id,race_runner_rows,blank_runner_rows,populated_runner_rows,blank_race_class
0,2016-05-07,Maisons-Laffitte (FR),3:30,651027,16,2,14,partial_race_blank
1,2015-05-06,Sonoda (JPN),11:07,627438,12,1,11,partial_race_blank
2,2018-09-10,Chantilly (FR),3:30,711601,18,1,17,partial_race_blank
3,2018-09-14,Saint-Cloud (FR),3:10,712047,18,1,17,partial_race_blank
4,2018-09-17,Maisons-Laffitte (FR),3:55,712142,15,1,14,partial_race_blank
5,2018-09-17,Maisons-Laffitte (FR),4:25,712151,15,1,14,partial_race_blank
6,2018-09-28,Saint-Cloud (FR),3:40,713069,16,1,15,partial_race_blank
7,2018-09-28,Saint-Cloud (FR),4:10,713071,16,1,15,partial_race_blank


Affected races for blank owner


,date,course,off,race_id,race_runner_rows,blank_runner_rows,populated_runner_rows,blank_race_class
0,2020-10-03,Newcastle (AUS),5:49,769163,11,6,5,partial_race_blank
1,2016-05-07,Maisons-Laffitte (FR),3:30,651027,16,2,14,partial_race_blank
2,2020-08-08,Flemington (AUS),3:05,768529,7,2,5,partial_race_blank
3,2021-12-15,Kawasaki (JPN),11:07,800932,5,2,3,partial_race_blank
4,2022-07-13,Ohi (JPN),11:07,817393,5,2,3,partial_race_blank
5,2022-07-18,Morioka (JPN),11:07,817570,5,2,3,partial_race_blank
6,2015-01-03,Santa Anita (USA),11:30,617158,9,1,8,partial_race_blank
7,2015-04-04,Caulfield (AUS),6:10,623280,11,1,10,partial_race_blank
8,2015-05-03,San Siro (ITY),12:07,631223,3,1,2,partial_race_blank
9,2015-06-18,Longchamp (FR),11:15,630083,6,1,5,partial_race_blank


## Stage 33 — Conclusion from blank connection-field coverage

Blank connection values are extremely rare:

- 2 blank jockey values;
- 9 blank trainer values;
- 35 blank owner values.

The race-level coverage patterns are:

### Jockey

Both blank jockey values occur in partially populated races:

- 2 affected races;
- 2 blank runner rows;
- 17 companion runner rows with populated jockey labels.

No race is wholly blank for jockey.

### Trainer

All blank trainer values occur in partially populated races:

- 8 affected races;
- 9 blank runner rows;
- 117 companion runner rows with populated trainer labels.

No race is wholly blank for trainer.

### Owner

Owner blanks occur in both partial- and whole-race patterns:

- 20 partially affected races containing 30 blank runner rows;
- 1 wholly blank race containing 5 runner rows;
- the wholly blank case is the 2019-10-31 Mombetsu race.

The blank rows otherwise retain ordinary race, horse and result information.

Therefore empty connection strings should be interpreted as source values not supplied.

They must:

- be converted to database nulls;
- not be interpreted as evidence that no connection existed;
- not be filled from companion runners;
- not be inferred from repeated horse, trainer or owner histories;
- remain eligible for later source repair only when supported by independent provenance.

## Stage 34 — Consolidated source-field governance

The preceding stages established the safe treatment of the runner-level connection fields.

This stage records one consolidated decision for each field across:

- source meaning;
- blank handling;
- exact-label identity;
- within-race cardinality;
- punctuation and compound labels;
- cross-role equality;
- normalisation;
- database storage;
- deferred identity resolution.

The table is a notebook-level governance output.

It does not create canonical connection entities.

Therefore empty connection strings should initially be interpreted as source values not supplied.

They must:

- be converted to database nulls in the uncorrected source representation;
- not be interpreted as evidence that no connection existed;
- not be filled from companion runners or inferred from repeated histories;
- be reviewed as a finite manual-repair population;
- be corrected only where an independent authoritative or reputable source identifies the missing value;
- retain repair provenance and the original source value;
- remain null where external verification is unavailable or conflicting.

In [35]:
# Consolidate the notebook's source-semantic findings into one governed table.
#
# These decisions describe safe source handling. They do not claim that exact
# labels are globally unique people, partnerships, organisations or licences.
#
# Blank source values remain preserved in the raw source layer. They may be
# repaired in a separate governed correction layer only where independent
# evidence supplies the missing value and complete provenance is retained.

connection_field_governance = pd.DataFrame(
    [
        {
            "source_field": "jockey",
            "source_level_meaning": (
                "runner-level source-presented jockey label"
            ),
            "blank_treatment": (
                "preserve original empty string in raw source; "
                "represent as null analytically; permit provenance-backed "
                "manual repair from independent evidence"
            ),
            "exact_label_treatment": (
                "preserve exactly as supplied"
            ),
            "within_race_cardinality": (
                "maximum one populated exact label occurrence "
                "per provisional race"
            ),
            "compound_label_treatment": (
                "preserve punctuation and complete raw label"
            ),
            "cross_role_equality": (
                "exact equality with trainer or owner is not "
                "proof of shared identity"
            ),
            "normalisation_policy": (
                "no automatic canonicalisation beyond explicit "
                "missing-value handling"
            ),
            "database_storage": (
                "raw source text plus nullable governed value; "
                "repairs require separate provenance"
            ),
            "deferred_work": (
                "canonical person identity, namesake resolution, "
                "licensing and historical identity changes"
            ),
            "governance_status": "confirmed",
        },
        {
            "source_field": "trainer",
            "source_level_meaning": (
                "runner-level source-presented trainer or "
                "training-connection label"
            ),
            "blank_treatment": (
                "preserve original empty string in raw source; "
                "represent as null analytically; permit provenance-backed "
                "manual repair from independent evidence"
            ),
            "exact_label_treatment": (
                "preserve exactly as supplied"
            ),
            "within_race_cardinality": (
                "one exact label may relate to multiple distinct "
                "runner entries in one provisional race"
            ),
            "compound_label_treatment": (
                "preserve ampersand and other compound labels "
                "atomically; decomposition requires separate evidence"
            ),
            "cross_role_equality": (
                "exact equality with jockey or owner is not "
                "proof of shared identity"
            ),
            "normalisation_policy": (
                "do not split, merge or expand trainer names "
                "automatically"
            ),
            "database_storage": (
                "raw source text plus nullable governed value; "
                "repairs require separate provenance"
            ),
            "deferred_work": (
                "joint-trainer resolution, compressed shared "
                "surnames, canonical people, licences and namesakes"
            ),
            "governance_status": "confirmed",
        },
        {
            "source_field": "owner",
            "source_level_meaning": (
                "runner-level source-presented owner or "
                "ownership-connection label"
            ),
            "blank_treatment": (
                "preserve original empty string in raw source; "
                "represent as null analytically; permit provenance-backed "
                "manual repair from independent evidence"
            ),
            "exact_label_treatment": (
                "preserve exactly as supplied"
            ),
            "within_race_cardinality": (
                "one exact label may relate to multiple distinct "
                "runner entries in one provisional race"
            ),
            "compound_label_treatment": (
                "preserve complete raw label; token order and "
                "punctuation may generate candidates only"
            ),
            "cross_role_equality": (
                "exact equality with jockey or trainer is not "
                "proof of shared identity"
            ),
            "normalisation_policy": (
                "do not classify, split, reorder or canonicalise "
                "owners automatically"
            ),
            "database_storage": (
                "raw source text plus nullable governed value; "
                "repairs require separate provenance"
            ),
            "deferred_work": (
                "people, partnerships, syndicates, companies, "
                "co-owners, beneficial ownership and namesakes"
            ),
            "governance_status": "confirmed",
        },
    ]
)


# Confirm all scoped fields are represented exactly once.
assert set(
    connection_field_governance["source_field"]
) == set(CONNECTION_IDENTITY_FIELDS)

assert connection_field_governance[
    "source_field"
].is_unique


print("Consolidated connection-field governance")
display(connection_field_governance)

Consolidated connection-field governance


,source_field,source_level_meaning,blank_treatment,exact_label_treatment,within_race_cardinality,compound_label_treatment,cross_role_equality,normalisation_policy,database_storage,deferred_work,governance_status
0,jockey,runner-level source-presented jockey label,preserve original empty string in raw source; ...,preserve exactly as supplied,maximum one populated exact label occurrence p...,preserve punctuation and complete raw label,exact equality with trainer or owner is not pr...,no automatic canonicalisation beyond explicit ...,raw source text plus nullable governed value; ...,"canonical person identity, namesake resolution...",confirmed
1,trainer,runner-level source-presented trainer or train...,preserve original empty string in raw source; ...,preserve exactly as supplied,one exact label may relate to multiple distinc...,preserve ampersand and other compound labels a...,exact equality with jockey or owner is not pro...,"do not split, merge or expand trainer names au...",raw source text plus nullable governed value; ...,"joint-trainer resolution, compressed shared su...",confirmed
2,owner,runner-level source-presented owner or ownersh...,preserve original empty string in raw source; ...,preserve exactly as supplied,one exact label may relate to multiple distinc...,preserve complete raw label; token order and p...,exact equality with jockey or trainer is not p...,"do not classify, split, reorder or canonicalis...",raw source text plus nullable governed value; ...,"people, partnerships, syndicates, companies, c...",confirmed


## Stage 35 — Manual verification queue for blank connection values

The source contains 46 blank field occurrences across 43 runner rows:

- 2 missing jockey values;
- 9 missing trainer values;
- 35 missing owner values.

Two rows are blank for both trainer and owner.

Because this is a small finite population, each missing field occurrence can be reviewed against independent race records.

The verification queue preserves one row per missing field occurrence rather than one row per runner. This allows trainer and owner repairs on the same runner to be researched, evidenced and decided separately.

Each queue row records:

- source row and race identifiers;
- race and runner context;
- the missing source field;
- the original source value;
- the companion connection labels;
- fields for the proposed repaired value;
- evidence source and locator;
- verification date;
- decision and confidence;
- reviewer notes.

Permitted decisions are:

- `pending`;
- `verified_repair`;
- `verified_unavailable`;
- `conflicting_evidence`;
- `not_researched`.

No repaired value should enter the governed database unless its decision is `verified_repair` and its provenance fields are populated.

In [36]:
# Create one manual-verification record per blank field occurrence.
#
# There are 43 affected runner rows but 46 missing field occurrences because
# two rows are blank for both trainer and owner.

manual_connection_repair_rows = []


for _, source_row in blank_connection_rows.iterrows():
    for field in CONNECTION_IDENTITY_FIELDS:
        if not source_row[f"{field}_is_blank"]:
            continue

        manual_connection_repair_rows.append(
            {
                "source_rowid": int(
                    source_row["source_rowid"]
                ),
                "race_id": source_row["race_id"],
                "date": source_row["date"],
                "course": source_row["course"],
                "off": source_row["off"],
                "race_name": source_row["race_name"],
                "race_type": source_row["type"],
                "horse": source_row["horse"],
                "position": source_row["pos"],
                "declared_runners": source_row["ran"],
                "missing_source_field": field,
                "original_source_value": source_row[field],
                "source_jockey": source_row["jockey"],
                "source_trainer": source_row["trainer"],
                "source_owner": source_row["owner"],
                "proposed_repaired_value": pd.NA,
                "evidence_source_name": pd.NA,
                "evidence_source_type": pd.NA,
                "evidence_locator": pd.NA,
                "evidence_accessed_date": pd.NA,
                "verification_decision": "pending",
                "verification_confidence": pd.NA,
                "reviewer_notes": pd.NA,
            }
        )


manual_connection_repair_queue = pd.DataFrame(
    manual_connection_repair_rows
)


manual_string_columns = [
    "proposed_repaired_value",
    "evidence_source_name",
    "evidence_source_type",
    "evidence_locator",
    "evidence_accessed_date",
    "verification_decision",
    "verification_confidence",
    "reviewer_notes",
]


for column in manual_string_columns:
    manual_connection_repair_queue[column] = (
        manual_connection_repair_queue[
            column
        ].astype("string")
    )


manual_connection_repair_queue = (
    manual_connection_repair_queue.sort_values(
        [
            "missing_source_field",
            "date",
            "course",
            "off",
            "source_rowid",
        ]
    )
    .reset_index(drop=True)
)


manual_connection_repair_queue.insert(
    0,
    "repair_record_id",
    [
        f"connection_blank_{number:03d}"
        for number in range(
            1,
            len(manual_connection_repair_queue) + 1,
        )
    ],
)


assert len(manual_connection_repair_queue) == 46

assert (
    manual_connection_repair_queue[
        "missing_source_field"
    ]
    .value_counts()
    .to_dict()
    == {
        "owner": 35,
        "trainer": 9,
        "jockey": 2,
    }
)

assert manual_connection_repair_queue[
    "original_source_value"
].eq("").all()

assert manual_connection_repair_queue[
    "repair_record_id"
].is_unique


manual_connection_repair_summary = (
    manual_connection_repair_queue.groupby(
        [
            "missing_source_field",
            "verification_decision",
        ],
        as_index=False,
    )
    .agg(
        repair_records=("repair_record_id", "size"),
        distinct_runner_rows=("source_rowid", "nunique"),
        distinct_races=("race_id", "nunique"),
        distinct_courses=("course", "nunique"),
        first_observed_date=("date", "min"),
        last_observed_date=("date", "max"),
    )
)


print("Manual connection-repair queue summary")
display(manual_connection_repair_summary)

print("Complete manual connection-repair queue")
display(manual_connection_repair_queue)

Manual connection-repair queue summary


,missing_source_field,verification_decision,repair_records,distinct_runner_rows,distinct_races,distinct_courses,first_observed_date,last_observed_date
0,jockey,pending,2,2,2,2,2016-04-08,2021-09-05
1,owner,pending,35,35,21,15,2015-01-03,2022-11-13
2,trainer,pending,9,9,8,4,2015-05-06,2018-09-28


Complete manual connection-repair queue


,repair_record_id,source_rowid,race_id,date,course,off,race_name,race_type,horse,position,...,source_trainer,source_owner,proposed_repaired_value,evidence_source_name,evidence_source_type,evidence_locator,evidence_accessed_date,verification_decision,verification_confidence,reviewer_notes
0,connection_blank_001,189632,648524,2016-04-08,Auteuil (FR),2:15,Prix Guy Hunault (Hurdle) (Conditions) (5yo) (...,Hurdle,Ahzana (FR),PU,...,L Viel,Laurent Viel,<NA>,<NA>,<NA>,<NA>,<NA>,pending,<NA>,<NA>
1,connection_blank_002,1068784,792961,2021-09-05,Baden-Baden (GER),2:15,149th Wettstar Grosser Preis von Baden (3yo+)...,Flat,Millebosc (FR),PU,...,Mlle Stephanie Nigge,Gerard Augustin Normand,<NA>,<NA>,<NA>,<NA>,<NA>,pending,<NA>,<NA>
2,connection_blank_003,1443,617158,2015-01-03,Santa Anita (USA),11:30,Santa Ynez Stakes (Fillies) (Dirt),Flat,Rattataptap (USA),8,...,Jeff Bonde,,<NA>,<NA>,<NA>,<NA>,<NA>,pending,<NA>,<NA>
3,connection_blank_004,33009,623280,2015-04-04,Caulfield (AUS),6:10,Le Pine Funerals Easter Cup Handicap) (2yo+) (...,Flat,Post DFrance (NZ),6,...,Stephen Brown,,<NA>,<NA>,<NA>,<NA>,<NA>,pending,<NA>,<NA>
4,connection_blank_005,48891,631223,2015-05-03,San Siro (ITY),12:07,Premio Razza Ticino (Turf),Flat,Balami Fan (ITY),3,...,V Cangiano,,<NA>,<NA>,<NA>,<NA>,<NA>,pending,<NA>,<NA>
5,connection_blank_006,71791,630083,2015-06-18,Longchamp (FR),11:15,Prix de Bougival (Maiden) (Unraced 3yo Colts &...,Flat,Rappeur Des Mottes (FR),5,...,E Lellouche,,<NA>,<NA>,<NA>,<NA>,<NA>,pending,<NA>,<NA>
6,connection_blank_007,203870,651027,2016-05-07,Maisons-Laffitte (FR),3:30,Prix de lEtoile dArtois (Handicap) (4yo) (Roun...,Flat,Star White (FR),7,...,,,<NA>,<NA>,<NA>,<NA>,<NA>,pending,<NA>,<NA>
7,connection_blank_008,203991,651027,2016-05-07,Maisons-Laffitte (FR),3:30,Prix de lEtoile dArtois (Handicap) (4yo) (Roun...,Flat,Colombia DEmra (FR),12,...,,,<NA>,<NA>,<NA>,<NA>,<NA>,pending,<NA>,<NA>
8,connection_blank_009,712612,730833,2019-05-13,Saint-Cloud (FR),4:55,Prix de la Vallee du Lot (Handicap) (4yo+) (Turf),Flat,Le Letty (FR),13,...,Mme M-C Chaalon,,<NA>,<NA>,<NA>,<NA>,<NA>,pending,<NA>,<NA>
9,connection_blank_010,725030,732516,2019-06-03,Longchamp (FR),4:55,Prix de la Porte de la Seine (Handicap) (4yo+)...,Flat,Le Letty (FR),9,...,Mme M-C Chaalon,,<NA>,<NA>,<NA>,<NA>,<NA>,pending,<NA>,<NA>


## Stage 36 — Persist and reload the manual connection-repair queue

The manual verification queue is a reusable notebook output.

It must be persisted before external research begins so that:

- the original 46-record pending population is preserved;
- research decisions can be edited outside the notebook if necessary;
- the queue can be reloaded in a fresh kernel;
- later database repairs can be traced to a stable evidence record;
- completed and unresolved cases remain distinguishable.

The persisted file contains source context and verification fields only.

It does not modify the raw source database.

In [37]:
# Persist the complete manual connection-repair queue.
#
# Keep this output separate from the immutable raw source database.
connection_repair_output_directory = (
    PROJECT_ROOT
    / "data"
    / "derived"
    / "connection_identity"
)

connection_repair_output_directory.mkdir(
    parents=True,
    exist_ok=True,
)


manual_connection_repair_queue_path = (
    connection_repair_output_directory
    / "manual_connection_repair_queue.csv"
)


manual_connection_repair_queue.to_csv(
    manual_connection_repair_queue_path,
    index=False,
)


# Reload immediately to verify that the persisted artifact is usable.
reloaded_manual_connection_repair_queue = (
    pd.read_csv(
        manual_connection_repair_queue_path,
        dtype={
            "repair_record_id": "string",
            "missing_source_field": "string",
            "original_source_value": "string",
            "source_jockey": "string",
            "source_trainer": "string",
            "source_owner": "string",
            "proposed_repaired_value": "string",
            "evidence_source_name": "string",
            "evidence_source_type": "string",
            "evidence_locator": "string",
            "evidence_accessed_date": "string",
            "verification_decision": "string",
            "verification_confidence": "string",
            "reviewer_notes": "string",
        },
        keep_default_na=True,
    )
)


# Confirm the persisted and reloaded queue retains its governing structure.
assert len(
    reloaded_manual_connection_repair_queue
) == 46

assert reloaded_manual_connection_repair_queue[
    "repair_record_id"
].is_unique

assert set(
    reloaded_manual_connection_repair_queue[
        "repair_record_id"
    ]
) == set(
    manual_connection_repair_queue[
        "repair_record_id"
    ]
)

assert (
    reloaded_manual_connection_repair_queue[
        "missing_source_field"
    ]
    .value_counts()
    .to_dict()
    == {
        "owner": 35,
        "trainer": 9,
        "jockey": 2,
    }
)

assert reloaded_manual_connection_repair_queue[
    "verification_decision"
].eq("pending").all()


manual_connection_repair_persistence_summary = (
    pd.DataFrame(
        [
            {
                "output_path": str(
                    manual_connection_repair_queue_path.relative_to(
                        PROJECT_ROOT
                    )
                ),
                "persisted_records": len(
                    manual_connection_repair_queue
                ),
                "reloaded_records": len(
                    reloaded_manual_connection_repair_queue
                ),
                "distinct_repair_record_ids": int(
                    reloaded_manual_connection_repair_queue[
                        "repair_record_id"
                    ].nunique()
                ),
                "pending_records": int(
                    reloaded_manual_connection_repair_queue[
                        "verification_decision"
                    ].eq("pending").sum()
                ),
                "reload_verified": True,
            }
        ]
    )
)


print("Manual connection-repair queue persistence")
display(manual_connection_repair_persistence_summary)

Manual connection-repair queue persistence


,output_path,persisted_records,reloaded_records,distinct_repair_record_ids,pending_records,reload_verified
0,data/derived/connection_identity/manual_connec...,46,46,46,46,True


## Stage 37 — External verification batch 1: missing jockey values

The two missing jockey values were checked against independent historical race records.

### `connection_blank_001`

- horse: `Ahzana (FR)`;
- race: Prix Guy Hunault, Auteuil, 2016-04-08;
- verified jockey: `T. Viel`;
- evidence: two independent historical race sources identify T. Viel as Ahzana's jockey in this race.

The abbreviated source presentation is retained rather than expanding the name to a canonical person identity.

### `connection_blank_002`

- horse: `Millebosc (FR)`;
- race: 149th Wettstar Grosser Preis von Baden, Baden-Baden, 2021-09-05;
- verified jockey: `Adrie de Vries`;
- evidence: the complete historical Racing Post result identifies Adrie de Vries as the jockey.

Both records qualify as provenance-backed repairs.

The original empty strings remain preserved in the raw source layer.

In [38]:
# Apply the first externally verified repair batch to the persisted manual
# queue.
#
# These updates affect the governed repair artifact only. They do not overwrite
# the original source database or remove the original empty-string evidence.

from datetime import date


verification_date = date.today().isoformat()


jockey_repair_updates = {
    "connection_blank_001": {
        "proposed_repaired_value": "T. Viel",
        "evidence_source_name": (
            "Canalturf and Coin-Turf"
        ),
        "evidence_source_type": (
            "independent historical horse record and racecard"
        ),
        "evidence_locator": (
            "https://www.canalturf.com/courses_fiche_cheval.php"
            "?idcheval=176640 | "
            "https://www.coin-turf.fr/programmes-courses/"
            "08042016/3080_auteuil/prix-guy-hunault"
        ),
        "evidence_accessed_date": verification_date,
        "verification_decision": "verified_repair",
        "verification_confidence": "high",
        "reviewer_notes": (
            "Both sources identify T. Viel as Ahzana's jockey in "
            "the Prix Guy Hunault on 2016-04-08. Preserve the "
            "source-style abbreviated label; no canonical person "
            "identity is asserted."
        ),
    },
    "connection_blank_002": {
        "proposed_repaired_value": "Adrie de Vries",
        "evidence_source_name": "Racing Post",
        "evidence_source_type": (
            "complete historical race result"
        ),
        "evidence_locator": (
            "https://www.racingpost.com/results/207/"
            "baden-baden/2021-09-05/792961"
        ),
        "evidence_accessed_date": verification_date,
        "verification_decision": "verified_repair",
        "verification_confidence": "high",
        "reviewer_notes": (
            "The complete result identifies Adrie de Vries as "
            "Millebosc's jockey in the Grosser Preis von Baden "
            "on 2021-09-05."
        ),
    },
}


for repair_record_id, update_values in (
    jockey_repair_updates.items()
):
    matching_rows = (
        manual_connection_repair_queue[
            "repair_record_id"
        ].eq(repair_record_id)
    )

    assert int(matching_rows.sum()) == 1

    assert (
        manual_connection_repair_queue.loc[
            matching_rows,
            "missing_source_field",
        ].iloc[0]
        == "jockey"
    )

    for column, value in update_values.items():
        manual_connection_repair_queue.loc[
            matching_rows,
            column,
        ] = value


# Validate every verified repair has the minimum required provenance.
verified_repairs = (
    manual_connection_repair_queue.loc[
        manual_connection_repair_queue[
            "verification_decision"
        ].eq("verified_repair")
    ]
)


required_verified_repair_columns = [
    "proposed_repaired_value",
    "evidence_source_name",
    "evidence_source_type",
    "evidence_locator",
    "evidence_accessed_date",
    "verification_confidence",
]


assert len(verified_repairs) == 2

assert verified_repairs[
    required_verified_repair_columns
].notna().all().all()

assert verified_repairs[
    "proposed_repaired_value"
].str.strip().ne("").all()


# Overwrite the derived queue artifact with the reviewed state.
manual_connection_repair_queue.to_csv(
    manual_connection_repair_queue_path,
    index=False,
)


# Reload and confirm that the completed decisions persist.
reloaded_manual_connection_repair_queue = (
    pd.read_csv(
        manual_connection_repair_queue_path,
        dtype="string",
        keep_default_na=True,
    )
)


assert len(
    reloaded_manual_connection_repair_queue
) == 46

assert int(
    reloaded_manual_connection_repair_queue[
        "verification_decision"
    ].eq("verified_repair").sum()
) == 2

assert int(
    reloaded_manual_connection_repair_queue[
        "verification_decision"
    ].eq("pending").sum()
) == 44


manual_connection_repair_status_summary = (
    reloaded_manual_connection_repair_queue.groupby(
        [
            "missing_source_field",
            "verification_decision",
        ],
        as_index=False,
        dropna=False,
    )
    .agg(
        repair_records=("repair_record_id", "size"),
        distinct_runner_rows=("source_rowid", "nunique"),
        distinct_races=("race_id", "nunique"),
    )
    .sort_values(
        [
            "missing_source_field",
            "verification_decision",
        ]
    )
    .reset_index(drop=True)
)


print("Manual connection-repair status after jockey review")
display(manual_connection_repair_status_summary)

print("Verified jockey repairs")
display(
    reloaded_manual_connection_repair_queue.loc[
        reloaded_manual_connection_repair_queue[
            "missing_source_field"
        ].eq("jockey")
    ]
)

Manual connection-repair status after jockey review


,missing_source_field,verification_decision,repair_records,distinct_runner_rows,distinct_races
0,jockey,verified_repair,2,2,2
1,owner,pending,35,35,21
2,trainer,pending,9,9,8


Verified jockey repairs


,repair_record_id,source_rowid,race_id,date,course,off,race_name,race_type,horse,position,...,source_trainer,source_owner,proposed_repaired_value,evidence_source_name,evidence_source_type,evidence_locator,evidence_accessed_date,verification_decision,verification_confidence,reviewer_notes
0,connection_blank_001,189632,648524,2016-04-08,Auteuil (FR),2:15,Prix Guy Hunault (Hurdle) (Conditions) (5yo) (...,Hurdle,Ahzana (FR),PU,...,L Viel,Laurent Viel,T. Viel,Canalturf and Coin-Turf,independent historical horse record and racecard,https://www.canalturf.com/courses_fiche_cheval...,2026-08-03,verified_repair,high,Both sources identify T. Viel as Ahzana's jock...
1,connection_blank_002,1068784,792961,2021-09-05,Baden-Baden (GER),2:15,149th Wettstar Grosser Preis von Baden (3yo+)...,Flat,Millebosc (FR),PU,...,Mlle Stephanie Nigge,Gerard Augustin Normand,Adrie de Vries,Racing Post,complete historical race result,https://www.racingpost.com/results/207/baden-b...,2026-08-03,verified_repair,high,The complete result identifies Adrie de Vries ...


## Stage 38 — Complete external review of blank connection values

The remaining 44 missing connection values were reviewed as one finite external-verification population.

The review produced:

- 26 additional `verified_repair` decisions;
- 5 `conflicting_evidence` decisions;
- 13 `insufficient_evidence` decisions;
- no records left `pending`.

Together with the two previously verified jockey repairs, the complete 46-record queue contains:

- 28 provenance-backed repairs;
- 5 records where exact-date sources disagree;
- 13 records where the available evidence does not establish the missing value at the target race date.

A `conflicting_evidence` decision means that two or more apparently relevant sources provide materially different connection labels for the same horse and race.

An `insufficient_evidence` decision does not mean that the value is permanently unknowable. It means that the sources reviewed did not support a race-date repair strongly enough to enter the governed database.

Neither category receives a proposed repaired value.

The raw source database remains unchanged.

Only records classified as `verified_repair`, with a populated repaired value and complete provenance, are eligible for a governed correction layer.

In [39]:
# Complete the external-evidence review for all 44 records that remained after
# the two verified jockey repairs.
#
# The raw source database is not modified. This cell updates only the governed
# repair queue and persists the completed evidence log.

from datetime import date


verification_date = date.today().isoformat()


# Reload the latest persisted queue so this cell remains reproducible after a
# fresh-kernel restart.
manual_connection_repair_queue = pd.read_csv(
    manual_connection_repair_queue_path,
    dtype={
        "repair_record_id": "string",
        "missing_source_field": "string",
        "original_source_value": "string",
        "source_jockey": "string",
        "source_trainer": "string",
        "source_owner": "string",
        "proposed_repaired_value": "string",
        "evidence_source_name": "string",
        "evidence_source_type": "string",
        "evidence_locator": "string",
        "evidence_accessed_date": "string",
        "verification_decision": "string",
        "verification_confidence": "string",
        "reviewer_notes": "string",
    },
    keep_default_na=True,
)


assert len(manual_connection_repair_queue) == 46

assert manual_connection_repair_queue[
    "repair_record_id"
].is_unique


# The two jockey repairs must already be present.
existing_jockey_decisions = (
    manual_connection_repair_queue.loc[
        manual_connection_repair_queue[
            "repair_record_id"
        ].isin(
            [
                "connection_blank_001",
                "connection_blank_002",
            ]
        ),
        "verification_decision",
    ]
)

assert existing_jockey_decisions.eq(
    "verified_repair"
).all()


def verified_repair(
    value: str,
    source_name: str,
    source_type: str,
    locator: str,
    notes: str,
    confidence: str = "high",
) -> dict:
    """
    Construct a provenance-backed verified-repair decision.
    """
    return {
        "proposed_repaired_value": value,
        "evidence_source_name": source_name,
        "evidence_source_type": source_type,
        "evidence_locator": locator,
        "evidence_accessed_date": verification_date,
        "verification_decision": "verified_repair",
        "verification_confidence": confidence,
        "reviewer_notes": notes,
    }


def conflicting_evidence(
    source_name: str,
    source_type: str,
    locator: str,
    notes: str,
) -> dict:
    """
    Construct a decision where materially different exact-date labels were
    found and no value can safely be selected.
    """
    return {
        "proposed_repaired_value": pd.NA,
        "evidence_source_name": source_name,
        "evidence_source_type": source_type,
        "evidence_locator": locator,
        "evidence_accessed_date": verification_date,
        "verification_decision": "conflicting_evidence",
        "verification_confidence": "conflict",
        "reviewer_notes": notes,
    }


def insufficient_evidence(
    source_name: str,
    source_type: str,
    locator: str,
    notes: str,
) -> dict:
    """
    Construct a decision where the reviewed evidence does not establish the
    missing value at the target race date.
    """
    return {
        "proposed_repaired_value": pd.NA,
        "evidence_source_name": source_name,
        "evidence_source_type": source_type,
        "evidence_locator": locator,
        "evidence_accessed_date": verification_date,
        "verification_decision": "insufficient_evidence",
        "verification_confidence": "insufficient",
        "reviewer_notes": notes,
    }


remaining_connection_review_updates = {
    # ------------------------------------------------------------------
    # Missing owner values
    # ------------------------------------------------------------------

    "connection_blank_003": insufficient_evidence(
        source_name=(
            "Santa Anita historical result coverage; "
            "Paulick Report contemporary profile"
        ),
        source_type=(
            "target-race result and near-date ownership reporting"
        ),
        locator=(
            "Santa Anita, 2015-01-03, Santa Ynez Stakes, "
            "race_id 617158 | Paulick Report, 2015-02-26, "
            "Rattataptap Santa Ysabel preview"
        ),
        notes=(
            "Near-date contemporary reporting identifies Mark "
            "DeDomenico LLC, Jerry Durant and Michael House, but an "
            "owner declaration tied directly enough to the target "
            "Santa Ynez record was not located. Do not infer backwards."
        ),
    ),

    "connection_blank_004": insufficient_evidence(
        source_name="Racing Post and Australian historical result searches",
        source_type="exact historical race result search",
        locator=(
            "Caulfield, 2015-04-04, Le Pine Funerals Easter Cup, "
            "race_id 623280, Post DFrance"
        ),
        notes=(
            "The target runner and result were located, but the target-date "
            "owner was not exposed in the reviewed records."
        ),
    ),

    "connection_blank_005": insufficient_evidence(
        source_name="Italian historical result and horse-profile searches",
        source_type="exact historical race and horse-record search",
        locator=(
            "San Siro, 2015-05-03, Premio Razza Ticino, "
            "race_id 631223, Balami Fan"
        ),
        notes=(
            "No sufficiently reliable target-date owner declaration was "
            "located."
        ),
    ),

    "connection_blank_006": insufficient_evidence(
        source_name="Canalturf and later horse-profile sources",
        source_type=(
            "exact historical result plus later ownership-profile evidence"
        ),
        locator=(
            "https://www.canalturf.com/resultats-PMU/2015-06-18/"
            "longchamp/120411_prix-de-bougival.html"
        ),
        notes=(
            "Later sources associate Rappeur Des Mottes with Ecurie Des "
            "Mouillotins, but the available material does not establish "
            "that ownership on 2015-06-18. A later leasing reference makes "
            "backward inference unsafe."
        ),
    ),

    "connection_blank_007": insufficient_evidence(
        source_name="Coin-Turf",
        source_type="exact historical racecard",
        locator=(
            "https://www.coin-turf.fr/programmes-courses/07052016/"
            "4437_maisons-laffitte/"
            "prix-de-l-etoile-d-artois-prix-de-l-etoile-du-bon-secours"
        ),
        notes=(
            "The exact racecard identifies Star White and its trainer but "
            "does not provide a sufficiently supported owner value."
        ),
    ),

    "connection_blank_008": insufficient_evidence(
        source_name="Coin-Turf and horse-history searches",
        source_type="exact historical racecard and horse-record search",
        locator=(
            "https://www.coin-turf.fr/programmes-courses/07052016/"
            "4437_maisons-laffitte/"
            "prix-de-l-etoile-d-artois-prix-de-l-etoile-du-bon-secours"
        ),
        notes=(
            "The exact racecard identifies Colombia DEmra and its trainer "
            "but does not establish the owner."
        ),
    ),

    "connection_blank_009": verified_repair(
        value="Edouard Desespringalle",
        source_name="Turfoo",
        source_type="dated horse profile covering the exact target result",
        locator="https://www.turfoo.fr/fiches/chevaux/le-letty/",
        notes=(
            "The dated Le Letty profile covers the Saint-Cloud result and "
            "identifies Edouard Desespringalle as owner."
        ),
    ),

    "connection_blank_010": verified_repair(
        value="Edouard Desespringalle",
        source_name="Turfoo",
        source_type="dated horse profile covering the exact target result",
        locator="https://www.turfoo.fr/fiches/chevaux/le-letty/",
        notes=(
            "The dated Le Letty profile covers the Longchamp result and "
            "identifies Edouard Desespringalle as owner."
        ),
    ),

    "connection_blank_011": verified_repair(
        value="Mastec Co Ltd",
        source_name="netkeiba",
        source_type="exact historical race result",
        locator="https://db.netkeiba.com/race/201930103111/",
        notes=(
            "The exact Hokkaido Nisai Yushun result lists 株式会社マステック "
            "as the owner of Abenin Dream."
        ),
    ),

    "connection_blank_012": verified_repair(
        value="Makoto Kato",
        source_name="netkeiba and Rakuten Keiba",
        source_type="exact historical race result",
        locator="https://db.netkeiba.com/race/201930103111/",
        notes=(
            "The exact Hokkaido Nisai Yushun result lists 加藤誠 as owner "
            "of Chimera Verite."
        ),
    ),

    "connection_blank_013": verified_repair(
        value="Thoroughbred Club Ruffian Co Ltd",
        source_name="netkeiba and Rakuten Keiba",
        source_type="exact historical race result",
        locator="https://db.netkeiba.com/race/201930103111/",
        notes=(
            "The exact result lists サラブレッドクラブ・ラフィアン as "
            "owner of Meiner Astoria."
        ),
    ),

    "connection_blank_014": verified_repair(
        value="Kaneko Makoto Holdings Co Ltd",
        source_name="netkeiba",
        source_type="exact historical race result",
        locator="https://db.netkeiba.com/race/201930103111/",
        notes=(
            "The exact result lists 金子真人ホールディングス as owner "
            "of Pionono."
        ),
    ),

    "connection_blank_015": verified_repair(
        value="Toshiaki Date",
        source_name="netkeiba",
        source_type="exact historical race result",
        locator="https://db.netkeiba.com/race/201930103111/",
        notes=(
            "The exact result lists 伊達敏明 as owner of Adjuvant."
        ),
    ),

    "connection_blank_016": verified_repair(
        value="R D Pfitzner",
        source_name="Victoria Racing Club",
        source_type="official race-day racebook",
        locator="https://online.flippingbook.com/view/72851",
        notes=(
            "The official Flemington racebook lists R D Pfitzner as owner "
            "of Bertwhistle in the target TAB Handicap."
        ),
    ),

    "connection_blank_017": verified_repair(
        value=(
            "N Mashni, C M Barrett, Mrs J L Ince, D N Kwasha, "
            "Ms A C Sherry, 21st Century Racing Pty Ltd, "
            "First Light Racing Pty Ltd, F M L M Racing, "
            "S J V & Sulie"
        ),
        source_name="Victoria Racing Club",
        source_type="official race-day racebook",
        locator="https://online.flippingbook.com/view/72851",
        notes=(
            "The official Flemington racebook supplies the complete "
            "ownership group for Flag Edition. Syndicate-manager details "
            "remain available in the cited source."
        ),
    ),

    "connection_blank_018": insufficient_evidence(
        source_name="Racing NSW form material and target-race searches",
        source_type="period form guide and exact historical result search",
        locator=(
            "Newcastle (AUS), 2020-10-03, Happy 60th Nern Maiden Plate, "
            "race_id 769163, Nu Jiang"
        ),
        notes=(
            "Period material associates the horse with Zhongli "
            "Thoroughbreds, but a sufficiently direct owner declaration "
            "for the target race was not located."
        ),
    ),

    "connection_blank_019": insufficient_evidence(
        source_name="Minervini Racing, Vinery and target-race searches",
        source_type="stable profile and historical race search",
        locator=(
            "Newcastle (AUS), 2020-10-03, race_id 769163, Media Man"
        ),
        notes=(
            "Later and stable-level ownership descriptions were located, "
            "but the exact ownership group on the target date was not "
            "established."
        ),
    ),

    "connection_blank_020": insufficient_evidence(
        source_name="Australian historical result and horse-profile searches",
        source_type="exact historical race and horse-record search",
        locator=(
            "Newcastle (AUS), 2020-10-03, race_id 769163, Luftwaffe"
        ),
        notes=(
            "No sufficiently direct target-date owner declaration was "
            "located."
        ),
    ),

    "connection_blank_021": insufficient_evidence(
        source_name="Australian historical result and sale-record searches",
        source_type="exact historical race and horse-record search",
        locator=(
            "Newcastle (AUS), 2020-10-03, race_id 769163, Stormwater"
        ),
        notes=(
            "Horse and sale evidence was found, but it does not establish "
            "the owner at the target race date."
        ),
    ),

    "connection_blank_022": insufficient_evidence(
        source_name="Australian historical result and later race records",
        source_type="target-race search and later ownership records",
        locator=(
            "Newcastle (AUS), 2020-10-03, race_id 769163, Deel With Me"
        ),
        notes=(
            "Later records show differing ownership descriptions. They "
            "cannot safely be projected back to the target date."
        ),
    ),

    "connection_blank_023": insufficient_evidence(
        source_name="Australian historical result and later horse records",
        source_type="target-race search and later ownership records",
        locator=(
            "Newcastle (AUS), 2020-10-03, race_id 769163, Imperial Grey"
        ),
        notes=(
            "Only later ownership information was located. Target-date "
            "ownership remains unsupported."
        ),
    ),

    "connection_blank_024": verified_repair(
        value="Kosei Yoshihashi",
        source_name="netkeiba and JBIS-Search",
        source_type="exact historical result and official horse record",
        locator="https://en.netkeiba.com/db/race/202044122910/",
        notes=(
            "The exact Tokyo Daishoten record identifies Kosei Yoshihashi "
            "as owner of Casino Fountain."
        ),
    ),

    "connection_blank_025": verified_repair(
        value="Mitsunari Hiromatsu",
        source_name="Rakuten Keiba",
        source_type="exact historical racecard",
        locator=(
            "Rakuten Keiba — Funabashi, 2021-04-07, Marine Cup, "
            "Absolute Queen"
        ),
        notes=(
            "The exact racecard lists 廣松光成 as owner of Absolute Queen."
        ),
    ),

    "connection_blank_026": verified_repair(
        value="Hiroshi Joichi",
        source_name="netkeiba and official Japanese racing records",
        source_type="exact historical race result",
        locator="https://en.netkeiba.com/db/race/202144071411/",
        notes=(
            "The exact Japan Dirt Derby record identifies Hiroshi Joichi "
            "as owner of Castle Top."
        ),
    ),

    "connection_blank_027": verified_repair(
        value="Yusuke Kobayashi",
        source_name="Rakuten Keiba",
        source_type="exact historical racecard",
        locator=(
            "Rakuten Keiba — Funabashi, 2021-12-01, Queen Sho, "
            "Dear Rickey"
        ),
        notes=(
            "The exact racecard lists 小林祐介 as owner of Dear Rickey."
        ),
    ),

    "connection_blank_028": verified_repair(
        value="Kumiko Hara",
        source_name="netkeiba",
        source_type="exact historical race result",
        locator="https://db.netkeiba.com/race/202145121511/",
        notes=(
            "The exact Zen-Nippon Nisai Yushun result lists 原久美子 as "
            "owner of SIl Te Plait."
        ),
    ),

    "connection_blank_029": verified_repair(
        value="Carrot Farm",
        source_name="netkeiba and Rakuten Keiba",
        source_type="exact historical result and racecard",
        locator="https://db.netkeiba.com/race/202145121511/",
        notes=(
            "The exact record identifies Carrot Farm as owner of Praelude."
        ),
    ),

    "connection_blank_030": verified_repair(
        value="Foret Bleu Co Ltd",
        source_name="Rakuten Keiba and Japanese owner records",
        source_type="exact historical racecard",
        locator=(
            "Rakuten Keiba — Ohi, 2022-04-20, Tokyo Sprint, Gishigishi"
        ),
        notes=(
            "The exact racecard lists 有限会社フォレブルー as owner of "
            "Gishigishi."
        ),
    ),

    "connection_blank_031": verified_repair(
        value="Tomohiro Iizuka",
        source_name="Rakuten Keiba, netkeiba and Japanese name record",
        source_type="exact racecard plus independent name-reading evidence",
        locator="https://en.netkeiba.com/db/race/202244071311/",
        notes=(
            "The exact Japanese racecard lists 飯塚倫尋 as owner of "
            "Cryogenic. Independent Japanese material supplies the reading "
            "Tomohiro Iizuka."
        ),
    ),

    "connection_blank_032": verified_repair(
        value="Toshio Terada",
        source_name="netkeiba",
        source_type="exact English historical race result",
        locator="https://en.netkeiba.com/db/race/202244071311/",
        notes=(
            "The exact Japan Dirt Derby result identifies Toshio Terada "
            "as owner of Hapi."
        ),
    ),

    "connection_blank_033": insufficient_evidence(
        source_name="Racing Post and historical Irish result searches",
        source_type="exact historical result plus later ownership evidence",
        locator=(
            "Killarney, 2022-07-15, Dawn Milk Omega Mares Maiden Hurdle, "
            "race_id 817263, Fiery Brown"
        ),
        notes=(
            "A later result lists Christy OConnell and William Tanner, but "
            "the reviewed target-date result does not establish that the "
            "same ownership applied on 2022-07-15."
        ),
    ),

    "connection_blank_034": verified_repair(
        value="Masatoshi Suzuki",
        source_name="Japanese target-race listing and bilingual race record",
        source_type="exact historical race listing",
        locator=(
            "Morioka, 2022-07-18, Mercury Cup, Vacation — "
            "OWNER 鈴木雅俊 / MASATOSHI SUZUKI"
        ),
        notes=(
            "The exact Mercury Cup listing supplies both the Japanese owner "
            "name and its Romanised presentation."
        ),
        confidence="medium",
    ),

    "connection_blank_035": verified_repair(
        value="Tomohiro Ozaki",
        source_name="Japanese target-race listing and bilingual race record",
        source_type="exact historical race listing",
        locator=(
            "Morioka, 2022-07-18, Mercury Cup, Giga King — "
            "OWNER 尾崎智大 / TOMOHIRO OZAKI"
        ),
        notes=(
            "The exact Mercury Cup listing supplies both the Japanese owner "
            "name and its Romanised presentation."
        ),
        confidence="medium",
    ),

    "connection_blank_036": verified_repair(
        value="Hero Racing Co Ltd",
        source_name="Rakuten Keiba and Japanese racing records",
        source_type="exact historical racecard",
        locator=(
            "Rakuten Keiba — Funabashi, 2022-09-28, Nippon TV Hai, "
            "Giga King"
        ),
        notes=(
            "The exact racecard identifies Hero Racing as owner of "
            "Giga King on the target date."
        ),
    ),

    "connection_blank_037": verified_repair(
        value="Lee Bon Hee",
        source_name="Racing Post",
        source_type="complete exact historical race result",
        locator=(
            "https://www.racingpost.com/results/1350/busan/"
            "2022-11-13/830140"
        ),
        notes=(
            "The complete result explicitly lists Lee Bon Hee as the "
            "second-place owner of Happy Fever."
        ),
    ),

    # ------------------------------------------------------------------
    # Missing trainer values
    # ------------------------------------------------------------------

    "connection_blank_038": verified_repair(
        value="Masaya Komura",
        source_name="JBIS-Search",
        source_type="official exact historical race result",
        locator="https://www.jbis.jp/race/result/20150506/227/10/",
        notes=(
            "The official Hyogo Championship result lists MASAYA KOMURA "
            "(SONODA) as trainer of Maximum Kaiser. The location suffix is "
            "not included in the governed label."
        ),
    ),

    "connection_blank_039": verified_repair(
        value="K Borgel",
        source_name="Coin-Turf",
        source_type="exact historical racecard",
        locator=(
            "https://www.coin-turf.fr/programmes-courses/07052016/"
            "4437_maisons-laffitte/"
            "prix-de-l-etoile-d-artois-prix-de-l-etoile-du-bon-secours"
        ),
        notes=(
            "The exact racecard lists K Borgel as trainer of Star White."
        ),
    ),

    "connection_blank_040": verified_repair(
        value="C Plisson",
        source_name="Coin-Turf",
        source_type="exact historical racecard",
        locator=(
            "https://www.coin-turf.fr/programmes-courses/07052016/"
            "4437_maisons-laffitte/"
            "prix-de-l-etoile-d-artois-prix-de-l-etoile-du-bon-secours"
        ),
        notes=(
            "The exact racecard lists C Plisson as trainer of Colombia "
            "DEmra."
        ),
    ),

    "connection_blank_041": conflicting_evidence(
        source_name="ZEturf, Timeform and The Horse's Mouth",
        source_type="conflicting exact-date race representations",
        locator=(
            "https://www.zeturf.fr/fr/course-du-jour/2018-09-10/"
            "R1C6-chantilly-prix-de-foulangues | "
            "https://www.timeform.com/horse-racing/horse-form/"
            "numbers-talk/000000303059"
        ),
        notes=(
            "Exact-date sources present Ecurie Avant-Garde, while another "
            "exact result representation attributes Numbers Talk to "
            "N Caullery. The difference may reflect stable/entity handling "
            "rather than a harmless spelling variation."
        ),
    ),

    "connection_blank_042": verified_repair(
        value="Stal't Neerhof",
        source_name="Coin-Turf",
        source_type="exact historical racecard",
        locator=(
            "https://www.coin-turf.fr/programmes-courses/14092018/"
            "43153_saint-cloud/prix-de-lamarque"
        ),
        notes=(
            "The exact racecard lists Stal't Neerhof as the training "
            "connection for Valley Kid."
        ),
    ),

    "connection_blank_043": conflicting_evidence(
        source_name="Coin-Turf and The Horse's Mouth",
        source_type="conflicting exact-date race representations",
        locator=(
            "https://www.coin-turf.fr/programmes-courses/17092018/"
            "43327_maisons-laffitte/prix-du-moulin-de-maisons-laffitte | "
            "https://thehorsesmouth.co.uk/races/"
            "112805-prix-du-moulin-de-maisons-laffitte-handicap-3yo-turf"
        ),
        notes=(
            "Coin-Turf lists Ecurie Avant-Garde for Unital; The Horse's "
            "Mouth lists J-M Lefebvre for the same horse and race. No value "
            "is selected."
        ),
    ),

    "connection_blank_044": conflicting_evidence(
        source_name="Coin-Turf and The Horse's Mouth",
        source_type="conflicting exact-date race representations",
        locator=(
            "Coin-Turf — Maisons-Laffitte, 2018-09-17, "
            "Prix de la Porte du Parc | "
            "https://thehorsesmouth.co.uk/races/"
            "112804-prix-de-la-porte-du-parc-handicap-3yo-turf"
        ),
        notes=(
            "Coin-Turf presents Ecurie Avant-Garde for Talento; The "
            "Horse's Mouth lists R Roels for the same horse and race."
        ),
    ),

    "connection_blank_045": conflicting_evidence(
        source_name="Coin-Turf, Geny and The Horse's Mouth",
        source_type="conflicting exact-date race representations",
        locator=(
            "https://www.coin-turf.fr/programmes-courses/28092018/"
            "43831_saint-cloud/prix-du-lieu-feral | "
            "https://thehorsesmouth.co.uk/races/"
            "112742-prix-du-lieu-feral-handicap-3yo-turf"
        ),
        notes=(
            "Coin-Turf and Geny present Hue & Lamotte dArgy as the "
            "training connection for Totem; The Horse's Mouth lists "
            "B Legros. No automatic preference is defensible."
        ),
    ),

    "connection_blank_046": conflicting_evidence(
        source_name="Coin-Turf and The Horse's Mouth",
        source_type="conflicting exact-date race representations",
        locator=(
            "https://www.coin-turf.fr/programmes-courses/28092018/"
            "43832_saint-cloud/prix-du-lieu-jourdain | "
            "https://thehorsesmouth.co.uk/races/"
            "112741-prix-du-lieu-jourdain-handicap-3yo-turf"
        ),
        notes=(
            "Coin-Turf lists Stal Luro for Jasmine A La Plage; The "
            "Horse's Mouth lists F Belmont for the same horse and race."
        ),
    ),
}


# Confirm the evidence table covers every remaining record exactly once.
expected_remaining_record_ids = {
    f"connection_blank_{number:03d}"
    for number in range(3, 47)
}


assert set(
    remaining_connection_review_updates
) == expected_remaining_record_ids

assert len(
    remaining_connection_review_updates
) == 44


# Apply the complete evidence decisions.
for repair_record_id, update_values in (
    remaining_connection_review_updates.items()
):
    matching_rows = (
        manual_connection_repair_queue[
            "repair_record_id"
        ].eq(repair_record_id)
    )

    assert int(matching_rows.sum()) == 1

    for column, value in update_values.items():
        manual_connection_repair_queue.loc[
            matching_rows,
            column,
        ] = value


# ----------------------------------------------------------------------
# Governance validation
# ----------------------------------------------------------------------

assert not manual_connection_repair_queue[
    "verification_decision"
].eq("pending").any()


expected_decision_counts = {
    "verified_repair": 28,
    "conflicting_evidence": 5,
    "insufficient_evidence": 13,
}


assert (
    manual_connection_repair_queue[
        "verification_decision"
    ]
    .value_counts()
    .to_dict()
    == expected_decision_counts
)


verified_repairs = (
    manual_connection_repair_queue.loc[
        manual_connection_repair_queue[
            "verification_decision"
        ].eq("verified_repair")
    ]
)


non_repair_decisions = (
    manual_connection_repair_queue.loc[
        manual_connection_repair_queue[
            "verification_decision"
        ].isin(
            [
                "conflicting_evidence",
                "insufficient_evidence",
            ]
        )
    ]
)


required_provenance_columns = [
    "evidence_source_name",
    "evidence_source_type",
    "evidence_locator",
    "evidence_accessed_date",
    "verification_confidence",
    "reviewer_notes",
]


# Every decision must have recorded evidence and notes.
assert manual_connection_repair_queue[
    required_provenance_columns
].notna().all().all()


# Every verified repair must have a nonblank proposed value.
assert verified_repairs[
    "proposed_repaired_value"
].notna().all()

assert verified_repairs[
    "proposed_repaired_value"
].str.strip().ne("").all()


# Conflict and insufficient-evidence records must remain unrepaired.
assert non_repair_decisions[
    "proposed_repaired_value"
].isna().all()


# Original source values remain blank or null after CSV reload.
assert (
    manual_connection_repair_queue[
        "original_source_value"
    ]
    .fillna("")
    .eq("")
    .all()
)


# ----------------------------------------------------------------------
# Persistence and reload verification
# ----------------------------------------------------------------------

manual_connection_repair_queue.to_csv(
    manual_connection_repair_queue_path,
    index=False,
)


manual_connection_repair_evidence_log_path = (
    manual_connection_repair_queue_path.parent
    / "manual_connection_repair_evidence_log.csv"
)


manual_connection_repair_queue.to_csv(
    manual_connection_repair_evidence_log_path,
    index=False,
)


reloaded_manual_connection_repair_queue = (
    pd.read_csv(
        manual_connection_repair_queue_path,
        dtype="string",
        keep_default_na=True,
    )
)


assert len(
    reloaded_manual_connection_repair_queue
) == 46

assert not reloaded_manual_connection_repair_queue[
    "verification_decision"
].eq("pending").any()

assert (
    reloaded_manual_connection_repair_queue[
        "verification_decision"
    ]
    .value_counts()
    .to_dict()
    == expected_decision_counts
)


# ----------------------------------------------------------------------
# Reader-facing summaries
# ----------------------------------------------------------------------

manual_connection_repair_status_summary = (
    reloaded_manual_connection_repair_queue.groupby(
        [
            "missing_source_field",
            "verification_decision",
        ],
        as_index=False,
        dropna=False,
    )
    .agg(
        repair_records=("repair_record_id", "size"),
        distinct_runner_rows=("source_rowid", "nunique"),
        distinct_races=("race_id", "nunique"),
    )
    .sort_values(
        [
            "missing_source_field",
            "verification_decision",
        ]
    )
    .reset_index(drop=True)
)


verified_connection_repairs_display = (
    reloaded_manual_connection_repair_queue.loc[
        reloaded_manual_connection_repair_queue[
            "verification_decision"
        ].eq("verified_repair"),
        [
            "repair_record_id",
            "missing_source_field",
            "date",
            "course",
            "horse",
            "proposed_repaired_value",
            "evidence_source_name",
            "verification_confidence",
        ],
    ]
    .sort_values("repair_record_id")
    .reset_index(drop=True)
)


conflicting_connection_evidence_display = (
    reloaded_manual_connection_repair_queue.loc[
        reloaded_manual_connection_repair_queue[
            "verification_decision"
        ].eq("conflicting_evidence"),
        [
            "repair_record_id",
            "missing_source_field",
            "date",
            "course",
            "horse",
            "evidence_source_name",
            "reviewer_notes",
        ],
    ]
    .sort_values("repair_record_id")
    .reset_index(drop=True)
)


insufficient_connection_evidence_display = (
    reloaded_manual_connection_repair_queue.loc[
        reloaded_manual_connection_repair_queue[
            "verification_decision"
        ].eq("insufficient_evidence"),
        [
            "repair_record_id",
            "missing_source_field",
            "date",
            "course",
            "horse",
            "evidence_source_name",
            "reviewer_notes",
        ],
    ]
    .sort_values("repair_record_id")
    .reset_index(drop=True)
)


manual_connection_repair_completion_summary = pd.DataFrame(
    [
        {
            "queue_path": str(
                manual_connection_repair_queue_path.relative_to(
                    PROJECT_ROOT
                )
            ),
            "evidence_log_path": str(
                manual_connection_repair_evidence_log_path.relative_to(
                    PROJECT_ROOT
                )
            ),
            "total_records": len(
                reloaded_manual_connection_repair_queue
            ),
            "verified_repairs": int(
                reloaded_manual_connection_repair_queue[
                    "verification_decision"
                ].eq("verified_repair").sum()
            ),
            "conflicting_evidence": int(
                reloaded_manual_connection_repair_queue[
                    "verification_decision"
                ].eq("conflicting_evidence").sum()
            ),
            "insufficient_evidence": int(
                reloaded_manual_connection_repair_queue[
                    "verification_decision"
                ].eq("insufficient_evidence").sum()
            ),
            "pending_records": int(
                reloaded_manual_connection_repair_queue[
                    "verification_decision"
                ].eq("pending").sum()
            ),
            "reload_verified": True,
        }
    ]
)


print("Completed manual connection-repair review")
display(manual_connection_repair_completion_summary)

print("Decision status by missing source field")
display(manual_connection_repair_status_summary)

print("Verified connection repairs")
display(verified_connection_repairs_display)

print("Conflicting connection evidence")
display(conflicting_connection_evidence_display)

print("Insufficient connection evidence")
display(insufficient_connection_evidence_display)

Completed manual connection-repair review


,queue_path,evidence_log_path,total_records,verified_repairs,conflicting_evidence,insufficient_evidence,pending_records,reload_verified
0,data/derived/connection_identity/manual_connec...,data/derived/connection_identity/manual_connec...,46,28,5,13,0,True


Decision status by missing source field


,missing_source_field,verification_decision,repair_records,distinct_runner_rows,distinct_races
0,jockey,verified_repair,2,2,2
1,owner,insufficient_evidence,13,13,7
2,owner,verified_repair,22,22,14
3,trainer,conflicting_evidence,5,5,5
4,trainer,verified_repair,4,4,3


Verified connection repairs


,repair_record_id,missing_source_field,date,course,horse,proposed_repaired_value,evidence_source_name,verification_confidence
0,connection_blank_001,jockey,2016-04-08,Auteuil (FR),Ahzana (FR),T. Viel,Canalturf and Coin-Turf,high
1,connection_blank_002,jockey,2021-09-05,Baden-Baden (GER),Millebosc (FR),Adrie de Vries,Racing Post,high
2,connection_blank_009,owner,2019-05-13,Saint-Cloud (FR),Le Letty (FR),Edouard Desespringalle,Turfoo,high
3,connection_blank_010,owner,2019-06-03,Longchamp (FR),Le Letty (FR),Edouard Desespringalle,Turfoo,high
4,connection_blank_011,owner,2019-10-31,Mombetsu (JPN),Abenin Dream (JPN),Mastec Co Ltd,netkeiba,high
5,connection_blank_012,owner,2019-10-31,Mombetsu (JPN),Chimera Verite (JPN),Makoto Kato,netkeiba and Rakuten Keiba,high
6,connection_blank_013,owner,2019-10-31,Mombetsu (JPN),Meiner Astoria (JPN),Thoroughbred Club Ruffian Co Ltd,netkeiba and Rakuten Keiba,high
7,connection_blank_014,owner,2019-10-31,Mombetsu (JPN),Pionono (JPN),Kaneko Makoto Holdings Co Ltd,netkeiba,high
8,connection_blank_015,owner,2019-10-31,Mombetsu (JPN),Adjuvant (JPN),Toshiaki Date,netkeiba,high
9,connection_blank_016,owner,2020-08-08,Flemington (AUS),Bertwhistle (AUS),R D Pfitzner,Victoria Racing Club,high


Conflicting connection evidence


,repair_record_id,missing_source_field,date,course,horse,evidence_source_name,reviewer_notes
0,connection_blank_041,trainer,2018-09-10,Chantilly (FR),Numbers Talk (IRE),"ZEturf, Timeform and The Horse's Mouth","Exact-date sources present Ecurie Avant-Garde,..."
1,connection_blank_043,trainer,2018-09-17,Maisons-Laffitte (FR),Unital (FR),Coin-Turf and The Horse's Mouth,Coin-Turf lists Ecurie Avant-Garde for Unital;...
2,connection_blank_044,trainer,2018-09-17,Maisons-Laffitte (FR),Talento (IRE),Coin-Turf and The Horse's Mouth,Coin-Turf presents Ecurie Avant-Garde for Tale...
3,connection_blank_045,trainer,2018-09-28,Saint-Cloud (FR),Totem (FR),"Coin-Turf, Geny and The Horse's Mouth",Coin-Turf and Geny present Hue & Lamotte dArgy...
4,connection_blank_046,trainer,2018-09-28,Saint-Cloud (FR),Jasmine A La Plage (FR),Coin-Turf and The Horse's Mouth,Coin-Turf lists Stal Luro for Jasmine A La Pla...


Insufficient connection evidence


,repair_record_id,missing_source_field,date,course,horse,evidence_source_name,reviewer_notes
0,connection_blank_003,owner,2015-01-03,Santa Anita (USA),Rattataptap (USA),Santa Anita historical result coverage; Paulic...,Near-date contemporary reporting identifies Ma...
1,connection_blank_004,owner,2015-04-04,Caulfield (AUS),Post DFrance (NZ),Racing Post and Australian historical result s...,"The target runner and result were located, but..."
2,connection_blank_005,owner,2015-05-03,San Siro (ITY),Balami Fan (ITY),Italian historical result and horse-profile se...,No sufficiently reliable target-date owner dec...
3,connection_blank_006,owner,2015-06-18,Longchamp (FR),Rappeur Des Mottes (FR),Canalturf and later horse-profile sources,Later sources associate Rappeur Des Mottes wit...
4,connection_blank_007,owner,2016-05-07,Maisons-Laffitte (FR),Star White (FR),Coin-Turf,The exact racecard identifies Star White and i...
5,connection_blank_008,owner,2016-05-07,Maisons-Laffitte (FR),Colombia DEmra (FR),Coin-Turf and horse-history searches,The exact racecard identifies Colombia DEmra a...
6,connection_blank_018,owner,2020-10-03,Newcastle (AUS),Nu Jiang (AUS),Racing NSW form material and target-race searches,Period material associates the horse with Zhon...
7,connection_blank_019,owner,2020-10-03,Newcastle (AUS),Media Man (AUS),"Minervini Racing, Vinery and target-race searches",Later and stable-level ownership descriptions ...
8,connection_blank_020,owner,2020-10-03,Newcastle (AUS),Luftwaffe (AUS),Australian historical result and horse-profile...,No sufficiently direct target-date owner decla...
9,connection_blank_021,owner,2020-10-03,Newcastle (AUS),Stormwater (AUS),Australian historical result and sale-record s...,"Horse and sale evidence was found, but it does..."


## Stage 39 — Provenance-quality audit of manual connection repairs

The completed evidence log contains a decision for every blank connection occurrence.

Before verified values are promoted into a governed correction artifact, this stage audits whether each decision is structurally reproducible.

The audit checks:

- every repair record has one permitted decision;
- verified repairs contain a nonblank proposed value;
- conflicting and insufficient-evidence records contain no proposed value;
- every decision contains source name, source type, evidence locator, access date, confidence and reviewer notes;
- access dates are parseable;
- verified-repair evidence locators contain a direct web address or require additional locator strengthening.

A descriptive locator may still identify the evidence used, but it is weaker than a direct URL, archived document path or stable publication identifier.

Records requiring stronger locators remain verified at the analytical level, but should not enter the final governed correction artifact until their evidence can be reproduced independently.

In [40]:
# Audit the completed manual evidence log before deriving a governed repair
# artifact.
#
# This stage does not alter any decisions or repaired values.

connection_repair_provenance_audit = (
    reloaded_manual_connection_repair_queue.copy()
)


permitted_verification_decisions = {
    "verified_repair",
    "conflicting_evidence",
    "insufficient_evidence",
}


# ----------------------------------------------------------------------
# Structural decision checks
# ----------------------------------------------------------------------

connection_repair_provenance_audit[
    "decision_is_permitted"
] = (
    connection_repair_provenance_audit[
        "verification_decision"
    ].isin(permitted_verification_decisions)
)


connection_repair_provenance_audit[
    "has_proposed_repaired_value"
] = (
    connection_repair_provenance_audit[
        "proposed_repaired_value"
    ]
    .fillna("")
    .str.strip()
    .ne("")
)


connection_repair_provenance_audit[
    "value_decision_is_consistent"
] = (
    (
        connection_repair_provenance_audit[
            "verification_decision"
        ].eq("verified_repair")
        & connection_repair_provenance_audit[
            "has_proposed_repaired_value"
        ]
    )
    |
    (
        connection_repair_provenance_audit[
            "verification_decision"
        ].isin(
            [
                "conflicting_evidence",
                "insufficient_evidence",
            ]
        )
        & ~connection_repair_provenance_audit[
            "has_proposed_repaired_value"
        ]
    )
)


# ----------------------------------------------------------------------
# Required provenance-field checks
# ----------------------------------------------------------------------

required_provenance_columns = [
    "evidence_source_name",
    "evidence_source_type",
    "evidence_locator",
    "evidence_accessed_date",
    "verification_confidence",
    "reviewer_notes",
]


for column in required_provenance_columns:
    connection_repair_provenance_audit[
        f"{column}_is_populated"
    ] = (
        connection_repair_provenance_audit[column]
        .fillna("")
        .str.strip()
        .ne("")
    )


provenance_population_columns = [
    f"{column}_is_populated"
    for column in required_provenance_columns
]


connection_repair_provenance_audit[
    "all_required_provenance_populated"
] = (
    connection_repair_provenance_audit[
        provenance_population_columns
    ].all(axis=1)
)


# ----------------------------------------------------------------------
# Evidence-date and locator checks
# ----------------------------------------------------------------------

connection_repair_provenance_audit[
    "parsed_evidence_accessed_date"
] = pd.to_datetime(
    connection_repair_provenance_audit[
        "evidence_accessed_date"
    ],
    errors="coerce",
)


connection_repair_provenance_audit[
    "evidence_accessed_date_is_parseable"
] = (
    connection_repair_provenance_audit[
        "parsed_evidence_accessed_date"
    ].notna()
)


# A direct URL is the strongest easily reproducible locator available in the
# current CSV structure.
connection_repair_provenance_audit[
    "locator_contains_direct_url"
] = (
    connection_repair_provenance_audit[
        "evidence_locator"
    ]
    .fillna("")
    .str.contains(
        r"https?://",
        regex=True,
        case=False,
    )
)


connection_repair_provenance_audit[
    "locator_review_status"
] = "descriptive_locator_only"


connection_repair_provenance_audit.loc[
    connection_repair_provenance_audit[
        "locator_contains_direct_url"
    ],
    "locator_review_status",
] = "direct_url_recorded"


# Only verified repairs are eligible for promotion. A descriptive-only
# locator must be strengthened first.
connection_repair_provenance_audit[
    "eligible_for_governed_repair_artifact"
] = (
    connection_repair_provenance_audit[
        "verification_decision"
    ].eq("verified_repair")
    & connection_repair_provenance_audit[
        "decision_is_permitted"
    ]
    & connection_repair_provenance_audit[
        "value_decision_is_consistent"
    ]
    & connection_repair_provenance_audit[
        "all_required_provenance_populated"
    ]
    & connection_repair_provenance_audit[
        "evidence_accessed_date_is_parseable"
    ]
    & connection_repair_provenance_audit[
        "locator_contains_direct_url"
    ]
)


# ----------------------------------------------------------------------
# Reconciliation assertions
# ----------------------------------------------------------------------

assert len(
    connection_repair_provenance_audit
) == 46

assert connection_repair_provenance_audit[
    "repair_record_id"
].is_unique

assert connection_repair_provenance_audit[
    "decision_is_permitted"
].all()

assert connection_repair_provenance_audit[
    "value_decision_is_consistent"
].all()

assert connection_repair_provenance_audit[
    "all_required_provenance_populated"
].all()

assert connection_repair_provenance_audit[
    "evidence_accessed_date_is_parseable"
].all()


# ----------------------------------------------------------------------
# Reader-facing summaries
# ----------------------------------------------------------------------

connection_repair_provenance_summary = (
    connection_repair_provenance_audit.groupby(
        [
            "verification_decision",
            "locator_review_status",
        ],
        as_index=False,
    )
    .agg(
        repair_records=("repair_record_id", "size"),
        distinct_runner_rows=("source_rowid", "nunique"),
        eligible_for_governed_repair_artifact=(
            "eligible_for_governed_repair_artifact",
            "sum",
        ),
    )
    .sort_values(
        [
            "verification_decision",
            "locator_review_status",
        ]
    )
    .reset_index(drop=True)
)


verified_repairs_requiring_locator_strengthening = (
    connection_repair_provenance_audit.loc[
        connection_repair_provenance_audit[
            "verification_decision"
        ].eq("verified_repair")
        & ~connection_repair_provenance_audit[
            "locator_contains_direct_url"
        ],
        [
            "repair_record_id",
            "missing_source_field",
            "date",
            "course",
            "horse",
            "proposed_repaired_value",
            "evidence_source_name",
            "evidence_source_type",
            "evidence_locator",
            "verification_confidence",
        ],
    ]
    .sort_values("repair_record_id")
    .reset_index(drop=True)
)


governed_repair_eligibility_summary = pd.DataFrame(
    [
        {
            "completed_evidence_records": len(
                connection_repair_provenance_audit
            ),
            "verified_repair_records": int(
                connection_repair_provenance_audit[
                    "verification_decision"
                ].eq("verified_repair").sum()
            ),
            "verified_repairs_with_direct_url": int(
                (
                    connection_repair_provenance_audit[
                        "verification_decision"
                    ].eq("verified_repair")
                    & connection_repair_provenance_audit[
                        "locator_contains_direct_url"
                    ]
                ).sum()
            ),
            "verified_repairs_requiring_locator_strengthening": int(
                len(
                    verified_repairs_requiring_locator_strengthening
                )
            ),
            "currently_eligible_governed_repairs": int(
                connection_repair_provenance_audit[
                    "eligible_for_governed_repair_artifact"
                ].sum()
            ),
        }
    ]
)


print("Connection-repair provenance audit")
display(governed_repair_eligibility_summary)

print("Provenance status by decision")
display(connection_repair_provenance_summary)

print("Verified repairs requiring stronger evidence locators")
display(verified_repairs_requiring_locator_strengthening)

Connection-repair provenance audit


,completed_evidence_records,verified_repair_records,verified_repairs_with_direct_url,verified_repairs_requiring_locator_strengthening,currently_eligible_governed_repairs
0,46,28,22,6,22


Provenance status by decision


,verification_decision,locator_review_status,repair_records,distinct_runner_rows,eligible_for_governed_repair_artifact
0,conflicting_evidence,direct_url_recorded,5,5,0
1,insufficient_evidence,descriptive_locator_only,10,10,0
2,insufficient_evidence,direct_url_recorded,3,3,0
3,verified_repair,descriptive_locator_only,6,6,0
4,verified_repair,direct_url_recorded,22,22,22


Verified repairs requiring stronger evidence locators


,repair_record_id,missing_source_field,date,course,horse,proposed_repaired_value,evidence_source_name,evidence_source_type,evidence_locator,verification_confidence
0,connection_blank_025,owner,2021-04-07,Funabashi (JPN),Absolute Queen (JPN),Mitsunari Hiromatsu,Rakuten Keiba,exact historical racecard,"Rakuten Keiba — Funabashi, 2021-04-07, Marine ...",high
1,connection_blank_027,owner,2021-12-01,Funabashi (JPN),Dear Rickey (JPN),Yusuke Kobayashi,Rakuten Keiba,exact historical racecard,"Rakuten Keiba — Funabashi, 2021-12-01, Queen S...",high
2,connection_blank_030,owner,2022-04-20,Ohi (JPN),Gishigishi (JPN),Foret Bleu Co Ltd,Rakuten Keiba and Japanese owner records,exact historical racecard,"Rakuten Keiba — Ohi, 2022-04-20, Tokyo Sprint,...",high
3,connection_blank_034,owner,2022-07-18,Morioka (JPN),Vacation (JPN),Masatoshi Suzuki,Japanese target-race listing and bilingual rac...,exact historical race listing,"Morioka, 2022-07-18, Mercury Cup, Vacation — O...",medium
4,connection_blank_035,owner,2022-07-18,Morioka (JPN),Giga King (JPN),Tomohiro Ozaki,Japanese target-race listing and bilingual rac...,exact historical race listing,"Morioka, 2022-07-18, Mercury Cup, Giga King — ...",medium
5,connection_blank_036,owner,2022-09-28,Funabashi (JPN),Giga King (JPN),Hero Racing Co Ltd,Rakuten Keiba and Japanese racing records,exact historical racecard,"Rakuten Keiba — Funabashi, 2022-09-28, Nippon ...",high


## Stage 40 — Strengthening verified-repair evidence locators

The provenance audit identified six verified repairs whose evidence locators were descriptive references rather than direct URLs.

Direct target-race evidence has now been recorded for all six:

- two Funabashi owner records;
- one Ohi owner record;
- two Morioka owner records;
- one further Funabashi owner record.

The strengthened evidence uses:

- official NAR racecards;
- an exact JBIS historical result;
- direct Rakuten Keiba racecards;
- a direct bilingual Mercury Cup listing where Romanised owner presentation was retained.

This stage changes only the evidence-source metadata.

It does not alter:

- the original source values;
- proposed repaired values;
- verification decisions;
- confidence classifications;
- reviewer conclusions.

The Giga King records also demonstrate why ownership must be verified at the individual race date:

- the 2022-07-18 Mercury Cup card lists `尾崎智大`;
- the 2022-09-28 Nippon TV Hai card lists `（株）Ｈｅｒｏレーシング`.

A later or general horse profile would therefore not be a safe substitute for exact-date evidence.

In [41]:
# Strengthen the six verified repairs whose evidence locators were previously
# descriptive rather than directly reproducible.
#
# Only provenance metadata is changed. The proposed repaired values,
# verification decisions and original source values remain untouched.

manual_connection_repair_queue = pd.read_csv(
    manual_connection_repair_queue_path,
    dtype="string",
    keep_default_na=True,
)


assert len(manual_connection_repair_queue) == 46

assert manual_connection_repair_queue[
    "repair_record_id"
].is_unique


# Record direct target-race locators for the six repairs isolated by the
# provenance audit.
verified_repair_locator_updates = {
    "connection_blank_025": {
        "evidence_source_name": (
            "NAR Local Racing Information Site and Rakuten Keiba"
        ),
        "evidence_source_type": (
            "official and independent exact-date racecards"
        ),
        "evidence_locator": (
            "https://www.keiba.go.jp/KeibaWeb/TodayRaceInfo/"
            "DebaTable?k_babaCode=19&k_raceDate=2021%2F04%2F07"
            "&k_raceNo=11"
            " | "
            "https://keiba.rakuten.co.jp/race_card/list/"
            "RACEID/202104071900000011"
        ),
    },
    "connection_blank_027": {
        "evidence_source_name": (
            "NAR Local Racing Information Site and Rakuten Keiba"
        ),
        "evidence_source_type": (
            "official and independent exact-date racecards"
        ),
        "evidence_locator": (
            "https://www.keiba.go.jp/KeibaWeb/TodayRaceInfo/"
            "DebaTable?k_babaCode=19&k_raceDate=2021%2F12%2F01"
            "&k_raceNo=11"
            " | "
            "https://keiba.rakuten.co.jp/race_card/list/"
            "RACEID/202112011914090311"
        ),
    },
    "connection_blank_030": {
        "evidence_source_name": "JBIS-Search",
        "evidence_source_type": (
            "official exact-date historical race result"
        ),
        "evidence_locator": (
            "https://www.jbis.jp/race/result/20220420/220/11/"
        ),
    },
    "connection_blank_034": {
        "evidence_source_name": (
            "NAR Local Racing Information Site and bilingual "
            "Mercury Cup listing"
        ),
        "evidence_source_type": (
            "official exact-date racecard plus bilingual "
            "target-race listing"
        ),
        "evidence_locator": (
            "https://www.keiba.go.jp/KeibaWeb/TodayRaceInfo/"
            "DebaTable?k_babaCode=10&k_raceDate=2022%2F07%2F18"
            "&k_raceNo=12"
            " | "
            "https://mixi.jp/view_bbs.pl"
            "?comm_id=751243&id=99203601"
        ),
    },
    "connection_blank_035": {
        "evidence_source_name": (
            "NAR Local Racing Information Site and bilingual "
            "Mercury Cup listing"
        ),
        "evidence_source_type": (
            "official exact-date racecard plus bilingual "
            "target-race listing"
        ),
        "evidence_locator": (
            "https://www.keiba.go.jp/KeibaWeb/TodayRaceInfo/"
            "DebaTable?k_babaCode=10&k_raceDate=2022%2F07%2F18"
            "&k_raceNo=12"
            " | "
            "https://mixi.jp/view_bbs.pl"
            "?comm_id=751243&id=99203601"
        ),
    },
    "connection_blank_036": {
        "evidence_source_name": (
            "NAR Local Racing Information Site and Rakuten Keiba"
        ),
        "evidence_source_type": (
            "official and independent exact-date racecards"
        ),
        "evidence_locator": (
            "https://www.keiba.go.jp/KeibaWeb/TodayRaceInfo/"
            "DebaTable?k_babaCode=19&k_raceDate=2022%2F09%2F28"
            "&k_raceNo=11"
            " | "
            "https://keiba.rakuten.co.jp/race_card/list/"
            "RACEID/202209281900000011"
        ),
    },
}


expected_strengthened_record_ids = {
    "connection_blank_025",
    "connection_blank_027",
    "connection_blank_030",
    "connection_blank_034",
    "connection_blank_035",
    "connection_blank_036",
}


assert set(
    verified_repair_locator_updates
) == expected_strengthened_record_ids


# Confirm that these are exactly the six verified repairs isolated by the
# preceding provenance audit.
observed_strengthening_record_ids = set(
    verified_repairs_requiring_locator_strengthening[
        "repair_record_id"
    ]
)


assert (
    observed_strengthening_record_ids
    == expected_strengthened_record_ids
)


# Apply provenance updates without altering any decision or repaired value.
for repair_record_id, provenance_update in (
    verified_repair_locator_updates.items()
):
    matching_rows = (
        manual_connection_repair_queue[
            "repair_record_id"
        ].eq(repair_record_id)
    )

    assert int(matching_rows.sum()) == 1

    assert (
        manual_connection_repair_queue.loc[
            matching_rows,
            "verification_decision",
        ].iloc[0]
        == "verified_repair"
    )

    existing_repaired_value = (
        manual_connection_repair_queue.loc[
            matching_rows,
            "proposed_repaired_value",
        ].iloc[0]
    )

    assert pd.notna(existing_repaired_value)

    assert str(existing_repaired_value).strip() != ""

    for column, value in provenance_update.items():
        manual_connection_repair_queue.loc[
            matching_rows,
            column,
        ] = value


# Confirm every strengthened locator now contains at least one direct URL.
strengthened_records = (
    manual_connection_repair_queue.loc[
        manual_connection_repair_queue[
            "repair_record_id"
        ].isin(expected_strengthened_record_ids)
    ]
    .copy()
)


assert strengthened_records[
    "evidence_locator"
].str.contains(
    r"https?://",
    regex=True,
    case=False,
).all()


# Persist both governed evidence artifacts.
manual_connection_repair_queue.to_csv(
    manual_connection_repair_queue_path,
    index=False,
)


manual_connection_repair_queue.to_csv(
    manual_connection_repair_evidence_log_path,
    index=False,
)


# Reload immediately so the strengthened evidence is tested from disk.
reloaded_manual_connection_repair_queue = pd.read_csv(
    manual_connection_repair_queue_path,
    dtype="string",
    keep_default_na=True,
)


assert len(
    reloaded_manual_connection_repair_queue
) == 46

assert reloaded_manual_connection_repair_queue[
    "repair_record_id"
].is_unique


# ----------------------------------------------------------------------
# Re-run the provenance eligibility audit
# ----------------------------------------------------------------------

connection_repair_provenance_audit = (
    reloaded_manual_connection_repair_queue.copy()
)


connection_repair_provenance_audit[
    "decision_is_permitted"
] = (
    connection_repair_provenance_audit[
        "verification_decision"
    ].isin(
        {
            "verified_repair",
            "conflicting_evidence",
            "insufficient_evidence",
        }
    )
)


connection_repair_provenance_audit[
    "has_proposed_repaired_value"
] = (
    connection_repair_provenance_audit[
        "proposed_repaired_value"
    ]
    .fillna("")
    .str.strip()
    .ne("")
)


connection_repair_provenance_audit[
    "value_decision_is_consistent"
] = (
    (
        connection_repair_provenance_audit[
            "verification_decision"
        ].eq("verified_repair")
        & connection_repair_provenance_audit[
            "has_proposed_repaired_value"
        ]
    )
    |
    (
        connection_repair_provenance_audit[
            "verification_decision"
        ].isin(
            [
                "conflicting_evidence",
                "insufficient_evidence",
            ]
        )
        & ~connection_repair_provenance_audit[
            "has_proposed_repaired_value"
        ]
    )
)


required_provenance_columns = [
    "evidence_source_name",
    "evidence_source_type",
    "evidence_locator",
    "evidence_accessed_date",
    "verification_confidence",
    "reviewer_notes",
]


connection_repair_provenance_audit[
    "all_required_provenance_populated"
] = (
    connection_repair_provenance_audit[
        required_provenance_columns
    ]
    .fillna("")
    .apply(
        lambda column: column.str.strip().ne("")
    )
    .all(axis=1)
)


connection_repair_provenance_audit[
    "evidence_accessed_date_is_parseable"
] = pd.to_datetime(
    connection_repair_provenance_audit[
        "evidence_accessed_date"
    ],
    errors="coerce",
).notna()


connection_repair_provenance_audit[
    "locator_contains_direct_url"
] = (
    connection_repair_provenance_audit[
        "evidence_locator"
    ]
    .fillna("")
    .str.contains(
        r"https?://",
        regex=True,
        case=False,
    )
)


connection_repair_provenance_audit[
    "eligible_for_governed_repair_artifact"
] = (
    connection_repair_provenance_audit[
        "verification_decision"
    ].eq("verified_repair")
    & connection_repair_provenance_audit[
        "decision_is_permitted"
    ]
    & connection_repair_provenance_audit[
        "value_decision_is_consistent"
    ]
    & connection_repair_provenance_audit[
        "all_required_provenance_populated"
    ]
    & connection_repair_provenance_audit[
        "evidence_accessed_date_is_parseable"
    ]
    & connection_repair_provenance_audit[
        "locator_contains_direct_url"
    ]
)


# ----------------------------------------------------------------------
# Final eligibility reconciliation
# ----------------------------------------------------------------------

verified_repair_rows = (
    connection_repair_provenance_audit.loc[
        connection_repair_provenance_audit[
            "verification_decision"
        ].eq("verified_repair")
    ]
)


assert len(verified_repair_rows) == 28

assert verified_repair_rows[
    "locator_contains_direct_url"
].all()

assert verified_repair_rows[
    "eligible_for_governed_repair_artifact"
].all()


assert int(
    connection_repair_provenance_audit[
        "eligible_for_governed_repair_artifact"
    ].sum()
) == 28


strengthened_locator_display = (
    connection_repair_provenance_audit.loc[
        connection_repair_provenance_audit[
            "repair_record_id"
        ].isin(expected_strengthened_record_ids),
        [
            "repair_record_id",
            "missing_source_field",
            "date",
            "course",
            "horse",
            "proposed_repaired_value",
            "evidence_source_name",
            "evidence_locator",
            "verification_confidence",
            "eligible_for_governed_repair_artifact",
        ],
    ]
    .sort_values("repair_record_id")
    .reset_index(drop=True)
)


governed_repair_eligibility_summary = pd.DataFrame(
    [
        {
            "completed_evidence_records": len(
                connection_repair_provenance_audit
            ),
            "verified_repair_records": len(
                verified_repair_rows
            ),
            "verified_repairs_with_direct_url": int(
                verified_repair_rows[
                    "locator_contains_direct_url"
                ].sum()
            ),
            "verified_repairs_requiring_locator_strengthening": int(
                (
                    ~verified_repair_rows[
                        "locator_contains_direct_url"
                    ]
                ).sum()
            ),
            "currently_eligible_governed_repairs": int(
                connection_repair_provenance_audit[
                    "eligible_for_governed_repair_artifact"
                ].sum()
            ),
            "reload_verified": True,
        }
    ]
)


print("Strengthened verified-repair evidence locators")
display(strengthened_locator_display)

print("Governed repair eligibility after locator strengthening")
display(governed_repair_eligibility_summary)

Strengthened verified-repair evidence locators


,repair_record_id,missing_source_field,date,course,horse,proposed_repaired_value,evidence_source_name,evidence_locator,verification_confidence,eligible_for_governed_repair_artifact
0,connection_blank_025,owner,2021-04-07,Funabashi (JPN),Absolute Queen (JPN),Mitsunari Hiromatsu,NAR Local Racing Information Site and Rakuten ...,https://www.keiba.go.jp/KeibaWeb/TodayRaceInfo...,high,True
1,connection_blank_027,owner,2021-12-01,Funabashi (JPN),Dear Rickey (JPN),Yusuke Kobayashi,NAR Local Racing Information Site and Rakuten ...,https://www.keiba.go.jp/KeibaWeb/TodayRaceInfo...,high,True
2,connection_blank_030,owner,2022-04-20,Ohi (JPN),Gishigishi (JPN),Foret Bleu Co Ltd,JBIS-Search,https://www.jbis.jp/race/result/20220420/220/11/,high,True
3,connection_blank_034,owner,2022-07-18,Morioka (JPN),Vacation (JPN),Masatoshi Suzuki,NAR Local Racing Information Site and bilingua...,https://www.keiba.go.jp/KeibaWeb/TodayRaceInfo...,medium,True
4,connection_blank_035,owner,2022-07-18,Morioka (JPN),Giga King (JPN),Tomohiro Ozaki,NAR Local Racing Information Site and bilingua...,https://www.keiba.go.jp/KeibaWeb/TodayRaceInfo...,medium,True
5,connection_blank_036,owner,2022-09-28,Funabashi (JPN),Giga King (JPN),Hero Racing Co Ltd,NAR Local Racing Information Site and Rakuten ...,https://www.keiba.go.jp/KeibaWeb/TodayRaceInfo...,high,True


Governed repair eligibility after locator strengthening


,completed_evidence_records,verified_repair_records,verified_repairs_with_direct_url,verified_repairs_requiring_locator_strengthening,currently_eligible_governed_repairs,reload_verified
0,46,28,28,0,28,True


## Stage 41 — Governed connection-value correction artifact

The completed evidence review contains:

- 28 verified repairs;
- 5 conflicting-evidence records;
- 13 insufficient-evidence records.

Only the 28 verified repairs satisfy the requirements for promotion into a governed correction layer.

The correction artifact uses a long structure with one row per:

- source runner row;
- repaired source field.

Each correction record retains:

- the immutable source-row identifier;
- race and runner context;
- the original source value;
- the governed repaired value;
- evidence source and locator;
- evidence-access date;
- confidence;
- reviewer notes;
- repair-record identifier.

The artifact does not overwrite the raw source database.

Downstream database construction may apply it by joining on:

- `source_rowid`;
- `source_field`.

The five conflicting records and thirteen insufficient-evidence records remain null in the governed connection fields.

In [42]:
# Derive the governed connection-value correction artifact from the completed
# and provenance-audited manual evidence log.
#
# Only verified repairs that passed every eligibility check are included.

governed_connection_repairs = (
    connection_repair_provenance_audit.loc[
        connection_repair_provenance_audit[
            "eligible_for_governed_repair_artifact"
        ],
        [
            "repair_record_id",
            "source_rowid",
            "race_id",
            "date",
            "course",
            "off",
            "race_name",
            "race_type",
            "horse",
            "missing_source_field",
            "original_source_value",
            "proposed_repaired_value",
            "evidence_source_name",
            "evidence_source_type",
            "evidence_locator",
            "evidence_accessed_date",
            "verification_confidence",
            "reviewer_notes",
        ],
    ]
    .rename(
        columns={
            "missing_source_field": "source_field",
            "proposed_repaired_value": "governed_repaired_value",
        }
    )
    .copy()
)


# Record the correction policy explicitly in every artifact row.
governed_connection_repairs[
    "correction_method"
] = "manual_external_verification"


governed_connection_repairs[
    "correction_status"
] = "approved"


# Stable column order for database ingestion and review.
governed_connection_repairs = (
    governed_connection_repairs[
        [
            "repair_record_id",
            "source_rowid",
            "race_id",
            "date",
            "course",
            "off",
            "race_name",
            "race_type",
            "horse",
            "source_field",
            "original_source_value",
            "governed_repaired_value",
            "correction_method",
            "correction_status",
            "evidence_source_name",
            "evidence_source_type",
            "evidence_locator",
            "evidence_accessed_date",
            "verification_confidence",
            "reviewer_notes",
        ]
    ]
    .sort_values(
        [
            "source_rowid",
            "source_field",
        ]
    )
    .reset_index(drop=True)
)


# ----------------------------------------------------------------------
# Governance validation
# ----------------------------------------------------------------------

assert len(governed_connection_repairs) == 28

assert governed_connection_repairs[
    "repair_record_id"
].is_unique


# A source row may contain more than one repaired field, so uniqueness is
# defined by source row plus source field.
assert not governed_connection_repairs.duplicated(
    subset=[
        "source_rowid",
        "source_field",
    ]
).any()


assert governed_connection_repairs[
    "source_field"
].isin(CONNECTION_IDENTITY_FIELDS).all()


assert governed_connection_repairs[
    "governed_repaired_value"
].notna().all()

assert governed_connection_repairs[
    "governed_repaired_value"
].str.strip().ne("").all()


# The correction artifact must originate only from source blanks.
assert (
    governed_connection_repairs[
        "original_source_value"
    ]
    .fillna("")
    .eq("")
    .all()
)


assert governed_connection_repairs[
    "correction_method"
].eq(
    "manual_external_verification"
).all()


assert governed_connection_repairs[
    "correction_status"
].eq("approved").all()


# Every governed correction must retain a direct reproducible evidence locator.
assert governed_connection_repairs[
    "evidence_locator"
].str.contains(
    r"https?://",
    regex=True,
    case=False,
).all()


# Reconcile field counts to the completed evidence review.
expected_governed_repair_counts = {
    "owner": 22,
    "trainer": 4,
    "jockey": 2,
}


assert (
    governed_connection_repairs[
        "source_field"
    ]
    .value_counts()
    .to_dict()
    == expected_governed_repair_counts
)


# ----------------------------------------------------------------------
# Persistence
# ----------------------------------------------------------------------

governed_connection_repairs_path = (
    connection_repair_output_directory
    / "governed_connection_repairs.csv"
)


governed_connection_repairs.to_csv(
    governed_connection_repairs_path,
    index=False,
)


# Reload immediately from disk.
reloaded_governed_connection_repairs = (
    pd.read_csv(
        governed_connection_repairs_path,
        dtype={
            "repair_record_id": "string",
            "source_field": "string",
            "original_source_value": "string",
            "governed_repaired_value": "string",
            "correction_method": "string",
            "correction_status": "string",
            "evidence_source_name": "string",
            "evidence_source_type": "string",
            "evidence_locator": "string",
            "evidence_accessed_date": "string",
            "verification_confidence": "string",
            "reviewer_notes": "string",
        },
        keep_default_na=True,
    )
)


# ----------------------------------------------------------------------
# Reload validation
# ----------------------------------------------------------------------

assert len(
    reloaded_governed_connection_repairs
) == 28

assert reloaded_governed_connection_repairs[
    "repair_record_id"
].is_unique

assert not reloaded_governed_connection_repairs.duplicated(
    subset=[
        "source_rowid",
        "source_field",
    ]
).any()

assert (
    reloaded_governed_connection_repairs[
        "source_field"
    ]
    .value_counts()
    .to_dict()
    == expected_governed_repair_counts
)


# ----------------------------------------------------------------------
# Reader-facing summaries
# ----------------------------------------------------------------------

governed_connection_repair_summary = (
    reloaded_governed_connection_repairs.groupby(
        [
            "source_field",
            "verification_confidence",
        ],
        as_index=False,
    )
    .agg(
        governed_repairs=("repair_record_id", "size"),
        distinct_runner_rows=("source_rowid", "nunique"),
        distinct_races=("race_id", "nunique"),
        distinct_courses=("course", "nunique"),
        first_observed_date=("date", "min"),
        last_observed_date=("date", "max"),
    )
    .sort_values(
        [
            "source_field",
            "verification_confidence",
        ]
    )
    .reset_index(drop=True)
)


governed_connection_repair_persistence_summary = pd.DataFrame(
    [
        {
            "output_path": str(
                governed_connection_repairs_path.relative_to(
                    PROJECT_ROOT
                )
            ),
            "persisted_records": len(
                governed_connection_repairs
            ),
            "reloaded_records": len(
                reloaded_governed_connection_repairs
            ),
            "jockey_repairs": int(
                reloaded_governed_connection_repairs[
                    "source_field"
                ].eq("jockey").sum()
            ),
            "trainer_repairs": int(
                reloaded_governed_connection_repairs[
                    "source_field"
                ].eq("trainer").sum()
            ),
            "owner_repairs": int(
                reloaded_governed_connection_repairs[
                    "source_field"
                ].eq("owner").sum()
            ),
            "reload_verified": True,
        }
    ]
)


print("Governed connection-repair artifact persistence")
display(governed_connection_repair_persistence_summary)

print("Governed connection repairs by field and confidence")
display(governed_connection_repair_summary)

print("Complete governed connection-repair artifact")
display(reloaded_governed_connection_repairs)

Governed connection-repair artifact persistence


,output_path,persisted_records,reloaded_records,jockey_repairs,trainer_repairs,owner_repairs,reload_verified
0,data/derived/connection_identity/governed_conn...,28,28,2,4,22,True


Governed connection repairs by field and confidence


,source_field,verification_confidence,governed_repairs,distinct_runner_rows,distinct_races,distinct_courses,first_observed_date,last_observed_date
0,jockey,high,2,2,2,2,2016-04-08,2021-09-05
1,owner,high,20,20,13,8,2019-05-13,2022-11-13
2,owner,medium,2,2,1,1,2022-07-18,2022-07-18
3,trainer,high,4,4,3,3,2015-05-06,2018-09-14


Complete governed connection-repair artifact


,repair_record_id,source_rowid,race_id,date,course,off,race_name,race_type,horse,source_field,original_source_value,governed_repaired_value,correction_method,correction_status,evidence_source_name,evidence_source_type,evidence_locator,evidence_accessed_date,verification_confidence,reviewer_notes
0,connection_blank_026,1044085,789638,2021-07-14,Ohi (JPN),11:07,Japan Dirt Derby (Local (3yo) (Dirt),Flat,Castle Top (JPN),owner,<NA>,Hiroshi Joichi,manual_external_verification,approved,netkeiba and official Japanese racing records,exact historical race result,https://en.netkeiba.com/db/race/202144071411/,2026-08-03,high,The exact Japan Dirt Derby record identifies H...
1,connection_blank_002,1068784,792961,2021-09-05,Baden-Baden (GER),2:15,149th Wettstar Grosser Preis von Baden (3yo+)...,Flat,Millebosc (FR),jockey,<NA>,Adrie de Vries,manual_external_verification,approved,Racing Post,complete historical race result,https://www.racingpost.com/results/207/baden-b...,2026-08-03,high,The complete result identifies Adrie de Vries ...
2,connection_blank_027,1112631,799985,2021-12-01,Funabashi (JPN),11:07,Queen Sho (Local (3yo+ Fillies & Mares) (Dirt),Flat,Dear Rickey (JPN),owner,<NA>,Yusuke Kobayashi,manual_external_verification,approved,NAR Local Racing Information Site and Rakuten ...,official and independent exact-date racecards,https://www.keiba.go.jp/KeibaWeb/TodayRaceInfo...,2026-08-03,high,The exact racecard lists 小林祐介 as owner of Dear...
3,connection_blank_028,1118779,800932,2021-12-15,Kawasaki (JPN),11:07,Zen-Nippon Nisai Yushun (Local (2yo) (Dirt),Flat,SIl Te Plait (JPN),owner,<NA>,Kumiko Hara,manual_external_verification,approved,netkeiba,exact historical race result,https://db.netkeiba.com/race/202145121511/,2026-08-03,high,The exact Zen-Nippon Nisai Yushun result lists...
4,connection_blank_029,1118781,800932,2021-12-15,Kawasaki (JPN),11:07,Zen-Nippon Nisai Yushun (Local (2yo) (Dirt),Flat,Praelude (JPN),owner,<NA>,Carrot Farm,manual_external_verification,approved,netkeiba and Rakuten Keiba,exact historical result and racecard,https://db.netkeiba.com/race/202145121511/,2026-08-03,high,The exact record identifies Carrot Farm as own...
5,connection_blank_030,1171417,810727,2022-04-20,Ohi (JPN),11:07,Tokyo Sprint (Local (4yo+) (Dirt),Flat,Gishigishi (JPN),owner,<NA>,Foret Bleu Co Ltd,manual_external_verification,approved,JBIS-Search,official exact-date historical race result,https://www.jbis.jp/race/result/20220420/220/11/,2026-08-03,high,The exact racecard lists 有限会社フォレブルー as owner o...
6,connection_blank_031,1214955,817393,2022-07-13,Ohi (JPN),11:07,Japan Dirt Derby (Local,Flat,Cryogenic (JPN),owner,<NA>,Tomohiro Iizuka,manual_external_verification,approved,"Rakuten Keiba, netkeiba and Japanese name record",exact racecard plus independent name-reading e...,https://en.netkeiba.com/db/race/202244071311/,2026-08-03,high,The exact Japanese racecard lists 飯塚倫尋 as owne...
7,connection_blank_032,1214956,817393,2022-07-13,Ohi (JPN),11:07,Japan Dirt Derby (Local,Flat,Hapi (JPN),owner,<NA>,Toshio Terada,manual_external_verification,approved,netkeiba,exact English historical race result,https://en.netkeiba.com/db/race/202244071311/,2026-08-03,high,The exact Japan Dirt Derby result identifies T...
8,connection_blank_034,1217234,817570,2022-07-18,Morioka (JPN),11:07,Mercury Cup (Local (3yo+) (Dirt),Flat,Vacation (JPN),owner,<NA>,Masatoshi Suzuki,manual_external_verification,approved,NAR Local Racing Information Site and bilingua...,official exact-date racecard plus bilingual ta...,https://www.keiba.go.jp/KeibaWeb/TodayRaceInfo...,2026-08-03,medium,The exact Mercury Cup listing supplies both th...
9,connection_blank_035,1217236,817570,2022-07-18,Morioka (JPN),11:07,Mercury Cup (Local (3yo+) (Dirt),Flat,Giga King (JPN),owner,<NA>,Tomohiro Ozaki,manual_external_verification,approved,NAR Local Racing Information Site and bilingua...,official exact-date racecard plus bilingual ta...,https://www.keiba.go.jp/KeibaWeb/TodayRaceInfo...,2026-08-03,med

## Stage 42 — Validate application of governed connection repairs

The governed repair artifact contains one approved correction for each verified missing connection value.

Before database integration, this stage applies the artifact to the corresponding source rows in memory and verifies that:

- every repair matches exactly one source runner row;
- the targeted raw field is blank;
- the governed value is populated;
- no populated source value is overwritten;
- fields not targeted by a repair remain unchanged;
- all 28 approved repairs are applied exactly once;
- unresolved and conflicting records remain blank.

This is an application test only.

The raw source database is opened read-only and is not modified.

In [43]:
# Validate that the governed correction artifact can be applied safely to the
# source rows without overwriting populated source values.
#
# The source database remains read-only.

repair_source_rowids = (
    reloaded_governed_connection_repairs[
        "source_rowid"
    ]
    .astype("int64")
    .drop_duplicates()
    .sort_values()
    .tolist()
)


# Build a bounded SQL placeholder list for only the affected source rows.
repair_row_placeholders = ", ".join(
    "?"
    for _ in repair_source_rowids
)


connection = sqlite3.connect(
    f"file:{SOURCE_DB_PATH}?mode=ro",
    uri=True,
)

try:
    governed_repair_source_rows = pd.read_sql_query(
        f"""
        SELECT
            rowid AS source_rowid,
            race_id,
            date,
            course,
            off,
            horse,
            jockey,
            trainer,
            owner
        FROM {quote_identifier(SOURCE_TABLE)}
        WHERE rowid IN ({repair_row_placeholders})
        ORDER BY rowid
        """,
        connection,
        params=repair_source_rowids,
    )

finally:
    connection.close()


# Every governed repair must match exactly one source row.
expected_distinct_source_rows = int(
    reloaded_governed_connection_repairs[
        "source_rowid"
    ].nunique()
)


assert len(
    governed_repair_source_rows
) == expected_distinct_source_rows

assert governed_repair_source_rows[
    "source_rowid"
].is_unique


# Preserve an untouched copy for before/after comparison.
governed_repair_source_rows_before = (
    governed_repair_source_rows.copy(deep=True)
)

governed_repair_source_rows_after = (
    governed_repair_source_rows.copy(deep=True)
)


# Record one application-audit row per governed repair.
governed_repair_application_rows = []


for repair in (
    reloaded_governed_connection_repairs
    .sort_values(
        [
            "source_rowid",
            "source_field",
        ]
    )
    .itertuples(index=False)
):
    source_rowid = int(repair.source_rowid)
    source_field = repair.source_field
    governed_value = repair.governed_repaired_value

    matching_source_rows = (
        governed_repair_source_rows_after[
            "source_rowid"
        ].eq(source_rowid)
    )

    assert int(matching_source_rows.sum()) == 1
    assert source_field in CONNECTION_IDENTITY_FIELDS

    original_value = (
        governed_repair_source_rows_after.loc[
            matching_source_rows,
            source_field,
        ].iloc[0]
    )

    # Approved repairs may only fill source blanks.
    assert pd.isna(original_value) or original_value == ""

    assert pd.notna(governed_value)
    assert str(governed_value).strip() != ""

    governed_repair_source_rows_after.loc[
        matching_source_rows,
        source_field,
    ] = governed_value

    applied_value = (
        governed_repair_source_rows_after.loc[
            matching_source_rows,
            source_field,
        ].iloc[0]
    )

    governed_repair_application_rows.append(
        {
            "repair_record_id": repair.repair_record_id,
            "source_rowid": source_rowid,
            "source_field": source_field,
            "original_source_value": original_value,
            "governed_repaired_value": governed_value,
            "applied_value": applied_value,
            "application_succeeded": (
                applied_value == governed_value
            ),
        }
    )


governed_repair_application_audit = pd.DataFrame(
    governed_repair_application_rows
)


# ----------------------------------------------------------------------
# Confirm every approved repair was applied exactly once
# ----------------------------------------------------------------------

assert len(
    governed_repair_application_audit
) == 28

assert governed_repair_application_audit[
    "repair_record_id"
].is_unique

assert governed_repair_application_audit[
    "application_succeeded"
].all()


assert not governed_repair_application_audit.duplicated(
    subset=[
        "source_rowid",
        "source_field",
    ]
).any()


# ----------------------------------------------------------------------
# Confirm only targeted fields changed
# ----------------------------------------------------------------------

targeted_repairs = set(
    zip(
        reloaded_governed_connection_repairs[
            "source_rowid"
        ].astype("int64"),
        reloaded_governed_connection_repairs[
            "source_field"
        ],
    )
)


unexpected_changes = []


for before_row, after_row in zip(
    governed_repair_source_rows_before.itertuples(
        index=False
    ),
    governed_repair_source_rows_after.itertuples(
        index=False
    ),
):
    assert before_row.source_rowid == after_row.source_rowid

    for field in CONNECTION_IDENTITY_FIELDS:
        before_value = getattr(before_row, field)
        after_value = getattr(after_row, field)

        value_changed = not (
            (
                pd.isna(before_value)
                and pd.isna(after_value)
            )
            or before_value == after_value
        )

        repair_key = (
            int(before_row.source_rowid),
            field,
        )

        if value_changed and repair_key not in targeted_repairs:
            unexpected_changes.append(
                {
                    "source_rowid": int(
                        before_row.source_rowid
                    ),
                    "source_field": field,
                    "before_value": before_value,
                    "after_value": after_value,
                }
            )


unexpected_connection_repair_changes = pd.DataFrame(
    unexpected_changes,
    columns=[
        "source_rowid",
        "source_field",
        "before_value",
        "after_value",
    ],
)


assert unexpected_connection_repair_changes.empty


# ----------------------------------------------------------------------
# Confirm all governed values now populate their targeted cells
# ----------------------------------------------------------------------

for repair in (
    reloaded_governed_connection_repairs
    .itertuples(index=False)
):
    matching_source_rows = (
        governed_repair_source_rows_after[
            "source_rowid"
        ].eq(int(repair.source_rowid))
    )

    observed_value = (
        governed_repair_source_rows_after.loc[
            matching_source_rows,
            repair.source_field,
        ].iloc[0]
    )

    assert observed_value == repair.governed_repaired_value


# ----------------------------------------------------------------------
# Reader-facing summaries
# ----------------------------------------------------------------------

governed_repair_application_summary = (
    governed_repair_application_audit.groupby(
        "source_field",
        as_index=False,
    )
    .agg(
        approved_repairs=("repair_record_id", "size"),
        distinct_runner_rows=("source_rowid", "nunique"),
        successful_applications=(
            "application_succeeded",
            "sum",
        ),
    )
    .sort_values("source_field")
    .reset_index(drop=True)
)


governed_repair_application_validation = pd.DataFrame(
    [
        {
            "approved_repairs": len(
                governed_repair_application_audit
            ),
            "distinct_source_rows": int(
                governed_repair_application_audit[
                    "source_rowid"
                ].nunique()
            ),
            "successful_applications": int(
                governed_repair_application_audit[
                    "application_succeeded"
                ].sum()
            ),
            "unexpected_field_changes": len(
                unexpected_connection_repair_changes
            ),
            "source_database_modified": False,
            "application_validation_passed": True,
        }
    ]
)


print("Governed connection-repair application validation")
display(governed_repair_application_validation)

print("Governed repair applications by field")
display(governed_repair_application_summary)

print("Applied governed connection repairs")
display(governed_repair_application_audit)

Governed connection-repair application validation


,approved_repairs,distinct_source_rows,successful_applications,unexpected_field_changes,source_database_modified,application_validation_passed
0,28,28,28,0,False,True


Governed repair applications by field


,source_field,approved_repairs,distinct_runner_rows,successful_applications
0,jockey,2,2,2
1,owner,22,22,22
2,trainer,4,4,4


Applied governed connection repairs


,repair_record_id,source_rowid,source_field,original_source_value,governed_repaired_value,applied_value,application_succeeded
0,connection_blank_038,50160,trainer,,Masaya Komura,Masaya Komura,True
1,connection_blank_001,189632,jockey,,T. Viel,T. Viel,True
2,connection_blank_039,203870,trainer,,K Borgel,K Borgel,True
3,connection_blank_040,203991,trainer,,C Plisson,C Plisson,True
4,connection_blank_042,600778,trainer,,Stal't Neerhof,Stal't Neerhof,True
5,connection_blank_009,712612,owner,,Edouard Desespringalle,Edouard Desespringalle,True
6,connection_blank_010,725030,owner,,Edouard Desespringalle,Edouard Desespringalle,True
7,connection_blank_011,801287,owner,,Mastec Co Ltd,Mastec Co Ltd,True
8,connection_blank_012,801290,owner,,Makoto Kato,Makoto Kato,True
9,connection_blank_013,801304,owner,,Thoroughbred Club Ruffian Co Ltd,Thoroughbred Club Ruffian Co Ltd,True


## Stage 43 — Conclusion from governed repair application

The governed connection-repair artifact was applied in memory to the corresponding raw source rows.

The validation confirmed:

- 28 approved repairs matched 28 distinct source runner rows;
- all targeted source values were blank;
- all 28 governed values were applied successfully;
- no populated source value was overwritten;
- no untargeted connection field changed;
- the raw source database remained unchanged.

The approved correction population comprises:

- 2 jockey repairs;
- 4 trainer repairs;
- 22 owner repairs.

The correction artifact is therefore suitable for controlled use during database construction.

Application must remain conditional on:

- an exact `source_rowid` match;
- the expected `source_field`;
- confirmation that the raw source value is blank;
- an approved correction status;
- retained evidence provenance.

The correction artifact must not be used to:

- overwrite a populated raw value;
- resolve conflicting trainer evidence;
- fill insufficient-evidence owner records;
- canonicalise connection labels;
- infer identity relationships between labels.

The five conflicting trainer records and thirteen insufficient-evidence owner records remain analytically null.

## Stage 44 — Analytical conclusion

The runner-level `jockey`, `trainer` and `owner` fields are source-presented connection labels.

They are highly complete, but they are not globally stable entity identifiers.

### Jockey

The `jockey` field behaves as a runner-level riding assertion:

- 1,851,283 of 1,851,285 runner rows are populated;
- 7,917 exact populated labels occur;
- no exact jockey label appears on more than one runner in the same provisional race;
- two blank values were independently verified and repaired;
- exact equality with trainer or owner labels does not establish shared identity.

The field can be stored safely as nullable raw text.

It cannot support canonical person identity without later namesake and identity-resolution work.

### Trainer

The `trainer` field behaves as a runner-level training-connection assertion:

- 1,851,276 runner rows are populated;
- 10,708 exact populated labels occur;
- one exact trainer label may relate to multiple runners in the same race;
- 120,906 repeated same-race trainer groups contain distinct exact horse labels on every row;
- the largest observed same-race trainer group contains 14 runners.

Trainer labels include individual-looking names, initials, hyphenated forms and compound ampersand structures.

Ampersand labels cannot be split safely by punctuation alone.

They may represent:

- complete individual names;
- compressed shared surnames;
- formal joint-training arrangements;
- stable or licence presentations;
- three or more apparent participants;
- structures that require external verification.

Four missing trainer values were verified and repaired.

Five further missing trainer values produced materially conflicting exact-date evidence and remain null.

The complete raw trainer label must therefore remain atomic unless separately governed identity-resolution evidence exists.

### Owner

The `owner` field behaves as a runner-level ownership-connection assertion:

- 1,851,250 runner rows are populated;
- 98,234 exact populated labels occur;
- one exact owner label may relate to multiple runners in one race;
- 28,080 repeated same-race owner groups contain distinct exact horse labels on every row;
- the largest observed same-race owner group contains 13 runners.

Owner labels may describe:

- named individuals;
- multiple named co-owners;
- partnerships;
- syndicates;
- companies;
- studs;
- racing organisations;
- other source-specific ownership presentations.

The absence of an organisation keyword does not prove that a label is one person.

Token-order comparison produced useful candidate groups, but repeated switching between alternative labels showed that reordered strings are not safe automatic replacements.

Twenty-two missing owner values were independently verified and repaired.

Thirteen remained unsupported at the target race date and therefore remain null.

### Cross-field identity

Exact equality across `jockey`, `trainer` and `owner` fields is text evidence only.

It does not prove that:

- the same real person is represented;
- one label is globally unique;
- the role strings can share a canonical entity identifier;
- namesakes have been resolved.

### Governed database treatment

The safe database treatment is:

- preserve each original source value exactly;
- represent empty strings as analytical nulls;
- retain `jockey`, `trainer` and `owner` as nullable runner-level text;
- apply only approved provenance-backed repairs through a separate correction artifact;
- preserve compound trainer and owner labels atomically;
- prohibit automatic splitting, token reordering, expansion or canonicalisation;
- defer canonical people, partnerships, organisations and licences to a separate identity-resolution programme.

The source supports reliable runner-level connection assertions.

It does not, by itself, support a complete canonical connections entity model.

## Stage 43 — Conclusion from governed repair application

The governed connection-repair artifact was applied in memory to the corresponding raw source rows.

The validation confirmed:

- 28 approved repairs matched 28 distinct source runner rows;
- all targeted source values were blank;
- all 28 governed values were applied successfully;
- no populated source value was overwritten;
- no untargeted `jockey`, `trainer` or `owner` field changed;
- the raw source database remained unchanged.

The approved correction population comprises:

- 2 jockey repairs;
- 4 trainer repairs;
- 22 owner repairs.

The correction artifact is therefore suitable for controlled use during database construction.

Application must remain conditional on:

- an exact `source_rowid` match;
- the expected `source_field`;
- confirmation that the raw source value is blank;
- an approved correction status;
- retained evidence provenance.

The correction artifact must not be used to:

- overwrite a populated raw value;
- resolve conflicting trainer evidence;
- fill insufficient-evidence owner records;
- canonicalise connection labels;
- infer identity relationships between labels.

The five conflicting trainer records and thirteen insufficient-evidence owner records remain analytically null.

## Stage 44 — Analytical conclusion

### Bounded question

What do the runner-level `jockey`, `trainer` and `owner` fields represent in the source, how stable and complete are their labels, and which source-internal identity relationships can be preserved safely without inventing equivalence between people, partnerships, syndicates or organisations?

### Executive conclusion

The three fields are reliable as **runner-level source-presented connection labels**.

They are not reliable as canonical identifiers for people, partnerships, licences, syndicates or organisations.

The database should therefore preserve:

- each original source value exactly;
- analytical nulls for empty strings;
- approved provenance-backed repairs in a separate correction layer;
- complete compound labels as atomic source assertions;
- unresolved and conflicting cases without forced resolution.

### Jockey

The `jockey` field behaves as a runner-level riding assertion.

- 1,851,283 of 1,851,285 runner rows are populated.
- 7,917 exact populated labels occur.
- No exact jockey label appears on more than one runner in the same provisional race.
- The two blank values were independently verified and repaired.
- Exact equality with a trainer or owner label does not establish that the same real person is represented.

The field can safely be stored as nullable raw text.

It cannot support canonical person identity without later namesake and identity-resolution work.

### Trainer

The `trainer` field behaves as a runner-level training-connection assertion.

- 1,851,276 runner rows are populated.
- 10,708 exact populated labels occur.
- One exact trainer label may relate to multiple runners in the same race.
- 120,906 repeated same-race trainer groups contain distinct exact horse labels on every row.
- The largest observed same-race trainer group contains 14 runners.

Trainer labels include:

- individual-looking names;
- initials;
- hyphenated forms;
- compound ampersand structures;
- compressed shared surnames;
- joint-training or stable-style presentations.

Ampersand labels cannot be split safely by punctuation alone.

Examples such as:

- `John & Thady Gosden`;
- `Michael & David Easterby`;
- `David A & B Hayes & Tom Dabernig`;

show that literal splitting can omit shared surnames or misrepresent the complete source assertion.

Four missing trainer values were independently verified and repaired.

Five further trainer blanks produced materially conflicting exact-date evidence and remain unresolved.

The complete raw trainer label must therefore remain atomic unless a separate governed identity-resolution process establishes verified component relationships.

### Owner

The `owner` field behaves as a runner-level ownership-connection assertion.

- 1,851,250 runner rows are populated.
- 98,234 exact populated labels occur.
- One exact owner label may relate to multiple runners in one race.
- 28,080 repeated same-race owner groups contain distinct exact horse labels on every row.
- The largest observed same-race owner group contains 13 runners.

Owner labels may represent:

- individuals;
- named co-owners;
- partnerships;
- syndicates;
- companies;
- studs;
- racing organisations;
- other source-specific ownership presentations.

The absence of an organisation keyword does not prove that a label represents one person.

Token-order comparison generated useful candidate groups, but repeated switching between alternative presentations showed that reordered labels are not safe automatic replacements.

Twenty-two missing owner values were independently verified and repaired.

Thirteen owner blanks remained unsupported at the target race date and therefore remain unresolved.

### Cross-field identity

Exact equality across `jockey`, `trainer` and `owner` fields is text evidence only.

It does not prove:

- that the same real person is represented;
- that one label is globally unique;
- that labels can share a canonical entity identifier;
- that namesakes have been resolved;
- that an owner label represents a person rather than a collective or organisation.

### Blank values and manual repair

The source contains 46 blank connection-field occurrences across 43 runner rows.

External review produced:

- 28 verified repairs;
- 5 conflicting-evidence cases;
- 13 insufficient-evidence cases;
- no unresolved review queue entries.

The governed correction artifact contains only the 28 approved repairs:

- 2 jockey;
- 4 trainer;
- 22 owner.

All 28:

- retain direct evidence locators;
- retain confidence and review notes;
- match exactly one source row and source field;
- apply only to a blank raw value;
- leave all untargeted values unchanged.

The raw source database remains immutable.

### Governed database treatment

The safe database treatment is:

- preserve raw `jockey`, `trainer` and `owner` values exactly;
- convert empty strings to analytical nulls;
- store the fields as nullable runner-level text;
- apply only approved provenance-backed repairs through a separate correction artifact;
- preserve compound trainer and owner labels atomically;
- prohibit automatic splitting, expansion, token reordering, merging or canonicalisation;
- retain conflicting and insufficient-evidence records as null;
- defer canonical people, partnerships, organisations, licences and namesakes to a separate identity-resolution programme.

### Confidence

Confidence is high in:

- raw-field completeness counts;
- exact-label counts;
- within-race cardinality;
- same-race distinct-horse consistency;
- the unsafety of automatic compound-label decomposition;
- the 28 governed repairs and their application behaviour.

Confidence is lower in any attempt to infer canonical real-world entities from source strings alone.

### Limitations

This study does not establish:

- globally unique jockey, trainer or owner identities;
- legal or beneficial ownership;
- formal joint-training licence structures;
- effective dates for all trainer partnerships;
- historical name changes;
- namesake resolution;
- complete cross-jurisdiction entity linkage;
- exhaustive correction of all source label variants.

The provisional race key remains based on `date`, `course` and `off`, with the limitations already established elsewhere in the project.

### Practical consequence

The source is suitable for analysis at the level of:

> this runner was presented by the source with this jockey, trainer and owner label.

It is not suitable, without further governed identity resolution, for claims such as:

> these two labels are definitely the same person, partnership, syndicate or organisation.

That distinction must remain explicit in all downstream analysis.

## Stage 45 — Notebook closure classification and manual-verification decision

### Closure route

This notebook is classified as a:

> non-rerunnable archival construction record

The notebook preserves the completed investigation, executed outputs, reasoning, anomalies, repair decisions and source lineage.

It is not intended to remain the durable production workflow.

A fresh-kernel top-to-bottom rerun is not required because:

- the analytical investigation is complete;
- governed outputs have been persisted and reloaded;
- the repair artifact has been application-tested against the immutable source;
- external evidence has already been captured with access dates and direct locators;
- rerunning the research cells would repeat finite manual-review work without improving reliability;
- reusable implementation, focused tests and independent validation will live outside the notebook.

Any material cell that performs external verification or overwrites the persisted repair queue should therefore be treated as construction history rather than a routine rerun step.

### Durable notebook outputs

The notebook created and reloaded:

- `data/derived/connection_identity/manual_connection_repair_queue.csv`;
- `data/derived/connection_identity/manual_connection_repair_evidence_log.csv`;
- `data/derived/connection_identity/governed_connection_repairs.csv`.

The governed repair artifact contains:

- 28 approved repairs;
- 2 jockey repairs;
- 4 trainer repairs;
- 22 owner repairs.

The full evidence log retains:

- 28 `verified_repair` decisions;
- 5 `conflicting_evidence` decisions;
- 13 `insufficient_evidence` decisions;
- no pending records.

### Manual-verification decision

Manual-verification status:

> `specialist_reference`

External evidence was used to verify missing jockey, trainer and owner values.

Equivalent governed evidence is preserved in the connection-specific repair queue and evidence log, including:

- the raw source row and field;
- the verification question implied by the missing value;
- the proposed repaired value where supported;
- the verification decision;
- evidence source and type;
- direct source locator;
- access date;
- confidence;
- reviewer notes;
- permitted database action.

The governed correction artifact contains only approved repairs and does not overwrite the immutable raw source.

Before formal repository closure, the reusable implementation and validator must either:

- consume this specialist reference directly; or
- register equivalent permanent verification identifiers in the project-wide manual-verification register if required by the final integration design.

### Current status

The notebook investigation and notebook-level governed outputs are complete.

Repository-level closeout remains outstanding for:

- reusable implementation;
- focused unit tests;
- independent source-wide validation;
- database integration documentation;
- reader-facing report;
- lessons learned;
- audit and field-governance updates;
- README and project-plan updates;
- local validation evidence.